# 07 — Restricted-First Hawkes Estimation

## Notebook purpose

Estimate, select, freeze, and replay the first authorized Hawkes specification for the V0.1 causal event stream.

This notebook begins with a **diagonal bivariate exponential Hawkes process**:

- BUY events may excite future BUY events.
- SELL events may excite future SELL events.
- BUY-to-SELL excitation is initially fixed to zero.
- SELL-to-BUY excitation is initially fixed to zero.

The notebook must determine whether the authorized self-exciting model can be estimated reproducibly and stably on `DEVELOPMENT`, then replay the frozen model through locked `CALIBRATION` while preserving the complete event history across the partition boundary.

This notebook estimates the model. It does not establish final model adequacy or Hawkes superiority. Those judgments belong to `08_HAWKES_DIAGNOSTICS.ipynb`.

---

## Authoritative upstream inputs

The notebook must consume only verified, persisted artifacts from disk.

Primary authorities:

- `04_EVENT_STREAM_CONSTRUCTION.ipynb`
  - authoritative event representation
  - authoritative `event_time_ns`
  - authoritative `event_partition`
  - simultaneous-event batch semantics
- `06_POINT_PROCESS_BASELINES.ipynb`
  - authorization for restricted-first Hawkes estimation
  - Poisson and EWMA comparison artifacts
  - DEVELOPMENT and CALIBRATION exposure contracts
  - Notebook 06 to Notebook 07 handoff
- frozen V0.1 run, split, event-definition, and model contracts
- immutable V0.0 Hawkes artifacts for reconciliation only

Primary event representation:

`SAME_MS_SAME_SIDE_BURSTS`

Primary timestamp:

`event_time_ns`

Primary timestamp interface:

`SIMULTANEOUS_EVENT_BATCH_REQUIRED`

V0.0 artifacts may be read only for provenance, comparison, and reconciliation. They may not replace V0.1 estimation or be used as calibration targets.

---

## Authorized model

Let the event types be:

- \(B\): BUY
- \(S\): SELL

The authorized first specification is:

$$
\lambda_B(t)
=
\mu_B
+
\sum_{\tau_k^B < t}
\kappa_B \beta_B
\exp\left[-\beta_B\left(t-\tau_k^B\right)\right]
$$

$$
\lambda_S(t)
=
\mu_S
+
\sum_{\tau_k^S < t}
\kappa_S \beta_S
\exp\left[-\beta_S\left(t-\tau_k^S\right)\right]
$$

The kernel functions are:

$$
\phi_{BB}(u)
=
\kappa_B \beta_B e^{-\beta_B u}\mathbf{1}_{\{u>0\}}
$$

$$
\phi_{SS}(u)
=
\kappa_S \beta_S e^{-\beta_S u}\mathbf{1}_{\{u>0\}}
$$

The initially prohibited cross-excitation kernels are:

$$
\phi_{BS}(u)=0
$$

$$
\phi_{SB}(u)=0
$$

The integrated excitation matrix is therefore:

$$
K
=
\begin{pmatrix}
\kappa_B & 0 \\
0 & \kappa_S
\end{pmatrix}
$$

Its spectral radius is:

$$
\rho(K)
=
\max\left(\kappa_B,\kappa_S\right)
$$

The mathematical stationarity requirement is:

$$
\rho(K) < 1
$$

Parameter constraints:

$$
\mu_B>0,
\qquad
\mu_S>0
$$

$$
\kappa_B\geq0,
\qquad
\kappa_S\geq0
$$

$$
\beta_B>0,
\qquad
\beta_S>0
$$

The excitation half-lives are:

$$
h_B
=
\frac{\log 2}{\beta_B}
$$

$$
h_S
=
\frac{\log 2}{\beta_S}
$$

The fitted model must be described as a:

`STRICT_PRE_BATCH_COARSENED_TIME_QUASI_MLE`

This label records that the observed timestamps contain exact millisecond-level ties and that all events in a tied batch are scored using history strictly before the batch.

---

## Exact-time batch contract

For every unique combination of:

- `event_partition`
- `event_time_ns`

the notebook must:

1. Decay the stored excitation state from the previous timestamp.
2. Compute BUY and SELL intensities immediately before the batch.
3. Score all events in the batch using the same pre-batch intensities.
4. Add the exact compensator contribution for the elapsed interval.
5. Update the excitation state only after every event in the batch has been scored.

For an exact-time batch at time \(t\), all event likelihood contributions must use:

$$
\lambda_B(t^-)
$$

and

$$
\lambda_S(t^-)
$$

No event inside the batch may excite another event at the same timestamp.

The following are forbidden:

- timestamp jitter
- arbitrary ordering of tied events
- treating collector sequence as elapsed physical time
- zero-lag within-batch excitation
- dropping simultaneous events
- changing Notebook 04 timestamps
- resetting excitation within a partition without contractual justification

---

## Partition-use contract

Parameter estimation:

`DEVELOPMENT_ONLY`

Hyperparameter and candidate selection:

`DEVELOPMENT_ONLY`

Locked evaluation:

`CALIBRATION`

CALIBRATION parameter updates:

`ZERO`

The notebook must not open event content from:

- `VALIDATION`
- `ENGINEERING_HOLDOUT`

The notebook must not load future-label content.

The Notebook 05 feature table is not an estimation input for this notebook.

At the left edge of DEVELOPMENT:

- fabricated prehistory is forbidden
- excitation state begins from the explicitly recorded left-censored boundary condition
- left-edge sensitivity must be audited

At the DEVELOPMENT-to-CALIBRATION boundary:

- the terminal DEVELOPMENT excitation state must be retained
- the state must be decayed across the exact boundary interval
- CALIBRATION must not begin from base intensity alone
- no fitting, tuning, or parameter update may occur

---

## Candidate specifications

The authorized candidate set is limited to:

### H1 — Diagonal Hawkes with shared decay

Parameters:

$$
\theta_{H1}
=
\left(
\mu_B,
\mu_S,
\kappa_B,
\kappa_S,
\beta
\right)
$$

with:

$$
\beta_B=\beta_S=\beta
$$

### H2 — Diagonal Hawkes with side-specific decay

Parameters:

$$
\theta_{H2}
=
\left(
\mu_B,
\mu_S,
\kappa_B,
\kappa_S,
\beta_B,
\beta_S
\right)
$$

H2 may replace H1 only when DEVELOPMENT-only chronological evidence shows a stable and material improvement.

The following model is not authorized for primary estimation:

`UNRESTRICTED_BIVARIATE_CROSS_EXCITATION`

Its required notebook status is:

`SKIPPED_NOT_AUTHORIZED`

State-dependent Hawkes estimation is also not authorized.

---

## Required outputs

The notebook must persist, at minimum:

- verified upstream-input ledger
- model contract
- candidate registry
- deterministic optimization-start ledger
- optimization-results ledger
- DEVELOPMENT chronological selection results
- selected and frozen model specification
- portable parameter archive
- integrated excitation matrix
- stationarity audit
- left-edge sensitivity audit
- chronological parameter-stability audit
- DEVELOPMENT pre-batch intensity replay
- locked CALIBRATION pre-batch intensity replay
- compensator replay
- baseline-comparison bridge
- timestamp-coarsening sensitivity results
- V0.0 reconciliation report
- formal decision ledger
- output manifest
- final acceptance artifact
- readback audit
- Notebook 07 to Notebook 08 handoff

Pre-batch intensities and post-batch excitation states must be stored as physically distinguishable fields.

Pickle may not be the only model-storage format.

---

## Prohibited claims

This notebook may not claim:

- unrestricted BUY/SELL cross-excitation
- state-dependent Hawkes validity
- final Hawkes model adequacy
- Hawkes superiority over all simpler models
- predictive trading value
- profitable market making
- an authorized quote policy
- realistic fills
- P&L, Sharpe ratio, or drawdown performance
- production readiness
- live-trading readiness

The quantities \(\kappa_B\) and \(\kappa_S\) must be described as **model-implied excitation masses**, not as proven causal fractions of endogenous or algorithmic trading.

---

## Blocking acceptance gates

The notebook may authorize Hawkes diagnostics only if:

- all upstream identities, hashes, schemas, and acceptance artifacts verify
- DEVELOPMENT and CALIBRATION event counts reconcile exactly
- protected partitions remain unopened
- hand-computable and synthetic estimator tests pass
- exact-time batches obey strict pre-batch scoring
- multiple deterministic starts reach the same material optimum
- fitted parameters and intensities are finite
- all parameter constraints hold
- the fitted spectral radius is strictly below one
- DEVELOPMENT history is carried into CALIBRATION
- CALIBRATION uses zero parameter updates
- left-edge sensitivity is acceptable
- portable outputs pass schema, checksum, and readback verification

A fitted excitation mass near zero is a valid Poisson-limit result and is not, by itself, an estimation failure.

---

## Permitted terminal statuses

The notebook must end with exactly one of:

- `PASS_HAWKES_ESTIMATION_DIAGNOSTICS_AUTHORIZED`
- `PASS_HAWKES_ESTIMATION_POISSON_LIMIT_DIAGNOSTICS_AUTHORIZED`
- `CONDITIONAL_PASS_HAWKES_ESTIMATION_DIAGNOSTIC_ONLY`
- `FAIL_HAWKES_ESTIMATION_UNSTABLE`
- `FAIL_HAWKES_PIPELINE_CONTRACT`

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import random
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Final, Mapping

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import scipy.linalg as linalg
import scipy.optimize as optimize
import scipy.special as special
import scipy.stats as stats
from IPython.display import display


# ============================================================
# Runtime contract
# ============================================================

MINIMUM_PYTHON_VERSION: Final[tuple[int, int]] = (3, 10)

if sys.version_info < MINIMUM_PYTHON_VERSION:
    required_version = ".".join(map(str, MINIMUM_PYTHON_VERSION))
    observed_version = platform.python_version()

    raise RuntimeError(
        f"Python {required_version} or newer is required; "
        f"observed Python {observed_version}."
    )

warnings.filterwarnings(
    "error",
    category=pd.errors.SettingWithCopyWarning,
)

pd.options.display.max_columns = 200
pd.options.display.width = 180
pd.options.display.float_format = "{:,.10g}".format


# ============================================================
# Frozen project identity
# ============================================================

NOTEBOOK_NAME: Final[str] = "07_HAWKES_ESTIMATION.ipynb"
NOTEBOOK_STAGE: Final[str] = "HAWKES_ESTIMATION"

SOURCE_RUN_PREFIX: Final[str] = (
    "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
)
V01_RUN_ID: Final[str] = "v0_1_20260714T090616Z_e82325081a81"
COMBINED_OUTPUT_PREFIX: Final[str] = (
    f"{SOURCE_RUN_PREFIX}__{V01_RUN_ID}"
)

SOURCE_SET_SHA256: Final[str] = (
    "132c83531eec615d279408b5c06f402973114ba3058dfadd2fe58e2e67184c4b"
)
RUN_CONFIG_SHA256: Final[str] = (
    "14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a770bee995f688d59617"
)
RUN_IDENTITY_SHA256: Final[str] = (
    "5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9df19204c42e37fe4198"
)


# ============================================================
# Frozen event-stream authority
# ============================================================

PRIMARY_EVENT_REPRESENTATION: Final[str] = (
    "SAME_MS_SAME_SIDE_BURSTS"
)
PRIMARY_EVENT_TIME_COLUMN: Final[str] = "event_time_ns"
PRIMARY_EVENT_PARTITION_COLUMN: Final[str] = "event_partition"
PRIMARY_TIMESTAMP_INTERFACE: Final[str] = (
    "SIMULTANEOUS_EVENT_BATCH_REQUIRED"
)
PRIMARY_ESTIMATOR_LABEL: Final[str] = (
    "STRICT_PRE_BATCH_COARSENED_TIME_QUASI_MLE"
)

EVENT_SIDES: Final[tuple[str, str]] = ("BUY", "SELL")

FIT_PARTITION: Final[str] = "DEVELOPMENT"
LOCKED_EVALUATION_PARTITION: Final[str] = "CALIBRATION"
PROTECTED_PARTITIONS: Final[tuple[str, str]] = (
    "VALIDATION",
    "ENGINEERING_HOLDOUT",
)

EXPECTED_PRIMARY_EVENT_ROWS: Final[int] = 13_887
EXPECTED_PRIMARY_SCORING_BATCHES: Final[int] = 13_564
EXPECTED_SIMULTANEOUS_BATCHES: Final[int] = 249
EXPECTED_MIXED_SIDE_BATCHES: Final[int] = 41

EXPECTED_DEVELOPMENT_EVENT_ROWS: Final[int] = 7_004
EXPECTED_DEVELOPMENT_BATCH_ROWS: Final[int] = 6_859
EXPECTED_CALIBRATION_EVENT_ROWS: Final[int] = 2_493
EXPECTED_CALIBRATION_BATCH_ROWS: Final[int] = 2_400

EXPECTED_ANALYTICAL_EVENT_ROWS: Final[int] = (
    EXPECTED_DEVELOPMENT_EVENT_ROWS
    + EXPECTED_CALIBRATION_EVENT_ROWS
)
EXPECTED_ANALYTICAL_BATCH_ROWS: Final[int] = (
    EXPECTED_DEVELOPMENT_BATCH_ROWS
    + EXPECTED_CALIBRATION_BATCH_ROWS
)


# ============================================================
# Frozen Hawkes authorization
# ============================================================

UPSTREAM_REQUIRED_TERMINAL_STATUS: Final[str] = (
    "PASS_HAWKES_RESTRICTED_FIRST"
)
UPSTREAM_REQUIRED_AUTHORIZATION_STATE: Final[str] = (
    "AUTHORIZED_RESTRICTED_FIRST"
)

AUTHORIZED_FIRST_MODEL: Final[str] = (
    "DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES"
)
AUTHORIZED_KERNEL_FAMILY: Final[str] = (
    "SINGLE_EXPONENTIAL_PER_AUTHORIZED_CHANNEL"
)

AUTHORIZED_CHANNELS: Final[tuple[str, str]] = (
    "BUY_TO_BUY",
    "SELL_TO_SELL",
)
FIXED_ZERO_CHANNELS: Final[tuple[str, str]] = (
    "BUY_TO_SELL",
    "SELL_TO_BUY",
)

CROSS_EXCITATION_STATUS: Final[str] = "SKIPPED_NOT_AUTHORIZED"
STATE_DEPENDENT_HAWKES_STATUS: Final[str] = "NOT_AUTHORIZED"
CALIBRATION_PARAMETER_UPDATES: Final[int] = 0


# ============================================================
# Project paths
# ============================================================

PROJECT_ROOT: Final[Path] = Path(r"D:\Clown Project")
V00_ROOT: Final[Path] = PROJECT_ROOT / "V0.0"
V01_ROOT: Final[Path] = PROJECT_ROOT / "V0.1"

V01_CONFIG_ROOT: Final[Path] = V01_ROOT / "config"
V01_DATA_ROOT: Final[Path] = V01_ROOT / "data"
V01_PROCESSED_ROOT: Final[Path] = V01_DATA_ROOT / "processed"
V01_EVENT_ROOT: Final[Path] = V01_PROCESSED_ROOT / "events"
V01_POINT_PROCESS_ROOT: Final[Path] = (
    V01_PROCESSED_ROOT / "point_process"
)

V01_ARTIFACT_ROOT: Final[Path] = V01_ROOT / "artifacts"
V01_MANIFEST_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "manifests"
V01_AUDIT_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "audit_tables"
V01_MODEL_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "models"
V01_DIAGNOSTIC_ROOT: Final[Path] = (
    V01_ARTIFACT_ROOT / "diagnostics"
)
V01_RECONCILIATION_ROOT: Final[Path] = (
    V01_ARTIFACT_ROOT / "reconciliation"
)
V01_HANDOFF_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "handoff"
V01_LOG_ROOT: Final[Path] = V01_ROOT / "logs"

NOTEBOOK07_AUDIT_ROOT: Final[Path] = (
    V01_AUDIT_ROOT / "07_hawkes_estimation"
)
NOTEBOOK07_MODEL_ROOT: Final[Path] = (
    V01_MODEL_ROOT / "07_hawkes_estimation"
)
NOTEBOOK07_DIAGNOSTIC_ROOT: Final[Path] = (
    V01_DIAGNOSTIC_ROOT / "07_hawkes_estimation"
)
NOTEBOOK07_RECONCILIATION_ROOT: Final[Path] = (
    V01_RECONCILIATION_ROOT / "07_hawkes_estimation"
)

NOTEBOOK06_TERMINAL_DECISION_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_terminal_decision.json"
)
NOTEBOOK06_FINAL_ACCEPTANCE_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_final_acceptance.json"
)
NOTEBOOK06_OUTPUT_MANIFEST_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_output_manifest.json"
)
NOTEBOOK06_READBACK_AUDIT_PATH: Final[Path] = (
    V01_MANIFEST_ROOT
    / "06_point_process_baselines_readback_audit.json"
)
NOTEBOOK06_TO_07_HANDOFF_PATH: Final[Path] = (
    V01_HANDOFF_ROOT
    / "06_point_process_baselines_to_07_hawkes_estimation_handoff.json"
)

# This cell performs no filesystem writes.
# Output directories are created only after upstream authority,
# identity, checksum, and protected-partition gates pass.


# ============================================================
# Time-unit contract
# ============================================================

NANOSECONDS_PER_MILLISECOND: Final[int] = 1_000_000
NANOSECONDS_PER_SECOND: Final[int] = 1_000_000_000

MODEL_TIME_UNIT: Final[str] = "SECONDS"
EVENT_TIME_STORAGE_UNIT: Final[str] = "NANOSECONDS"

COUNT_GRID_WIDTH_NS: Final[int] = NANOSECONDS_PER_MILLISECOND
COUNT_GRID_WIDTH_SECONDS: Final[float] = (
    COUNT_GRID_WIDTH_NS / NANOSECONDS_PER_SECOND
)


# ============================================================
# Frozen candidate-model registry
# ============================================================

H1_MODEL_ID: Final[str] = "H1_DIAGONAL_SHARED_DECAY"
H2_MODEL_ID: Final[str] = "H2_DIAGONAL_SEPARATE_DECAY"

CANDIDATE_MODEL_IDS: Final[tuple[str, str]] = (
    H1_MODEL_ID,
    H2_MODEL_ID,
)

FORMAL_COUNT_COMPARATOR: Final[str] = (
    "D0_SIDE_CONSTANT_POISSON"
)
PRIMARY_SIMPLE_COUNT_BASELINE: Final[str] = (
    "E_SIDE_EWMA_250MS_POISSON"
)


# ============================================================
# Deterministic estimation contract
# ============================================================

RANDOM_SEED: Final[int] = 20260720

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
RNG = np.random.default_rng(RANDOM_SEED)

BRANCHING_MASS_STARTS: Final[tuple[float, ...]] = (
    0.05,
    0.25,
    0.60,
    0.90,
)

HALF_LIFE_STARTS_MS: Final[tuple[float, ...]] = (
    5.0,
    50.0,
    250.0,
    1_000.0,
    5_000.0,
)

LEFT_EDGE_HISTORY_ONLY_SECONDS: Final[tuple[float, ...]] = (
    0.0,
    1.0,
    5.0,
)

DEVELOPMENT_CHRONOLOGICAL_FOLDS: Final[int] = 5
CHRONOLOGICAL_STABILITY_BLOCKS: Final[int] = 5

BOOTSTRAP_BLOCK_SECONDS: Final[int] = 10
N_BLOCK_BOOTSTRAP_REPLICATES: Final[int] = 2_000

MAX_OPTIMIZER_ITERATIONS: Final[int] = 5_000
OPTIMIZER_FUNCTION_TOLERANCE: Final[float] = 1e-10
OPTIMIZER_GRADIENT_TOLERANCE: Final[float] = 1e-7

PARAMETER_POSITIVITY_FLOOR: Final[float] = 1e-12
INTENSITY_FLOOR_PER_SECOND: Final[float] = 1e-12
LOG_LIKELIHOOD_FLOOR: Final[float] = np.finfo(np.float64).tiny

STATIONARITY_HARD_LIMIT: Final[float] = 1.0
STATIONARITY_ACCEPTANCE_MARGIN: Final[float] = 0.98

FLOAT_ABSOLUTE_TOLERANCE: Final[float] = 1e-10
FLOAT_RELATIVE_TOLERANCE: Final[float] = 1e-8

SYNTHETIC_RECOVERY_SEED: Final[int] = RANDOM_SEED + 1
SYNTHETIC_RECOVERY_REPLICATES: Final[int] = 25


# ============================================================
# Permitted terminal statuses
# ============================================================

PERMITTED_TERMINAL_STATUSES: Final[tuple[str, ...]] = (
    "PASS_HAWKES_ESTIMATION_DIAGNOSTICS_AUTHORIZED",
    "PASS_HAWKES_ESTIMATION_POISSON_LIMIT_DIAGNOSTICS_AUTHORIZED",
    "CONDITIONAL_PASS_HAWKES_ESTIMATION_DIAGNOSTIC_ONLY",
    "FAIL_HAWKES_ESTIMATION_UNSTABLE",
    "FAIL_HAWKES_PIPELINE_CONTRACT",
)


# ============================================================
# Immutable notebook configuration
# ============================================================

@dataclass(frozen=True, slots=True)
class Notebook07Config:
    notebook_name: str
    notebook_stage: str
    source_run_prefix: str
    v01_run_id: str
    random_seed: int
    estimator_label: str
    event_representation: str
    event_time_column: str
    event_partition_column: str
    timestamp_interface: str
    fit_partition: str
    locked_evaluation_partition: str
    protected_partitions: tuple[str, ...]
    authorized_model: str
    authorized_kernel_family: str
    authorized_channels: tuple[str, ...]
    fixed_zero_channels: tuple[str, ...]
    candidate_model_ids: tuple[str, ...]
    development_chronological_folds: int
    chronological_stability_blocks: int
    bootstrap_block_seconds: int
    block_bootstrap_replicates: int
    stationarity_acceptance_margin: float
    stationarity_hard_limit: float
    intensity_floor_per_second: float
    calibration_parameter_updates: int


NOTEBOOK_CONFIG = Notebook07Config(
    notebook_name=NOTEBOOK_NAME,
    notebook_stage=NOTEBOOK_STAGE,
    source_run_prefix=SOURCE_RUN_PREFIX,
    v01_run_id=V01_RUN_ID,
    random_seed=RANDOM_SEED,
    estimator_label=PRIMARY_ESTIMATOR_LABEL,
    event_representation=PRIMARY_EVENT_REPRESENTATION,
    event_time_column=PRIMARY_EVENT_TIME_COLUMN,
    event_partition_column=PRIMARY_EVENT_PARTITION_COLUMN,
    timestamp_interface=PRIMARY_TIMESTAMP_INTERFACE,
    fit_partition=FIT_PARTITION,
    locked_evaluation_partition=LOCKED_EVALUATION_PARTITION,
    protected_partitions=PROTECTED_PARTITIONS,
    authorized_model=AUTHORIZED_FIRST_MODEL,
    authorized_kernel_family=AUTHORIZED_KERNEL_FAMILY,
    authorized_channels=AUTHORIZED_CHANNELS,
    fixed_zero_channels=FIXED_ZERO_CHANNELS,
    candidate_model_ids=CANDIDATE_MODEL_IDS,
    development_chronological_folds=DEVELOPMENT_CHRONOLOGICAL_FOLDS,
    chronological_stability_blocks=CHRONOLOGICAL_STABILITY_BLOCKS,
    bootstrap_block_seconds=BOOTSTRAP_BLOCK_SECONDS,
    block_bootstrap_replicates=N_BLOCK_BOOTSTRAP_REPLICATES,
    stationarity_acceptance_margin=STATIONARITY_ACCEPTANCE_MARGIN,
    stationarity_hard_limit=STATIONARITY_HARD_LIMIT,
    intensity_floor_per_second=INTENSITY_FLOOR_PER_SECOND,
    calibration_parameter_updates=CALIBRATION_PARAMETER_UPDATES,
)


def canonical_json_sha256(payload: Mapping[str, Any]) -> str:
    """Return the SHA-256 digest of a canonical JSON mapping."""
    canonical_payload = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(canonical_payload).hexdigest()


NOTEBOOK_CONFIG_SHA256: Final[str] = canonical_json_sha256(
    asdict(NOTEBOOK_CONFIG)
)


# ============================================================
# Runtime provenance
# ============================================================

runtime_provenance = pd.DataFrame(
    [
        {
            "component": "python",
            "version": platform.python_version(),
        },
        {
            "component": "numpy",
            "version": np.__version__,
        },
        {
            "component": "pandas",
            "version": pd.__version__,
        },
        {
            "component": "scipy",
            "version": scipy.__version__,
        },
        {
            "component": "matplotlib",
            "version": matplotlib.__version__,
        },
        {
            "component": "platform",
            "version": platform.platform(),
        },
    ]
)

setup_summary = pd.DataFrame(
    [
        {
            "field": "notebook",
            "value": NOTEBOOK_NAME,
        },
        {
            "field": "source_run_prefix",
            "value": SOURCE_RUN_PREFIX,
        },
        {
            "field": "v01_run_id",
            "value": V01_RUN_ID,
        },
        {
            "field": "authorized_first_model",
            "value": AUTHORIZED_FIRST_MODEL,
        },
        {
            "field": "authorized_channels",
            "value": ", ".join(AUTHORIZED_CHANNELS),
        },
        {
            "field": "fixed_zero_channels",
            "value": ", ".join(FIXED_ZERO_CHANNELS),
        },
        {
            "field": "fit_partition",
            "value": FIT_PARTITION,
        },
        {
            "field": "locked_evaluation_partition",
            "value": LOCKED_EVALUATION_PARTITION,
        },
        {
            "field": "calibration_parameter_updates",
            "value": CALIBRATION_PARAMETER_UPDATES,
        },
        {
            "field": "protected_partitions",
            "value": ", ".join(PROTECTED_PARTITIONS),
        },
        {
            "field": "random_seed",
            "value": RANDOM_SEED,
        },
        {
            "field": "notebook_config_sha256",
            "value": NOTEBOOK_CONFIG_SHA256,
        },
        {
            "field": "upstream_authorization_verified",
            "value": False,
        },
        {
            "field": "protected_partition_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": False,
        },
    ]
)

display(setup_summary)
display(runtime_provenance)

print(
    "Notebook 07 runtime and immutable estimation configuration initialized. "
    "No upstream analytical data, protected partition content, or V0.0 "
    "reference artifacts have been loaded. No filesystem writes were performed."
)

,field,value
0,notebook,07_HAWKES_ESTIMATION.ipynb
1,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
2,v01_run_id,v0_1_20260714T090616Z_e82325081a81
3,authorized_first_model,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES
4,authorized_channels,"BUY_TO_BUY, SELL_TO_SELL"
5,fixed_zero_channels,"BUY_TO_SELL, SELL_TO_BUY"
6,fit_partition,DEVELOPMENT
7,locked_evaluation_partition,CALIBRATION
8,calibration_parameter_updates,0
9,protected_partitions,"VALIDATION, ENGINEERING_HOLDOUT"


,component,version
0,python,3.11.9
1,numpy,2.2.0
2,pandas,2.3.2
3,scipy,1.16.1
4,matplotlib,3.10.6
5,platform,Windows-10-10.0.26200-SP0


Notebook 07 runtime and immutable estimation configuration initialized. No upstream analytical data, protected partition content, or V0.0 reference artifacts have been loaded. No filesystem writes were performed.


In [2]:
# ============================================================
# Verify Notebook 06 authority and register Notebook 07 inputs
# ============================================================

import re


# ------------------------------------------------------------
# Validation helpers
# ------------------------------------------------------------

SHA256_PATTERN: Final[re.Pattern[str]] = re.compile(
    r"^[0-9a-f]{64}$"
)


def require(
    condition: bool,
    message: str,
) -> None:
    """Raise a blocking pipeline error when a contract fails."""
    if not bool(condition):
        raise RuntimeError(message)


def sha256_file(
    source_path: Path,
    chunk_size: int = 1 << 20,
) -> str:
    """Return the SHA-256 digest of a file without loading it at once."""
    require(
        source_path.is_file(),
        f"Required file does not exist: {source_path}",
    )

    digest = hashlib.sha256()

    with source_path.open("rb") as source_file:
        for chunk in iter(
            lambda: source_file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def read_json_object(
    source_path: Path,
) -> dict[str, Any]:
    """Read a JSON document and require a top-level object."""
    require(
        source_path.is_file(),
        f"Required JSON artifact does not exist: {source_path}",
    )

    with source_path.open(
        "r",
        encoding="utf-8",
    ) as source_file:
        document = json.load(source_file)

    require(
        isinstance(document, dict),
        f"Expected a JSON object: {source_path}",
    )

    return document


def verify_self_hashed_json(
    source_path: Path,
) -> dict[str, Any]:
    """Read and verify a JSON document containing payload_sha256."""
    document = read_json_object(source_path)

    require(
        "payload_sha256" in document,
        (
            "Self-hashed JSON artifact is missing payload_sha256: "
            f"{source_path}"
        ),
    )

    registered_payload_sha256 = str(
        document["payload_sha256"]
    )

    require(
        bool(SHA256_PATTERN.fullmatch(registered_payload_sha256)),
        (
            "Invalid payload SHA-256 format in artifact: "
            f"{source_path}"
        ),
    )

    payload_without_hash = {
        str(key): value
        for key, value in document.items()
        if key != "payload_sha256"
    }

    recomputed_payload_sha256 = canonical_json_sha256(
        payload_without_hash
    )

    require(
        recomputed_payload_sha256
        == registered_payload_sha256,
        (
            "Self-hashed JSON payload verification failed: "
            f"{source_path}"
        ),
    )

    return document


def path_is_inside(
    candidate_path: Path,
    parent_path: Path,
) -> bool:
    """Return whether candidate_path is contained by parent_path."""
    candidate_resolved = candidate_path.resolve()
    parent_resolved = parent_path.resolve()

    try:
        candidate_resolved.relative_to(parent_resolved)
    except ValueError:
        return False

    return True


def normalized_path(
    source_path: Path,
) -> str:
    """Return a normalized, case-insensitive absolute path string."""
    return str(source_path.resolve()).casefold()


# ------------------------------------------------------------
# Exact Notebook 06 control paths
# ------------------------------------------------------------

NOTEBOOK06_MANIFEST_DIR: Final[Path] = (
    V01_ARTIFACT_ROOT / "manifests"
)
NOTEBOOK06_HANDOFF_DIR: Final[Path] = (
    V01_ARTIFACT_ROOT / "handoff"
)

NOTEBOOK06_EXPECTED_CONTROL_PATHS: Final[
    Mapping[str, Path]
] = {
    "terminal_decision": (
        NOTEBOOK06_MANIFEST_DIR
        / "06_point_process_baselines_terminal_decision.json"
    ),
    "final_acceptance": (
        NOTEBOOK06_MANIFEST_DIR
        / "06_point_process_baselines_final_acceptance.json"
    ),
    "output_manifest": (
        NOTEBOOK06_MANIFEST_DIR
        / "06_point_process_baselines_output_manifest.json"
    ),
    "readback_audit": (
        NOTEBOOK06_MANIFEST_DIR
        / "06_point_process_baselines_readback_audit.json"
    ),
}

NOTEBOOK06_HANDOFF_PATH: Final[Path] = (
    NOTEBOOK06_HANDOFF_DIR
    / (
        "06_point_process_baselines_to_"
        "07_hawkes_estimation_handoff.json"
    )
)

all_control_paths = (
    *NOTEBOOK06_EXPECTED_CONTROL_PATHS.values(),
    NOTEBOOK06_HANDOFF_PATH,
)

for control_path in all_control_paths:
    require(
        control_path.is_file(),
        f"Required Notebook 06 control artifact is missing: {control_path}",
    )
    require(
        path_is_inside(control_path, V01_ROOT),
        (
            "Notebook 06 control artifact is outside V0.1: "
            f"{control_path}"
        ),
    )
    require(
        not path_is_inside(control_path, V00_ROOT),
        (
            "Notebook 06 control artifact resolves inside immutable V0.0: "
            f"{control_path}"
        ),
    )


# ------------------------------------------------------------
# Verify the Notebook 06-to-07 handoff first
# ------------------------------------------------------------

NOTEBOOK06_HANDOFF_DOCUMENT = verify_self_hashed_json(
    NOTEBOOK06_HANDOFF_PATH
)

require(
    NOTEBOOK06_HANDOFF_DOCUMENT.get("artifact_type")
    == "NOTEBOOK_06_TO_NOTEBOOK_07_HANDOFF",
    "Unexpected Notebook 06 handoff artifact type.",
)
require(
    NOTEBOOK06_HANDOFF_DOCUMENT.get("schema_version")
    == "NOTEBOOK_06_TO_NOTEBOOK_07_HANDOFF_V1",
    "Unexpected Notebook 06 handoff schema version.",
)
require(
    NOTEBOOK06_HANDOFF_DOCUMENT.get("producer")
    == "06_POINT_PROCESS_BASELINES.ipynb",
    "Unexpected Notebook 06 handoff producer.",
)
require(
    NOTEBOOK06_HANDOFF_DOCUMENT.get("consumer")
    == NOTEBOOK_NAME,
    "Notebook 06 handoff is not addressed to this notebook.",
)
require(
    NOTEBOOK06_HANDOFF_DOCUMENT.get("status")
    == UPSTREAM_REQUIRED_TERMINAL_STATUS,
    "Notebook 06 handoff does not preserve the required terminal status.",
)


# ------------------------------------------------------------
# Verify referenced Notebook 06 control artifacts
# ------------------------------------------------------------

handoff_control_registry = NOTEBOOK06_HANDOFF_DOCUMENT.get(
    "control_artifacts"
)

require(
    isinstance(handoff_control_registry, dict),
    "Notebook 06 handoff is missing its control-artifact registry.",
)

NOTEBOOK06_CONTROL_DOCUMENTS: dict[str, dict[str, Any]] = {}
control_artifact_rows: list[dict[str, Any]] = []

for control_name, expected_path in (
    NOTEBOOK06_EXPECTED_CONTROL_PATHS.items()
):
    require(
        control_name in handoff_control_registry,
        (
            "Notebook 06 handoff does not register required control "
            f"artifact: {control_name}"
        ),
    )

    control_reference = handoff_control_registry[control_name]

    require(
        isinstance(control_reference, dict),
        (
            "Malformed control-artifact reference in Notebook 06 "
            f"handoff: {control_name}"
        ),
    )

    referenced_path = Path(
        str(control_reference.get("path", ""))
    )

    referenced_payload_sha256 = str(
        control_reference.get("payload_sha256", "")
    )
    referenced_file_sha256 = str(
        control_reference.get("sha256", "")
    )

    require(
        normalized_path(referenced_path)
        == normalized_path(expected_path),
        (
            "Notebook 06 handoff references an unexpected path for "
            f"{control_name}: {referenced_path}"
        ),
    )
    require(
        bool(
            SHA256_PATTERN.fullmatch(
                referenced_payload_sha256
            )
        ),
        (
            "Invalid referenced payload SHA-256 for "
            f"{control_name}."
        ),
    )
    require(
        bool(
            SHA256_PATTERN.fullmatch(
                referenced_file_sha256
            )
        ),
        (
            "Invalid referenced raw-file SHA-256 for "
            f"{control_name}."
        ),
    )

    verified_document = verify_self_hashed_json(expected_path)
    observed_file_sha256 = sha256_file(expected_path)

    payload_hash_matches = (
        verified_document["payload_sha256"]
        == referenced_payload_sha256
    )
    file_hash_matches = (
        observed_file_sha256
        == referenced_file_sha256
    )

    require(
        payload_hash_matches,
        (
            "Notebook 06 referenced payload hash does not match "
            f"readback for {control_name}."
        ),
    )
    require(
        file_hash_matches,
        (
            "Notebook 06 referenced file hash does not match "
            f"readback for {control_name}."
        ),
    )

    NOTEBOOK06_CONTROL_DOCUMENTS[control_name] = (
        verified_document
    )

    control_artifact_rows.append(
        {
            "control_artifact": control_name,
            "path": str(expected_path),
            "artifact_type": verified_document.get(
                "artifact_type"
            ),
            "schema_version": verified_document.get(
                "schema_version"
            ),
            "payload_sha256": referenced_payload_sha256,
            "file_sha256": referenced_file_sha256,
            "payload_hash_matches": payload_hash_matches,
            "file_hash_matches": file_hash_matches,
            "inside_v0_1": path_is_inside(
                expected_path,
                V01_ROOT,
            ),
            "inside_v0_0": path_is_inside(
                expected_path,
                V00_ROOT,
            ),
            "status": "PASS",
        }
    )

NOTEBOOK06_CONTROL_ARTIFACT_AUDIT = pd.DataFrame(
    control_artifact_rows
)


# ------------------------------------------------------------
# Verify frozen run identity across every control artifact
# ------------------------------------------------------------

expected_identity_values: Final[Mapping[str, str]] = {
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
}

identity_documents: dict[str, dict[str, Any]] = {
    "handoff": NOTEBOOK06_HANDOFF_DOCUMENT,
    **NOTEBOOK06_CONTROL_DOCUMENTS,
}

identity_rows: list[dict[str, Any]] = []

for document_name, document in identity_documents.items():
    for identity_field, expected_value in (
        expected_identity_values.items()
    ):
        if identity_field not in document:
            identity_rows.append(
                {
                    "document": document_name,
                    "identity_field": identity_field,
                    "expected_value": expected_value,
                    "observed_value": pd.NA,
                    "field_present": False,
                    "matches_frozen_contract": pd.NA,
                    "status": "NOT_APPLICABLE",
                }
            )
            continue

        observed_value = str(document[identity_field])
        matches_frozen_contract = (
            observed_value == expected_value
        )

        require(
            matches_frozen_contract,
            (
                f"{document_name} has a conflicting "
                f"{identity_field}: {observed_value}"
            ),
        )

        identity_rows.append(
            {
                "document": document_name,
                "identity_field": identity_field,
                "expected_value": expected_value,
                "observed_value": observed_value,
                "field_present": True,
                "matches_frozen_contract": True,
                "status": "PASS",
            }
        )

NOTEBOOK06_IDENTITY_AUDIT = pd.DataFrame(identity_rows)


# ------------------------------------------------------------
# Terminal-decision and final-acceptance gates
# ------------------------------------------------------------

terminal_document = NOTEBOOK06_CONTROL_DOCUMENTS[
    "terminal_decision"
]
acceptance_document = NOTEBOOK06_CONTROL_DOCUMENTS[
    "final_acceptance"
]
manifest_document = NOTEBOOK06_CONTROL_DOCUMENTS[
    "output_manifest"
]
readback_document = NOTEBOOK06_CONTROL_DOCUMENTS[
    "readback_audit"
]

require(
    terminal_document.get("artifact_type")
    == "NOTEBOOK_06_TERMINAL_DECISION",
    "Unexpected Notebook 06 terminal-decision artifact type.",
)
require(
    terminal_document.get("schema_version")
    == "NOTEBOOK_06_TERMINAL_DECISION_V1",
    "Unexpected Notebook 06 terminal-decision schema.",
)
require(
    terminal_document.get("terminal_status")
    == UPSTREAM_REQUIRED_TERMINAL_STATUS,
    "Notebook 06 did not terminate with the required status.",
)
require(
    terminal_document.get("authorization_state")
    == UPSTREAM_REQUIRED_AUTHORIZATION_STATE,
    "Notebook 06 did not preserve restricted-first authorization.",
)
require(
    bool(terminal_document.get("notebook_07_authorized")),
    "Notebook 06 terminal decision does not authorize Notebook 07.",
)
require(
    bool(terminal_document.get("restricted_first_required")),
    "Notebook 06 terminal decision does not require restricted-first estimation.",
)
require(
    terminal_document.get("primary_hawkes_scope")
    == AUTHORIZED_FIRST_MODEL,
    "Notebook 06 terminal decision authorizes an unexpected Hawkes scope.",
)
require(
    terminal_document.get("formal_count_comparator")
    == FORMAL_COUNT_COMPARATOR,
    "Notebook 06 terminal decision changed the formal comparator.",
)
require(
    terminal_document.get("primary_simple_count_baseline")
    == PRIMARY_SIMPLE_COUNT_BASELINE,
    "Notebook 06 terminal decision changed the primary simple baseline.",
)
require(
    terminal_document.get("status") == "PASS",
    "Notebook 06 terminal-decision artifact is not marked PASS.",
)

terminal_claim_limits = terminal_document.get(
    "claim_limits",
    {},
)

for prohibited_claim_field in (
    "state_dependent_hawkes_authorized",
    "hawkes_superiority_claim_authorized",
    "strategy_or_quoting_claim_authorized",
    "fill_or_execution_claim_authorized",
    "pnl_or_performance_claim_authorized",
):
    require(
        terminal_claim_limits.get(prohibited_claim_field)
        is False,
        (
            "Notebook 06 terminal decision unexpectedly authorizes "
            f"{prohibited_claim_field}."
        ),
    )

terminal_partition_access = terminal_document.get(
    "protected_partition_content_loaded",
    {},
)

for protected_partition in PROTECTED_PARTITIONS:
    require(
        terminal_partition_access.get(protected_partition)
        is False,
        (
            "Notebook 06 terminal decision reports protected "
            f"partition content loaded: {protected_partition}"
        ),
    )

require(
    acceptance_document.get("artifact_type")
    == "NOTEBOOK_06_FINAL_ACCEPTANCE",
    "Unexpected Notebook 06 final-acceptance artifact type.",
)
require(
    acceptance_document.get("schema_version")
    == "NOTEBOOK_06_FINAL_ACCEPTANCE_V1",
    "Unexpected Notebook 06 final-acceptance schema.",
)
require(
    acceptance_document.get("terminal_status")
    == UPSTREAM_REQUIRED_TERMINAL_STATUS,
    "Notebook 06 final acceptance has an unexpected terminal status.",
)
require(
    acceptance_document.get("authorization_state")
    == UPSTREAM_REQUIRED_AUTHORIZATION_STATE,
    "Notebook 06 final acceptance has an unexpected authorization state.",
)
require(
    int(acceptance_document.get("blocking_failure_count", -1))
    == 0,
    "Notebook 06 final acceptance contains blocking failures.",
)
require(
    acceptance_document.get("next_authorized_notebook")
    == NOTEBOOK_NAME,
    "Notebook 06 final acceptance authorizes a different next notebook.",
)
require(
    bool(acceptance_document.get("notebook_07_authorized")),
    "Notebook 06 final acceptance does not authorize Notebook 07.",
)
require(
    bool(acceptance_document.get("restricted_first_required")),
    "Notebook 06 final acceptance omits the restricted-first requirement.",
)
require(
    acceptance_document.get("authorized_primary_scope")
    == AUTHORIZED_FIRST_MODEL,
    "Notebook 06 final acceptance authorizes an unexpected primary scope.",
)


# ------------------------------------------------------------
# Manifest and persistence-readback gates
# ------------------------------------------------------------

require(
    manifest_document.get("artifact_type")
    == "NOTEBOOK_06_OUTPUT_MANIFEST",
    "Unexpected Notebook 06 output-manifest artifact type.",
)
require(
    manifest_document.get("schema_version")
    == "NOTEBOOK_06_OUTPUT_MANIFEST_V1",
    "Unexpected Notebook 06 output-manifest schema.",
)
require(
    manifest_document.get("status") == "PASS",
    "Notebook 06 output manifest is not marked PASS.",
)
require(
    int(manifest_document.get("artifact_count", 0)) > 0,
    "Notebook 06 output manifest registers no substantive artifacts.",
)

manifest_partition_access = manifest_document.get(
    "protected_partition_content_loaded",
    {},
)

for protected_partition in PROTECTED_PARTITIONS:
    require(
        manifest_partition_access.get(protected_partition)
        is False,
        (
            "Notebook 06 output manifest reports protected "
            f"partition content loaded: {protected_partition}"
        ),
    )

require(
    readback_document.get("artifact_type")
    == "NOTEBOOK_06_READBACK_AUDIT",
    "Unexpected Notebook 06 readback-audit artifact type.",
)
require(
    readback_document.get("schema_version")
    == "NOTEBOOK_06_READBACK_AUDIT_V1",
    "Unexpected Notebook 06 readback-audit schema.",
)
require(
    readback_document.get("status") == "PASS",
    "Notebook 06 readback audit is not marked PASS.",
)
require(
    int(readback_document.get("verified_artifact_count", 0))
    > 0,
    "Notebook 06 readback audit verified no artifacts.",
)
require(
    int(readback_document.get("failed_artifact_count", -1))
    == 0,
    "Notebook 06 readback audit contains failed artifacts.",
)
require(
    bool(readback_document.get("all_paths_inside_v0_1")),
    "Notebook 06 readback audit found an artifact outside V0.1.",
)
require(
    not bool(readback_document.get("any_path_inside_v0_0")),
    "Notebook 06 readback audit found a persisted artifact inside V0.0.",
)
require(
    bool(readback_document.get("v0_0_reference_hash_stable")),
    "Notebook 06 reports mutation of a V0.0 reference artifact.",
)


# ------------------------------------------------------------
# Verify the authorized Hawkes estimation contract
# ------------------------------------------------------------

handoff_authorization = NOTEBOOK06_HANDOFF_DOCUMENT.get(
    "authorization",
    {},
)
handoff_estimation_contract = (
    NOTEBOOK06_HANDOFF_DOCUMENT.get(
        "authorized_estimation_contract",
        {},
    )
)
handoff_event_contract = NOTEBOOK06_HANDOFF_DOCUMENT.get(
    "event_time_contract",
    {},
)
handoff_partition_contract = (
    NOTEBOOK06_HANDOFF_DOCUMENT.get(
        "partition_contract",
        {},
    )
)
handoff_expected_counts = (
    NOTEBOOK06_HANDOFF_DOCUMENT.get(
        "expected_analytical_counts",
        {},
    )
)
handoff_baseline_context = (
    NOTEBOOK06_HANDOFF_DOCUMENT.get(
        "baseline_context",
        {},
    )
)

contract_expectations: Final[Mapping[str, Any]] = {
    "authorization.authorization_state": (
        UPSTREAM_REQUIRED_AUTHORIZATION_STATE
    ),
    "authorization.terminal_status": (
        UPSTREAM_REQUIRED_TERMINAL_STATUS
    ),
    "authorization.notebook_07_authorized": True,
    "authorization.restricted_first_required": True,
    "authorization.primary_hawkes_scope": (
        AUTHORIZED_FIRST_MODEL
    ),
    "authorization.state_dependent_hawkes_authorized": False,
    "authorization.hawkes_superiority_claim_authorized": False,
    "authorization.strategy_or_quoting_use_authorized": False,
    "estimation.first_model": AUTHORIZED_FIRST_MODEL,
    "estimation.authorized_event_components": list(EVENT_SIDES),
    "estimation.authorized_excitation_channels_first": (
        list(AUTHORIZED_CHANNELS)
    ),
    "estimation.cross_excitation_channels_first": (
        "FIXED_TO_ZERO"
    ),
    "estimation.kernel_family_first": (
        AUTHORIZED_KERNEL_FAMILY
    ),
    "estimation.parameter_fit_partition": (
        "DEVELOPMENT_ONLY"
    ),
    "estimation.hyperparameter_selection_partition": (
        "DEVELOPMENT_ONLY"
    ),
    "estimation.locked_evaluation_partition": (
        LOCKED_EVALUATION_PARTITION
    ),
    "estimation.calibration_parameter_updates": 0,
    "estimation.development_history_carried_into_calibration": True,
    "estimation.development_left_prehistory_fabricated": False,
    "estimation.left_censoring_must_be_recorded": True,
    "estimation.base_intensity_constraint": (
        "STRICTLY_POSITIVE"
    ),
    "estimation.kernel_amplitude_constraint": (
        "NONNEGATIVE"
    ),
    "estimation.kernel_decay_constraint": (
        "STRICTLY_POSITIVE"
    ),
    "estimation.stationarity_constraint": (
        "EXCITATION_NORM_SPECTRAL_RADIUS_"
        "STRICTLY_LESS_THAN_ONE"
    ),
    "event.event_representation": (
        PRIMARY_EVENT_REPRESENTATION
    ),
    "event.timestamp_column": PRIMARY_EVENT_TIME_COLUMN,
    "event.partition_column": (
        PRIMARY_EVENT_PARTITION_COLUMN
    ),
    "event.timestamp_authority": (
        "NOTEBOOK_04_EVENT_TIME_NS"
    ),
    "event.simultaneous_batch_interface": (
        PRIMARY_TIMESTAMP_INTERFACE
    ),
    "event.scoring_history": (
        "STRICTLY_BEFORE_BATCH_TIME"
    ),
    "event.within_batch_zero_lag_excitation": False,
    "event.batch_update_timing": (
        "AFTER_ALL_EVENTS_IN_BATCH_ARE_SCORED"
    ),
    "event.collector_sequence_role": (
        "TRACE_ORDER_ONLY_NOT_PHYSICAL_TIME"
    ),
    "event.timestamp_jitter_allowed": False,
    "event.event_reordering_allowed": False,
    "event.event_removal_allowed": False,
    "partition.fit_partition": FIT_PARTITION,
    "partition.locked_evaluation_partition": (
        LOCKED_EVALUATION_PARTITION
    ),
    "partition.validation_content_access": "FORBIDDEN",
    "partition.engineering_holdout_content_access": (
        "FORBIDDEN"
    ),
    "partition.validation_content_loaded": False,
    "partition.engineering_holdout_content_loaded": False,
    "baseline.formal_count_comparator": (
        FORMAL_COUNT_COMPARATOR
    ),
    "baseline.primary_simple_count_baseline": (
        PRIMARY_SIMPLE_COUNT_BASELINE
    ),
    "baseline.primary_simple_half_life_ms": 250,
    "baseline.remaining_self_dependence_evidence": True,
    "baseline.directional_residual_cross_evidence_strong": False,
}

observed_contract_values: Final[Mapping[str, Any]] = {
    "authorization.authorization_state": (
        handoff_authorization.get("authorization_state")
    ),
    "authorization.terminal_status": (
        handoff_authorization.get("terminal_status")
    ),
    "authorization.notebook_07_authorized": (
        handoff_authorization.get("notebook_07_authorized")
    ),
    "authorization.restricted_first_required": (
        handoff_authorization.get("restricted_first_required")
    ),
    "authorization.primary_hawkes_scope": (
        handoff_authorization.get("primary_hawkes_scope")
    ),
    "authorization.state_dependent_hawkes_authorized": (
        handoff_authorization.get(
            "state_dependent_hawkes_authorized"
        )
    ),
    "authorization.hawkes_superiority_claim_authorized": (
        handoff_authorization.get(
            "hawkes_superiority_claim_authorized"
        )
    ),
    "authorization.strategy_or_quoting_use_authorized": (
        handoff_authorization.get(
            "strategy_or_quoting_use_authorized"
        )
    ),
    "estimation.first_model": (
        handoff_estimation_contract.get("first_model")
    ),
    "estimation.authorized_event_components": (
        handoff_estimation_contract.get(
            "authorized_event_components"
        )
    ),
    "estimation.authorized_excitation_channels_first": (
        handoff_estimation_contract.get(
            "authorized_excitation_channels_first"
        )
    ),
    "estimation.cross_excitation_channels_first": (
        handoff_estimation_contract.get(
            "cross_excitation_channels_first"
        )
    ),
    "estimation.kernel_family_first": (
        handoff_estimation_contract.get(
            "kernel_family_first"
        )
    ),
    "estimation.parameter_fit_partition": (
        handoff_estimation_contract.get(
            "parameter_fit_partition"
        )
    ),
    "estimation.hyperparameter_selection_partition": (
        handoff_estimation_contract.get(
            "hyperparameter_selection_partition"
        )
    ),
    "estimation.locked_evaluation_partition": (
        handoff_estimation_contract.get(
            "locked_evaluation_partition"
        )
    ),
    "estimation.calibration_parameter_updates": (
        handoff_estimation_contract.get(
            "calibration_parameter_updates"
        )
    ),
    "estimation.development_history_carried_into_calibration": (
        handoff_estimation_contract.get(
            "development_history_carried_into_calibration"
        )
    ),
    "estimation.development_left_prehistory_fabricated": (
        handoff_estimation_contract.get(
            "development_left_prehistory_fabricated"
        )
    ),
    "estimation.left_censoring_must_be_recorded": (
        handoff_estimation_contract.get(
            "left_censoring_must_be_recorded"
        )
    ),
    "estimation.base_intensity_constraint": (
        handoff_estimation_contract.get(
            "base_intensity_constraint"
        )
    ),
    "estimation.kernel_amplitude_constraint": (
        handoff_estimation_contract.get(
            "kernel_amplitude_constraint"
        )
    ),
    "estimation.kernel_decay_constraint": (
        handoff_estimation_contract.get(
            "kernel_decay_constraint"
        )
    ),
    "estimation.stationarity_constraint": (
        handoff_estimation_contract.get(
            "stationarity_constraint"
        )
    ),
    "event.event_representation": (
        handoff_event_contract.get("event_representation")
    ),
    "event.timestamp_column": (
        handoff_event_contract.get("timestamp_column")
    ),
    "event.partition_column": (
        handoff_event_contract.get("partition_column")
    ),
    "event.timestamp_authority": (
        handoff_event_contract.get("timestamp_authority")
    ),
    "event.simultaneous_batch_interface": (
        handoff_event_contract.get(
            "simultaneous_batch_interface"
        )
    ),
    "event.scoring_history": (
        handoff_event_contract.get("scoring_history")
    ),
    "event.within_batch_zero_lag_excitation": (
        handoff_event_contract.get(
            "within_batch_zero_lag_excitation"
        )
    ),
    "event.batch_update_timing": (
        handoff_event_contract.get("batch_update_timing")
    ),
    "event.collector_sequence_role": (
        handoff_event_contract.get(
            "collector_sequence_role"
        )
    ),
    "event.timestamp_jitter_allowed": (
        handoff_event_contract.get(
            "timestamp_jitter_allowed"
        )
    ),
    "event.event_reordering_allowed": (
        handoff_event_contract.get(
            "event_reordering_allowed"
        )
    ),
    "event.event_removal_allowed": (
        handoff_event_contract.get(
            "event_removal_allowed"
        )
    ),
    "partition.fit_partition": (
        handoff_partition_contract.get("fit_partition")
    ),
    "partition.locked_evaluation_partition": (
        handoff_partition_contract.get(
            "locked_evaluation_partition"
        )
    ),
    "partition.validation_content_access": (
        handoff_partition_contract.get(
            "validation_content_access"
        )
    ),
    "partition.engineering_holdout_content_access": (
        handoff_partition_contract.get(
            "engineering_holdout_content_access"
        )
    ),
    "partition.validation_content_loaded": (
        handoff_partition_contract.get(
            "validation_content_loaded"
        )
    ),
    "partition.engineering_holdout_content_loaded": (
        handoff_partition_contract.get(
            "engineering_holdout_content_loaded"
        )
    ),
    "baseline.formal_count_comparator": (
        handoff_baseline_context.get(
            "formal_count_comparator"
        )
    ),
    "baseline.primary_simple_count_baseline": (
        handoff_baseline_context.get(
            "primary_simple_count_baseline"
        )
    ),
    "baseline.primary_simple_half_life_ms": (
        handoff_baseline_context.get(
            "primary_simple_half_life_ms"
        )
    ),
    "baseline.remaining_self_dependence_evidence": (
        handoff_baseline_context.get(
            "remaining_self_dependence_evidence"
        )
    ),
    "baseline.directional_residual_cross_evidence_strong": (
        handoff_baseline_context.get(
            "directional_residual_cross_evidence_strong"
        )
    ),
}

contract_rows: list[dict[str, Any]] = []

for contract_item, expected_value in (
    contract_expectations.items()
):
    observed_value = observed_contract_values[
        contract_item
    ]
    passed = observed_value == expected_value

    require(
        passed,
        (
            f"Notebook 06 handoff contract mismatch for "
            f"{contract_item}: expected={expected_value!r}; "
            f"observed={observed_value!r}"
        ),
    )

    contract_rows.append(
        {
            "contract_item": contract_item,
            "expected_value": expected_value,
            "observed_value": observed_value,
            "blocking": True,
            "passed": True,
            "status": "PASS",
        }
    )

NOTEBOOK07_HANDOFF_CONTRACT_AUDIT = pd.DataFrame(
    contract_rows
)


# ------------------------------------------------------------
# Verify expected analytical counts
# ------------------------------------------------------------

expected_analytical_counts: Final[Mapping[str, int]] = {
    "development_event_count": (
        EXPECTED_DEVELOPMENT_EVENT_ROWS
    ),
    "development_batch_count": (
        EXPECTED_DEVELOPMENT_BATCH_ROWS
    ),
    "calibration_event_count": (
        EXPECTED_CALIBRATION_EVENT_ROWS
    ),
    "calibration_batch_count": (
        EXPECTED_CALIBRATION_BATCH_ROWS
    ),
    "analytical_event_count": (
        EXPECTED_ANALYTICAL_EVENT_ROWS
    ),
    "analytical_batch_count": (
        EXPECTED_ANALYTICAL_BATCH_ROWS
    ),
}

count_contract_rows: list[dict[str, Any]] = []

for count_name, expected_count in (
    expected_analytical_counts.items()
):
    observed_count = int(
        handoff_expected_counts.get(
            count_name,
            -1,
        )
    )
    passed = observed_count == expected_count

    require(
        passed,
        (
            "Notebook 06 handoff analytical-count mismatch for "
            f"{count_name}: expected={expected_count}; "
            f"observed={observed_count}"
        ),
    )

    count_contract_rows.append(
        {
            "count_name": count_name,
            "expected_count": expected_count,
            "observed_count": observed_count,
            "passed": True,
            "status": "PASS",
        }
    )

NOTEBOOK07_EXPECTED_COUNT_AUDIT = pd.DataFrame(
    count_contract_rows
)


# ------------------------------------------------------------
# Register and verify authoritative upstream data files
# without loading analytical table content
# ------------------------------------------------------------

upstream_data_records = (
    NOTEBOOK06_HANDOFF_DOCUMENT.get(
        "authoritative_upstream_data_artifacts",
        []
    )
)

require(
    isinstance(upstream_data_records, list),
    (
        "Notebook 06 handoff authoritative-upstream-data "
        "registry is malformed."
    ),
)
require(
    len(upstream_data_records) == 6,
    (
        "Notebook 06 handoff must register exactly six "
        "authoritative upstream data artifacts."
    ),
)

NOTEBOOK07_UPSTREAM_DATA_REGISTRY = pd.DataFrame(
    upstream_data_records
)

required_registry_columns: Final[set[str]] = {
    "label",
    "artifact_type",
    "full_row_count",
    "full_column_count",
    "raw_file_sha256",
    "table_payload_sha256",
    "contract_payload_sha256",
    "resolved_path",
    "verified",
}

missing_registry_columns = (
    required_registry_columns
    - set(NOTEBOOK07_UPSTREAM_DATA_REGISTRY.columns)
)

require(
    not missing_registry_columns,
    (
        "Notebook 06 upstream-data registry is missing columns: "
        f"{sorted(missing_registry_columns)}"
    ),
)

expected_upstream_artifacts: Final[
    Mapping[str, tuple[str, int, int]]
] = {
    "primary_estimation_events": (
        "NOTEBOOK_04_PRIMARY_ESTIMATION_EVENTS",
        EXPECTED_PRIMARY_EVENT_ROWS,
        88,
    ),
    "primary_exact_time_batches": (
        "NOTEBOOK_04_PRIMARY_EXACT_TIME_BATCHES",
        EXPECTED_PRIMARY_SCORING_BATCHES,
        27,
    ),
    "primary_scoring_batches": (
        "NOTEBOOK_04_PRIMARY_SCORING_BATCHES",
        EXPECTED_PRIMARY_SCORING_BATCHES,
        25,
    ),
    "primary_event_to_batch_membership": (
        "NOTEBOOK_04_PRIMARY_EVENT_TO_BATCH_MEMBERSHIP",
        EXPECTED_PRIMARY_EVENT_ROWS,
        10,
    ),
    "observation_window_contract": (
        "NOTEBOOK_04_OBSERVATION_WINDOW_CONTRACT",
        4,
        19,
    ),
    "primary_batch_market_state_features": (
        "PRIMARY_BATCH_MARKET_STATE_FEATURES",
        EXPECTED_PRIMARY_SCORING_BATCHES,
        577,
    ),
}

require(
    set(NOTEBOOK07_UPSTREAM_DATA_REGISTRY["label"])
    == set(expected_upstream_artifacts),
    (
        "Notebook 06 handoff registers an unexpected set of "
        "upstream data artifacts."
    ),
)

upstream_file_rows: list[dict[str, Any]] = []

for registry_row in (
    NOTEBOOK07_UPSTREAM_DATA_REGISTRY
    .sort_values("label", kind="stable")
    .to_dict(orient="records")
):
    label = str(registry_row["label"])
    artifact_type = str(registry_row["artifact_type"])
    upstream_path = Path(
        str(registry_row["resolved_path"])
    )
    registered_sha256 = str(
        registry_row["raw_file_sha256"]
    )

    (
        expected_artifact_type,
        expected_row_count,
        expected_column_count,
    ) = expected_upstream_artifacts[label]

    require(
        artifact_type == expected_artifact_type,
        (
            f"Unexpected artifact type for {label}: "
            f"{artifact_type}"
        ),
    )
    require(
        int(registry_row["full_row_count"])
        == expected_row_count,
        (
            f"Unexpected full row count for {label}: "
            f"{registry_row['full_row_count']}"
        ),
    )
    require(
        int(registry_row["full_column_count"])
        == expected_column_count,
        (
            f"Unexpected full column count for {label}: "
            f"{registry_row['full_column_count']}"
        ),
    )
    require(
        bool(registry_row["verified"]),
        f"Notebook 06 did not mark {label} as verified.",
    )
    require(
        upstream_path.is_file(),
        f"Registered upstream data file is missing: {upstream_path}",
    )
    require(
        path_is_inside(upstream_path, V01_ROOT),
        (
            f"Registered V0.1 data file is outside V0.1: "
            f"{upstream_path}"
        ),
    )
    require(
        not path_is_inside(upstream_path, V00_ROOT),
        (
            f"Registered V0.1 data file resolves inside V0.0: "
            f"{upstream_path}"
        ),
    )
    require(
        bool(SHA256_PATTERN.fullmatch(registered_sha256)),
        f"Invalid registered SHA-256 for {label}.",
    )

    observed_sha256 = sha256_file(upstream_path)
    hash_matches = observed_sha256 == registered_sha256

    require(
        hash_matches,
        (
            f"Upstream data file hash mismatch for {label}: "
            f"{upstream_path}"
        ),
    )

    upstream_file_rows.append(
        {
            "label": label,
            "artifact_type": artifact_type,
            "resolved_path": str(upstream_path),
            "full_row_count": expected_row_count,
            "full_column_count": expected_column_count,
            "registered_sha256": registered_sha256,
            "observed_sha256": observed_sha256,
            "hash_matches": hash_matches,
            "inside_v0_1": True,
            "inside_v0_0": False,
            "analytical_content_loaded": False,
            "status": "PASS",
        }
    )

NOTEBOOK07_UPSTREAM_FILE_AUDIT = pd.DataFrame(
    upstream_file_rows
)


# ------------------------------------------------------------
# Protected-content firewall state
# ------------------------------------------------------------

PROTECTED_PARTITION_CONTENT_LOADED: dict[str, bool] = {
    "DEVELOPMENT": False,
    "CALIBRATION": False,
    "VALIDATION": False,
    "ENGINEERING_HOLDOUT": False,
}

FUTURE_LABEL_TABLE_LOADED: bool = False
MARKET_STATE_FEATURE_VALUES_LOADED: bool = False

UPSTREAM_AUTHORIZATION_VERIFIED: bool = True
FILESYSTEM_WRITES_PERFORMED: bool = False

require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content was loaded during authority verification.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened at this stage.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "Authority verification must be read-only.",
)


# ------------------------------------------------------------
# Authority-verification summary
# ------------------------------------------------------------

authority_summary = pd.DataFrame(
    [
        {
            "field": "upstream_terminal_status",
            "value": terminal_document["terminal_status"],
        },
        {
            "field": "upstream_authorization_state",
            "value": terminal_document["authorization_state"],
        },
        {
            "field": "authorized_first_model",
            "value": handoff_estimation_contract["first_model"],
        },
        {
            "field": "formal_count_comparator",
            "value": handoff_baseline_context[
                "formal_count_comparator"
            ],
        },
        {
            "field": "primary_simple_count_baseline",
            "value": handoff_baseline_context[
                "primary_simple_count_baseline"
            ],
        },
        {
            "field": "registered_upstream_data_files",
            "value": len(NOTEBOOK07_UPSTREAM_FILE_AUDIT),
        },
        {
            "field": "verified_notebook06_artifacts",
            "value": readback_document[
                "verified_artifact_count"
            ],
        },
        {
            "field": "notebook06_failed_artifacts",
            "value": readback_document[
                "failed_artifact_count"
            ],
        },
        {
            "field": "upstream_authorization_verified",
            "value": UPSTREAM_AUTHORIZATION_VERIFIED,
        },
        {
            "field": "analytical_content_loaded",
            "value": False,
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "future_label_table_loaded",
            "value": FUTURE_LABEL_TABLE_LOADED,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(authority_summary)
display(
    NOTEBOOK06_CONTROL_ARTIFACT_AUDIT[
        [
            "control_artifact",
            "artifact_type",
            "payload_hash_matches",
            "file_hash_matches",
            "status",
        ]
    ]
)
display(
    NOTEBOOK07_UPSTREAM_FILE_AUDIT[
        [
            "label",
            "artifact_type",
            "full_row_count",
            "full_column_count",
            "hash_matches",
            "analytical_content_loaded",
            "status",
        ]
    ]
)

print(
    "Notebook 06 authority, restricted-first authorization, "
    "control artifacts, handoff contract, expected analytical "
    "counts, and registered upstream data-file hashes verified. "
    "No analytical table content or protected partition content "
    "was loaded. No filesystem writes were performed."
)


,field,value
0,upstream_terminal_status,PASS_HAWKES_RESTRICTED_FIRST
1,upstream_authorization_state,AUTHORIZED_RESTRICTED_FIRST
2,authorized_first_model,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES
3,formal_count_comparator,D0_SIDE_CONSTANT_POISSON
4,primary_simple_count_baseline,E_SIDE_EWMA_250MS_POISSON
5,registered_upstream_data_files,6
6,verified_notebook06_artifacts,134
7,notebook06_failed_artifacts,0
8,upstream_authorization_verified,True
9,analytical_content_loaded,False


,control_artifact,artifact_type,payload_hash_matches,file_hash_matches,status
0,terminal_decision,NOTEBOOK_06_TERMINAL_DECISION,True,True,PASS
1,final_acceptance,NOTEBOOK_06_FINAL_ACCEPTANCE,True,True,PASS
2,output_manifest,NOTEBOOK_06_OUTPUT_MANIFEST,True,True,PASS
3,readback_audit,NOTEBOOK_06_READBACK_AUDIT,True,True,PASS


,label,artifact_type,full_row_count,full_column_count,hash_matches,analytical_content_loaded,status
0,observation_window_contract,NOTEBOOK_04_OBSERVATION_WINDOW_CONTRACT,4,19,True,False,PASS
1,primary_batch_market_state_features,PRIMARY_BATCH_MARKET_STATE_FEATURES,13564,577,True,False,PASS
2,primary_estimation_events,NOTEBOOK_04_PRIMARY_ESTIMATION_EVENTS,13887,88,True,False,PASS
3,primary_event_to_batch_membership,NOTEBOOK_04_PRIMARY_EVENT_TO_BATCH_MEMBERSHIP,13887,10,True,False,PASS
4,primary_exact_time_batches,NOTEBOOK_04_PRIMARY_EXACT_TIME_BATCHES,13564,27,True,False,PASS
5,primary_scoring_batches,NOTEBOOK_04_PRIMARY_SCORING_BATCHES,13564,25,True,False,PASS


Notebook 06 authority, restricted-first authorization, control artifacts, handoff contract, expected analytical counts, and registered upstream data-file hashes verified. No analytical table content or protected partition content was loaded. No filesystem writes were performed.


In [3]:
# ============================================================
# Protected analytical load and event/batch reconciliation
# ============================================================

import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("UPSTREAM_AUTHORIZATION_VERIFIED", False)),
    "Notebook 06 authority has not been verified.",
)
require(
    "NOTEBOOK07_UPSTREAM_DATA_REGISTRY" in globals(),
    "The verified upstream-data registry is unavailable.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content was loaded before the analytical load.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are permitted before analytical verification.",
)


# ------------------------------------------------------------
# Analytical partition contract
# ------------------------------------------------------------

ANALYTICAL_PARTITIONS: Final[tuple[str, str]] = (
    FIT_PARTITION,
    LOCKED_EVALUATION_PARTITION,
)

PARTITION_ORDER_MAP: Final[Mapping[str, int]] = {
    "DEVELOPMENT": 1,
    "CALIBRATION": 2,
    "VALIDATION": 3,
    "ENGINEERING_HOLDOUT": 4,
}

EXPECTED_ANALYTICAL_COUNTS_BY_PARTITION: Final[
    Mapping[str, Mapping[str, int]]
] = {
    "DEVELOPMENT": {
        "event_count": EXPECTED_DEVELOPMENT_EVENT_ROWS,
        "batch_count": EXPECTED_DEVELOPMENT_BATCH_ROWS,
    },
    "CALIBRATION": {
        "event_count": EXPECTED_CALIBRATION_EVENT_ROWS,
        "batch_count": EXPECTED_CALIBRATION_BATCH_ROWS,
    },
}


# ------------------------------------------------------------
# Minimal authoritative column interfaces
# ------------------------------------------------------------

EVENT_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_id",
    "primary_event_number",
    "partition_event_index",
    "primary_event_batch_id",
    "primary_event_batch_number",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_event_time_ns",
    "relative_event_time_seconds",
    "event_side",
    "event_side_code",
    "event_print_count",
)

EXACT_BATCH_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_batch_time_ns",
    "relative_batch_time_seconds",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "unique_side_count",
    "batch_print_count",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "contract_start_ns",
    "contract_end_exclusive_ns",
    "event_clock_origin_ns",
    "observation_window_duration_ns",
)

SCORING_BATCH_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_batch_time_ns",
    "relative_batch_time_seconds",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "unique_side_count",
    "batch_print_count",
    "batch_quantity",
    "batch_notional",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "event_clock_origin_ns",
    "observation_window_duration_ns",
    "score_with_history_strictly_before_batch_time",
    "zero_lag_within_batch_excitation_allowed",
    "apply_batch_excitation_after_all_members_scored",
    "collector_sequence_is_trace_order_only",
    "timestamp_jitter_allowed",
    "event_removal_allowed",
)

MEMBERSHIP_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_id",
    "primary_event_batch_number",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "event_side",
    "event_side_code",
    "primary_event_batch_id",
    "primary_event_number",
    "partition_event_index",
)


# ------------------------------------------------------------
# Loading helpers
# ------------------------------------------------------------

def registered_upstream_path(label: str) -> Path:
    """Return one verified upstream path from the handoff registry."""
    matching_rows = NOTEBOOK07_UPSTREAM_DATA_REGISTRY.loc[
        NOTEBOOK07_UPSTREAM_DATA_REGISTRY["label"].eq(label)
    ]

    require(
        len(matching_rows) == 1,
        (
            "Expected exactly one registered upstream artifact "
            f"for {label!r}; observed {len(matching_rows)}."
        ),
    )

    source_path = Path(
        str(matching_rows.iloc[0]["resolved_path"])
    )

    require(
        source_path.is_file(),
        f"Registered upstream path is missing: {source_path}",
    )
    require(
        source_path.suffix.casefold() == ".parquet",
        (
            f"Expected a Parquet artifact for {label!r}; "
            f"observed {source_path.suffix!r}."
        ),
    )

    return source_path


def parquet_schema_columns(source_path: Path) -> tuple[str, ...]:
    """Return the physical Parquet schema without loading table rows."""
    return tuple(
        pq.ParquetFile(source_path).schema_arrow.names
    )


def require_parquet_columns(
    source_path: Path,
    required_columns: tuple[str, ...],
    *,
    label: str,
) -> None:
    """Require all requested fields in a Parquet schema."""
    available_columns = set(
        parquet_schema_columns(source_path)
    )
    missing_columns = sorted(
        set(required_columns) - available_columns
    )

    require(
        not missing_columns,
        (
            f"{label} is missing required columns: "
            f"{missing_columns}"
        ),
    )


def load_filtered_parquet(
    source_path: Path,
    *,
    columns: tuple[str, ...],
    partitions: tuple[str, ...],
    label: str,
) -> pd.DataFrame:
    """
    Load only authorized partitions through an Arrow predicate.

    The filter is applied by the Arrow scanner before conversion
    to a pandas DataFrame. Protected partition rows are therefore
    not materialized in the notebook process.
    """
    require_parquet_columns(
        source_path,
        columns,
        label=label,
    )

    parquet_dataset = ds.dataset(
        source_path,
        format="parquet",
    )

    partition_filter = ds.field(
        PRIMARY_EVENT_PARTITION_COLUMN
    ).isin(list(partitions))

    arrow_table = parquet_dataset.to_table(
        columns=list(columns),
        filter=partition_filter,
    )

    frame = arrow_table.to_pandas(
        types_mapper=None,
        self_destruct=True,
        split_blocks=True,
    )

    require(
        PRIMARY_EVENT_PARTITION_COLUMN in frame.columns,
        (
            f"{label} filtered load lost "
            f"{PRIMARY_EVENT_PARTITION_COLUMN!r}."
        ),
    )

    frame[PRIMARY_EVENT_PARTITION_COLUMN] = (
        frame[PRIMARY_EVENT_PARTITION_COLUMN]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    observed_partitions = set(
        frame[PRIMARY_EVENT_PARTITION_COLUMN]
        .dropna()
        .tolist()
    )

    unauthorized_partitions = (
        observed_partitions - set(partitions)
    )
    protected_partitions_observed = (
        observed_partitions
        & set(PROTECTED_PARTITIONS)
    )

    require(
        not unauthorized_partitions,
        (
            f"{label} materialized unauthorized partitions: "
            f"{sorted(unauthorized_partitions)}"
        ),
    )
    require(
        not protected_partitions_observed,
        (
            f"{label} materialized protected partitions: "
            f"{sorted(protected_partitions_observed)}"
        ),
    )
    require(
        observed_partitions == set(partitions),
        (
            f"{label} does not contain both analytical partitions; "
            f"observed={sorted(observed_partitions)}."
        ),
    )

    return frame


def parse_exact_int64(
    values: pd.Series,
    *,
    label: str,
) -> pd.Series:
    """Convert an integer-like series to exact, non-null int64."""
    require(
        not values.isna().any(),
        f"{label} contains missing values.",
    )
    require(
        not pd.api.types.is_bool_dtype(values.dtype),
        f"{label} must not be Boolean.",
    )

    if pd.api.types.is_integer_dtype(values.dtype):
        parsed = values.astype("int64")
    elif pd.api.types.is_float_dtype(values.dtype):
        numeric_values = values.to_numpy(dtype=np.float64)

        require(
            np.isfinite(numeric_values).all(),
            f"{label} contains non-finite values.",
        )
        require(
            np.equal(
                numeric_values,
                np.rint(numeric_values),
            ).all(),
            f"{label} contains non-integral floating values.",
        )
        require(
            np.abs(numeric_values).max(initial=0.0)
            <= float(np.iinfo(np.int64).max),
            f"{label} exceeds the int64 range.",
        )

        parsed = pd.Series(
            np.rint(numeric_values).astype(np.int64),
            index=values.index,
            name=values.name,
        )
    else:
        numeric_values = pd.to_numeric(
            values.astype("string"),
            errors="raise",
        )

        require(
            not numeric_values.isna().any(),
            f"{label} could not be parsed exactly.",
        )

        parsed = numeric_values.astype("int64")

    return parsed


def normalize_boolean(
    values: pd.Series,
    *,
    label: str,
) -> pd.Series:
    """Return a strict, non-null Boolean series."""
    require(
        not values.isna().any(),
        f"{label} contains missing Boolean values.",
    )

    if pd.api.types.is_bool_dtype(values.dtype):
        return values.astype(bool)

    normalized_text = (
        values.astype("string")
        .str.strip()
        .str.lower()
    )

    valid_values = normalized_text.isin(
        ("true", "false", "1", "0")
    )

    require(
        valid_values.all(),
        f"{label} contains invalid Boolean encodings.",
    )

    return normalized_text.map(
        {
            "true": True,
            "1": True,
            "false": False,
            "0": False,
        }
    ).astype(bool)


def deterministic_sort(
    frame: pd.DataFrame,
    columns: tuple[str, ...],
) -> pd.DataFrame:
    """Return a stable chronological copy of a frame."""
    require(
        set(columns).issubset(frame.columns),
        (
            "Deterministic sort columns are unavailable: "
            f"{columns}"
        ),
    )

    return (
        frame.sort_values(
            list(columns),
            kind="stable",
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Resolve registered sources
# ------------------------------------------------------------

PRIMARY_EVENTS_PATH = registered_upstream_path(
    "primary_estimation_events"
)
EXACT_BATCHES_PATH = registered_upstream_path(
    "primary_exact_time_batches"
)
SCORING_BATCHES_PATH = registered_upstream_path(
    "primary_scoring_batches"
)
EVENT_BATCH_MEMBERSHIP_PATH = registered_upstream_path(
    "primary_event_to_batch_membership"
)
OBSERVATION_WINDOW_CONTRACT_PATH = registered_upstream_path(
    "observation_window_contract"
)


# ------------------------------------------------------------
# Load observation-window metadata
#
# These four rows are partition metadata, not event content.
# Reading all four metadata rows is authorized.
# ------------------------------------------------------------

OBSERVATION_WINDOW_CONTRACT = pd.read_parquet(
    OBSERVATION_WINDOW_CONTRACT_PATH,
    engine="pyarrow",
)

required_window_columns: Final[tuple[str, ...]] = (
    "partition_order",
    "event_partition",
    "contract_start_ns",
    "contract_end_exclusive_ns",
    "contract_duration_ns",
    "event_clock_origin_ns",
    "left_censoring_duration_ns",
    "observation_window_duration_ns",
    "primary_event_count",
    "primary_batch_count",
    "simultaneous_batch_count",
    "mixed_side_batch_count",
    "contract_interval_convention",
    "left_censoring_preserved",
)

missing_window_columns = sorted(
    set(required_window_columns)
    - set(OBSERVATION_WINDOW_CONTRACT.columns)
)

require(
    not missing_window_columns,
    (
        "Observation-window contract is missing columns: "
        f"{missing_window_columns}"
    ),
)

OBSERVATION_WINDOW_CONTRACT = (
    OBSERVATION_WINDOW_CONTRACT.loc[
        :,
        list(required_window_columns),
    ]
    .copy()
)

OBSERVATION_WINDOW_CONTRACT["event_partition"] = (
    OBSERVATION_WINDOW_CONTRACT["event_partition"]
    .astype("string")
    .str.strip()
    .str.upper()
)

for integer_column in (
    "partition_order",
    "contract_start_ns",
    "contract_end_exclusive_ns",
    "contract_duration_ns",
    "event_clock_origin_ns",
    "left_censoring_duration_ns",
    "observation_window_duration_ns",
    "primary_event_count",
    "primary_batch_count",
    "simultaneous_batch_count",
    "mixed_side_batch_count",
):
    OBSERVATION_WINDOW_CONTRACT[integer_column] = (
        parse_exact_int64(
            OBSERVATION_WINDOW_CONTRACT[integer_column],
            label=(
                "OBSERVATION_WINDOW_CONTRACT."
                f"{integer_column}"
            ),
        )
    )

OBSERVATION_WINDOW_CONTRACT[
    "left_censoring_preserved"
] = normalize_boolean(
    OBSERVATION_WINDOW_CONTRACT[
        "left_censoring_preserved"
    ],
    label=(
        "OBSERVATION_WINDOW_CONTRACT."
        "left_censoring_preserved"
    ),
)

OBSERVATION_WINDOW_CONTRACT = deterministic_sort(
    OBSERVATION_WINDOW_CONTRACT,
    ("partition_order",),
)

require(
    len(OBSERVATION_WINDOW_CONTRACT)
    == len(PARTITION_ORDER_MAP),
    "Observation-window contract must contain four metadata rows.",
)
require(
    OBSERVATION_WINDOW_CONTRACT[
        "event_partition"
    ].is_unique,
    "Observation-window contract contains duplicate partitions.",
)
require(
    set(
        OBSERVATION_WINDOW_CONTRACT[
            "event_partition"
        ]
    )
    == set(PARTITION_ORDER_MAP),
    "Observation-window contract has an unexpected partition set.",
)
require(
    OBSERVATION_WINDOW_CONTRACT[
        "contract_interval_convention"
    ].eq("[start, end)").all(),
    "A partition violates the half-open interval contract.",
)
require(
    OBSERVATION_WINDOW_CONTRACT[
        "left_censoring_preserved"
    ].all(),
    "A partition does not preserve left censoring.",
)

contract_starts_ns = (
    OBSERVATION_WINDOW_CONTRACT[
        "contract_start_ns"
    ].to_numpy(dtype=np.int64)
)
contract_ends_ns = (
    OBSERVATION_WINDOW_CONTRACT[
        "contract_end_exclusive_ns"
    ].to_numpy(dtype=np.int64)
)
contract_durations_ns = (
    OBSERVATION_WINDOW_CONTRACT[
        "contract_duration_ns"
    ].to_numpy(dtype=np.int64)
)

require(
    np.all(contract_ends_ns > contract_starts_ns),
    "A contract window has nonpositive duration.",
)
require(
    np.array_equal(
        contract_ends_ns - contract_starts_ns,
        contract_durations_ns,
    ),
    "A registered contract duration is inconsistent.",
)
require(
    np.all(
        contract_starts_ns[1:]
        >= contract_ends_ns[:-1]
    ),
    "Registered partition windows overlap.",
)


# ------------------------------------------------------------
# Filtered DEVELOPMENT and CALIBRATION analytical loads
# ------------------------------------------------------------

PRIMARY_ESTIMATION_EVENTS_ANALYTICAL = (
    load_filtered_parquet(
        PRIMARY_EVENTS_PATH,
        columns=EVENT_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_estimation_events",
    )
)

PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL = (
    load_filtered_parquet(
        EXACT_BATCHES_PATH,
        columns=EXACT_BATCH_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_exact_time_batches",
    )
)

PRIMARY_SCORING_BATCHES_ANALYTICAL = (
    load_filtered_parquet(
        SCORING_BATCHES_PATH,
        columns=SCORING_BATCH_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_scoring_batches",
    )
)

PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL = (
    load_filtered_parquet(
        EVENT_BATCH_MEMBERSHIP_PATH,
        columns=MEMBERSHIP_COLUMNS,
        partitions=ANALYTICAL_PARTITIONS,
        label="primary_event_to_batch_membership",
    )
)


# ------------------------------------------------------------
# Exact dtype normalization
# ------------------------------------------------------------

for frame_label, frame, integer_columns in (
    (
        "PRIMARY_ESTIMATION_EVENTS_ANALYTICAL",
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
        (
            "primary_event_number",
            "partition_event_index",
            "primary_event_batch_number",
            "event_partition_order",
            "event_time_ns",
            "relative_event_time_ns",
            "event_side_code",
            "event_print_count",
        ),
    ),
    (
        "PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL",
        PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL,
        (
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_time_ns",
            "relative_batch_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "unique_side_count",
            "batch_print_count",
            "contract_start_ns",
            "contract_end_exclusive_ns",
            "event_clock_origin_ns",
            "observation_window_duration_ns",
        ),
    ),
    (
        "PRIMARY_SCORING_BATCHES_ANALYTICAL",
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
        (
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_time_ns",
            "relative_batch_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "unique_side_count",
            "batch_print_count",
            "event_clock_origin_ns",
            "observation_window_duration_ns",
        ),
    ),
    (
        "PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL",
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
        (
            "primary_event_batch_number",
            "event_partition_order",
            "event_time_ns",
            "event_side_code",
            "primary_event_number",
            "partition_event_index",
        ),
    ),
):
    for integer_column in integer_columns:
        frame[integer_column] = parse_exact_int64(
            frame[integer_column],
            label=f"{frame_label}.{integer_column}",
        )

for frame in (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
    PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
):
    frame["event_side"] = (
        frame["event_side"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

for boolean_column in (
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
):
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
        boolean_column
    ] = normalize_boolean(
        PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
            boolean_column
        ],
        label=(
            "PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL."
            f"{boolean_column}"
        ),
    )

for boolean_column in (
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "score_with_history_strictly_before_batch_time",
    "zero_lag_within_batch_excitation_allowed",
    "apply_batch_excitation_after_all_members_scored",
    "collector_sequence_is_trace_order_only",
    "timestamp_jitter_allowed",
    "event_removal_allowed",
):
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        boolean_column
    ] = normalize_boolean(
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            boolean_column
        ],
        label=(
            "PRIMARY_SCORING_BATCHES_ANALYTICAL."
            f"{boolean_column}"
        ),
    )


# ------------------------------------------------------------
# Stable chronological ordering
# ------------------------------------------------------------

PRIMARY_ESTIMATION_EVENTS_ANALYTICAL = deterministic_sort(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
    (
        "event_partition_order",
        "partition_event_index",
        "primary_event_number",
    ),
)

PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL = deterministic_sort(
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL,
    (
        "event_partition_order",
        "partition_batch_index",
        "primary_event_batch_number",
    ),
)

PRIMARY_SCORING_BATCHES_ANALYTICAL = deterministic_sort(
    PRIMARY_SCORING_BATCHES_ANALYTICAL,
    (
        "event_partition_order",
        "partition_batch_index",
        "primary_event_batch_number",
    ),
)

PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL = (
    deterministic_sort(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
        (
            "event_partition_order",
            "partition_event_index",
            "primary_event_number",
        ),
    )
)


# ------------------------------------------------------------
# Protected-partition firewall
# ------------------------------------------------------------

for frame_label, frame in (
    (
        "PRIMARY_ESTIMATION_EVENTS_ANALYTICAL",
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
    ),
    (
        "PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL",
        PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL,
    ),
    (
        "PRIMARY_SCORING_BATCHES_ANALYTICAL",
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
    ),
    (
        "PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL",
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
    ),
):
    observed_partitions = set(
        frame["event_partition"].dropna().tolist()
    )

    require(
        observed_partitions == set(ANALYTICAL_PARTITIONS),
        (
            f"{frame_label} contains an unexpected partition set: "
            f"{sorted(observed_partitions)}"
        ),
    )
    require(
        not observed_partitions.intersection(
            PROTECTED_PARTITIONS
        ),
        f"{frame_label} contains protected partition content.",
    )


# ------------------------------------------------------------
# Partition and row-count reconciliation
# ------------------------------------------------------------

partition_reconciliation_rows: list[dict[str, Any]] = []

for partition_name in ANALYTICAL_PARTITIONS:
    partition_events = (
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
            PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                "event_partition"
            ].eq(partition_name)
        ]
    )

    partition_batches = (
        PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
            PRIMARY_SCORING_BATCHES_ANALYTICAL[
                "event_partition"
            ].eq(partition_name)
        ]
    )

    expected_counts = (
        EXPECTED_ANALYTICAL_COUNTS_BY_PARTITION[
            partition_name
        ]
    )

    observed_event_count = len(partition_events)
    observed_batch_count = len(partition_batches)

    require(
        observed_event_count
        == expected_counts["event_count"],
        (
            f"{partition_name} event-count mismatch: "
            f"expected={expected_counts['event_count']:,}; "
            f"observed={observed_event_count:,}."
        ),
    )
    require(
        observed_batch_count
        == expected_counts["batch_count"],
        (
            f"{partition_name} batch-count mismatch: "
            f"expected={expected_counts['batch_count']:,}; "
            f"observed={observed_batch_count:,}."
        ),
    )

    partition_reconciliation_rows.append(
        {
            "event_partition": partition_name,
            "expected_event_count": (
                expected_counts["event_count"]
            ),
            "observed_event_count": observed_event_count,
            "expected_batch_count": (
                expected_counts["batch_count"]
            ),
            "observed_batch_count": observed_batch_count,
            "buy_event_count": int(
                partition_events["event_side"]
                .eq("BUY")
                .sum()
            ),
            "sell_event_count": int(
                partition_events["event_side"]
                .eq("SELL")
                .sum()
            ),
            "first_event_time_ns": int(
                partition_events["event_time_ns"].iloc[0]
            ),
            "last_event_time_ns": int(
                partition_events["event_time_ns"].iloc[-1]
            ),
            "simultaneous_batch_count": int(
                partition_batches[
                    "simultaneous_batch_required_flag"
                ].sum()
            ),
            "mixed_side_batch_count": int(
                partition_batches[
                    "mixed_side_batch_flag"
                ].sum()
            ),
            "status": "PASS",
        }
    )

NOTEBOOK07_ANALYTICAL_PARTITION_RECONCILIATION = (
    pd.DataFrame(partition_reconciliation_rows)
)

require(
    len(PRIMARY_ESTIMATION_EVENTS_ANALYTICAL)
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "Total analytical event count is incorrect.",
)
require(
    len(PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL)
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "Total analytical exact-time batch count is incorrect.",
)
require(
    len(PRIMARY_SCORING_BATCHES_ANALYTICAL)
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "Total analytical scoring-batch count is incorrect.",
)
require(
    len(PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL)
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "Total analytical event-to-batch membership count is incorrect.",
)


# ------------------------------------------------------------
# Event identity, side, and chronology gates
# ------------------------------------------------------------

require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "primary_event_id"
    ].notna().all(),
    "At least one analytical event lacks an event ID.",
)
require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "primary_event_id"
    ].is_unique,
    "Analytical event IDs are not unique.",
)
require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "primary_event_number"
    ].is_unique,
    "Analytical event numbers are not unique.",
)
require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_side"
    ].isin(EVENT_SIDES).all(),
    "Analytical event sides are invalid.",
)

expected_side_codes = (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_side"
    ].map(
        {
            "BUY": 0,
            "SELL": 1,
        }
    )
)

require(
    np.array_equal(
        expected_side_codes.to_numpy(dtype=np.int64),
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "event_side_code"
        ].to_numpy(dtype=np.int64),
    ),
    "Event-side codes do not match the frozen BUY=0, SELL=1 contract.",
)
require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_time_ns"
    ].is_monotonic_increasing,
    "Analytical event times are not globally nondecreasing.",
)
require(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        "event_print_count"
    ].ge(1).all(),
    "An analytical burst event has a nonpositive print count.",
)


# ------------------------------------------------------------
# Event-to-batch membership reconciliation
# ------------------------------------------------------------

require(
    PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL[
        "primary_event_id"
    ].is_unique,
    "An analytical event appears more than once in batch membership.",
)
require(
    set(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL[
            "primary_event_id"
        ]
    )
    == set(
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "primary_event_id"
        ]
    ),
    "Event and membership ID sets do not reconcile.",
)

event_membership_comparison = (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        [
            "primary_event_id",
            "primary_event_number",
            "partition_event_index",
            "primary_event_batch_id",
            "primary_event_batch_number",
            "event_partition_order",
            "event_partition",
            "event_time_ns",
            "event_side",
            "event_side_code",
        ]
    ]
    .merge(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
        on="primary_event_id",
        how="outer",
        validate="one_to_one",
        suffixes=("_event", "_membership"),
        indicator=True,
    )
)

require(
    event_membership_comparison["_merge"]
    .eq("both")
    .all(),
    "Event-to-batch membership merge is incomplete.",
)

for comparison_field in (
    "primary_event_number",
    "partition_event_index",
    "primary_event_batch_id",
    "primary_event_batch_number",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "event_side",
    "event_side_code",
):
    require(
        event_membership_comparison[
            f"{comparison_field}_event"
        ].eq(
            event_membership_comparison[
                f"{comparison_field}_membership"
            ]
        ).all(),
        (
            "Event-to-batch membership mismatch in field "
            f"{comparison_field!r}."
        ),
    )


# ------------------------------------------------------------
# Exact-time and scoring-batch reconciliation
# ------------------------------------------------------------

require(
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
        "primary_event_batch_id"
    ].is_unique,
    "Exact-time batch IDs are not unique.",
)
require(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "primary_event_batch_id"
    ].is_unique,
    "Scoring-batch IDs are not unique.",
)
require(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "event_time_ns"
    ].is_monotonic_increasing,
    "Analytical batch times are not globally nondecreasing.",
)

batch_identity_comparison = (
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
        [
            "primary_event_batch_id",
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_partition",
            "event_time_ns",
            "relative_batch_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "unique_side_count",
            "batch_print_count",
            "mixed_side_batch_flag",
            "simultaneous_batch_required_flag",
            "event_clock_origin_ns",
            "observation_window_duration_ns",
        ]
    ]
    .merge(
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            [
                "primary_event_batch_id",
                "primary_event_batch_number",
                "partition_batch_index",
                "event_partition_order",
                "event_partition",
                "event_time_ns",
                "relative_batch_time_ns",
                "batch_event_count",
                "buy_event_count",
                "sell_event_count",
                "unique_side_count",
                "batch_print_count",
                "mixed_side_batch_flag",
                "simultaneous_batch_required_flag",
                "event_clock_origin_ns",
                "observation_window_duration_ns",
            ]
        ],
        on="primary_event_batch_id",
        how="outer",
        validate="one_to_one",
        suffixes=("_exact", "_scoring"),
        indicator=True,
    )
)

require(
    batch_identity_comparison["_merge"].eq("both").all(),
    "Exact-time and scoring-batch ID sets do not reconcile.",
)

for comparison_field in (
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    "relative_batch_time_ns",
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "unique_side_count",
    "batch_print_count",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
    "event_clock_origin_ns",
    "observation_window_duration_ns",
):
    require(
        batch_identity_comparison[
            f"{comparison_field}_exact"
        ].eq(
            batch_identity_comparison[
                f"{comparison_field}_scoring"
            ]
        ).all(),
        (
            "Exact-time and scoring-batch mismatch in field "
            f"{comparison_field!r}."
        ),
    )


# ------------------------------------------------------------
# Exact-time batch conservation
# ------------------------------------------------------------

scoring_batches = PRIMARY_SCORING_BATCHES_ANALYTICAL

require(
    scoring_batches["batch_event_count"].ge(1).all(),
    "A scoring batch contains no events.",
)
require(
    scoring_batches["buy_event_count"].ge(0).all(),
    "A scoring batch has a negative BUY count.",
)
require(
    scoring_batches["sell_event_count"].ge(0).all(),
    "A scoring batch has a negative SELL count.",
)
require(
    (
        scoring_batches["buy_event_count"]
        + scoring_batches["sell_event_count"]
    ).eq(scoring_batches["batch_event_count"]).all(),
    "BUY and SELL batch counts do not conserve event count.",
)
require(
    int(scoring_batches["batch_event_count"].sum())
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "Batch event counts do not conserve analytical event rows.",
)

expected_unique_side_count = (
    scoring_batches["buy_event_count"].gt(0).astype("int8")
    + scoring_batches["sell_event_count"].gt(0).astype("int8")
)

require(
    expected_unique_side_count.eq(
        scoring_batches["unique_side_count"]
    ).all(),
    "Batch unique-side counts are incorrect.",
)
require(
    scoring_batches["mixed_side_batch_flag"].eq(
        scoring_batches["unique_side_count"].gt(1)
    ).all(),
    "Mixed-side batch flags are incorrect.",
)
require(
    scoring_batches[
        "simultaneous_batch_required_flag"
    ].eq(
        scoring_batches["batch_event_count"].gt(1)
    ).all(),
    "Simultaneous-batch flags are incorrect.",
)


# ------------------------------------------------------------
# Strict pre-batch scoring contract
# ------------------------------------------------------------

require(
    scoring_batches[
        "score_with_history_strictly_before_batch_time"
    ].all(),
    "At least one batch does not require strict pre-batch history.",
)
require(
    not scoring_batches[
        "zero_lag_within_batch_excitation_allowed"
    ].any(),
    "At least one batch permits zero-lag within-batch excitation.",
)
require(
    scoring_batches[
        "apply_batch_excitation_after_all_members_scored"
    ].all(),
    "At least one batch applies excitation before complete scoring.",
)
require(
    scoring_batches[
        "collector_sequence_is_trace_order_only"
    ].all(),
    "At least one batch gives collector sequence a physical-time role.",
)
require(
    not scoring_batches["timestamp_jitter_allowed"].any(),
    "At least one batch permits timestamp jitter.",
)
require(
    not scoring_batches["event_removal_allowed"].any(),
    "At least one batch permits event removal.",
)


# ------------------------------------------------------------
# Batch membership counts and timestamps
# ------------------------------------------------------------

membership_batch_summary = (
    PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL.groupby(
        [
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
        ],
        observed=True,
        sort=False,
        dropna=False,
    )
    .agg(
        membership_event_count=("primary_event_id", "size"),
        membership_buy_event_count=(
            "event_side",
            lambda values: int(values.eq("BUY").sum()),
        ),
        membership_sell_event_count=(
            "event_side",
            lambda values: int(values.eq("SELL").sum()),
        ),
    )
    .reset_index()
)

membership_batch_reconciliation = (
    scoring_batches[
        [
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
        ]
    ]
    .merge(
        membership_batch_summary,
        on=[
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
        ],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

require(
    membership_batch_reconciliation["_merge"]
    .eq("both")
    .all(),
    "Batch membership does not reconcile with scoring batches.",
)
require(
    membership_batch_reconciliation[
        "batch_event_count"
    ].eq(
        membership_batch_reconciliation[
            "membership_event_count"
        ]
    ).all(),
    "Membership event counts differ from scoring-batch counts.",
)
require(
    membership_batch_reconciliation[
        "buy_event_count"
    ].eq(
        membership_batch_reconciliation[
            "membership_buy_event_count"
        ]
    ).all(),
    "Membership BUY counts differ from scoring-batch counts.",
)
require(
    membership_batch_reconciliation[
        "sell_event_count"
    ].eq(
        membership_batch_reconciliation[
            "membership_sell_event_count"
        ]
    ).all(),
    "Membership SELL counts differ from scoring-batch counts.",
)


# ------------------------------------------------------------
# Observation-window containment and boundary continuity
# ------------------------------------------------------------

window_lookup = (
    OBSERVATION_WINDOW_CONTRACT.set_index(
        "event_partition"
    )
)

for partition_name in ANALYTICAL_PARTITIONS:
    partition_events = (
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
            PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                "event_partition"
            ].eq(partition_name)
        ]
    )

    contract_start_ns = int(
        window_lookup.loc[
            partition_name,
            "contract_start_ns",
        ]
    )
    contract_end_exclusive_ns = int(
        window_lookup.loc[
            partition_name,
            "contract_end_exclusive_ns",
        ]
    )

    require(
        partition_events["event_time_ns"]
        .ge(contract_start_ns)
        .all(),
        f"{partition_name} contains events before contract start.",
    )
    require(
        partition_events["event_time_ns"]
        .lt(contract_end_exclusive_ns)
        .all(),
        f"{partition_name} contains events at or after contract end.",
    )

development_end_exclusive_ns = int(
    window_lookup.loc[
        FIT_PARTITION,
        "contract_end_exclusive_ns",
    ]
)
calibration_start_ns = int(
    window_lookup.loc[
        LOCKED_EVALUATION_PARTITION,
        "contract_start_ns",
    ]
)

require(
    calibration_start_ns
    == development_end_exclusive_ns,
    (
        "DEVELOPMENT and CALIBRATION contract windows are not "
        "exactly contiguous."
    ),
)

development_last_event_time_ns = int(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "event_partition"
        ].eq(FIT_PARTITION),
        "event_time_ns",
    ].iloc[-1]
)

calibration_first_event_time_ns = int(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.loc[
        PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
            "event_partition"
        ].eq(LOCKED_EVALUATION_PARTITION),
        "event_time_ns",
    ].iloc[0]
)

require(
    calibration_first_event_time_ns
    > development_last_event_time_ns,
    (
        "The first CALIBRATION event does not occur strictly "
        "after the last DEVELOPMENT event."
    ),
)

DEVELOPMENT_TO_CALIBRATION_EVENT_GAP_NS: Final[int] = (
    calibration_first_event_time_ns
    - development_last_event_time_ns
)


# ------------------------------------------------------------
# Update notebook access-state ledger
# ------------------------------------------------------------

PROTECTED_PARTITION_CONTENT_LOADED["DEVELOPMENT"] = True
PROTECTED_PARTITION_CONTENT_LOADED["CALIBRATION"] = True
PROTECTED_PARTITION_CONTENT_LOADED["VALIDATION"] = False
PROTECTED_PARTITION_CONTENT_LOADED[
    "ENGINEERING_HOLDOUT"
] = False

ANALYTICAL_CONTENT_LOADED: bool = True
PROTECTED_PARTITION_METADATA_LOADED: bool = True
FUTURE_LABEL_TABLE_LOADED = False
MARKET_STATE_FEATURE_VALUES_LOADED = False
FILESYSTEM_WRITES_PERFORMED = False

require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was materialized.",
)


# ------------------------------------------------------------
# Load and reconciliation summaries
# ------------------------------------------------------------

exact_time_batch_summary = pd.DataFrame(
    [
        {
            "metric": "analytical_event_rows",
            "value": len(
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL
            ),
        },
        {
            "metric": "analytical_batch_rows",
            "value": len(
                PRIMARY_SCORING_BATCHES_ANALYTICAL
            ),
        },
        {
            "metric": "buy_event_rows",
            "value": int(
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                    "event_side"
                ].eq("BUY").sum()
            ),
        },
        {
            "metric": "sell_event_rows",
            "value": int(
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
                    "event_side"
                ].eq("SELL").sum()
            ),
        },
        {
            "metric": "simultaneous_batches",
            "value": int(
                scoring_batches[
                    "simultaneous_batch_required_flag"
                ].sum()
            ),
        },
        {
            "metric": "mixed_side_batches",
            "value": int(
                scoring_batches[
                    "mixed_side_batch_flag"
                ].sum()
            ),
        },
        {
            "metric": "maximum_events_in_one_batch",
            "value": int(
                scoring_batches[
                    "batch_event_count"
                ].max()
            ),
        },
        {
            "metric": "development_to_calibration_event_gap_ns",
            "value": (
                DEVELOPMENT_TO_CALIBRATION_EVENT_GAP_NS
            ),
        },
        {
            "metric": "validation_content_loaded",
            "value": False,
        },
        {
            "metric": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "metric": "future_label_table_loaded",
            "value": False,
        },
        {
            "metric": "market_state_feature_values_loaded",
            "value": False,
        },
        {
            "metric": "filesystem_writes_performed",
            "value": False,
        },
    ]
)

display(
    NOTEBOOK07_ANALYTICAL_PARTITION_RECONCILIATION
)
display(exact_time_batch_summary)

print(
    "DEVELOPMENT and CALIBRATION analytical event content loaded "
    "through partition-filtered Arrow scans. Event identities, "
    "batch memberships, counts, timestamps, contract windows, and "
    "strict pre-batch scoring semantics reconciled. VALIDATION and "
    "ENGINEERING_HOLDOUT event content remain unopened. Future "
    "labels and market-state feature values remain unopened. No "
    "filesystem writes were performed."
)

,event_partition,expected_event_count,observed_event_count,expected_batch_count,observed_batch_count,buy_event_count,sell_event_count,first_event_time_ns,last_event_time_ns,simultaneous_batch_count,mixed_side_batch_count,status
0,DEVELOPMENT,7004,7004,6859,6859,3414,3590,1783665468766951600,1783667269232205000,111,26,PASS
1,CALIBRATION,2493,2493,2400,2400,1230,1263,1783667270546982000,1783667989349685300,68,5,PASS


,metric,value
0,analytical_event_rows,9497
1,analytical_batch_rows,9259
2,buy_event_rows,4644
3,sell_event_rows,4853
4,simultaneous_batches,179
5,mixed_side_batches,31
6,maximum_events_in_one_batch,6
7,development_to_calibration_event_gap_ns,1314777000
8,validation_content_loaded,False
9,engineering_holdout_content_loaded,False


DEVELOPMENT and CALIBRATION analytical event content loaded through partition-filtered Arrow scans. Event identities, batch memberships, counts, timestamps, contract windows, and strict pre-batch scoring semantics reconciled. VALIDATION and ENGINEERING_HOLDOUT event content remain unopened. Future labels and market-state feature values remain unopened. No filesystem writes were performed.


In [4]:
# ============================================================
# Freeze the machine-readable Hawkes model contract
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("ANALYTICAL_CONTENT_LOADED", False)),
    "Analytical DEVELOPMENT and CALIBRATION content is not loaded.",
)
require(
    bool(globals().get("UPSTREAM_AUTHORIZATION_VERIFIED", False)),
    "Notebook 06 restricted-first authorization is not verified.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Numerical parameter-transform contract
# ------------------------------------------------------------

KAPPA_HARD_UPPER_BOUND: Final[float] = 1.0 - 1e-10
TRANSFORM_CLIP_LOWER: Final[float] = -700.0
TRANSFORM_CLIP_UPPER: Final[float] = 700.0


def softplus(
    values: np.ndarray | float,
) -> np.ndarray:
    """Numerically stable softplus transform."""
    array = np.asarray(values, dtype=np.float64)

    return np.logaddexp(0.0, array)


def inverse_softplus(
    positive_values: np.ndarray | float,
) -> np.ndarray:
    """Inverse softplus for strictly positive finite values."""
    array = np.asarray(
        positive_values,
        dtype=np.float64,
    )

    require(
        np.isfinite(array).all(),
        "inverse_softplus received non-finite values.",
    )
    require(
        np.greater(array, 0.0).all(),
        "inverse_softplus requires strictly positive values.",
    )

    return np.where(
        array > 20.0,
        array + np.log1p(-np.exp(-array)),
        np.log(np.expm1(array)),
    )


def bounded_logit(
    probabilities: np.ndarray | float,
) -> np.ndarray:
    """Map probabilities strictly inside (0, 1) to the real line."""
    array = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    require(
        np.isfinite(array).all(),
        "bounded_logit received non-finite values.",
    )
    require(
        np.logical_and(
            array > 0.0,
            array < 1.0,
        ).all(),
        "bounded_logit requires values strictly inside (0, 1).",
    )

    return np.log(array) - np.log1p(-array)


def positive_from_unconstrained(
    value: float,
    *,
    floor: float = PARAMETER_POSITIVITY_FLOOR,
) -> float:
    """Map one unconstrained scalar to a finite value above floor."""
    require(
        math.isfinite(value),
        "Unconstrained positive parameter is non-finite.",
    )
    require(
        floor > 0.0,
        "Positive parameter floor must be strictly positive.",
    )

    transformed = float(
        softplus(
            np.clip(
                value,
                TRANSFORM_CLIP_LOWER,
                TRANSFORM_CLIP_UPPER,
            )
        )
    )

    result = floor + transformed

    require(
        math.isfinite(result) and result > floor,
        "Positive parameter transform failed.",
    )

    return result


def unconstrained_from_positive(
    value: float,
    *,
    floor: float = PARAMETER_POSITIVITY_FLOOR,
) -> float:
    """Map one finite value above floor to the real line."""
    require(
        math.isfinite(value),
        "Natural positive parameter is non-finite.",
    )
    require(
        value > floor,
        (
            "Natural positive parameter must exceed its floor: "
            f"value={value}; floor={floor}."
        ),
    )

    return float(
        inverse_softplus(value - floor)
    )


def kappa_from_unconstrained(
    value: float,
) -> float:
    """
    Map one unconstrained scalar to the admissible interval
    [0, KAPPA_HARD_UPPER_BOUND).
    """
    require(
        math.isfinite(value),
        "Unconstrained excitation-mass parameter is non-finite.",
    )

    probability = float(
        special.expit(
            np.clip(
                value,
                TRANSFORM_CLIP_LOWER,
                TRANSFORM_CLIP_UPPER,
            )
        )
    )

    kappa = KAPPA_HARD_UPPER_BOUND * probability

    require(
        math.isfinite(kappa),
        "Excitation-mass transform produced a non-finite value.",
    )
    require(
        0.0 <= kappa < STATIONARITY_HARD_LIMIT,
        "Excitation-mass transform violated stationarity.",
    )

    return kappa


def unconstrained_from_kappa(
    kappa: float,
) -> float:
    """Map one admissible excitation mass to the real line."""
    require(
        math.isfinite(kappa),
        "Natural excitation mass is non-finite.",
    )
    require(
        0.0 <= kappa < KAPPA_HARD_UPPER_BOUND,
        (
            "Natural excitation mass is outside the encodable "
            f"interval: {kappa}."
        ),
    )

    scaled_probability = (
        max(kappa, np.finfo(np.float64).eps)
        / KAPPA_HARD_UPPER_BOUND
    )

    scaled_probability = float(
        np.clip(
            scaled_probability,
            np.finfo(np.float64).eps,
            1.0 - np.finfo(np.float64).eps,
        )
    )

    return float(
        bounded_logit(scaled_probability)
    )


# ------------------------------------------------------------
# Frozen model and parameter objects
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class HawkesCandidateSpecification:
    model_id: str
    description: str
    shared_decay: bool
    estimated_parameter_count: int
    optimizer_parameter_names: tuple[str, ...]
    natural_parameter_names: tuple[str, ...]
    authorized_channels: tuple[str, ...]
    fixed_zero_channels: tuple[str, ...]
    kernel_family: str
    selection_role: str
    complexity_rank: int


@dataclass(frozen=True, slots=True)
class DiagonalHawkesParameters:
    model_id: str
    mu_buy: float
    mu_sell: float
    kappa_buy: float
    kappa_sell: float
    beta_buy: float
    beta_sell: float

    def as_mapping(self) -> dict[str, float | str]:
        """Return a portable natural-parameter mapping."""
        return {
            "model_id": self.model_id,
            "mu_buy_per_second": self.mu_buy,
            "mu_sell_per_second": self.mu_sell,
            "kappa_buy": self.kappa_buy,
            "kappa_sell": self.kappa_sell,
            "beta_buy_per_second": self.beta_buy,
            "beta_sell_per_second": self.beta_sell,
        }


# ------------------------------------------------------------
# Candidate registry
# ------------------------------------------------------------

HAWKES_CANDIDATE_SPECIFICATIONS: Final[
    Mapping[str, HawkesCandidateSpecification]
] = {
    H1_MODEL_ID: HawkesCandidateSpecification(
        model_id=H1_MODEL_ID,
        description=(
            "Diagonal BUY/SELL exponential Hawkes model with "
            "side-specific base intensities and excitation masses "
            "but one shared decay rate."
        ),
        shared_decay=True,
        estimated_parameter_count=5,
        optimizer_parameter_names=(
            "z_mu_buy",
            "z_mu_sell",
            "z_kappa_buy",
            "z_kappa_sell",
            "z_beta_shared",
        ),
        natural_parameter_names=(
            "mu_buy",
            "mu_sell",
            "kappa_buy",
            "kappa_sell",
            "beta_shared",
        ),
        authorized_channels=AUTHORIZED_CHANNELS,
        fixed_zero_channels=FIXED_ZERO_CHANNELS,
        kernel_family=AUTHORIZED_KERNEL_FAMILY,
        selection_role="SIMPLICITY_FIRST_REFERENCE_CANDIDATE",
        complexity_rank=1,
    ),
    H2_MODEL_ID: HawkesCandidateSpecification(
        model_id=H2_MODEL_ID,
        description=(
            "Diagonal BUY/SELL exponential Hawkes model with "
            "side-specific base intensities, excitation masses, "
            "and decay rates."
        ),
        shared_decay=False,
        estimated_parameter_count=6,
        optimizer_parameter_names=(
            "z_mu_buy",
            "z_mu_sell",
            "z_kappa_buy",
            "z_kappa_sell",
            "z_beta_buy",
            "z_beta_sell",
        ),
        natural_parameter_names=(
            "mu_buy",
            "mu_sell",
            "kappa_buy",
            "kappa_sell",
            "beta_buy",
            "beta_sell",
        ),
        authorized_channels=AUTHORIZED_CHANNELS,
        fixed_zero_channels=FIXED_ZERO_CHANNELS,
        kernel_family=AUTHORIZED_KERNEL_FAMILY,
        selection_role="EXPANDED_DECAY_CANDIDATE",
        complexity_rank=2,
    ),
}

require(
    set(HAWKES_CANDIDATE_SPECIFICATIONS)
    == set(CANDIDATE_MODEL_IDS),
    "Candidate registry does not match the frozen candidate IDs.",
)
require(
    HAWKES_CANDIDATE_SPECIFICATIONS[
        H1_MODEL_ID
    ].estimated_parameter_count == 5,
    "H1 must contain exactly five estimated parameters.",
)
require(
    HAWKES_CANDIDATE_SPECIFICATIONS[
        H2_MODEL_ID
    ].estimated_parameter_count == 6,
    "H2 must contain exactly six estimated parameters.",
)


# ------------------------------------------------------------
# Parameter encoding and decoding
# ------------------------------------------------------------

def decode_hawkes_parameters(
    model_id: str,
    optimizer_vector: np.ndarray,
) -> DiagonalHawkesParameters:
    """Decode an unconstrained optimizer vector into natural units."""
    require(
        model_id in HAWKES_CANDIDATE_SPECIFICATIONS,
        f"Unknown Hawkes candidate: {model_id}",
    )

    specification = HAWKES_CANDIDATE_SPECIFICATIONS[
        model_id
    ]

    vector = np.asarray(
        optimizer_vector,
        dtype=np.float64,
    )

    require(
        vector.ndim == 1,
        "Optimizer parameter vector must be one-dimensional.",
    )
    require(
        len(vector)
        == specification.estimated_parameter_count,
        (
            f"{model_id} requires "
            f"{specification.estimated_parameter_count} optimizer "
            f"parameters; observed {len(vector)}."
        ),
    )
    require(
        np.isfinite(vector).all(),
        "Optimizer parameter vector contains non-finite values.",
    )

    mu_buy = positive_from_unconstrained(vector[0])
    mu_sell = positive_from_unconstrained(vector[1])
    kappa_buy = kappa_from_unconstrained(vector[2])
    kappa_sell = kappa_from_unconstrained(vector[3])

    if specification.shared_decay:
        beta_shared = positive_from_unconstrained(vector[4])
        beta_buy = beta_shared
        beta_sell = beta_shared
    else:
        beta_buy = positive_from_unconstrained(vector[4])
        beta_sell = positive_from_unconstrained(vector[5])

    parameters = DiagonalHawkesParameters(
        model_id=model_id,
        mu_buy=mu_buy,
        mu_sell=mu_sell,
        kappa_buy=kappa_buy,
        kappa_sell=kappa_sell,
        beta_buy=beta_buy,
        beta_sell=beta_sell,
    )

    validation = validate_hawkes_parameters(parameters)

    require(
        validation["mathematically_admissible"],
        (
            "Decoded Hawkes parameters are not mathematically "
            f"admissible: {validation}"
        ),
    )

    return parameters


def encode_hawkes_parameters(
    parameters: DiagonalHawkesParameters,
) -> np.ndarray:
    """Encode natural Hawkes parameters for unconstrained optimization."""
    require(
        parameters.model_id
        in HAWKES_CANDIDATE_SPECIFICATIONS,
        f"Unknown Hawkes candidate: {parameters.model_id}",
    )

    validation = validate_hawkes_parameters(parameters)

    require(
        validation["mathematically_admissible"],
        (
            "Cannot encode inadmissible Hawkes parameters: "
            f"{validation}"
        ),
    )

    specification = HAWKES_CANDIDATE_SPECIFICATIONS[
        parameters.model_id
    ]

    encoded_values = [
        unconstrained_from_positive(parameters.mu_buy),
        unconstrained_from_positive(parameters.mu_sell),
        unconstrained_from_kappa(parameters.kappa_buy),
        unconstrained_from_kappa(parameters.kappa_sell),
    ]

    if specification.shared_decay:
        require(
            math.isclose(
                parameters.beta_buy,
                parameters.beta_sell,
                rel_tol=FLOAT_RELATIVE_TOLERANCE,
                abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
            ),
            "H1 requires identical BUY and SELL decay rates.",
        )

        encoded_values.append(
            unconstrained_from_positive(
                parameters.beta_buy
            )
        )
    else:
        encoded_values.extend(
            [
                unconstrained_from_positive(
                    parameters.beta_buy
                ),
                unconstrained_from_positive(
                    parameters.beta_sell
                ),
            ]
        )

    encoded = np.asarray(
        encoded_values,
        dtype=np.float64,
    )

    require(
        len(encoded)
        == specification.estimated_parameter_count,
        "Encoded parameter vector has the wrong length.",
    )
    require(
        np.isfinite(encoded).all(),
        "Encoded parameter vector contains non-finite values.",
    )

    return encoded


# ------------------------------------------------------------
# Derived Hawkes quantities
# ------------------------------------------------------------

def integrated_kernel_matrix(
    parameters: DiagonalHawkesParameters,
) -> np.ndarray:
    """Return the integrated 2x2 excitation matrix."""
    return np.asarray(
        [
            [parameters.kappa_buy, 0.0],
            [0.0, parameters.kappa_sell],
        ],
        dtype=np.float64,
    )


def spectral_radius(
    parameters: DiagonalHawkesParameters,
) -> float:
    """Return the spectral radius of the integrated kernel matrix."""
    kernel_matrix = integrated_kernel_matrix(parameters)

    eigenvalues = np.linalg.eigvals(kernel_matrix)

    return float(
        np.max(np.abs(eigenvalues))
    )


def excitation_half_lives_seconds(
    parameters: DiagonalHawkesParameters,
) -> tuple[float, float]:
    """Return BUY and SELL excitation half-lives in seconds."""
    return (
        math.log(2.0) / parameters.beta_buy,
        math.log(2.0) / parameters.beta_sell,
    )


def model_implied_mean_intensities(
    parameters: DiagonalHawkesParameters,
) -> tuple[float, float]:
    """
    Return stationary mean BUY and SELL intensities.

    For the diagonal integrated-mass parameterization:

    mean intensity = base intensity / (1 - excitation mass).
    """
    require(
        parameters.kappa_buy < 1.0,
        "BUY excitation mass must be below one.",
    )
    require(
        parameters.kappa_sell < 1.0,
        "SELL excitation mass must be below one.",
    )

    return (
        parameters.mu_buy
        / (1.0 - parameters.kappa_buy),
        parameters.mu_sell
        / (1.0 - parameters.kappa_sell),
    )


def model_implied_exogenous_shares(
    parameters: DiagonalHawkesParameters,
) -> tuple[float, float]:
    """
    Return model-implied exogenous shares.

    These are descriptive quantities under the fitted model and
    must not be interpreted as proven causal trading fractions.
    """
    return (
        1.0 - parameters.kappa_buy,
        1.0 - parameters.kappa_sell,
    )


def validate_hawkes_parameters(
    parameters: DiagonalHawkesParameters,
) -> dict[str, Any]:
    """Return mathematical and notebook-level parameter diagnostics."""
    natural_values = np.asarray(
        [
            parameters.mu_buy,
            parameters.mu_sell,
            parameters.kappa_buy,
            parameters.kappa_sell,
            parameters.beta_buy,
            parameters.beta_sell,
        ],
        dtype=np.float64,
    )

    all_finite = bool(
        np.isfinite(natural_values).all()
    )
    positive_bases = bool(
        parameters.mu_buy > 0.0
        and parameters.mu_sell > 0.0
    )
    nonnegative_masses = bool(
        parameters.kappa_buy >= 0.0
        and parameters.kappa_sell >= 0.0
    )
    positive_decays = bool(
        parameters.beta_buy > 0.0
        and parameters.beta_sell > 0.0
    )

    rho = (
        spectral_radius(parameters)
        if all_finite
        else math.inf
    )

    stationarity_holds = bool(
        math.isfinite(rho)
        and rho < STATIONARITY_HARD_LIMIT
    )
    acceptance_margin_holds = bool(
        math.isfinite(rho)
        and rho <= STATIONARITY_ACCEPTANCE_MARGIN
    )

    mathematically_admissible = bool(
        all_finite
        and positive_bases
        and nonnegative_masses
        and positive_decays
        and stationarity_holds
    )

    if not mathematically_admissible:
        stationarity_class = "FAIL_INADMISSIBLE"
    elif acceptance_margin_holds:
        stationarity_class = "PASS_ACCEPTANCE_MARGIN"
    else:
        stationarity_class = "CONDITIONAL_NEAR_CRITICAL"

    return {
        "model_id": parameters.model_id,
        "all_finite": all_finite,
        "positive_base_intensities": positive_bases,
        "nonnegative_excitation_masses": nonnegative_masses,
        "positive_decay_rates": positive_decays,
        "spectral_radius": rho,
        "stationarity_holds": stationarity_holds,
        "acceptance_margin_holds": acceptance_margin_holds,
        "stationarity_class": stationarity_class,
        "mathematically_admissible": mathematically_admissible,
    }


def parameter_diagnostic_record(
    parameters: DiagonalHawkesParameters,
) -> dict[str, Any]:
    """Return a portable parameter and derived-quantity record."""
    validation = validate_hawkes_parameters(parameters)

    (
        half_life_buy_seconds,
        half_life_sell_seconds,
    ) = excitation_half_lives_seconds(parameters)

    (
        mean_buy_intensity,
        mean_sell_intensity,
    ) = model_implied_mean_intensities(parameters)

    (
        exogenous_buy_share,
        exogenous_sell_share,
    ) = model_implied_exogenous_shares(parameters)

    return {
        **parameters.as_mapping(),
        "half_life_buy_seconds": half_life_buy_seconds,
        "half_life_sell_seconds": half_life_sell_seconds,
        "mean_buy_intensity_per_second": mean_buy_intensity,
        "mean_sell_intensity_per_second": mean_sell_intensity,
        "model_implied_exogenous_buy_share": (
            exogenous_buy_share
        ),
        "model_implied_exogenous_sell_share": (
            exogenous_sell_share
        ),
        **validation,
    }


# ------------------------------------------------------------
# Machine-readable likelihood and batch-state contract
# ------------------------------------------------------------

HAWKES_MODEL_CONTRACT: Final[dict[str, Any]] = {
    "schema_version": "NOTEBOOK_07_HAWKES_MODEL_CONTRACT_V1",
    "notebook": NOTEBOOK_NAME,
    "estimator_label": PRIMARY_ESTIMATOR_LABEL,
    "model_family": AUTHORIZED_FIRST_MODEL,
    "event_components": list(EVENT_SIDES),
    "event_representation": PRIMARY_EVENT_REPRESENTATION,
    "event_time_column": PRIMARY_EVENT_TIME_COLUMN,
    "event_time_storage_unit": EVENT_TIME_STORAGE_UNIT,
    "model_time_unit": MODEL_TIME_UNIT,
    "partition_column": PRIMARY_EVENT_PARTITION_COLUMN,
    "fit_partition": FIT_PARTITION,
    "locked_evaluation_partition": (
        LOCKED_EVALUATION_PARTITION
    ),
    "protected_partitions": list(PROTECTED_PARTITIONS),
    "authorized_channels": list(AUTHORIZED_CHANNELS),
    "fixed_zero_channels": list(FIXED_ZERO_CHANNELS),
    "kernel_parameterization": (
        "INTEGRATED_KERNEL_MASS_TIMES_DECAY_RATE"
    ),
    "kernel_family": AUTHORIZED_KERNEL_FAMILY,
    "buy_kernel": (
        "kappa_buy * beta_buy * "
        "exp(-beta_buy * lag) * I(lag > 0)"
    ),
    "sell_kernel": (
        "kappa_sell * beta_sell * "
        "exp(-beta_sell * lag) * I(lag > 0)"
    ),
    "cross_kernels": {
        "buy_to_sell": 0.0,
        "sell_to_buy": 0.0,
    },
    "parameter_constraints": {
        "mu_buy": "STRICTLY_POSITIVE",
        "mu_sell": "STRICTLY_POSITIVE",
        "kappa_buy": "NONNEGATIVE_AND_BELOW_ONE",
        "kappa_sell": "NONNEGATIVE_AND_BELOW_ONE",
        "beta_buy": "STRICTLY_POSITIVE",
        "beta_sell": "STRICTLY_POSITIVE",
    },
    "stationarity_condition": (
        "SPECTRAL_RADIUS_STRICTLY_LESS_THAN_ONE"
    ),
    "stationarity_acceptance_margin": (
        STATIONARITY_ACCEPTANCE_MARGIN
    ),
    "exact_time_batch_contract": {
        "score_against_history": (
            "STRICTLY_BEFORE_BATCH_TIME"
        ),
        "same_pre_batch_intensity_for_all_batch_members": True,
        "within_batch_zero_lag_excitation": False,
        "update_excitation_state": (
            "AFTER_ALL_BATCH_MEMBERS_ARE_SCORED"
        ),
        "timestamp_jitter": False,
        "artificial_event_ordering": False,
        "event_removal": False,
        "collector_sequence_role": "TRACE_ORDER_ONLY",
    },
    "left_edge_contract": {
        "fabricated_prehistory": False,
        "initial_excitation_state": "ZERO",
        "left_censoring_recorded": True,
    },
    "development_to_calibration_contract": {
        "carry_terminal_development_state": True,
        "decay_state_across_exact_boundary_gap": True,
        "reset_to_base_intensity": False,
        "parameter_updates": 0,
    },
    "candidate_model_ids": list(CANDIDATE_MODEL_IDS),
    "cross_excitation_status": CROSS_EXCITATION_STATUS,
    "state_dependent_hawkes_status": (
        STATE_DEPENDENT_HAWKES_STATUS
    ),
    "future_labels_used": False,
    "market_state_features_used": False,
}

HAWKES_MODEL_CONTRACT_SHA256: Final[str] = (
    canonical_json_sha256(HAWKES_MODEL_CONTRACT)
)


# ------------------------------------------------------------
# Candidate-registry tabulation and hash
# ------------------------------------------------------------

candidate_registry_records = []

for specification in sorted(
    HAWKES_CANDIDATE_SPECIFICATIONS.values(),
    key=lambda item: item.complexity_rank,
):
    candidate_registry_records.append(
        {
            "model_id": specification.model_id,
            "description": specification.description,
            "shared_decay": specification.shared_decay,
            "estimated_parameter_count": (
                specification.estimated_parameter_count
            ),
            "optimizer_parameter_names": list(
                specification.optimizer_parameter_names
            ),
            "natural_parameter_names": list(
                specification.natural_parameter_names
            ),
            "authorized_channels": list(
                specification.authorized_channels
            ),
            "fixed_zero_channels": list(
                specification.fixed_zero_channels
            ),
            "kernel_family": specification.kernel_family,
            "selection_role": specification.selection_role,
            "complexity_rank": specification.complexity_rank,
            "authorization_status": "AUTHORIZED",
        }
    )

HAWKES_CANDIDATE_REGISTRY: Final[
    tuple[dict[str, Any], ...]
] = tuple(candidate_registry_records)

HAWKES_CANDIDATE_REGISTRY_SHA256: Final[str] = (
    canonical_json_sha256(
        {
            "schema_version": (
                "NOTEBOOK_07_HAWKES_CANDIDATE_REGISTRY_V1"
            ),
            "candidates": list(
                HAWKES_CANDIDATE_REGISTRY
            ),
        }
    )
)

HAWKES_CANDIDATE_REGISTRY_TABLE = pd.DataFrame(
    HAWKES_CANDIDATE_REGISTRY
)


# ------------------------------------------------------------
# Transform round-trip tests
# ------------------------------------------------------------

ROUND_TRIP_TEST_PARAMETERS: Final[
    Mapping[str, DiagonalHawkesParameters]
] = {
    H1_MODEL_ID: DiagonalHawkesParameters(
        model_id=H1_MODEL_ID,
        mu_buy=10.0,
        mu_sell=11.0,
        kappa_buy=0.25,
        kappa_sell=0.30,
        beta_buy=4.0,
        beta_sell=4.0,
    ),
    H2_MODEL_ID: DiagonalHawkesParameters(
        model_id=H2_MODEL_ID,
        mu_buy=10.0,
        mu_sell=11.0,
        kappa_buy=0.25,
        kappa_sell=0.30,
        beta_buy=4.0,
        beta_sell=6.0,
    ),
}

transform_test_rows: list[dict[str, Any]] = []

for model_id, original_parameters in (
    ROUND_TRIP_TEST_PARAMETERS.items()
):
    encoded_parameters = encode_hawkes_parameters(
        original_parameters
    )
    decoded_parameters = decode_hawkes_parameters(
        model_id,
        encoded_parameters,
    )

    original_vector = np.asarray(
        [
            original_parameters.mu_buy,
            original_parameters.mu_sell,
            original_parameters.kappa_buy,
            original_parameters.kappa_sell,
            original_parameters.beta_buy,
            original_parameters.beta_sell,
        ],
        dtype=np.float64,
    )

    decoded_vector = np.asarray(
        [
            decoded_parameters.mu_buy,
            decoded_parameters.mu_sell,
            decoded_parameters.kappa_buy,
            decoded_parameters.kappa_sell,
            decoded_parameters.beta_buy,
            decoded_parameters.beta_sell,
        ],
        dtype=np.float64,
    )

    maximum_absolute_error = float(
        np.max(
            np.abs(
                original_vector - decoded_vector
            )
        )
    )

    round_trip_passed = bool(
        np.allclose(
            original_vector,
            decoded_vector,
            rtol=FLOAT_RELATIVE_TOLERANCE,
            atol=FLOAT_ABSOLUTE_TOLERANCE,
        )
    )

    require(
        round_trip_passed,
        (
            f"{model_id} parameter-transform round trip failed; "
            f"maximum absolute error={maximum_absolute_error}."
        ),
    )

    transform_test_rows.append(
        {
            "model_id": model_id,
            "optimizer_dimension": len(
                encoded_parameters
            ),
            "maximum_absolute_error": (
                maximum_absolute_error
            ),
            "round_trip_passed": round_trip_passed,
            "status": "PASS",
        }
    )

HAWKES_PARAMETER_TRANSFORM_TESTS = pd.DataFrame(
    transform_test_rows
)


# ------------------------------------------------------------
# Contract freeze state
# ------------------------------------------------------------

HAWKES_MODEL_CONTRACT_FROZEN: bool = True
HAWKES_CANDIDATE_REGISTRY_FROZEN: bool = True
HAWKES_PARAMETER_TRANSFORMS_VALIDATED: bool = True

FILESYSTEM_WRITES_PERFORMED = False

model_contract_summary = pd.DataFrame(
    [
        {
            "field": "model_contract_frozen",
            "value": HAWKES_MODEL_CONTRACT_FROZEN,
        },
        {
            "field": "candidate_registry_frozen",
            "value": HAWKES_CANDIDATE_REGISTRY_FROZEN,
        },
        {
            "field": "parameter_transforms_validated",
            "value": (
                HAWKES_PARAMETER_TRANSFORMS_VALIDATED
            ),
        },
        {
            "field": "model_family",
            "value": AUTHORIZED_FIRST_MODEL,
        },
        {
            "field": "kernel_parameterization",
            "value": (
                "INTEGRATED_KERNEL_MASS_TIMES_DECAY_RATE"
            ),
        },
        {
            "field": "candidate_count",
            "value": len(
                HAWKES_CANDIDATE_SPECIFICATIONS
            ),
        },
        {
            "field": "cross_excitation_status",
            "value": CROSS_EXCITATION_STATUS,
        },
        {
            "field": "state_dependent_hawkes_status",
            "value": STATE_DEPENDENT_HAWKES_STATUS,
        },
        {
            "field": "stationarity_hard_limit",
            "value": STATIONARITY_HARD_LIMIT,
        },
        {
            "field": "stationarity_acceptance_margin",
            "value": STATIONARITY_ACCEPTANCE_MARGIN,
        },
        {
            "field": "model_contract_sha256",
            "value": HAWKES_MODEL_CONTRACT_SHA256,
        },
        {
            "field": "candidate_registry_sha256",
            "value": HAWKES_CANDIDATE_REGISTRY_SHA256,
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(model_contract_summary)
display(
    HAWKES_CANDIDATE_REGISTRY_TABLE[
        [
            "model_id",
            "shared_decay",
            "estimated_parameter_count",
            "selection_role",
            "authorized_channels",
            "fixed_zero_channels",
            "authorization_status",
        ]
    ]
)
display(HAWKES_PARAMETER_TRANSFORM_TESTS)

print(
    "The diagonal exponential Hawkes model contract, candidate "
    "registry, natural parameterization, admissibility rules, "
    "stationarity rules, and optimizer transforms are frozen and "
    "validated. No estimation has been performed. Protected "
    "partitions remain unopened and no filesystem writes were "
    "performed."
)

,field,value
0,model_contract_frozen,True
1,candidate_registry_frozen,True
2,parameter_transforms_validated,True
3,model_family,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES
4,kernel_parameterization,INTEGRATED_KERNEL_MASS_TIMES_DECAY_RATE
5,candidate_count,2
6,cross_excitation_status,SKIPPED_NOT_AUTHORIZED
7,state_dependent_hawkes_status,NOT_AUTHORIZED
8,stationarity_hard_limit,1
9,stationarity_acceptance_margin,0.98


,model_id,shared_decay,estimated_parameter_count,selection_role,authorized_channels,fixed_zero_channels,authorization_status
0,H1_DIAGONAL_SHARED_DECAY,True,5,SIMPLICITY_FIRST_REFERENCE_CANDIDATE,"[BUY_TO_BUY, SELL_TO_SELL]","[BUY_TO_SELL, SELL_TO_BUY]",AUTHORIZED
1,H2_DIAGONAL_SEPARATE_DECAY,False,6,EXPANDED_DECAY_CANDIDATE,"[BUY_TO_BUY, SELL_TO_SELL]","[BUY_TO_SELL, SELL_TO_BUY]",AUTHORIZED


,model_id,optimizer_dimension,maximum_absolute_error,round_trip_passed,status
0,H1_DIAGONAL_SHARED_DECAY,5,5.551115123e-17,True,PASS
1,H2_DIAGONAL_SEPARATE_DECAY,6,5.551115123e-17,True,PASS


The diagonal exponential Hawkes model contract, candidate registry, natural parameterization, admissibility rules, stationarity rules, and optimizer transforms are frozen and validated. No estimation has been performed. Protected partitions remain unopened and no filesystem writes were performed.


In [5]:
# ============================================================
# Exact pre-batch Hawkes likelihood engine and deterministic tests
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("HAWKES_MODEL_CONTRACT_FROZEN", False)),
    "The Hawkes model contract has not been frozen.",
)
require(
    bool(globals().get("HAWKES_CANDIDATE_REGISTRY_FROZEN", False)),
    "The Hawkes candidate registry has not been frozen.",
)
require(
    bool(
        globals().get(
            "HAWKES_PARAMETER_TRANSFORMS_VALIDATED",
            False,
        )
    ),
    "The Hawkes parameter transforms have not been validated.",
)
require(
    bool(globals().get("ANALYTICAL_CONTENT_LOADED", False)),
    "Analytical DEVELOPMENT and CALIBRATION content is unavailable.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Replay-state and result containers
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class DiagonalHawkesState:
    state_time_ns: int
    buy_excitation: float
    sell_excitation: float


@dataclass(frozen=True, slots=True)
class DiagonalHawkesReplayResult:
    model_id: str
    observation_start_ns: int
    observation_end_exclusive_ns: int
    event_count: int
    batch_count: int
    log_event_term: float
    compensator_buy: float
    compensator_sell: float
    total_compensator: float
    log_likelihood: float
    negative_log_likelihood: float
    final_state: DiagonalHawkesState
    replay: pd.DataFrame


# ------------------------------------------------------------
# Batch-interface validation
# ------------------------------------------------------------

HAWKES_BATCH_REQUIRED_COLUMNS: Final[tuple[str, ...]] = (
    "event_time_ns",
    "buy_event_count",
    "sell_event_count",
)


def validate_hawkes_batch_input(
    batch_frame: pd.DataFrame,
    *,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
) -> pd.DataFrame:
    """
    Validate and return a stable exact-time Hawkes batch table.

    Each row must represent one unique exact event timestamp.
    """
    require(
        isinstance(batch_frame, pd.DataFrame),
        "Hawkes batch input must be a pandas DataFrame.",
    )

    missing_columns = sorted(
        set(HAWKES_BATCH_REQUIRED_COLUMNS)
        - set(batch_frame.columns)
    )

    require(
        not missing_columns,
        (
            "Hawkes batch input is missing columns: "
            f"{missing_columns}"
        ),
    )

    start_ns = int(observation_start_ns)
    end_ns = int(observation_end_exclusive_ns)

    require(
        end_ns > start_ns,
        "The observation interval must have positive duration.",
    )

    validated = batch_frame.loc[
        :,
        list(HAWKES_BATCH_REQUIRED_COLUMNS),
    ].copy()

    for integer_column in HAWKES_BATCH_REQUIRED_COLUMNS:
        validated[integer_column] = parse_exact_int64(
            validated[integer_column],
            label=f"HAWKES_BATCH_INPUT.{integer_column}",
        )

    validated = deterministic_sort(
        validated,
        ("event_time_ns",),
    )

    require(
        validated["event_time_ns"].is_unique,
        (
            "Hawkes batch input contains duplicate timestamps. "
            "Exact-time events must be aggregated before replay."
        ),
    )
    require(
        validated["event_time_ns"].is_monotonic_increasing,
        "Hawkes batch timestamps are not increasing.",
    )
    require(
        validated["buy_event_count"].ge(0).all(),
        "A Hawkes batch has a negative BUY count.",
    )
    require(
        validated["sell_event_count"].ge(0).all(),
        "A Hawkes batch has a negative SELL count.",
    )
    require(
        (
            validated["buy_event_count"]
            + validated["sell_event_count"]
        ).gt(0).all(),
        "A Hawkes batch contains no events.",
    )

    if not validated.empty:
        require(
            int(validated["event_time_ns"].iloc[0])
            >= start_ns,
            "A Hawkes batch occurs before the observation start.",
        )
        require(
            int(validated["event_time_ns"].iloc[-1])
            < end_ns,
            (
                "A Hawkes batch occurs at or after the exclusive "
                "observation end."
            ),
        )

    return validated


# ------------------------------------------------------------
# Exact compensator helper
# ------------------------------------------------------------

def excitation_interval_integral(
    post_event_excitation: float,
    decay_rate: float,
    elapsed_seconds: float,
) -> float:
    """
    Integrate exponentially decaying excitation over one interval.
    """
    require(
        math.isfinite(post_event_excitation)
        and post_event_excitation >= 0.0,
        "Post-event excitation must be finite and nonnegative.",
    )
    require(
        math.isfinite(decay_rate)
        and decay_rate > 0.0,
        "Decay rate must be finite and strictly positive.",
    )
    require(
        math.isfinite(elapsed_seconds)
        and elapsed_seconds >= 0.0,
        "Elapsed time must be finite and nonnegative.",
    )

    if elapsed_seconds == 0.0:
        return 0.0

    return (
        post_event_excitation
        * (-math.expm1(-decay_rate * elapsed_seconds))
        / decay_rate
    )


# ------------------------------------------------------------
# Exact strict-pre-batch likelihood replay
# ------------------------------------------------------------

def replay_diagonal_hawkes(
    batch_frame: pd.DataFrame,
    parameters: DiagonalHawkesParameters,
    *,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
    initial_state: DiagonalHawkesState | None = None,
    return_replay: bool = True,
) -> DiagonalHawkesReplayResult:
    """
    Replay one diagonal exponential Hawkes model.

    Every event in an exact-time batch is scored with the same
    intensity based on history strictly before the batch time.
    Excitation from the batch is applied only after all members
    have been scored.
    """
    parameter_validation = validate_hawkes_parameters(
        parameters
    )

    require(
        parameter_validation["mathematically_admissible"],
        (
            "Cannot replay mathematically inadmissible parameters: "
            f"{parameter_validation}"
        ),
    )

    start_ns = int(observation_start_ns)
    end_ns = int(observation_end_exclusive_ns)

    batches = validate_hawkes_batch_input(
        batch_frame,
        observation_start_ns=start_ns,
        observation_end_exclusive_ns=end_ns,
    )

    if initial_state is None:
        state = DiagonalHawkesState(
            state_time_ns=start_ns,
            buy_excitation=0.0,
            sell_excitation=0.0,
        )
    else:
        state = initial_state

    require(
        int(state.state_time_ns) == start_ns,
        (
            "Initial Hawkes state must be expressed exactly at the "
            "observation start."
        ),
    )
    require(
        math.isfinite(state.buy_excitation)
        and state.buy_excitation >= 0.0,
        "Initial BUY excitation must be finite and nonnegative.",
    )
    require(
        math.isfinite(state.sell_excitation)
        and state.sell_excitation >= 0.0,
        "Initial SELL excitation must be finite and nonnegative.",
    )

    previous_time_ns = start_ns
    post_buy_excitation = float(state.buy_excitation)
    post_sell_excitation = float(state.sell_excitation)

    cumulative_log_event_term = 0.0
    cumulative_compensator_buy = 0.0
    cumulative_compensator_sell = 0.0

    replay_rows: list[dict[str, Any]] = []

    for batch_number, batch in enumerate(
        batches.itertuples(index=False),
        start=1,
    ):
        event_time_ns = int(batch.event_time_ns)
        buy_event_count = int(batch.buy_event_count)
        sell_event_count = int(batch.sell_event_count)

        elapsed_ns = event_time_ns - previous_time_ns

        require(
            elapsed_ns >= 0,
            "Encountered negative elapsed Hawkes time.",
        )

        elapsed_seconds = (
            elapsed_ns / NANOSECONDS_PER_SECOND
        )

        buy_decay_factor = math.exp(
            -parameters.beta_buy * elapsed_seconds
        )
        sell_decay_factor = math.exp(
            -parameters.beta_sell * elapsed_seconds
        )

        pre_buy_excitation = (
            post_buy_excitation * buy_decay_factor
        )
        pre_sell_excitation = (
            post_sell_excitation * sell_decay_factor
        )

        lambda_buy_pre = (
            parameters.mu_buy + pre_buy_excitation
        )
        lambda_sell_pre = (
            parameters.mu_sell + pre_sell_excitation
        )

        require(
            math.isfinite(lambda_buy_pre)
            and lambda_buy_pre
            >= INTENSITY_FLOOR_PER_SECOND,
            (
                "BUY pre-batch intensity is invalid at "
                f"{event_time_ns}."
            ),
        )
        require(
            math.isfinite(lambda_sell_pre)
            and lambda_sell_pre
            >= INTENSITY_FLOOR_PER_SECOND,
            (
                "SELL pre-batch intensity is invalid at "
                f"{event_time_ns}."
            ),
        )

        interval_compensator_buy = (
            parameters.mu_buy * elapsed_seconds
            + excitation_interval_integral(
                post_buy_excitation,
                parameters.beta_buy,
                elapsed_seconds,
            )
        )

        interval_compensator_sell = (
            parameters.mu_sell * elapsed_seconds
            + excitation_interval_integral(
                post_sell_excitation,
                parameters.beta_sell,
                elapsed_seconds,
            )
        )

        log_event_term_buy = (
            buy_event_count * math.log(lambda_buy_pre)
        )
        log_event_term_sell = (
            sell_event_count * math.log(lambda_sell_pre)
        )

        post_buy_excitation_after_batch = (
            pre_buy_excitation
            + buy_event_count
            * parameters.kappa_buy
            * parameters.beta_buy
        )

        post_sell_excitation_after_batch = (
            pre_sell_excitation
            + sell_event_count
            * parameters.kappa_sell
            * parameters.beta_sell
        )

        cumulative_log_event_term += (
            log_event_term_buy + log_event_term_sell
        )
        cumulative_compensator_buy += (
            interval_compensator_buy
        )
        cumulative_compensator_sell += (
            interval_compensator_sell
        )

        if return_replay:
            replay_rows.append(
                {
                    "batch_number": batch_number,
                    "event_time_ns": event_time_ns,
                    "elapsed_from_previous_batch_ns": (
                        elapsed_ns
                    ),
                    "elapsed_from_previous_batch_seconds": (
                        elapsed_seconds
                    ),
                    "buy_event_count": buy_event_count,
                    "sell_event_count": sell_event_count,
                    "pre_buy_excitation": (
                        pre_buy_excitation
                    ),
                    "pre_sell_excitation": (
                        pre_sell_excitation
                    ),
                    "lambda_buy_pre": lambda_buy_pre,
                    "lambda_sell_pre": lambda_sell_pre,
                    "interval_compensator_buy": (
                        interval_compensator_buy
                    ),
                    "interval_compensator_sell": (
                        interval_compensator_sell
                    ),
                    "log_event_term_buy": (
                        log_event_term_buy
                    ),
                    "log_event_term_sell": (
                        log_event_term_sell
                    ),
                    "post_buy_excitation": (
                        post_buy_excitation_after_batch
                    ),
                    "post_sell_excitation": (
                        post_sell_excitation_after_batch
                    ),
                    "strict_pre_batch_scoring": True,
                    "zero_lag_within_batch_excitation": False,
                }
            )

        post_buy_excitation = (
            post_buy_excitation_after_batch
        )
        post_sell_excitation = (
            post_sell_excitation_after_batch
        )
        previous_time_ns = event_time_ns

    tail_elapsed_ns = end_ns - previous_time_ns

    require(
        tail_elapsed_ns >= 0,
        "Observation end precedes the final Hawkes batch.",
    )

    tail_elapsed_seconds = (
        tail_elapsed_ns / NANOSECONDS_PER_SECOND
    )

    tail_compensator_buy = (
        parameters.mu_buy * tail_elapsed_seconds
        + excitation_interval_integral(
            post_buy_excitation,
            parameters.beta_buy,
            tail_elapsed_seconds,
        )
    )

    tail_compensator_sell = (
        parameters.mu_sell * tail_elapsed_seconds
        + excitation_interval_integral(
            post_sell_excitation,
            parameters.beta_sell,
            tail_elapsed_seconds,
        )
    )

    cumulative_compensator_buy += tail_compensator_buy
    cumulative_compensator_sell += tail_compensator_sell

    final_buy_excitation = (
        post_buy_excitation
        * math.exp(
            -parameters.beta_buy * tail_elapsed_seconds
        )
    )
    final_sell_excitation = (
        post_sell_excitation
        * math.exp(
            -parameters.beta_sell * tail_elapsed_seconds
        )
    )

    total_compensator = (
        cumulative_compensator_buy
        + cumulative_compensator_sell
    )

    log_likelihood = (
        cumulative_log_event_term - total_compensator
    )

    require(
        math.isfinite(log_likelihood),
        "Hawkes replay produced a non-finite log likelihood.",
    )

    replay_table = pd.DataFrame(replay_rows)

    if return_replay and not replay_table.empty:
        replay_table["cumulative_log_event_term"] = (
            replay_table[
                [
                    "log_event_term_buy",
                    "log_event_term_sell",
                ]
            ]
            .sum(axis=1)
            .cumsum()
        )

        replay_table["cumulative_compensator"] = (
            replay_table[
                [
                    "interval_compensator_buy",
                    "interval_compensator_sell",
                ]
            ]
            .sum(axis=1)
            .cumsum()
        )

        replay_table["cumulative_log_likelihood_before_tail"] = (
            replay_table["cumulative_log_event_term"]
            - replay_table["cumulative_compensator"]
        )

    return DiagonalHawkesReplayResult(
        model_id=parameters.model_id,
        observation_start_ns=start_ns,
        observation_end_exclusive_ns=end_ns,
        event_count=int(
            batches[
                [
                    "buy_event_count",
                    "sell_event_count",
                ]
            ].to_numpy(dtype=np.int64).sum()
        ),
        batch_count=len(batches),
        log_event_term=float(
            cumulative_log_event_term
        ),
        compensator_buy=float(
            cumulative_compensator_buy
        ),
        compensator_sell=float(
            cumulative_compensator_sell
        ),
        total_compensator=float(total_compensator),
        log_likelihood=float(log_likelihood),
        negative_log_likelihood=float(
            -log_likelihood
        ),
        final_state=DiagonalHawkesState(
            state_time_ns=end_ns,
            buy_excitation=float(
                final_buy_excitation
            ),
            sell_excitation=float(
                final_sell_excitation
            ),
        ),
        replay=replay_table,
    )


# ------------------------------------------------------------
# O(n) natural-parameter score recursion
# ------------------------------------------------------------

def diagonal_hawkes_log_likelihood_and_natural_score(
    batch_frame: pd.DataFrame,
    parameters: DiagonalHawkesParameters,
    *,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
) -> tuple[float, dict[str, float]]:
    """
    Return log likelihood and exact recursive score.

    This score implementation assumes zero excitation state at the
    left observation boundary. That is the DEVELOPMENT fitting
    contract. Locked CALIBRATION replay does not require gradients.
    """
    batches = validate_hawkes_batch_input(
        batch_frame,
        observation_start_ns=observation_start_ns,
        observation_end_exclusive_ns=(
            observation_end_exclusive_ns
        ),
    )

    validation = validate_hawkes_parameters(parameters)

    require(
        validation["mathematically_admissible"],
        "Natural-score recursion received invalid parameters.",
    )

    previous_time_ns = int(observation_start_ns)

    post_excitation = {
        "BUY": 0.0,
        "SELL": 0.0,
    }
    post_derivative_kappa = {
        "BUY": 0.0,
        "SELL": 0.0,
    }
    post_derivative_beta = {
        "BUY": 0.0,
        "SELL": 0.0,
    }

    side_parameter_map = {
        "BUY": {
            "mu": parameters.mu_buy,
            "kappa": parameters.kappa_buy,
            "beta": parameters.beta_buy,
            "count_column": "buy_event_count",
        },
        "SELL": {
            "mu": parameters.mu_sell,
            "kappa": parameters.kappa_sell,
            "beta": parameters.beta_sell,
            "count_column": "sell_event_count",
        },
    }

    log_likelihood = 0.0

    natural_score = {
        "mu_buy": 0.0,
        "mu_sell": 0.0,
        "kappa_buy": 0.0,
        "kappa_sell": 0.0,
        "beta_buy": 0.0,
        "beta_sell": 0.0,
    }

    for batch in batches.itertuples(index=False):
        event_time_ns = int(batch.event_time_ns)

        elapsed_seconds = (
            event_time_ns - previous_time_ns
        ) / NANOSECONDS_PER_SECOND

        for side_name, side_contract in (
            side_parameter_map.items()
        ):
            side_key = side_name.casefold()
            mu = float(side_contract["mu"])
            kappa = float(side_contract["kappa"])
            beta = float(side_contract["beta"])
            count = int(
                getattr(
                    batch,
                    str(side_contract["count_column"]),
                )
            )

            previous_post = post_excitation[side_name]
            previous_d_kappa = (
                post_derivative_kappa[side_name]
            )
            previous_d_beta = (
                post_derivative_beta[side_name]
            )

            decay_factor = math.exp(
                -beta * elapsed_seconds
            )

            pre_excitation = (
                previous_post * decay_factor
            )
            pre_d_kappa = (
                previous_d_kappa * decay_factor
            )
            pre_d_beta = (
                previous_d_beta * decay_factor
                - previous_post
                * elapsed_seconds
                * decay_factor
            )

            intensity = mu + pre_excitation

            require(
                intensity >= INTENSITY_FLOOR_PER_SECOND,
                "Natural-score recursion encountered invalid intensity.",
            )

            one_minus_decay = -math.expm1(
                -beta * elapsed_seconds
            )

            excitation_integral_factor = (
                one_minus_decay / beta
            )

            integral_factor_beta_derivative = (
                (
                    beta
                    * elapsed_seconds
                    * decay_factor
                )
                - one_minus_decay
            ) / (beta * beta)

            interval_compensator = (
                mu * elapsed_seconds
                + previous_post
                * excitation_integral_factor
            )

            compensator_d_kappa = (
                previous_d_kappa
                * excitation_integral_factor
            )

            compensator_d_beta = (
                previous_d_beta
                * excitation_integral_factor
                + previous_post
                * integral_factor_beta_derivative
            )

            log_likelihood += (
                count * math.log(intensity)
                - interval_compensator
            )

            natural_score[f"mu_{side_key}"] += (
                count / intensity
                - elapsed_seconds
            )

            natural_score[f"kappa_{side_key}"] += (
                count * pre_d_kappa / intensity
                - compensator_d_kappa
            )

            natural_score[f"beta_{side_key}"] += (
                count * pre_d_beta / intensity
                - compensator_d_beta
            )

            post_excitation[side_name] = (
                pre_excitation + count * kappa * beta
            )
            post_derivative_kappa[side_name] = (
                pre_d_kappa + count * beta
            )
            post_derivative_beta[side_name] = (
                pre_d_beta + count * kappa
            )

        previous_time_ns = event_time_ns

    tail_elapsed_seconds = (
        int(observation_end_exclusive_ns)
        - previous_time_ns
    ) / NANOSECONDS_PER_SECOND

    require(
        tail_elapsed_seconds >= 0.0,
        "Natural-score recursion has a negative tail interval.",
    )

    for side_name, side_contract in (
        side_parameter_map.items()
    ):
        side_key = side_name.casefold()
        mu = float(side_contract["mu"])
        beta = float(side_contract["beta"])

        previous_post = post_excitation[side_name]
        previous_d_kappa = (
            post_derivative_kappa[side_name]
        )
        previous_d_beta = (
            post_derivative_beta[side_name]
        )

        decay_factor = math.exp(
            -beta * tail_elapsed_seconds
        )
        one_minus_decay = -math.expm1(
            -beta * tail_elapsed_seconds
        )

        excitation_integral_factor = (
            one_minus_decay / beta
        )

        integral_factor_beta_derivative = (
            (
                beta
                * tail_elapsed_seconds
                * decay_factor
            )
            - one_minus_decay
        ) / (beta * beta)

        tail_compensator = (
            mu * tail_elapsed_seconds
            + previous_post
            * excitation_integral_factor
        )

        log_likelihood -= tail_compensator

        natural_score[f"mu_{side_key}"] -= (
            tail_elapsed_seconds
        )

        natural_score[f"kappa_{side_key}"] -= (
            previous_d_kappa
            * excitation_integral_factor
        )

        natural_score[f"beta_{side_key}"] -= (
            previous_d_beta
            * excitation_integral_factor
            + previous_post
            * integral_factor_beta_derivative
        )

    require(
        math.isfinite(log_likelihood),
        "Natural-score recursion produced a non-finite likelihood.",
    )
    require(
        all(
            math.isfinite(value)
            for value in natural_score.values()
        ),
        "Natural-score recursion produced a non-finite derivative.",
    )

    return float(log_likelihood), natural_score


# ------------------------------------------------------------
# Optimizer-space objective and exact gradient
# ------------------------------------------------------------

def positive_transform_derivative(
    unconstrained_value: float,
) -> float:
    """Derivative of floor plus softplus away from clipping edges."""
    clipped_value = float(
        np.clip(
            unconstrained_value,
            TRANSFORM_CLIP_LOWER,
            TRANSFORM_CLIP_UPPER,
        )
    )

    if clipped_value != unconstrained_value:
        return 0.0

    return float(special.expit(clipped_value))


def kappa_transform_derivative(
    unconstrained_value: float,
) -> float:
    """Derivative of the bounded excitation-mass transform."""
    clipped_value = float(
        np.clip(
            unconstrained_value,
            TRANSFORM_CLIP_LOWER,
            TRANSFORM_CLIP_UPPER,
        )
    )

    if clipped_value != unconstrained_value:
        return 0.0

    probability = float(
        special.expit(clipped_value)
    )

    return (
        KAPPA_HARD_UPPER_BOUND
        * probability
        * (1.0 - probability)
    )


def negative_log_likelihood_and_gradient(
    optimizer_vector: np.ndarray,
    *,
    model_id: str,
    batch_frame: pd.DataFrame,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
) -> tuple[float, np.ndarray]:
    """Return DEVELOPMENT NLL and exact optimizer-space gradient."""
    vector = np.asarray(
        optimizer_vector,
        dtype=np.float64,
    )

    parameters = decode_hawkes_parameters(
        model_id,
        vector,
    )

    (
        log_likelihood,
        natural_score,
    ) = diagonal_hawkes_log_likelihood_and_natural_score(
        batch_frame,
        parameters,
        observation_start_ns=observation_start_ns,
        observation_end_exclusive_ns=(
            observation_end_exclusive_ns
        ),
    )

    specification = HAWKES_CANDIDATE_SPECIFICATIONS[
        model_id
    ]

    optimizer_gradient = np.zeros(
        specification.estimated_parameter_count,
        dtype=np.float64,
    )

    optimizer_gradient[0] = (
        -natural_score["mu_buy"]
        * positive_transform_derivative(vector[0])
    )

    optimizer_gradient[1] = (
        -natural_score["mu_sell"]
        * positive_transform_derivative(vector[1])
    )

    optimizer_gradient[2] = (
        -natural_score["kappa_buy"]
        * kappa_transform_derivative(vector[2])
    )

    optimizer_gradient[3] = (
        -natural_score["kappa_sell"]
        * kappa_transform_derivative(vector[3])
    )

    if specification.shared_decay:
        optimizer_gradient[4] = (
            -(
                natural_score["beta_buy"]
                + natural_score["beta_sell"]
            )
            * positive_transform_derivative(vector[4])
        )
    else:
        optimizer_gradient[4] = (
            -natural_score["beta_buy"]
            * positive_transform_derivative(vector[4])
        )

        optimizer_gradient[5] = (
            -natural_score["beta_sell"]
            * positive_transform_derivative(vector[5])
        )

    negative_log_likelihood = -log_likelihood

    require(
        math.isfinite(negative_log_likelihood),
        "Optimizer objective is non-finite.",
    )
    require(
        np.isfinite(optimizer_gradient).all(),
        "Optimizer gradient is non-finite.",
    )

    return (
        float(negative_log_likelihood),
        optimizer_gradient,
    )


def negative_log_likelihood_only(
    optimizer_vector: np.ndarray,
    *,
    model_id: str,
    batch_frame: pd.DataFrame,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
) -> float:
    """Return only the optimizer-space negative log likelihood."""
    value, _ = negative_log_likelihood_and_gradient(
        optimizer_vector,
        model_id=model_id,
        batch_frame=batch_frame,
        observation_start_ns=observation_start_ns,
        observation_end_exclusive_ns=(
            observation_end_exclusive_ns
        ),
    )

    return value


# ------------------------------------------------------------
# Deterministic test fixtures
# ------------------------------------------------------------

TEST_SECOND_NS: Final[int] = NANOSECONDS_PER_SECOND


def seconds_to_ns(seconds: float) -> int:
    """Convert an exactly specified test time to integer nanoseconds."""
    return int(
        round(seconds * NANOSECONDS_PER_SECOND)
    )


EMPTY_TEST_BATCHES = pd.DataFrame(
    {
        "event_time_ns": pd.Series(dtype="int64"),
        "buy_event_count": pd.Series(dtype="int64"),
        "sell_event_count": pd.Series(dtype="int64"),
    }
)

TEST_PARAMETERS = DiagonalHawkesParameters(
    model_id=H2_MODEL_ID,
    mu_buy=2.0,
    mu_sell=3.0,
    kappa_buy=0.4,
    kappa_sell=0.2,
    beta_buy=3.0,
    beta_sell=5.0,
)


# ------------------------------------------------------------
# Test 1 — no-event interval
# ------------------------------------------------------------

no_event_result = replay_diagonal_hawkes(
    EMPTY_TEST_BATCHES,
    TEST_PARAMETERS,
    observation_start_ns=0,
    observation_end_exclusive_ns=seconds_to_ns(2.0),
)

expected_no_event_compensator = (
    TEST_PARAMETERS.mu_buy
    + TEST_PARAMETERS.mu_sell
) * 2.0

expected_no_event_log_likelihood = (
    -expected_no_event_compensator
)

require(
    math.isclose(
        no_event_result.log_likelihood,
        expected_no_event_log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "No-event likelihood test failed.",
)


# ------------------------------------------------------------
# Test 2 — one-event hand calculation
# ------------------------------------------------------------

one_event_batches = pd.DataFrame(
    {
        "event_time_ns": [seconds_to_ns(1.0)],
        "buy_event_count": [1],
        "sell_event_count": [0],
    }
)

one_event_result = replay_diagonal_hawkes(
    one_event_batches,
    TEST_PARAMETERS,
    observation_start_ns=0,
    observation_end_exclusive_ns=seconds_to_ns(2.0),
)

expected_one_event_buy_compensator = (
    TEST_PARAMETERS.mu_buy * 2.0
    + TEST_PARAMETERS.kappa_buy
    * (
        1.0
        - math.exp(-TEST_PARAMETERS.beta_buy)
    )
)

expected_one_event_sell_compensator = (
    TEST_PARAMETERS.mu_sell * 2.0
)

expected_one_event_log_likelihood = (
    math.log(TEST_PARAMETERS.mu_buy)
    - expected_one_event_buy_compensator
    - expected_one_event_sell_compensator
)

require(
    math.isclose(
        one_event_result.log_likelihood,
        expected_one_event_log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "One-event hand-calculation test failed.",
)


# ------------------------------------------------------------
# Test 3 — tied batch uses one pre-batch intensity
# ------------------------------------------------------------

tied_batch = pd.DataFrame(
    {
        "event_time_ns": [seconds_to_ns(1.0)],
        "buy_event_count": [2],
        "sell_event_count": [1],
    }
)

tied_batch_result = replay_diagonal_hawkes(
    tied_batch,
    TEST_PARAMETERS,
    observation_start_ns=0,
    observation_end_exclusive_ns=seconds_to_ns(1.5),
)

tied_row = tied_batch_result.replay.iloc[0]

require(
    math.isclose(
        float(tied_row["lambda_buy_pre"]),
        TEST_PARAMETERS.mu_buy,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "Tied BUY events did not use the base pre-batch intensity.",
)
require(
    math.isclose(
        float(tied_row["lambda_sell_pre"]),
        TEST_PARAMETERS.mu_sell,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "Tied SELL events did not use the base pre-batch intensity.",
)

expected_tied_post_buy = (
    2
    * TEST_PARAMETERS.kappa_buy
    * TEST_PARAMETERS.beta_buy
)

expected_tied_post_sell = (
    TEST_PARAMETERS.kappa_sell
    * TEST_PARAMETERS.beta_sell
)

require(
    math.isclose(
        float(tied_row["post_buy_excitation"]),
        expected_tied_post_buy,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "Tied BUY excitation update is incorrect.",
)
require(
    math.isclose(
        float(tied_row["post_sell_excitation"]),
        expected_tied_post_sell,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "Tied SELL excitation update is incorrect.",
)
require(
    not bool(
        tied_row["zero_lag_within_batch_excitation"]
    ),
    "Tied batch incorrectly permits zero-lag excitation.",
)


# ------------------------------------------------------------
# Test 4 — Poisson limit equivalence
# ------------------------------------------------------------

poisson_parameters = DiagonalHawkesParameters(
    model_id=H2_MODEL_ID,
    mu_buy=2.5,
    mu_sell=1.5,
    kappa_buy=0.0,
    kappa_sell=0.0,
    beta_buy=4.0,
    beta_sell=7.0,
)

poisson_test_batches = pd.DataFrame(
    {
        "event_time_ns": [
            seconds_to_ns(0.2),
            seconds_to_ns(0.8),
            seconds_to_ns(1.3),
        ],
        "buy_event_count": [1, 2, 0],
        "sell_event_count": [0, 1, 1],
    }
)

poisson_result = replay_diagonal_hawkes(
    poisson_test_batches,
    poisson_parameters,
    observation_start_ns=0,
    observation_end_exclusive_ns=seconds_to_ns(2.0),
)

poisson_buy_count = int(
    poisson_test_batches["buy_event_count"].sum()
)
poisson_sell_count = int(
    poisson_test_batches["sell_event_count"].sum()
)

expected_poisson_log_likelihood = (
    poisson_buy_count
    * math.log(poisson_parameters.mu_buy)
    + poisson_sell_count
    * math.log(poisson_parameters.mu_sell)
    - poisson_parameters.mu_buy * 2.0
    - poisson_parameters.mu_sell * 2.0
)

require(
    math.isclose(
        poisson_result.log_likelihood,
        expected_poisson_log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "Zero-excitation Poisson equivalence test failed.",
)


# ------------------------------------------------------------
# Test 5 — long-gap state decay
# ------------------------------------------------------------

long_gap_batches = pd.DataFrame(
    {
        "event_time_ns": [
            seconds_to_ns(1.0),
            seconds_to_ns(11.0),
        ],
        "buy_event_count": [1, 1],
        "sell_event_count": [0, 0],
    }
)

long_gap_result = replay_diagonal_hawkes(
    long_gap_batches,
    TEST_PARAMETERS,
    observation_start_ns=0,
    observation_end_exclusive_ns=seconds_to_ns(12.0),
)

first_post_buy_excitation = float(
    long_gap_result.replay.iloc[0][
        "post_buy_excitation"
    ]
)

second_pre_buy_excitation = float(
    long_gap_result.replay.iloc[1][
        "pre_buy_excitation"
    ]
)

expected_second_pre_buy_excitation = (
    first_post_buy_excitation
    * math.exp(-TEST_PARAMETERS.beta_buy * 10.0)
)

require(
    math.isclose(
        second_pre_buy_excitation,
        expected_second_pre_buy_excitation,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    "Long-gap excitation decay test failed.",
)


# ------------------------------------------------------------
# Test 6 — exact compensator versus numerical quadrature
# ------------------------------------------------------------

quadrature_batches = pd.DataFrame(
    {
        "event_time_ns": [
            seconds_to_ns(0.5),
            seconds_to_ns(1.2),
            seconds_to_ns(1.8),
        ],
        "buy_event_count": [1, 2, 1],
        "sell_event_count": [0, 1, 0],
    }
)

quadrature_end_seconds: Final[float] = 2.5

quadrature_result = replay_diagonal_hawkes(
    quadrature_batches,
    TEST_PARAMETERS,
    observation_start_ns=0,
    observation_end_exclusive_ns=seconds_to_ns(
        quadrature_end_seconds
    ),
)

buy_event_times_and_counts = tuple(
    zip(
        quadrature_batches["event_time_ns"]
        .to_numpy(dtype=np.int64)
        / NANOSECONDS_PER_SECOND,
        quadrature_batches["buy_event_count"]
        .to_numpy(dtype=np.int64),
    )
)

sell_event_times_and_counts = tuple(
    zip(
        quadrature_batches["event_time_ns"]
        .to_numpy(dtype=np.int64)
        / NANOSECONDS_PER_SECOND,
        quadrature_batches["sell_event_count"]
        .to_numpy(dtype=np.int64),
    )
)


def direct_side_intensity(
    time_seconds: float,
    *,
    mu: float,
    kappa: float,
    beta: float,
    events: tuple[tuple[float, int], ...],
) -> float:
    """Direct intensity using events strictly before time_seconds."""
    excitation = sum(
        event_count
        * kappa
        * beta
        * math.exp(
            -beta * (time_seconds - event_time_seconds)
        )
        for event_time_seconds, event_count in events
        if event_time_seconds < time_seconds
    )

    return mu + excitation


quadrature_points = sorted(
    {
        float(value)
        for value in (
            quadrature_batches["event_time_ns"]
            .to_numpy(dtype=np.int64)
            / NANOSECONDS_PER_SECOND
        )
    }
)

numerical_buy_compensator, _ = (
    scipy.integrate.quad(
        lambda time_value: direct_side_intensity(
            time_value,
            mu=TEST_PARAMETERS.mu_buy,
            kappa=TEST_PARAMETERS.kappa_buy,
            beta=TEST_PARAMETERS.beta_buy,
            events=buy_event_times_and_counts,
        ),
        0.0,
        quadrature_end_seconds,
        points=quadrature_points,
        epsabs=1e-11,
        epsrel=1e-11,
        limit=200,
    )
)

numerical_sell_compensator, _ = (
    scipy.integrate.quad(
        lambda time_value: direct_side_intensity(
            time_value,
            mu=TEST_PARAMETERS.mu_sell,
            kappa=TEST_PARAMETERS.kappa_sell,
            beta=TEST_PARAMETERS.beta_sell,
            events=sell_event_times_and_counts,
        ),
        0.0,
        quadrature_end_seconds,
        points=quadrature_points,
        epsabs=1e-11,
        epsrel=1e-11,
        limit=200,
    )
)

require(
    math.isclose(
        quadrature_result.compensator_buy,
        numerical_buy_compensator,
        rel_tol=1e-9,
        abs_tol=1e-9,
    ),
    "BUY analytical compensator differs from numerical integration.",
)
require(
    math.isclose(
        quadrature_result.compensator_sell,
        numerical_sell_compensator,
        rel_tol=1e-9,
        abs_tol=1e-9,
    ),
    "SELL analytical compensator differs from numerical integration.",
)


# ------------------------------------------------------------
# Test 7 — recursive likelihood matches score likelihood
# ------------------------------------------------------------

for model_id, test_parameters in (
    ROUND_TRIP_TEST_PARAMETERS.items()
):
    recursive_result = replay_diagonal_hawkes(
        quadrature_batches,
        test_parameters,
        observation_start_ns=0,
        observation_end_exclusive_ns=seconds_to_ns(
            quadrature_end_seconds
        ),
        return_replay=False,
    )

    score_log_likelihood, _ = (
        diagonal_hawkes_log_likelihood_and_natural_score(
            quadrature_batches,
            test_parameters,
            observation_start_ns=0,
            observation_end_exclusive_ns=seconds_to_ns(
                quadrature_end_seconds
            ),
        )
    )

    require(
        math.isclose(
            recursive_result.log_likelihood,
            score_log_likelihood,
            rel_tol=FLOAT_RELATIVE_TOLERANCE,
            abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
        ),
        (
            f"{model_id} replay likelihood and score-recursion "
            "likelihood do not match."
        ),
    )


# ------------------------------------------------------------
# Test 8 — exact gradient versus central finite differences
# ------------------------------------------------------------

def central_finite_difference_gradient(
    objective_function: Any,
    parameter_vector: np.ndarray,
    *,
    relative_step: float = 1e-6,
) -> np.ndarray:
    """Return a deterministic central finite-difference gradient."""
    vector = np.asarray(
        parameter_vector,
        dtype=np.float64,
    )

    gradient = np.empty_like(vector)

    for parameter_index in range(len(vector)):
        step = relative_step * max(
            1.0,
            abs(float(vector[parameter_index])),
        )

        upper_vector = vector.copy()
        lower_vector = vector.copy()

        upper_vector[parameter_index] += step
        lower_vector[parameter_index] -= step

        upper_value = float(
            objective_function(upper_vector)
        )
        lower_value = float(
            objective_function(lower_vector)
        )

        gradient[parameter_index] = (
            upper_value - lower_value
        ) / (2.0 * step)

    return gradient


gradient_test_rows: list[dict[str, Any]] = []

for model_id, natural_parameters in (
    ROUND_TRIP_TEST_PARAMETERS.items()
):
    optimizer_parameters = encode_hawkes_parameters(
        natural_parameters
    )

    objective_value, exact_gradient = (
        negative_log_likelihood_and_gradient(
            optimizer_parameters,
            model_id=model_id,
            batch_frame=quadrature_batches,
            observation_start_ns=0,
            observation_end_exclusive_ns=seconds_to_ns(
                quadrature_end_seconds
            ),
        )
    )

    numerical_gradient = (
        central_finite_difference_gradient(
            lambda vector: negative_log_likelihood_only(
                vector,
                model_id=model_id,
                batch_frame=quadrature_batches,
                observation_start_ns=0,
                observation_end_exclusive_ns=seconds_to_ns(
                    quadrature_end_seconds
                ),
            ),
            optimizer_parameters,
        )
    )

    maximum_absolute_error = float(
        np.max(
            np.abs(
                exact_gradient - numerical_gradient
            )
        )
    )

    maximum_relative_error = float(
        np.max(
            np.abs(
                exact_gradient - numerical_gradient
            )
            / np.maximum(
                1.0,
                np.abs(numerical_gradient),
            )
        )
    )

    gradient_passed = bool(
        np.allclose(
            exact_gradient,
            numerical_gradient,
            rtol=2e-5,
            atol=2e-6,
        )
    )

    require(
        gradient_passed,
        (
            f"{model_id} gradient check failed; "
            f"maximum absolute error={maximum_absolute_error}; "
            f"maximum relative error={maximum_relative_error}."
        ),
    )

    gradient_test_rows.append(
        {
            "model_id": model_id,
            "negative_log_likelihood": objective_value,
            "gradient_dimension": len(exact_gradient),
            "maximum_absolute_error": (
                maximum_absolute_error
            ),
            "maximum_relative_error": (
                maximum_relative_error
            ),
            "gradient_check_passed": gradient_passed,
            "status": "PASS",
        }
    )

HAWKES_GRADIENT_VALIDATION = pd.DataFrame(
    gradient_test_rows
)


# ------------------------------------------------------------
# Deterministic validation ledger
# ------------------------------------------------------------

validation_test_rows = [
    {
        "test_id": "T01_NO_EVENT_INTERVAL",
        "test_scope": "LIKELIHOOD",
        "observed_value": no_event_result.log_likelihood,
        "expected_value": expected_no_event_log_likelihood,
        "absolute_error": abs(
            no_event_result.log_likelihood
            - expected_no_event_log_likelihood
        ),
        "status": "PASS",
    },
    {
        "test_id": "T02_ONE_EVENT_HAND_CALCULATION",
        "test_scope": "LIKELIHOOD",
        "observed_value": one_event_result.log_likelihood,
        "expected_value": expected_one_event_log_likelihood,
        "absolute_error": abs(
            one_event_result.log_likelihood
            - expected_one_event_log_likelihood
        ),
        "status": "PASS",
    },
    {
        "test_id": "T03_TIED_BATCH_PRE_INTENSITY",
        "test_scope": "SIMULTANEOUS_BATCH_CAUSALITY",
        "observed_value": float(
            tied_row["lambda_buy_pre"]
        ),
        "expected_value": TEST_PARAMETERS.mu_buy,
        "absolute_error": abs(
            float(tied_row["lambda_buy_pre"])
            - TEST_PARAMETERS.mu_buy
        ),
        "status": "PASS",
    },
    {
        "test_id": "T04_POISSON_LIMIT",
        "test_scope": "LIKELIHOOD",
        "observed_value": poisson_result.log_likelihood,
        "expected_value": expected_poisson_log_likelihood,
        "absolute_error": abs(
            poisson_result.log_likelihood
            - expected_poisson_log_likelihood
        ),
        "status": "PASS",
    },
    {
        "test_id": "T05_LONG_GAP_DECAY",
        "test_scope": "STATE_RECURSION",
        "observed_value": second_pre_buy_excitation,
        "expected_value": (
            expected_second_pre_buy_excitation
        ),
        "absolute_error": abs(
            second_pre_buy_excitation
            - expected_second_pre_buy_excitation
        ),
        "status": "PASS",
    },
    {
        "test_id": "T06_BUY_COMPENSATOR_QUADRATURE",
        "test_scope": "COMPENSATOR",
        "observed_value": (
            quadrature_result.compensator_buy
        ),
        "expected_value": numerical_buy_compensator,
        "absolute_error": abs(
            quadrature_result.compensator_buy
            - numerical_buy_compensator
        ),
        "status": "PASS",
    },
    {
        "test_id": "T07_SELL_COMPENSATOR_QUADRATURE",
        "test_scope": "COMPENSATOR",
        "observed_value": (
            quadrature_result.compensator_sell
        ),
        "expected_value": numerical_sell_compensator,
        "absolute_error": abs(
            quadrature_result.compensator_sell
            - numerical_sell_compensator
        ),
        "status": "PASS",
    },
    {
        "test_id": "T08_REPLAY_SCORE_RECURSION",
        "test_scope": "LIKELIHOOD_IMPLEMENTATION",
        "observed_value": 1.0,
        "expected_value": 1.0,
        "absolute_error": 0.0,
        "status": "PASS",
    },
]

HAWKES_DETERMINISTIC_ENGINE_TESTS = pd.DataFrame(
    validation_test_rows
)

require(
    HAWKES_DETERMINISTIC_ENGINE_TESTS[
        "status"
    ].eq("PASS").all(),
    "At least one deterministic Hawkes-engine test failed.",
)
require(
    HAWKES_GRADIENT_VALIDATION[
        "status"
    ].eq("PASS").all(),
    "At least one Hawkes gradient test failed.",
)


# ------------------------------------------------------------
# Empirical DEVELOPMENT smoke replay
#
# This uses non-authoritative provisional parameters only to prove
# that the engine can traverse the complete DEVELOPMENT contract.
# It is not a fitted Hawkes result.
# ------------------------------------------------------------

development_contract_row = window_lookup.loc[
    FIT_PARTITION
]

DEVELOPMENT_START_NS: Final[int] = int(
    development_contract_row["contract_start_ns"]
)
DEVELOPMENT_END_EXCLUSIVE_NS: Final[int] = int(
    development_contract_row["contract_end_exclusive_ns"]
)
DEVELOPMENT_DURATION_SECONDS: Final[float] = (
    int(development_contract_row["contract_duration_ns"])
    / NANOSECONDS_PER_SECOND
)

DEVELOPMENT_BATCHES_FOR_HAWKES = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            "event_partition"
        ].eq(FIT_PARTITION),
        [
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

development_buy_rate = (
    DEVELOPMENT_BATCHES_FOR_HAWKES[
        "buy_event_count"
    ].sum()
    / DEVELOPMENT_DURATION_SECONDS
)

development_sell_rate = (
    DEVELOPMENT_BATCHES_FOR_HAWKES[
        "sell_event_count"
    ].sum()
    / DEVELOPMENT_DURATION_SECONDS
)

PROVISIONAL_SMOKE_PARAMETERS = (
    DiagonalHawkesParameters(
        model_id=H1_MODEL_ID,
        mu_buy=max(
            float(development_buy_rate) * 0.8,
            PARAMETER_POSITIVITY_FLOOR * 10.0,
        ),
        mu_sell=max(
            float(development_sell_rate) * 0.8,
            PARAMETER_POSITIVITY_FLOOR * 10.0,
        ),
        kappa_buy=0.20,
        kappa_sell=0.20,
        beta_buy=math.log(2.0) / 0.250,
        beta_sell=math.log(2.0) / 0.250,
    )
)

development_smoke_result = replay_diagonal_hawkes(
    DEVELOPMENT_BATCHES_FOR_HAWKES,
    PROVISIONAL_SMOKE_PARAMETERS,
    observation_start_ns=DEVELOPMENT_START_NS,
    observation_end_exclusive_ns=(
        DEVELOPMENT_END_EXCLUSIVE_NS
    ),
    return_replay=False,
)

require(
    development_smoke_result.event_count
    == EXPECTED_DEVELOPMENT_EVENT_ROWS,
    "DEVELOPMENT smoke replay lost events.",
)
require(
    development_smoke_result.batch_count
    == EXPECTED_DEVELOPMENT_BATCH_ROWS,
    "DEVELOPMENT smoke replay lost batches.",
)
require(
    math.isfinite(
        development_smoke_result.log_likelihood
    ),
    "DEVELOPMENT smoke replay produced a non-finite likelihood.",
)
require(
    development_smoke_result.final_state.state_time_ns
    == DEVELOPMENT_END_EXCLUSIVE_NS,
    "DEVELOPMENT smoke replay ended at the wrong boundary.",
)


# ------------------------------------------------------------
# Validation state
# ------------------------------------------------------------

HAWKES_LIKELIHOOD_ENGINE_VALIDATED: bool = True
HAWKES_EXACT_GRADIENT_VALIDATED: bool = True
HAWKES_DEVELOPMENT_SMOKE_REPLAY_PASSED: bool = True
FILESYSTEM_WRITES_PERFORMED = False

engine_validation_summary = pd.DataFrame(
    [
        {
            "field": "likelihood_engine_validated",
            "value": HAWKES_LIKELIHOOD_ENGINE_VALIDATED,
        },
        {
            "field": "exact_gradient_validated",
            "value": HAWKES_EXACT_GRADIENT_VALIDATED,
        },
        {
            "field": "development_smoke_replay_passed",
            "value": (
                HAWKES_DEVELOPMENT_SMOKE_REPLAY_PASSED
            ),
        },
        {
            "field": "deterministic_engine_tests",
            "value": len(
                HAWKES_DETERMINISTIC_ENGINE_TESTS
            ),
        },
        {
            "field": "gradient_tests",
            "value": len(
                HAWKES_GRADIENT_VALIDATION
            ),
        },
        {
            "field": "development_smoke_event_count",
            "value": development_smoke_result.event_count,
        },
        {
            "field": "development_smoke_batch_count",
            "value": development_smoke_result.batch_count,
        },
        {
            "field": "development_smoke_log_likelihood",
            "value": (
                development_smoke_result.log_likelihood
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(engine_validation_summary)
display(HAWKES_DETERMINISTIC_ENGINE_TESTS)
display(HAWKES_GRADIENT_VALIDATION)

print(
    "The exact strict-pre-batch Hawkes likelihood, compensator, "
    "state recursion, Poisson limit, simultaneous-batch update, "
    "and optimizer-space gradient passed deterministic validation. "
    "The engine also traversed the complete DEVELOPMENT interval "
    "without losing events or batches. The smoke parameters are "
    "provisional and are not fitted results. Protected partitions "
    "remain unopened and no filesystem writes were performed."
)

,field,value
0,likelihood_engine_validated,True
1,exact_gradient_validated,True
2,development_smoke_replay_passed,True
3,deterministic_engine_tests,8
4,gradient_tests,2
5,development_smoke_event_count,7004
6,development_smoke_batch_count,6859
7,development_smoke_log_likelihood,"-1,717.274244"
8,validation_content_loaded,False
9,engineering_holdout_content_loaded,False


,test_id,test_scope,observed_value,expected_value,absolute_error,status
0,T01_NO_EVENT_INTERVAL,LIKELIHOOD,-10,-10,0,PASS
1,T02_ONE_EVENT_HAND_CALCULATION,LIKELIHOOD,-9.686937992,-9.686937992,0,PASS
2,T03_TIED_BATCH_PRE_INTENSITY,SIMULTANEOUS_BATCH_CAUSALITY,2,2,0,PASS
3,T04_POISSON_LIMIT,LIKELIHOOD,-4.440197588,-4.440197588,8.881784197e-16,PASS
4,T05_LONG_GAP_DECAY,STATE_RECURSION,1.122914756e-13,1.122914756e-13,0,PASS
5,T06_BUY_COMPENSATOR_QUADRATURE,COMPENSATOR,6.533832399,6.533832399,8.881784197e-16,PASS
6,T07_SELL_COMPENSATOR_QUADRATURE,COMPENSATOR,7.699699312,7.699699312,8.881784197e-16,PASS
7,T08_REPLAY_SCORE_RECURSION,LIKELIHOOD_IMPLEMENTATION,1,1,0,PASS


,model_id,negative_log_likelihood,gradient_dimension,maximum_absolute_error,maximum_relative_error,gradient_check_passed,status
0,H1_DIAGONAL_SHARED_DECAY,42.1414169,5,1.385304649e-09,1.385304649e-09,True,PASS
1,H2_DIAGONAL_SEPARATE_DECAY,42.14294895,6,2.743691718e-09,2.743691718e-09,True,PASS


The exact strict-pre-batch Hawkes likelihood, compensator, state recursion, Poisson limit, simultaneous-batch update, and optimizer-space gradient passed deterministic validation. The engine also traversed the complete DEVELOPMENT interval without losing events or batches. The smoke parameters are provisional and are not fitted results. Protected partitions remain unopened and no filesystem writes were performed.


In [6]:
# ============================================================
# Left-edge sensitivity and DEVELOPMENT-to-CALIBRATION history contract
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("HAWKES_LIKELIHOOD_ENGINE_VALIDATED", False)),
    "The Hawkes likelihood engine has not been validated.",
)
require(
    bool(globals().get("HAWKES_EXACT_GRADIENT_VALIDATED", False)),
    "The Hawkes gradient has not been validated.",
)
require(
    bool(
        globals().get(
            "HAWKES_DEVELOPMENT_SMOKE_REPLAY_PASSED",
            False,
        )
    ),
    "The complete DEVELOPMENT smoke replay has not passed.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Frozen boundary definitions
# ------------------------------------------------------------

calibration_contract_row = window_lookup.loc[
    LOCKED_EVALUATION_PARTITION
]

CALIBRATION_START_NS: Final[int] = int(
    calibration_contract_row["contract_start_ns"]
)
CALIBRATION_END_EXCLUSIVE_NS: Final[int] = int(
    calibration_contract_row["contract_end_exclusive_ns"]
)
CALIBRATION_DURATION_SECONDS: Final[float] = (
    int(calibration_contract_row["contract_duration_ns"])
    / NANOSECONDS_PER_SECOND
)

require(
    DEVELOPMENT_END_EXCLUSIVE_NS == CALIBRATION_START_NS,
    (
        "The DEVELOPMENT-to-CALIBRATION boundary is not exactly "
        "contiguous."
    ),
)

CALIBRATION_BATCHES_FOR_HAWKES = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            "event_partition"
        ].eq(LOCKED_EVALUATION_PARTITION),
        [
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

ANALYTICAL_BATCHES_FOR_HAWKES = (
    pd.concat(
        [
            DEVELOPMENT_BATCHES_FOR_HAWKES,
            CALIBRATION_BATCHES_FOR_HAWKES,
        ],
        axis=0,
        ignore_index=True,
    )
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    len(CALIBRATION_BATCHES_FOR_HAWKES)
    == EXPECTED_CALIBRATION_BATCH_ROWS,
    "The CALIBRATION Hawkes batch view has the wrong row count.",
)
require(
    int(
        CALIBRATION_BATCHES_FOR_HAWKES[
            ["buy_event_count", "sell_event_count"]
        ]
        .to_numpy(dtype=np.int64)
        .sum()
    )
    == EXPECTED_CALIBRATION_EVENT_ROWS,
    "The CALIBRATION Hawkes batch view does not conserve events.",
)
require(
    len(ANALYTICAL_BATCHES_FOR_HAWKES)
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "The combined analytical Hawkes batch view has the wrong row count.",
)
require(
    CALIBRATION_PARAMETER_UPDATES == 0,
    "The frozen contract requires zero CALIBRATION parameter updates.",
)


# ------------------------------------------------------------
# History-prefix and scoring-window objects
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class LeftEdgeSensitivityWindow:
    prefix_seconds: float
    observation_start_ns: int
    scoring_start_ns: int
    observation_end_exclusive_ns: int
    history_batch_count: int
    history_event_count: int
    scoring_batch_count: int
    scoring_event_count: int
    left_censoring_recorded: bool
    fabricated_prehistory: bool


@dataclass(frozen=True, slots=True)
class LeftEdgePreparedReplay:
    window: LeftEdgeSensitivityWindow
    history_batches: pd.DataFrame
    scoring_batches: pd.DataFrame
    initial_scoring_state: DiagonalHawkesState


def batch_event_count(
    batch_frame: pd.DataFrame,
) -> int:
    """Return the conserved event count of an exact-time batch table."""
    if batch_frame.empty:
        return 0

    return int(
        batch_frame[
            ["buy_event_count", "sell_event_count"]
        ]
        .to_numpy(dtype=np.int64)
        .sum()
    )


def prepare_left_edge_replay(
    development_batches: pd.DataFrame,
    parameters: DiagonalHawkesParameters,
    *,
    prefix_seconds: float,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
) -> LeftEdgePreparedReplay:
    """
    Build a DEVELOPMENT scoring window after a history-only prefix.

    Events before scoring_start_ns update the Hawkes state but do not
    contribute to the returned scoring likelihood.
    """
    require(
        math.isfinite(prefix_seconds)
        and prefix_seconds >= 0.0,
        "History-only prefix must be finite and nonnegative.",
    )

    start_ns = int(observation_start_ns)
    end_ns = int(observation_end_exclusive_ns)

    prefix_ns = int(
        round(prefix_seconds * NANOSECONDS_PER_SECOND)
    )
    scoring_start_ns = start_ns + prefix_ns

    require(
        scoring_start_ns < end_ns,
        (
            "The left-edge history prefix consumes the entire "
            "DEVELOPMENT interval."
        ),
    )

    validated_batches = validate_hawkes_batch_input(
        development_batches,
        observation_start_ns=start_ns,
        observation_end_exclusive_ns=end_ns,
    )

    history_batches = (
        validated_batches.loc[
            validated_batches["event_time_ns"].lt(
                scoring_start_ns
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    scoring_batches = (
        validated_batches.loc[
            validated_batches["event_time_ns"].ge(
                scoring_start_ns
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    require(
        not scoring_batches.empty,
        (
            "The left-edge sensitivity scoring window contains "
            "no event batches."
        ),
    )

    if prefix_ns == 0:
        require(
            history_batches.empty,
            "A zero-second prefix unexpectedly contains history.",
        )

        initial_scoring_state = DiagonalHawkesState(
            state_time_ns=scoring_start_ns,
            buy_excitation=0.0,
            sell_excitation=0.0,
        )
    else:
        history_replay = replay_diagonal_hawkes(
            history_batches,
            parameters,
            observation_start_ns=start_ns,
            observation_end_exclusive_ns=scoring_start_ns,
            initial_state=None,
            return_replay=False,
        )

        initial_scoring_state = history_replay.final_state

    require(
        initial_scoring_state.state_time_ns
        == scoring_start_ns,
        "Prepared scoring state is expressed at the wrong time.",
    )
    require(
        initial_scoring_state.buy_excitation >= 0.0,
        "Prepared BUY excitation is negative.",
    )
    require(
        initial_scoring_state.sell_excitation >= 0.0,
        "Prepared SELL excitation is negative.",
    )

    window = LeftEdgeSensitivityWindow(
        prefix_seconds=float(prefix_seconds),
        observation_start_ns=start_ns,
        scoring_start_ns=scoring_start_ns,
        observation_end_exclusive_ns=end_ns,
        history_batch_count=len(history_batches),
        history_event_count=batch_event_count(
            history_batches
        ),
        scoring_batch_count=len(scoring_batches),
        scoring_event_count=batch_event_count(
            scoring_batches
        ),
        left_censoring_recorded=True,
        fabricated_prehistory=False,
    )

    require(
        window.history_event_count
        + window.scoring_event_count
        == EXPECTED_DEVELOPMENT_EVENT_ROWS,
        (
            "Left-edge prefix and scoring windows do not conserve "
            "DEVELOPMENT events."
        ),
    )
    require(
        window.history_batch_count
        + window.scoring_batch_count
        == EXPECTED_DEVELOPMENT_BATCH_ROWS,
        (
            "Left-edge prefix and scoring windows do not conserve "
            "DEVELOPMENT batches."
        ),
    )

    return LeftEdgePreparedReplay(
        window=window,
        history_batches=history_batches,
        scoring_batches=scoring_batches,
        initial_scoring_state=initial_scoring_state,
    )


def score_prepared_left_edge_replay(
    prepared_replay: LeftEdgePreparedReplay,
    parameters: DiagonalHawkesParameters,
) -> DiagonalHawkesReplayResult:
    """Score only the post-prefix DEVELOPMENT interval."""
    return replay_diagonal_hawkes(
        prepared_replay.scoring_batches,
        parameters,
        observation_start_ns=(
            prepared_replay.window.scoring_start_ns
        ),
        observation_end_exclusive_ns=(
            prepared_replay.window
            .observation_end_exclusive_ns
        ),
        initial_state=(
            prepared_replay.initial_scoring_state
        ),
        return_replay=False,
    )


# ------------------------------------------------------------
# Build frozen left-edge sensitivity windows
#
# The provisional parameters are used only to validate state
# preparation and split-window mechanics. Candidate fitting later
# repeats these audits with fitted parameters.
# ------------------------------------------------------------

LEFT_EDGE_PREPARED_REPLAYS: dict[
    float,
    LeftEdgePreparedReplay,
] = {}

left_edge_contract_rows: list[dict[str, Any]] = []

for prefix_seconds in LEFT_EDGE_HISTORY_ONLY_SECONDS:
    prepared_replay = prepare_left_edge_replay(
        DEVELOPMENT_BATCHES_FOR_HAWKES,
        PROVISIONAL_SMOKE_PARAMETERS,
        prefix_seconds=prefix_seconds,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
    )

    scoring_result = score_prepared_left_edge_replay(
        prepared_replay,
        PROVISIONAL_SMOKE_PARAMETERS,
    )

    LEFT_EDGE_PREPARED_REPLAYS[
        float(prefix_seconds)
    ] = prepared_replay

    scoring_duration_seconds = (
        prepared_replay.window
        .observation_end_exclusive_ns
        - prepared_replay.window.scoring_start_ns
    ) / NANOSECONDS_PER_SECOND

    require(
        scoring_duration_seconds > 0.0,
        "A left-edge scoring duration is nonpositive.",
    )
    require(
        scoring_result.event_count
        == prepared_replay.window.scoring_event_count,
        "A left-edge replay lost scoring events.",
    )
    require(
        scoring_result.batch_count
        == prepared_replay.window.scoring_batch_count,
        "A left-edge replay lost scoring batches.",
    )
    require(
        math.isfinite(scoring_result.log_likelihood),
        "A left-edge replay produced a non-finite likelihood.",
    )

    left_edge_contract_rows.append(
        {
            "prefix_seconds": prefix_seconds,
            "scoring_start_ns": (
                prepared_replay.window.scoring_start_ns
            ),
            "history_batch_count": (
                prepared_replay.window.history_batch_count
            ),
            "history_event_count": (
                prepared_replay.window.history_event_count
            ),
            "scoring_batch_count": (
                prepared_replay.window.scoring_batch_count
            ),
            "scoring_event_count": (
                prepared_replay.window.scoring_event_count
            ),
            "initial_buy_excitation": (
                prepared_replay
                .initial_scoring_state
                .buy_excitation
            ),
            "initial_sell_excitation": (
                prepared_replay
                .initial_scoring_state
                .sell_excitation
            ),
            "scoring_duration_seconds": (
                scoring_duration_seconds
            ),
            "provisional_log_likelihood": (
                scoring_result.log_likelihood
            ),
            "provisional_log_score_per_event": (
                scoring_result.log_likelihood
                / scoring_result.event_count
            ),
            "left_censoring_recorded": True,
            "fabricated_prehistory": False,
            "status": "PASS",
        }
    )

NOTEBOOK07_LEFT_EDGE_CONTRACT_AUDIT = pd.DataFrame(
    left_edge_contract_rows
)


# ------------------------------------------------------------
# DEVELOPMENT-to-CALIBRATION inherited-state replay
# ------------------------------------------------------------

DEVELOPMENT_BOUNDARY_REPLAY = replay_diagonal_hawkes(
    DEVELOPMENT_BATCHES_FOR_HAWKES,
    PROVISIONAL_SMOKE_PARAMETERS,
    observation_start_ns=DEVELOPMENT_START_NS,
    observation_end_exclusive_ns=(
        DEVELOPMENT_END_EXCLUSIVE_NS
    ),
    initial_state=None,
    return_replay=False,
)

require(
    DEVELOPMENT_BOUNDARY_REPLAY.final_state.state_time_ns
    == CALIBRATION_START_NS,
    (
        "The terminal DEVELOPMENT state is not expressed at the "
        "CALIBRATION boundary."
    ),
)

CALIBRATION_INHERITED_REPLAY = replay_diagonal_hawkes(
    CALIBRATION_BATCHES_FOR_HAWKES,
    PROVISIONAL_SMOKE_PARAMETERS,
    observation_start_ns=CALIBRATION_START_NS,
    observation_end_exclusive_ns=(
        CALIBRATION_END_EXCLUSIVE_NS
    ),
    initial_state=(
        DEVELOPMENT_BOUNDARY_REPLAY.final_state
    ),
    return_replay=False,
)

ANALYTICAL_CONTINUOUS_REPLAY = replay_diagonal_hawkes(
    ANALYTICAL_BATCHES_FOR_HAWKES,
    PROVISIONAL_SMOKE_PARAMETERS,
    observation_start_ns=DEVELOPMENT_START_NS,
    observation_end_exclusive_ns=(
        CALIBRATION_END_EXCLUSIVE_NS
    ),
    initial_state=None,
    return_replay=False,
)


# ------------------------------------------------------------
# Split-versus-continuous replay equivalence
# ------------------------------------------------------------

split_log_likelihood = (
    DEVELOPMENT_BOUNDARY_REPLAY.log_likelihood
    + CALIBRATION_INHERITED_REPLAY.log_likelihood
)

split_log_event_term = (
    DEVELOPMENT_BOUNDARY_REPLAY.log_event_term
    + CALIBRATION_INHERITED_REPLAY.log_event_term
)

split_compensator_buy = (
    DEVELOPMENT_BOUNDARY_REPLAY.compensator_buy
    + CALIBRATION_INHERITED_REPLAY.compensator_buy
)

split_compensator_sell = (
    DEVELOPMENT_BOUNDARY_REPLAY.compensator_sell
    + CALIBRATION_INHERITED_REPLAY.compensator_sell
)

require(
    math.isclose(
        split_log_likelihood,
        ANALYTICAL_CONTINUOUS_REPLAY.log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "Inherited-state split likelihood does not match the "
        "continuous analytical replay."
    ),
)
require(
    math.isclose(
        split_log_event_term,
        ANALYTICAL_CONTINUOUS_REPLAY.log_event_term,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "Inherited-state split event term does not match the "
        "continuous analytical replay."
    ),
)
require(
    math.isclose(
        split_compensator_buy,
        ANALYTICAL_CONTINUOUS_REPLAY.compensator_buy,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "Inherited-state split BUY compensator does not match the "
        "continuous analytical replay."
    ),
)
require(
    math.isclose(
        split_compensator_sell,
        ANALYTICAL_CONTINUOUS_REPLAY.compensator_sell,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "Inherited-state split SELL compensator does not match the "
        "continuous analytical replay."
    ),
)

require(
    math.isclose(
        CALIBRATION_INHERITED_REPLAY
        .final_state
        .buy_excitation,
        ANALYTICAL_CONTINUOUS_REPLAY
        .final_state
        .buy_excitation,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "Inherited-state split BUY terminal state does not match "
        "the continuous replay."
    ),
)
require(
    math.isclose(
        CALIBRATION_INHERITED_REPLAY
        .final_state
        .sell_excitation,
        ANALYTICAL_CONTINUOUS_REPLAY
        .final_state
        .sell_excitation,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "Inherited-state split SELL terminal state does not match "
        "the continuous replay."
    ),
)

require(
    (
        DEVELOPMENT_BOUNDARY_REPLAY.event_count
        + CALIBRATION_INHERITED_REPLAY.event_count
    )
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "The split replay does not conserve analytical events.",
)
require(
    (
        DEVELOPMENT_BOUNDARY_REPLAY.batch_count
        + CALIBRATION_INHERITED_REPLAY.batch_count
    )
    == EXPECTED_ANALYTICAL_BATCH_ROWS,
    "The split replay does not conserve analytical batches.",
)
require(
    CALIBRATION_INHERITED_REPLAY.event_count
    == EXPECTED_CALIBRATION_EVENT_ROWS,
    "The inherited CALIBRATION replay lost events.",
)
require(
    CALIBRATION_INHERITED_REPLAY.batch_count
    == EXPECTED_CALIBRATION_BATCH_ROWS,
    "The inherited CALIBRATION replay lost batches.",
)


# ------------------------------------------------------------
# Demonstrate that resetting at CALIBRATION is a distinct,
# prohibited replay path
# ------------------------------------------------------------

CALIBRATION_RESET_REPLAY_FOR_AUDIT_ONLY = (
    replay_diagonal_hawkes(
        CALIBRATION_BATCHES_FOR_HAWKES,
        PROVISIONAL_SMOKE_PARAMETERS,
        observation_start_ns=CALIBRATION_START_NS,
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        initial_state=None,
        return_replay=False,
    )
)

calibration_reset_log_likelihood_difference = (
    CALIBRATION_RESET_REPLAY_FOR_AUDIT_ONLY.log_likelihood
    - CALIBRATION_INHERITED_REPLAY.log_likelihood
)

calibration_boundary_state_nonzero = bool(
    DEVELOPMENT_BOUNDARY_REPLAY
    .final_state
    .buy_excitation
    > FLOAT_ABSOLUTE_TOLERANCE
    or DEVELOPMENT_BOUNDARY_REPLAY
    .final_state
    .sell_excitation
    > FLOAT_ABSOLUTE_TOLERANCE
)

require(
    calibration_boundary_state_nonzero,
    (
        "The provisional DEVELOPMENT boundary state is unexpectedly "
        "zero on both sides; the inherited-history test is not "
        "informative."
    ),
)
require(
    not math.isclose(
        CALIBRATION_RESET_REPLAY_FOR_AUDIT_ONLY
        .log_likelihood,
        CALIBRATION_INHERITED_REPLAY.log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "Reset and inherited CALIBRATION replays unexpectedly "
        "produce indistinguishable likelihoods."
    ),
)


# ------------------------------------------------------------
# Boundary contract audit
# ------------------------------------------------------------

NOTEBOOK07_BOUNDARY_HISTORY_AUDIT = pd.DataFrame(
    [
        {
            "audit_item": "DEVELOPMENT_LEFT_BOUNDARY",
            "required_behavior": (
                "ZERO_INITIAL_STATE_WITH_LEFT_CENSORING_RECORDED"
            ),
            "observed_behavior": (
                "ZERO_INITIAL_STATE_WITH_LEFT_CENSORING_RECORDED"
            ),
            "passed": True,
            "status": "PASS",
        },
        {
            "audit_item": "FABRICATED_PREHISTORY",
            "required_behavior": "FORBIDDEN",
            "observed_behavior": "NOT_USED",
            "passed": True,
            "status": "PASS",
        },
        {
            "audit_item": "DEVELOPMENT_CALIBRATION_CONTIGUITY",
            "required_behavior": "EXACT_CONTIGUOUS_BOUNDARY",
            "observed_behavior": (
                "EXACT_CONTIGUOUS_BOUNDARY"
            ),
            "passed": True,
            "status": "PASS",
        },
        {
            "audit_item": "CALIBRATION_INITIAL_STATE",
            "required_behavior": (
                "INHERIT_TERMINAL_DEVELOPMENT_STATE"
            ),
            "observed_behavior": (
                "INHERIT_TERMINAL_DEVELOPMENT_STATE"
            ),
            "passed": True,
            "status": "PASS",
        },
        {
            "audit_item": "CALIBRATION_PARAMETER_UPDATES",
            "required_behavior": "ZERO",
            "observed_behavior": str(
                CALIBRATION_PARAMETER_UPDATES
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES == 0
            ),
            "status": "PASS",
        },
        {
            "audit_item": "SPLIT_CONTINUOUS_LIKELIHOOD",
            "required_behavior": "NUMERICALLY_IDENTICAL",
            "observed_behavior": (
                "NUMERICALLY_IDENTICAL"
            ),
            "passed": True,
            "status": "PASS",
        },
        {
            "audit_item": "SPLIT_CONTINUOUS_TERMINAL_STATE",
            "required_behavior": "NUMERICALLY_IDENTICAL",
            "observed_behavior": (
                "NUMERICALLY_IDENTICAL"
            ),
            "passed": True,
            "status": "PASS",
        },
        {
            "audit_item": "CALIBRATION_RESET_PATH",
            "required_behavior": "PROHIBITED",
            "observed_behavior": (
                "COMPUTED_FOR_AUDIT_ONLY_AND_NOT_AUTHORIZED"
            ),
            "passed": True,
            "status": "PASS",
        },
    ]
)

require(
    NOTEBOOK07_BOUNDARY_HISTORY_AUDIT[
        "passed"
    ].all(),
    "At least one boundary-history contract item failed.",
)


# ------------------------------------------------------------
# Frozen history-treatment state
# ------------------------------------------------------------

LEFT_EDGE_SENSITIVITY_CONTRACT_FROZEN: bool = True
DEVELOPMENT_CALIBRATION_HISTORY_CONTRACT_VALIDATED: bool = True
CALIBRATION_RESET_AUTHORIZED: bool = False
CALIBRATION_FITTING_AUTHORIZED: bool = False
FILESYSTEM_WRITES_PERFORMED = False

boundary_summary = pd.DataFrame(
    [
        {
            "field": "left_edge_sensitivity_contract_frozen",
            "value": (
                LEFT_EDGE_SENSITIVITY_CONTRACT_FROZEN
            ),
        },
        {
            "field": (
                "development_calibration_history_contract_validated"
            ),
            "value": (
                DEVELOPMENT_CALIBRATION_HISTORY_CONTRACT_VALIDATED
            ),
        },
        {
            "field": "development_start_ns",
            "value": DEVELOPMENT_START_NS,
        },
        {
            "field": "development_end_exclusive_ns",
            "value": DEVELOPMENT_END_EXCLUSIVE_NS,
        },
        {
            "field": "calibration_start_ns",
            "value": CALIBRATION_START_NS,
        },
        {
            "field": "calibration_end_exclusive_ns",
            "value": CALIBRATION_END_EXCLUSIVE_NS,
        },
        {
            "field": "boundary_buy_excitation",
            "value": (
                DEVELOPMENT_BOUNDARY_REPLAY
                .final_state
                .buy_excitation
            ),
        },
        {
            "field": "boundary_sell_excitation",
            "value": (
                DEVELOPMENT_BOUNDARY_REPLAY
                .final_state
                .sell_excitation
            ),
        },
        {
            "field": "split_log_likelihood",
            "value": split_log_likelihood,
        },
        {
            "field": "continuous_log_likelihood",
            "value": (
                ANALYTICAL_CONTINUOUS_REPLAY.log_likelihood
            ),
        },
        {
            "field": (
                "reset_minus_inherited_calibration_log_likelihood"
            ),
            "value": (
                calibration_reset_log_likelihood_difference
            ),
        },
        {
            "field": "calibration_reset_authorized",
            "value": CALIBRATION_RESET_AUTHORIZED,
        },
        {
            "field": "calibration_fitting_authorized",
            "value": CALIBRATION_FITTING_AUTHORIZED,
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(boundary_summary)
display(NOTEBOOK07_LEFT_EDGE_CONTRACT_AUDIT)
display(NOTEBOOK07_BOUNDARY_HISTORY_AUDIT)

print(
    "The DEVELOPMENT left-edge treatment, history-only sensitivity "
    "windows, exact DEVELOPMENT-to-CALIBRATION state transfer, and "
    "zero-update CALIBRATION contract are frozen and validated. "
    "Inherited-state split replay matches one continuous analytical "
    "replay, while a CALIBRATION reset produces a distinct and "
    "prohibited result. All values in this cell use provisional "
    "smoke parameters only; no candidate has been fitted or selected. "
    "Protected partitions remain unopened and no filesystem writes "
    "were performed."
)

,field,value
0,left_edge_sensitivity_contract_frozen,True
1,development_calibration_history_contract_valid...,True
2,development_start_ns,1783665467531985400
3,development_end_exclusive_ns,1783667269391572100
4,calibration_start_ns,1783667269391572100
5,calibration_end_exclusive_ns,1783667989690751200
6,boundary_buy_excitation,0.1491959403
7,boundary_sell_excitation,0.7000046537
8,split_log_likelihood,"-2,560.219411"
9,continuous_log_likelihood,"-2,560.219411"


,prefix_seconds,scoring_start_ns,history_batch_count,history_event_count,scoring_batch_count,scoring_event_count,initial_buy_excitation,initial_sell_excitation,scoring_duration_seconds,provisional_log_likelihood,provisional_log_score_per_event,left_censoring_recorded,fabricated_prehistory,status
0,0,1783665467531985400,0,0,6859,7004,0,0,"1,801.859587","-1,717.274244",-0.2451847864,True,False,PASS
1,1,1783665468531985400,0,0,6859,7004,0,0,"1,800.859587","-1,714.164568",-0.2447408006,True,False,PASS
2,5,1783665472531985400,12,12,6847,6992,0.3347141942,0.3327179795,"1,796.859587","-1,706.520556",-0.2440675853,True,False,PASS


,audit_item,required_behavior,observed_behavior,passed,status
0,DEVELOPMENT_LEFT_BOUNDARY,ZERO_INITIAL_STATE_WITH_LEFT_CENSORING_RECORDED,ZERO_INITIAL_STATE_WITH_LEFT_CENSORING_RECORDED,True,PASS
1,FABRICATED_PREHISTORY,FORBIDDEN,NOT_USED,True,PASS
2,DEVELOPMENT_CALIBRATION_CONTIGUITY,EXACT_CONTIGUOUS_BOUNDARY,EXACT_CONTIGUOUS_BOUNDARY,True,PASS
3,CALIBRATION_INITIAL_STATE,INHERIT_TERMINAL_DEVELOPMENT_STATE,INHERIT_TERMINAL_DEVELOPMENT_STATE,True,PASS
4,CALIBRATION_PARAMETER_UPDATES,ZERO,0,True,PASS
5,SPLIT_CONTINUOUS_LIKELIHOOD,NUMERICALLY_IDENTICAL,NUMERICALLY_IDENTICAL,True,PASS
6,SPLIT_CONTINUOUS_TERMINAL_STATE,NUMERICALLY_IDENTICAL,NUMERICALLY_IDENTICAL,True,PASS
7,CALIBRATION_RESET_PATH,PROHIBITED,COMPUTED_FOR_AUDIT_ONLY_AND_NOT_AUTHORIZED,True,PASS


The DEVELOPMENT left-edge treatment, history-only sensitivity windows, exact DEVELOPMENT-to-CALIBRATION state transfer, and zero-update CALIBRATION contract are frozen and validated. Inherited-state split replay matches one continuous analytical replay, while a CALIBRATION reset produces a distinct and prohibited result. All values in this cell use provisional smoke parameters only; no candidate has been fitted or selected. Protected partitions remain unopened and no filesystem writes were performed.


In [7]:
# ============================================================
# Deterministic multi-start design and DEVELOPMENT pre-screen
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "LEFT_EDGE_SENSITIVITY_CONTRACT_FROZEN",
            False,
        )
    ),
    "The left-edge sensitivity contract has not been frozen.",
)
require(
    bool(
        globals().get(
            "DEVELOPMENT_CALIBRATION_HISTORY_CONTRACT_VALIDATED",
            False,
        )
    ),
    "The DEVELOPMENT-to-CALIBRATION history contract is invalid.",
)
require(
    bool(globals().get("HAWKES_EXACT_GRADIENT_VALIDATED", False)),
    "The exact Hawkes gradient has not been validated.",
)
require(
    CALIBRATION_FITTING_AUTHORIZED is False,
    "CALIBRATION must not be used for fitting.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Frozen deterministic starting-value design
# ------------------------------------------------------------

GLOBAL_OBJECTIVE_STARTS_PER_MODEL: Final[int] = 12

KAPPA_START_PAIRS: Final[
    tuple[tuple[float, float], ...]
] = tuple(
    (float(kappa_buy), float(kappa_sell))
    for kappa_buy in BRANCHING_MASS_STARTS
    for kappa_sell in BRANCHING_MASS_STARTS
)

H1_HALF_LIFE_PAIRS_MS: Final[
    tuple[tuple[float, float], ...]
] = tuple(
    (float(half_life_ms), float(half_life_ms))
    for half_life_ms in HALF_LIFE_STARTS_MS
)

H2_HALF_LIFE_PAIRS_MS: Final[
    tuple[tuple[float, float], ...]
] = tuple(
    (
        float(half_life_buy_ms),
        float(half_life_sell_ms),
    )
    for half_life_buy_ms in HALF_LIFE_STARTS_MS
    for half_life_sell_ms in HALF_LIFE_STARTS_MS
)

require(
    len(KAPPA_START_PAIRS)
    == len(BRANCHING_MASS_STARTS) ** 2,
    "The branching-mass start grid is incomplete.",
)
require(
    len(H1_HALF_LIFE_PAIRS_MS)
    == len(HALF_LIFE_STARTS_MS),
    "The H1 decay-start grid is incomplete.",
)
require(
    len(H2_HALF_LIFE_PAIRS_MS)
    == len(HALF_LIFE_STARTS_MS) ** 2,
    "The H2 decay-start grid is incomplete.",
)


# ------------------------------------------------------------
# Empirical DEVELOPMENT rate anchors
# ------------------------------------------------------------

DEVELOPMENT_BUY_EVENT_COUNT: Final[int] = int(
    DEVELOPMENT_BATCHES_FOR_HAWKES[
        "buy_event_count"
    ].sum()
)

DEVELOPMENT_SELL_EVENT_COUNT: Final[int] = int(
    DEVELOPMENT_BATCHES_FOR_HAWKES[
        "sell_event_count"
    ].sum()
)

DEVELOPMENT_EMPIRICAL_BUY_RATE: Final[float] = (
    DEVELOPMENT_BUY_EVENT_COUNT
    / DEVELOPMENT_DURATION_SECONDS
)

DEVELOPMENT_EMPIRICAL_SELL_RATE: Final[float] = (
    DEVELOPMENT_SELL_EVENT_COUNT
    / DEVELOPMENT_DURATION_SECONDS
)

require(
    DEVELOPMENT_BUY_EVENT_COUNT
    == int(
        NOTEBOOK07_ANALYTICAL_PARTITION_RECONCILIATION.loc[
            NOTEBOOK07_ANALYTICAL_PARTITION_RECONCILIATION[
                "event_partition"
            ].eq(FIT_PARTITION),
            "buy_event_count",
        ].iloc[0]
    ),
    "The DEVELOPMENT BUY count does not reconcile.",
)
require(
    DEVELOPMENT_SELL_EVENT_COUNT
    == int(
        NOTEBOOK07_ANALYTICAL_PARTITION_RECONCILIATION.loc[
            NOTEBOOK07_ANALYTICAL_PARTITION_RECONCILIATION[
                "event_partition"
            ].eq(FIT_PARTITION),
            "sell_event_count",
        ].iloc[0]
    ),
    "The DEVELOPMENT SELL count does not reconcile.",
)
require(
    DEVELOPMENT_EMPIRICAL_BUY_RATE > 0.0,
    "The DEVELOPMENT empirical BUY rate is nonpositive.",
)
require(
    DEVELOPMENT_EMPIRICAL_SELL_RATE > 0.0,
    "The DEVELOPMENT empirical SELL rate is nonpositive.",
)


# ------------------------------------------------------------
# Starting-value helpers
# ------------------------------------------------------------

def half_life_ms_to_beta_per_second(
    half_life_ms: float,
) -> float:
    """Convert a strictly positive half-life in milliseconds to beta."""
    require(
        math.isfinite(half_life_ms)
        and half_life_ms > 0.0,
        "Half-life must be finite and strictly positive.",
    )

    half_life_seconds = half_life_ms / 1_000.0

    return math.log(2.0) / half_life_seconds


def stationary_rate_anchored_mu(
    empirical_rate: float,
    kappa: float,
) -> float:
    """
    Choose a base intensity whose stationary mean equals the
    empirical DEVELOPMENT event rate at the proposed kappa.
    """
    require(
        math.isfinite(empirical_rate)
        and empirical_rate > 0.0,
        "Empirical event rate must be finite and positive.",
    )
    require(
        math.isfinite(kappa)
        and 0.0 <= kappa < 1.0,
        "Starting excitation mass must lie in [0, 1).",
    )

    anchored_mu = empirical_rate * (1.0 - kappa)

    return max(
        float(anchored_mu),
        PARAMETER_POSITIVITY_FLOOR * 10.0,
    )


def kappa_pair_identifier(
    kappa_buy: float,
    kappa_sell: float,
) -> str:
    """Return a deterministic branching-mass pair identifier."""
    return (
        f"KB{kappa_buy:.2f}_"
        f"KS{kappa_sell:.2f}"
    )


def half_life_pair_identifier(
    half_life_buy_ms: float,
    half_life_sell_ms: float,
) -> str:
    """Return a deterministic half-life pair identifier."""
    return (
        f"HLB{half_life_buy_ms:g}MS_"
        f"HLS{half_life_sell_ms:g}MS"
    )


def excitation_regime(
    kappa_buy: float,
    kappa_sell: float,
) -> str:
    """Classify a starting excitation pair for audit summaries."""
    maximum_kappa = max(kappa_buy, kappa_sell)

    if maximum_kappa <= 0.05:
        return "NEAR_POISSON"
    if maximum_kappa <= 0.25:
        return "LOW_EXCITATION"
    if maximum_kappa <= 0.60:
        return "MODERATE_EXCITATION"

    return "HIGH_NEAR_CRITICAL_START"


def decay_regime(
    half_life_buy_ms: float,
    half_life_sell_ms: float,
) -> str:
    """Classify a starting half-life pair by geometric mean."""
    geometric_mean_ms = math.sqrt(
        half_life_buy_ms * half_life_sell_ms
    )

    if geometric_mean_ms <= 10.0:
        return "VERY_FAST"
    if geometric_mean_ms <= 100.0:
        return "FAST"
    if geometric_mean_ms <= 500.0:
        return "MEDIUM"
    if geometric_mean_ms <= 2_000.0:
        return "SLOW"

    return "VERY_SLOW"


def construct_start_parameters(
    *,
    model_id: str,
    kappa_buy: float,
    kappa_sell: float,
    half_life_buy_ms: float,
    half_life_sell_ms: float,
) -> DiagonalHawkesParameters:
    """Construct one stationary-rate-anchored natural start."""
    require(
        model_id in HAWKES_CANDIDATE_SPECIFICATIONS,
        f"Unknown candidate model: {model_id}",
    )

    specification = HAWKES_CANDIDATE_SPECIFICATIONS[
        model_id
    ]

    if specification.shared_decay:
        require(
            math.isclose(
                half_life_buy_ms,
                half_life_sell_ms,
                rel_tol=0.0,
                abs_tol=0.0,
            ),
            "H1 requires one shared starting half-life.",
        )

    beta_buy = half_life_ms_to_beta_per_second(
        half_life_buy_ms
    )
    beta_sell = half_life_ms_to_beta_per_second(
        half_life_sell_ms
    )

    parameters = DiagonalHawkesParameters(
        model_id=model_id,
        mu_buy=stationary_rate_anchored_mu(
            DEVELOPMENT_EMPIRICAL_BUY_RATE,
            kappa_buy,
        ),
        mu_sell=stationary_rate_anchored_mu(
            DEVELOPMENT_EMPIRICAL_SELL_RATE,
            kappa_sell,
        ),
        kappa_buy=float(kappa_buy),
        kappa_sell=float(kappa_sell),
        beta_buy=beta_buy,
        beta_sell=beta_sell,
    )

    validation = validate_hawkes_parameters(
        parameters
    )

    require(
        validation["mathematically_admissible"],
        (
            "Constructed starting parameters are inadmissible: "
            f"{validation}"
        ),
    )

    return parameters


# ------------------------------------------------------------
# Generate and DEVELOPMENT-score the complete start grid
# ------------------------------------------------------------

candidate_half_life_grids: Final[
    Mapping[str, tuple[tuple[float, float], ...]]
] = {
    H1_MODEL_ID: H1_HALF_LIFE_PAIRS_MS,
    H2_MODEL_ID: H2_HALF_LIFE_PAIRS_MS,
}

start_records: list[dict[str, Any]] = []
model_start_counters: dict[str, int] = {
    model_id: 0
    for model_id in CANDIDATE_MODEL_IDS
}

for model_id in CANDIDATE_MODEL_IDS:
    for (
        kappa_buy,
        kappa_sell,
    ) in KAPPA_START_PAIRS:
        for (
            half_life_buy_ms,
            half_life_sell_ms,
        ) in candidate_half_life_grids[model_id]:
            model_start_counters[model_id] += 1

            start_number = model_start_counters[
                model_id
            ]

            start_id = (
                f"{model_id}__START_{start_number:04d}"
            )

            natural_parameters = construct_start_parameters(
                model_id=model_id,
                kappa_buy=kappa_buy,
                kappa_sell=kappa_sell,
                half_life_buy_ms=half_life_buy_ms,
                half_life_sell_ms=half_life_sell_ms,
            )

            optimizer_vector = encode_hawkes_parameters(
                natural_parameters
            )

            (
                initial_negative_log_likelihood,
                initial_gradient,
            ) = negative_log_likelihood_and_gradient(
                optimizer_vector,
                model_id=model_id,
                batch_frame=(
                    DEVELOPMENT_BATCHES_FOR_HAWKES
                ),
                observation_start_ns=(
                    DEVELOPMENT_START_NS
                ),
                observation_end_exclusive_ns=(
                    DEVELOPMENT_END_EXCLUSIVE_NS
                ),
            )

            initial_gradient_norm = float(
                np.linalg.norm(
                    initial_gradient,
                    ord=2,
                )
            )

            require(
                math.isfinite(
                    initial_negative_log_likelihood
                ),
                (
                    f"{start_id} produced a non-finite "
                    "DEVELOPMENT objective."
                ),
            )
            require(
                math.isfinite(initial_gradient_norm),
                (
                    f"{start_id} produced a non-finite "
                    "DEVELOPMENT gradient norm."
                ),
            )

            start_payload = {
                "start_id": start_id,
                "model_id": model_id,
                "mu_buy": natural_parameters.mu_buy,
                "mu_sell": natural_parameters.mu_sell,
                "kappa_buy": (
                    natural_parameters.kappa_buy
                ),
                "kappa_sell": (
                    natural_parameters.kappa_sell
                ),
                "beta_buy": natural_parameters.beta_buy,
                "beta_sell": natural_parameters.beta_sell,
                "half_life_buy_ms": (
                    half_life_buy_ms
                ),
                "half_life_sell_ms": (
                    half_life_sell_ms
                ),
                "optimizer_vector": [
                    float(value)
                    for value in optimizer_vector
                ],
            }

            start_records.append(
                {
                    "start_id": start_id,
                    "model_id": model_id,
                    "start_number": start_number,
                    "kappa_pair_id": (
                        kappa_pair_identifier(
                            kappa_buy,
                            kappa_sell,
                        )
                    ),
                    "half_life_pair_id": (
                        half_life_pair_identifier(
                            half_life_buy_ms,
                            half_life_sell_ms,
                        )
                    ),
                    "excitation_regime": excitation_regime(
                        kappa_buy,
                        kappa_sell,
                    ),
                    "decay_regime": decay_regime(
                        half_life_buy_ms,
                        half_life_sell_ms,
                    ),
                    "mu_buy": natural_parameters.mu_buy,
                    "mu_sell": natural_parameters.mu_sell,
                    "kappa_buy": (
                        natural_parameters.kappa_buy
                    ),
                    "kappa_sell": (
                        natural_parameters.kappa_sell
                    ),
                    "beta_buy": (
                        natural_parameters.beta_buy
                    ),
                    "beta_sell": (
                        natural_parameters.beta_sell
                    ),
                    "half_life_buy_ms": (
                        half_life_buy_ms
                    ),
                    "half_life_sell_ms": (
                        half_life_sell_ms
                    ),
                    "spectral_radius": spectral_radius(
                        natural_parameters
                    ),
                    "optimizer_vector": tuple(
                        float(value)
                        for value in optimizer_vector
                    ),
                    "optimizer_dimension": len(
                        optimizer_vector
                    ),
                    "initial_negative_log_likelihood": (
                        initial_negative_log_likelihood
                    ),
                    "initial_log_likelihood": (
                        -initial_negative_log_likelihood
                    ),
                    "initial_log_score_per_event": (
                        -initial_negative_log_likelihood
                        / EXPECTED_DEVELOPMENT_EVENT_ROWS
                    ),
                    "initial_gradient_norm": (
                        initial_gradient_norm
                    ),
                    "start_payload_sha256": (
                        canonical_json_sha256(
                            start_payload
                        )
                    ),
                    "development_only": True,
                    "calibration_used": False,
                    "validation_used": False,
                    "engineering_holdout_used": False,
                    "status": "PASS",
                }
            )

HAWKES_FULL_START_LEDGER = pd.DataFrame(
    start_records
)

require(
    len(HAWKES_FULL_START_LEDGER)
    == (
        len(KAPPA_START_PAIRS)
        * (
            len(H1_HALF_LIFE_PAIRS_MS)
            + len(H2_HALF_LIFE_PAIRS_MS)
        )
    ),
    "The complete Hawkes starting-value ledger has the wrong size.",
)
require(
    HAWKES_FULL_START_LEDGER[
        "start_id"
    ].is_unique,
    "Starting-value IDs are not unique.",
)
require(
    HAWKES_FULL_START_LEDGER[
        "start_payload_sha256"
    ].is_unique,
    "Starting-value payload hashes are not unique.",
)
require(
    HAWKES_FULL_START_LEDGER[
        "status"
    ].eq("PASS").all(),
    "At least one deterministic starting value failed.",
)
require(
    not HAWKES_FULL_START_LEDGER[
        [
            "initial_negative_log_likelihood",
            "initial_gradient_norm",
        ]
    ].isna().any().any(),
    "The starting-value ledger contains missing objective results.",
)


# ------------------------------------------------------------
# Rank starts within each candidate
# ------------------------------------------------------------

HAWKES_FULL_START_LEDGER[
    "objective_rank_within_model"
] = (
    HAWKES_FULL_START_LEDGER.groupby(
        "model_id",
        observed=True,
    )["initial_negative_log_likelihood"]
    .rank(
        method="first",
        ascending=True,
    )
    .astype("int64")
)

HAWKES_FULL_START_LEDGER[
    "selected_global_objective"
] = (
    HAWKES_FULL_START_LEDGER[
        "objective_rank_within_model"
    ].le(GLOBAL_OBJECTIVE_STARTS_PER_MODEL)
)

HAWKES_FULL_START_LEDGER[
    "selected_kappa_pair_coverage"
] = False

HAWKES_FULL_START_LEDGER[
    "selected_half_life_pair_coverage"
] = False


# ------------------------------------------------------------
# Select the best DEVELOPMENT objective within every structural
# start category, then union those anchors with the global top
# starts. This preserves wide deterministic coverage without
# optimizing all 480 grid points.
# ------------------------------------------------------------

for model_id in CANDIDATE_MODEL_IDS:
    model_mask = HAWKES_FULL_START_LEDGER[
        "model_id"
    ].eq(model_id)

    model_ledger = (
        HAWKES_FULL_START_LEDGER.loc[model_mask]
    )

    best_kappa_pair_indices = (
        model_ledger.sort_values(
            [
                "initial_negative_log_likelihood",
                "start_number",
            ],
            kind="stable",
        )
        .groupby(
            "kappa_pair_id",
            observed=True,
            sort=False,
        )
        .head(1)
        .index
    )

    best_half_life_pair_indices = (
        model_ledger.sort_values(
            [
                "initial_negative_log_likelihood",
                "start_number",
            ],
            kind="stable",
        )
        .groupby(
            "half_life_pair_id",
            observed=True,
            sort=False,
        )
        .head(1)
        .index
    )

    HAWKES_FULL_START_LEDGER.loc[
        best_kappa_pair_indices,
        "selected_kappa_pair_coverage",
    ] = True

    HAWKES_FULL_START_LEDGER.loc[
        best_half_life_pair_indices,
        "selected_half_life_pair_coverage",
    ] = True


HAWKES_FULL_START_LEDGER[
    "selected_for_optimization"
] = (
    HAWKES_FULL_START_LEDGER[
        [
            "selected_global_objective",
            "selected_kappa_pair_coverage",
            "selected_half_life_pair_coverage",
        ]
    ].any(axis=1)
)


def start_selection_reason(
    row: pd.Series,
) -> str:
    """Return the deterministic reason ledger for one selected start."""
    reasons: list[str] = []

    if bool(row["selected_global_objective"]):
        reasons.append("GLOBAL_OBJECTIVE_TOP")
    if bool(row["selected_kappa_pair_coverage"]):
        reasons.append("KAPPA_PAIR_COVERAGE")
    if bool(row["selected_half_life_pair_coverage"]):
        reasons.append("HALF_LIFE_PAIR_COVERAGE")

    return (
        ";".join(reasons)
        if reasons
        else "NOT_SELECTED"
    )


HAWKES_FULL_START_LEDGER[
    "selection_reason"
] = HAWKES_FULL_START_LEDGER.apply(
    start_selection_reason,
    axis=1,
)

HAWKES_OPTIMIZATION_START_LEDGER = (
    HAWKES_FULL_START_LEDGER.loc[
        HAWKES_FULL_START_LEDGER[
            "selected_for_optimization"
        ]
    ]
    .sort_values(
        [
            "model_id",
            "initial_negative_log_likelihood",
            "start_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Coverage gates
# ------------------------------------------------------------

for model_id in CANDIDATE_MODEL_IDS:
    full_model_ledger = (
        HAWKES_FULL_START_LEDGER.loc[
            HAWKES_FULL_START_LEDGER[
                "model_id"
            ].eq(model_id)
        ]
    )

    selected_model_ledger = (
        HAWKES_OPTIMIZATION_START_LEDGER.loc[
            HAWKES_OPTIMIZATION_START_LEDGER[
                "model_id"
            ].eq(model_id)
        ]
    )

    require(
        len(selected_model_ledger)
        >= GLOBAL_OBJECTIVE_STARTS_PER_MODEL,
        (
            f"{model_id} has too few selected optimizer starts."
        ),
    )
    require(
        set(
            selected_model_ledger[
                "kappa_pair_id"
            ]
        )
        == set(
            full_model_ledger[
                "kappa_pair_id"
            ]
        ),
        (
            f"{model_id} optimization starts do not cover every "
            "branching-mass pair."
        ),
    )
    require(
        set(
            selected_model_ledger[
                "half_life_pair_id"
            ]
        )
        == set(
            full_model_ledger[
                "half_life_pair_id"
            ]
        ),
        (
            f"{model_id} optimization starts do not cover every "
            "half-life pair."
        ),
    )
    require(
        set(
            selected_model_ledger[
                "excitation_regime"
            ]
        )
        == set(
            full_model_ledger[
                "excitation_regime"
            ]
        ),
        (
            f"{model_id} optimization starts do not cover every "
            "excitation regime."
        ),
    )
    require(
        set(
            selected_model_ledger[
                "decay_regime"
            ]
        )
        == set(
            full_model_ledger[
                "decay_regime"
            ]
        ),
        (
            f"{model_id} optimization starts do not cover every "
            "decay regime."
        ),
    )


# ------------------------------------------------------------
# Freeze portable ledger hashes
# ------------------------------------------------------------

def portable_start_ledger_records(
    ledger: pd.DataFrame,
) -> list[dict[str, Any]]:
    """Return JSON-safe records for deterministic ledger hashing."""
    portable_records: list[dict[str, Any]] = []

    ordered_ledger = ledger.sort_values(
        [
            "model_id",
            "start_number",
        ],
        kind="stable",
    )

    for row in ordered_ledger.itertuples(index=False):
        portable_records.append(
            {
                "start_id": str(row.start_id),
                "model_id": str(row.model_id),
                "start_number": int(row.start_number),
                "kappa_pair_id": str(row.kappa_pair_id),
                "half_life_pair_id": str(
                    row.half_life_pair_id
                ),
                "mu_buy": float(row.mu_buy),
                "mu_sell": float(row.mu_sell),
                "kappa_buy": float(row.kappa_buy),
                "kappa_sell": float(row.kappa_sell),
                "beta_buy": float(row.beta_buy),
                "beta_sell": float(row.beta_sell),
                "half_life_buy_ms": float(
                    row.half_life_buy_ms
                ),
                "half_life_sell_ms": float(
                    row.half_life_sell_ms
                ),
                "optimizer_vector": [
                    float(value)
                    for value in row.optimizer_vector
                ],
                "initial_negative_log_likelihood": float(
                    row.initial_negative_log_likelihood
                ),
                "initial_gradient_norm": float(
                    row.initial_gradient_norm
                ),
                "selected_for_optimization": bool(
                    row.selected_for_optimization
                ),
                "selection_reason": str(
                    row.selection_reason
                ),
                "start_payload_sha256": str(
                    row.start_payload_sha256
                ),
            }
        )

    return portable_records


HAWKES_FULL_START_LEDGER_SHA256: Final[str] = (
    canonical_json_sha256(
        {
            "schema_version": (
                "NOTEBOOK_07_FULL_START_LEDGER_V1"
            ),
            "records": portable_start_ledger_records(
                HAWKES_FULL_START_LEDGER
            ),
        }
    )
)

HAWKES_OPTIMIZATION_START_LEDGER_SHA256: Final[str] = (
    canonical_json_sha256(
        {
            "schema_version": (
                "NOTEBOOK_07_OPTIMIZATION_START_LEDGER_V1"
            ),
            "records": portable_start_ledger_records(
                HAWKES_OPTIMIZATION_START_LEDGER
            ),
        }
    )
)


# ------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------

start_design_summary_rows: list[dict[str, Any]] = []

for model_id in CANDIDATE_MODEL_IDS:
    model_full = HAWKES_FULL_START_LEDGER.loc[
        HAWKES_FULL_START_LEDGER[
            "model_id"
        ].eq(model_id)
    ]

    model_selected = (
        HAWKES_OPTIMIZATION_START_LEDGER.loc[
            HAWKES_OPTIMIZATION_START_LEDGER[
                "model_id"
            ].eq(model_id)
        ]
    )

    best_start = model_full.sort_values(
        [
            "initial_negative_log_likelihood",
            "start_number",
        ],
        kind="stable",
    ).iloc[0]

    start_design_summary_rows.append(
        {
            "model_id": model_id,
            "full_grid_start_count": len(model_full),
            "selected_optimization_start_count": (
                len(model_selected)
            ),
            "kappa_pair_count": (
                model_full["kappa_pair_id"].nunique()
            ),
            "half_life_pair_count": (
                model_full[
                    "half_life_pair_id"
                ].nunique()
            ),
            "best_start_id": best_start["start_id"],
            "best_initial_negative_log_likelihood": (
                best_start[
                    "initial_negative_log_likelihood"
                ]
            ),
            "best_initial_log_score_per_event": (
                best_start[
                    "initial_log_score_per_event"
                ]
            ),
            "all_objectives_finite": bool(
                np.isfinite(
                    model_full[
                        "initial_negative_log_likelihood"
                    ].to_numpy(dtype=np.float64)
                ).all()
            ),
            "all_gradients_finite": bool(
                np.isfinite(
                    model_full[
                        "initial_gradient_norm"
                    ].to_numpy(dtype=np.float64)
                ).all()
            ),
            "status": "PASS",
        }
    )

HAWKES_START_DESIGN_SUMMARY = pd.DataFrame(
    start_design_summary_rows
)

HAWKES_START_LEDGER_FROZEN: bool = True
HAWKES_OPTIMIZATION_STARTS_FROZEN: bool = True
HAWKES_CANDIDATE_FITTING_STARTED: bool = False
FILESYSTEM_WRITES_PERFORMED = False

start_contract_summary = pd.DataFrame(
    [
        {
            "field": "development_buy_event_count",
            "value": DEVELOPMENT_BUY_EVENT_COUNT,
        },
        {
            "field": "development_sell_event_count",
            "value": DEVELOPMENT_SELL_EVENT_COUNT,
        },
        {
            "field": "development_empirical_buy_rate",
            "value": DEVELOPMENT_EMPIRICAL_BUY_RATE,
        },
        {
            "field": "development_empirical_sell_rate",
            "value": DEVELOPMENT_EMPIRICAL_SELL_RATE,
        },
        {
            "field": "complete_start_grid_size",
            "value": len(HAWKES_FULL_START_LEDGER),
        },
        {
            "field": "selected_optimizer_start_count",
            "value": len(
                HAWKES_OPTIMIZATION_START_LEDGER
            ),
        },
        {
            "field": "start_ledger_frozen",
            "value": HAWKES_START_LEDGER_FROZEN,
        },
        {
            "field": "optimization_starts_frozen",
            "value": HAWKES_OPTIMIZATION_STARTS_FROZEN,
        },
        {
            "field": "candidate_fitting_started",
            "value": HAWKES_CANDIDATE_FITTING_STARTED,
        },
        {
            "field": "full_start_ledger_sha256",
            "value": HAWKES_FULL_START_LEDGER_SHA256,
        },
        {
            "field": "optimization_start_ledger_sha256",
            "value": (
                HAWKES_OPTIMIZATION_START_LEDGER_SHA256
            ),
        },
        {
            "field": "calibration_used",
            "value": False,
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(start_contract_summary)
display(HAWKES_START_DESIGN_SUMMARY)
display(
    HAWKES_OPTIMIZATION_START_LEDGER[
        [
            "start_id",
            "model_id",
            "kappa_buy",
            "kappa_sell",
            "half_life_buy_ms",
            "half_life_sell_ms",
            "initial_negative_log_likelihood",
            "initial_gradient_norm",
            "selection_reason",
        ]
    ]
    .groupby(
        "model_id",
        observed=True,
        group_keys=False,
    )
    .head(10)
    .reset_index(drop=True)
)

print(
    "The complete deterministic Hawkes starting-value grid was "
    "constructed and scored using DEVELOPMENT only. Starting base "
    "intensities were anchored to empirical DEVELOPMENT side rates, "
    "and the optimization shortlist preserves every branching-mass "
    "and half-life pair while retaining the best DEVELOPMENT "
    "objective starts. No optimizer has been run, CALIBRATION was "
    "not used, protected partitions remain unopened, and no "
    "filesystem writes were performed."
)

,field,value
0,development_buy_event_count,3414
1,development_sell_event_count,3590
2,development_empirical_buy_rate,1.894709236
3,development_empirical_sell_rate,1.992386103
4,complete_start_grid_size,480
5,selected_optimizer_start_count,66
6,start_ledger_frozen,True
7,optimization_starts_frozen,True
8,candidate_fitting_started,False
9,full_start_ledger_sha256,72de8ddc69dac8fe3019b941feeeba22b787442533cf0f...


,model_id,full_grid_start_count,selected_optimization_start_count,kappa_pair_count,half_life_pair_count,best_start_id,best_initial_negative_log_likelihood,best_initial_log_score_per_event,all_objectives_finite,all_gradients_finite,status
0,H1_DIAGONAL_SHARED_DECAY,80,25,16,5,H1_DIAGONAL_SHARED_DECAY__START_0021,"1,149.042845",-0.1640552321,True,True,PASS
1,H2_DIAGONAL_SEPARATE_DECAY,400,41,16,25,H2_DIAGONAL_SEPARATE_DECAY__START_0101,"1,149.042845",-0.1640552321,True,True,PASS


,start_id,model_id,kappa_buy,kappa_sell,half_life_buy_ms,half_life_sell_ms,initial_negative_log_likelihood,initial_gradient_norm,selection_reason
0,H1_DIAGONAL_SHARED_DECAY__START_0021,H1_DIAGONAL_SHARED_DECAY,0.25,0.05,5,5,"1,149.042845",253.2825094,GLOBAL_OBJECTIVE_TOP;KAPPA_PAIR_COVERAGE;HALF_...
1,H1_DIAGONAL_SHARED_DECAY__START_0027,H1_DIAGONAL_SHARED_DECAY,0.25,0.25,50,50,"1,252.08854",134.4236553,GLOBAL_OBJECTIVE_TOP;KAPPA_PAIR_COVERAGE;HALF_...
2,H1_DIAGONAL_SHARED_DECAY__START_0001,H1_DIAGONAL_SHARED_DECAY,0.05,0.05,5,5,"1,315.965825",303.3485425,GLOBAL_OBJECTIVE_TOP;KAPPA_PAIR_COVERAGE
3,H1_DIAGONAL_SHARED_DECAY__START_0022,H1_DIAGONAL_SHARED_DECAY,0.25,0.05,50,50,"1,340.129373",125.0938147,GLOBAL_OBJECTIVE_TOP
4,H1_DIAGONAL_SHARED_DECAY__START_0026,H1_DIAGONAL_SHARED_DECAY,0.25,0.25,5,5,"1,349.199562",492.602971,GLOBAL_OBJECTIVE_TOP
5,H1_DIAGONAL_SHARED_DECAY__START_0006,H1_DIAGONAL_SHARED_DECAY,0.05,0.25,5,5,"1,516.122541",520.118751,GLOBAL_OBJECTIVE_TOP;KAPPA_PAIR_COVERAGE
6,H1_DIAGONAL_SHARED_DECAY__START_0007,H1_DIAGONAL_SHARED_DECAY,0.05,0.25,50,50,"1,641.377608",293.6827683,GLOBAL_OBJECTIVE_TOP
7,H1_DIAGONAL_SHARED_DECAY__START_0028,H1_DIAGONAL_SHARED_DECAY,0.25,0.25,250,250,"1,663.108904",167.7035427,GLOBAL_OBJECTIVE_TOP;HALF_LIFE_PAIR_COVERAGE
8,H1_DIAGONAL_SHARED_DECAY__START_0048,H1_DIAGONAL_SHARED_DECAY,0.6,0.25,250,250,"1,673.503982",166.6089721,GLOBAL_OBJECTIVE_TOP;KAPPA_PAIR_COVERAGE
9,H1_DIAGONAL_SHARED_DECAY__START_0002,H1_DIAGONAL_SHARED_DECAY,0.05,0.05,50,50,"1,729.418441",289.5469897,GLOBAL_OBJECTIVE_TOP


The complete deterministic Hawkes starting-value grid was constructed and scored using DEVELOPMENT only. Starting base intensities were anchored to empirical DEVELOPMENT side rates, and the optimization shortlist preserves every branching-mass and half-life pair while retaining the best DEVELOPMENT objective starts. No optimizer has been run, CALIBRATION was not used, protected partitions remain unopened, and no filesystem writes were performed.


In [8]:
# ============================================================
# Fit H1 and H2 on DEVELOPMENT using deterministic multi-start MLE
# ============================================================

import time


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("HAWKES_START_LEDGER_FROZEN", False)),
    "The deterministic Hawkes start ledger has not been frozen.",
)
require(
    bool(
        globals().get(
            "HAWKES_OPTIMIZATION_STARTS_FROZEN",
            False,
        )
    ),
    "The Hawkes optimization shortlist has not been frozen.",
)
require(
    bool(globals().get("HAWKES_EXACT_GRADIENT_VALIDATED", False)),
    "The exact Hawkes gradient has not been validated.",
)
require(
    CALIBRATION_FITTING_AUTHORIZED is False,
    "CALIBRATION must not be used for candidate fitting.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Numerical optimization guardrails
#
# These bounds prevent numerically meaningless excursions while
# remaining broad relative to the observed event rate and the
# millisecond timestamp interface.
# ------------------------------------------------------------

MINIMUM_BASE_INTENSITY_PER_SECOND: Final[float] = 1e-10
MAXIMUM_BASE_INTENSITY_PER_SECOND: Final[float] = 500.0

MINIMUM_OPTIMIZATION_KAPPA: Final[float] = 1e-10
MAXIMUM_OPTIMIZATION_KAPPA: Final[float] = 0.999

MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS: Final[float] = 0.001
MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS: Final[float] = 300.0

MAXIMUM_OPTIMIZATION_BETA_PER_SECOND: Final[float] = (
    math.log(2.0)
    / MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
)

MINIMUM_OPTIMIZATION_BETA_PER_SECOND: Final[float] = (
    math.log(2.0)
    / MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
)

OPTIMIZER_ACCEPTANCE_GRADIENT_INF_PER_EVENT: Final[float] = (
    1e-5
)

MULTISTART_OBJECTIVE_AGREEMENT_PER_EVENT: Final[float] = (
    1e-6
)

MINIMUM_MATERIAL_OPTIMUM_CLUSTER_SIZE: Final[int] = 2

BOUNDARY_RELATIVE_TOLERANCE: Final[float] = 0.01

require(
    MINIMUM_BASE_INTENSITY_PER_SECOND
    > PARAMETER_POSITIVITY_FLOOR,
    "The optimizer base-intensity floor must exceed the transform floor.",
)
require(
    MAXIMUM_BASE_INTENSITY_PER_SECOND
    > max(
        DEVELOPMENT_EMPIRICAL_BUY_RATE,
        DEVELOPMENT_EMPIRICAL_SELL_RATE,
    ),
    "The optimizer base-intensity ceiling is too restrictive.",
)
require(
    0.0
    < MINIMUM_OPTIMIZATION_KAPPA
    < MAXIMUM_OPTIMIZATION_KAPPA
    < 1.0,
    "The optimizer excitation-mass bounds are invalid.",
)
require(
    0.0
    < MINIMUM_OPTIMIZATION_BETA_PER_SECOND
    < MAXIMUM_OPTIMIZATION_BETA_PER_SECOND,
    "The optimizer decay-rate bounds are invalid.",
)


# ------------------------------------------------------------
# Optimizer-space bounds
# ------------------------------------------------------------

MU_OPTIMIZER_BOUNDS: Final[tuple[float, float]] = (
    unconstrained_from_positive(
        MINIMUM_BASE_INTENSITY_PER_SECOND
    ),
    unconstrained_from_positive(
        MAXIMUM_BASE_INTENSITY_PER_SECOND
    ),
)

KAPPA_OPTIMIZER_BOUNDS: Final[tuple[float, float]] = (
    unconstrained_from_kappa(
        MINIMUM_OPTIMIZATION_KAPPA
    ),
    unconstrained_from_kappa(
        MAXIMUM_OPTIMIZATION_KAPPA
    ),
)

BETA_OPTIMIZER_BOUNDS: Final[tuple[float, float]] = (
    unconstrained_from_positive(
        MINIMUM_OPTIMIZATION_BETA_PER_SECOND
    ),
    unconstrained_from_positive(
        MAXIMUM_OPTIMIZATION_BETA_PER_SECOND
    ),
)


def optimizer_bounds_for_model(
    model_id: str,
) -> tuple[tuple[float, float], ...]:
    """Return frozen L-BFGS-B bounds for one candidate."""
    specification = HAWKES_CANDIDATE_SPECIFICATIONS[
        model_id
    ]

    bounds: list[tuple[float, float]] = [
        MU_OPTIMIZER_BOUNDS,
        MU_OPTIMIZER_BOUNDS,
        KAPPA_OPTIMIZER_BOUNDS,
        KAPPA_OPTIMIZER_BOUNDS,
        BETA_OPTIMIZER_BOUNDS,
    ]

    if not specification.shared_decay:
        bounds.append(BETA_OPTIMIZER_BOUNDS)

    require(
        len(bounds)
        == specification.estimated_parameter_count,
        f"Optimizer-bound dimension mismatch for {model_id}.",
    )

    return tuple(bounds)


HAWKES_OPTIMIZER_BOUNDS: Final[
    Mapping[str, tuple[tuple[float, float], ...]]
] = {
    model_id: optimizer_bounds_for_model(model_id)
    for model_id in CANDIDATE_MODEL_IDS
}


# ------------------------------------------------------------
# Scaled DEVELOPMENT objective
# ------------------------------------------------------------

def development_average_nll_and_gradient(
    optimizer_vector: np.ndarray,
    *,
    model_id: str,
) -> tuple[float, np.ndarray]:
    """
    Return average DEVELOPMENT negative log likelihood and gradient.

    Scaling by the fixed DEVELOPMENT event count improves numerical
    conditioning without changing the optimizer.
    """
    total_nll, total_gradient = (
        negative_log_likelihood_and_gradient(
            optimizer_vector,
            model_id=model_id,
            batch_frame=DEVELOPMENT_BATCHES_FOR_HAWKES,
            observation_start_ns=DEVELOPMENT_START_NS,
            observation_end_exclusive_ns=(
                DEVELOPMENT_END_EXCLUSIVE_NS
            ),
        )
    )

    return (
        total_nll / EXPECTED_DEVELOPMENT_EVENT_ROWS,
        total_gradient / EXPECTED_DEVELOPMENT_EVENT_ROWS,
    )


# ------------------------------------------------------------
# Optimizer-result diagnostics
# ------------------------------------------------------------

def approximate_inverse_hessian_diagnostics(
    optimizer_result: optimize.OptimizeResult,
) -> dict[str, float | bool]:
    """Inspect the L-BFGS inverse-Hessian approximation when available."""
    default_result: dict[str, float | bool] = {
        "inverse_hessian_available": False,
        "inverse_hessian_min_eigenvalue": math.nan,
        "inverse_hessian_max_eigenvalue": math.nan,
        "inverse_hessian_condition_number": math.nan,
    }

    inverse_hessian_object = getattr(
        optimizer_result,
        "hess_inv",
        None,
    )

    if inverse_hessian_object is None:
        return default_result

    try:
        if hasattr(inverse_hessian_object, "todense"):
            inverse_hessian = np.asarray(
                inverse_hessian_object.todense(),
                dtype=np.float64,
            )
        else:
            inverse_hessian = np.asarray(
                inverse_hessian_object,
                dtype=np.float64,
            )

        require(
            inverse_hessian.ndim == 2,
            "The inverse-Hessian approximation is not a matrix.",
        )
        require(
            inverse_hessian.shape[0]
            == inverse_hessian.shape[1],
            "The inverse-Hessian approximation is not square.",
        )
        require(
            np.isfinite(inverse_hessian).all(),
            "The inverse-Hessian approximation is non-finite.",
        )

        symmetric_inverse_hessian = (
            inverse_hessian + inverse_hessian.T
        ) / 2.0

        eigenvalues = np.linalg.eigvalsh(
            symmetric_inverse_hessian
        )

        minimum_eigenvalue = float(
            np.min(eigenvalues)
        )
        maximum_eigenvalue = float(
            np.max(eigenvalues)
        )

        positive_eigenvalues = eigenvalues[
            eigenvalues > np.finfo(np.float64).eps
        ]

        if len(positive_eigenvalues) == 0:
            condition_number = math.inf
        else:
            condition_number = float(
                np.max(positive_eigenvalues)
                / np.min(positive_eigenvalues)
            )

        return {
            "inverse_hessian_available": True,
            "inverse_hessian_min_eigenvalue": (
                minimum_eigenvalue
            ),
            "inverse_hessian_max_eigenvalue": (
                maximum_eigenvalue
            ),
            "inverse_hessian_condition_number": (
                condition_number
            ),
        }

    except Exception:
        return default_result


def near_lower_boundary(
    value: float,
    lower_bound: float,
    upper_bound: float,
) -> bool:
    """Return whether a natural parameter is near its lower guardrail."""
    range_width = upper_bound - lower_bound

    return bool(
        value
        <= lower_bound
        + BOUNDARY_RELATIVE_TOLERANCE * range_width
    )


def near_upper_boundary(
    value: float,
    lower_bound: float,
    upper_bound: float,
) -> bool:
    """Return whether a natural parameter is near its upper guardrail."""
    range_width = upper_bound - lower_bound

    return bool(
        value
        >= upper_bound
        - BOUNDARY_RELATIVE_TOLERANCE * range_width
    )


def natural_boundary_diagnostics(
    parameters: DiagonalHawkesParameters,
) -> dict[str, bool]:
    """Return numerical-guardrail proximity indicators."""
    buy_half_life_seconds = (
        math.log(2.0) / parameters.beta_buy
    )
    sell_half_life_seconds = (
        math.log(2.0) / parameters.beta_sell
    )

    return {
        "mu_buy_near_lower_bound": near_lower_boundary(
            parameters.mu_buy,
            MINIMUM_BASE_INTENSITY_PER_SECOND,
            MAXIMUM_BASE_INTENSITY_PER_SECOND,
        ),
        "mu_buy_near_upper_bound": near_upper_boundary(
            parameters.mu_buy,
            MINIMUM_BASE_INTENSITY_PER_SECOND,
            MAXIMUM_BASE_INTENSITY_PER_SECOND,
        ),
        "mu_sell_near_lower_bound": near_lower_boundary(
            parameters.mu_sell,
            MINIMUM_BASE_INTENSITY_PER_SECOND,
            MAXIMUM_BASE_INTENSITY_PER_SECOND,
        ),
        "mu_sell_near_upper_bound": near_upper_boundary(
            parameters.mu_sell,
            MINIMUM_BASE_INTENSITY_PER_SECOND,
            MAXIMUM_BASE_INTENSITY_PER_SECOND,
        ),
        "kappa_buy_near_lower_bound": (
            parameters.kappa_buy <= 1e-6
        ),
        "kappa_buy_near_upper_bound": (
            parameters.kappa_buy
            >= STATIONARITY_ACCEPTANCE_MARGIN
        ),
        "kappa_sell_near_lower_bound": (
            parameters.kappa_sell <= 1e-6
        ),
        "kappa_sell_near_upper_bound": (
            parameters.kappa_sell
            >= STATIONARITY_ACCEPTANCE_MARGIN
        ),
        "buy_half_life_near_minimum": near_lower_boundary(
            buy_half_life_seconds,
            MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
            MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
        ),
        "buy_half_life_near_maximum": near_upper_boundary(
            buy_half_life_seconds,
            MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
            MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
        ),
        "sell_half_life_near_minimum": near_lower_boundary(
            sell_half_life_seconds,
            MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
            MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
        ),
        "sell_half_life_near_maximum": near_upper_boundary(
            sell_half_life_seconds,
            MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
            MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS,
        ),
    }


# ------------------------------------------------------------
# Fit one deterministic start
# ------------------------------------------------------------

def fit_one_hawkes_start(
    start_row: pd.Series,
) -> tuple[dict[str, Any], optimize.OptimizeResult | None]:
    """Fit one candidate from one frozen DEVELOPMENT start."""
    model_id = str(start_row["model_id"])
    start_id = str(start_row["start_id"])

    initial_vector = np.asarray(
        start_row["optimizer_vector"],
        dtype=np.float64,
    )

    specification = HAWKES_CANDIDATE_SPECIFICATIONS[
        model_id
    ]

    require(
        len(initial_vector)
        == specification.estimated_parameter_count,
        f"{start_id} has the wrong optimizer dimension.",
    )

    start_clock = time.perf_counter()

    try:
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always")

            optimizer_result = optimize.minimize(
                fun=lambda vector: (
                    development_average_nll_and_gradient(
                        vector,
                        model_id=model_id,
                    )
                ),
                x0=initial_vector,
                method="L-BFGS-B",
                jac=True,
                bounds=HAWKES_OPTIMIZER_BOUNDS[
                    model_id
                ],
                options={
                    "maxiter": MAX_OPTIMIZER_ITERATIONS,
                    "maxfun": (
                        MAX_OPTIMIZER_ITERATIONS * 20
                    ),
                    "ftol": OPTIMIZER_FUNCTION_TOLERANCE,
                    "gtol": OPTIMIZER_GRADIENT_TOLERANCE,
                    "maxls": 50,
                    "maxcor": 20,
                },
            )

        elapsed_seconds = (
            time.perf_counter() - start_clock
        )

        optimized_vector = np.asarray(
            optimizer_result.x,
            dtype=np.float64,
        )

        optimized_parameters = decode_hawkes_parameters(
            model_id,
            optimized_vector,
        )

        (
            average_negative_log_likelihood,
            average_gradient,
        ) = development_average_nll_and_gradient(
            optimized_vector,
            model_id=model_id,
        )

        total_negative_log_likelihood = (
            average_negative_log_likelihood
            * EXPECTED_DEVELOPMENT_EVENT_ROWS
        )

        total_log_likelihood = (
            -total_negative_log_likelihood
        )

        optimizer_gradient_inf_per_event = float(
            np.linalg.norm(
                average_gradient,
                ord=np.inf,
            )
        )

        optimizer_gradient_l2_per_event = float(
            np.linalg.norm(
                average_gradient,
                ord=2,
            )
        )

        parameter_validation = validate_hawkes_parameters(
            optimized_parameters
        )

        replay_result = replay_diagonal_hawkes(
            DEVELOPMENT_BATCHES_FOR_HAWKES,
            optimized_parameters,
            observation_start_ns=DEVELOPMENT_START_NS,
            observation_end_exclusive_ns=(
                DEVELOPMENT_END_EXCLUSIVE_NS
            ),
            return_replay=False,
        )

        objective_replay_error = abs(
            replay_result.negative_log_likelihood
            - total_negative_log_likelihood
        )

        objective_replay_matches = math.isclose(
            replay_result.negative_log_likelihood,
            total_negative_log_likelihood,
            rel_tol=FLOAT_RELATIVE_TOLERANCE,
            abs_tol=1e-7,
        )

        finite_solution = bool(
            np.isfinite(optimized_vector).all()
            and math.isfinite(
                total_negative_log_likelihood
            )
            and math.isfinite(
                optimizer_gradient_inf_per_event
            )
        )

        gradient_acceptable = bool(
            optimizer_gradient_inf_per_event
            <= OPTIMIZER_ACCEPTANCE_GRADIENT_INF_PER_EVENT
        )

        optimizer_reported_success = bool(
            optimizer_result.success
        )

        accepted_solution = bool(
            finite_solution
            and parameter_validation[
                "mathematically_admissible"
            ]
            and objective_replay_matches
            and (
                optimizer_reported_success
                or gradient_acceptable
            )
        )

        (
            half_life_buy_seconds,
            half_life_sell_seconds,
        ) = excitation_half_lives_seconds(
            optimized_parameters
        )

        (
            mean_buy_intensity,
            mean_sell_intensity,
        ) = model_implied_mean_intensities(
            optimized_parameters
        )

        boundary_diagnostics = (
            natural_boundary_diagnostics(
                optimized_parameters
            )
        )

        inverse_hessian_diagnostics = (
            approximate_inverse_hessian_diagnostics(
                optimizer_result
            )
        )

        warning_messages = tuple(
            str(warning.message)
            for warning in caught_warnings
        )

        result_record = {
            "start_id": start_id,
            "model_id": model_id,
            "selection_reason": str(
                start_row["selection_reason"]
            ),
            "initial_negative_log_likelihood": float(
                start_row[
                    "initial_negative_log_likelihood"
                ]
            ),
            "optimized_negative_log_likelihood": float(
                total_negative_log_likelihood
            ),
            "optimized_log_likelihood": float(
                total_log_likelihood
            ),
            "optimized_log_score_per_event": float(
                total_log_likelihood
                / EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "objective_improvement": float(
                start_row[
                    "initial_negative_log_likelihood"
                ]
                - total_negative_log_likelihood
            ),
            "objective_replay_error": float(
                objective_replay_error
            ),
            "objective_replay_matches": bool(
                objective_replay_matches
            ),
            "optimizer_reported_success": (
                optimizer_reported_success
            ),
            "accepted_solution": accepted_solution,
            "gradient_acceptable": gradient_acceptable,
            "optimizer_status_code": int(
                optimizer_result.status
            ),
            "optimizer_message": str(
                optimizer_result.message
            ),
            "optimizer_iterations": int(
                optimizer_result.nit
            ),
            "optimizer_function_evaluations": int(
                optimizer_result.nfev
            ),
            "optimizer_gradient_evaluations": int(
                getattr(
                    optimizer_result,
                    "njev",
                    optimizer_result.nfev,
                )
            ),
            "optimizer_gradient_inf_per_event": (
                optimizer_gradient_inf_per_event
            ),
            "optimizer_gradient_l2_per_event": (
                optimizer_gradient_l2_per_event
            ),
            "runtime_seconds": float(
                elapsed_seconds
            ),
            "warning_count": len(
                warning_messages
            ),
            "warning_messages": warning_messages,
            "optimized_vector": tuple(
                float(value)
                for value in optimized_vector
            ),
            "mu_buy": optimized_parameters.mu_buy,
            "mu_sell": optimized_parameters.mu_sell,
            "kappa_buy": optimized_parameters.kappa_buy,
            "kappa_sell": optimized_parameters.kappa_sell,
            "beta_buy": optimized_parameters.beta_buy,
            "beta_sell": optimized_parameters.beta_sell,
            "half_life_buy_seconds": (
                half_life_buy_seconds
            ),
            "half_life_sell_seconds": (
                half_life_sell_seconds
            ),
            "model_implied_mean_buy_intensity": (
                mean_buy_intensity
            ),
            "model_implied_mean_sell_intensity": (
                mean_sell_intensity
            ),
            "spectral_radius": parameter_validation[
                "spectral_radius"
            ],
            "stationarity_holds": parameter_validation[
                "stationarity_holds"
            ],
            "acceptance_margin_holds": (
                parameter_validation[
                    "acceptance_margin_holds"
                ]
            ),
            "stationarity_class": parameter_validation[
                "stationarity_class"
            ],
            "mathematically_admissible": (
                parameter_validation[
                    "mathematically_admissible"
                ]
            ),
            **boundary_diagnostics,
            **inverse_hessian_diagnostics,
            "development_only": True,
            "calibration_used": False,
            "validation_used": False,
            "engineering_holdout_used": False,
            "exception_type": pd.NA,
            "exception_message": pd.NA,
            "status": (
                "ACCEPTED"
                if accepted_solution
                else "REJECTED"
            ),
        }

        return result_record, optimizer_result

    except Exception as exception:
        elapsed_seconds = (
            time.perf_counter() - start_clock
        )

        failure_record = {
            "start_id": start_id,
            "model_id": model_id,
            "selection_reason": str(
                start_row["selection_reason"]
            ),
            "initial_negative_log_likelihood": float(
                start_row[
                    "initial_negative_log_likelihood"
                ]
            ),
            "optimized_negative_log_likelihood": math.nan,
            "optimized_log_likelihood": math.nan,
            "optimized_log_score_per_event": math.nan,
            "objective_improvement": math.nan,
            "objective_replay_error": math.nan,
            "objective_replay_matches": False,
            "optimizer_reported_success": False,
            "accepted_solution": False,
            "gradient_acceptable": False,
            "optimizer_status_code": pd.NA,
            "optimizer_message": "EXCEPTION",
            "optimizer_iterations": pd.NA,
            "optimizer_function_evaluations": pd.NA,
            "optimizer_gradient_evaluations": pd.NA,
            "optimizer_gradient_inf_per_event": math.nan,
            "optimizer_gradient_l2_per_event": math.nan,
            "runtime_seconds": float(
                elapsed_seconds
            ),
            "warning_count": 0,
            "warning_messages": tuple(),
            "optimized_vector": tuple(),
            "mu_buy": math.nan,
            "mu_sell": math.nan,
            "kappa_buy": math.nan,
            "kappa_sell": math.nan,
            "beta_buy": math.nan,
            "beta_sell": math.nan,
            "half_life_buy_seconds": math.nan,
            "half_life_sell_seconds": math.nan,
            "model_implied_mean_buy_intensity": math.nan,
            "model_implied_mean_sell_intensity": math.nan,
            "spectral_radius": math.nan,
            "stationarity_holds": False,
            "acceptance_margin_holds": False,
            "stationarity_class": "FAIL_EXCEPTION",
            "mathematically_admissible": False,
            "inverse_hessian_available": False,
            "inverse_hessian_min_eigenvalue": math.nan,
            "inverse_hessian_max_eigenvalue": math.nan,
            "inverse_hessian_condition_number": math.nan,
            "development_only": True,
            "calibration_used": False,
            "validation_used": False,
            "engineering_holdout_used": False,
            "exception_type": type(
                exception
            ).__name__,
            "exception_message": str(
                exception
            ),
            "status": "FAILED_EXCEPTION",
        }

        for boundary_field in (
            "mu_buy_near_lower_bound",
            "mu_buy_near_upper_bound",
            "mu_sell_near_lower_bound",
            "mu_sell_near_upper_bound",
            "kappa_buy_near_lower_bound",
            "kappa_buy_near_upper_bound",
            "kappa_sell_near_lower_bound",
            "kappa_sell_near_upper_bound",
            "buy_half_life_near_minimum",
            "buy_half_life_near_maximum",
            "sell_half_life_near_minimum",
            "sell_half_life_near_maximum",
        ):
            failure_record[boundary_field] = False

        return failure_record, None


# ------------------------------------------------------------
# Run the deterministic multi-start fits
# ------------------------------------------------------------

optimization_records: list[dict[str, Any]] = []

HAWKES_RAW_OPTIMIZER_RESULTS: dict[
    str,
    optimize.OptimizeResult,
] = {}

optimization_run_start = time.perf_counter()

for model_id in CANDIDATE_MODEL_IDS:
    model_starts = (
        HAWKES_OPTIMIZATION_START_LEDGER.loc[
            HAWKES_OPTIMIZATION_START_LEDGER[
                "model_id"
            ].eq(model_id)
        ]
        .sort_values(
            [
                "initial_negative_log_likelihood",
                "start_number",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    print(
        f"Fitting {model_id}: "
        f"{len(model_starts)} deterministic starts."
    )

    for start_position, (_, start_row) in enumerate(
        model_starts.iterrows(),
        start=1,
    ):
        result_record, optimizer_result = (
            fit_one_hawkes_start(start_row)
        )

        optimization_records.append(
            result_record
        )

        if optimizer_result is not None:
            HAWKES_RAW_OPTIMIZER_RESULTS[
                str(start_row["start_id"])
            ] = optimizer_result

        if (
            start_position == 1
            or start_position % 5 == 0
            or start_position == len(model_starts)
        ):
            print(
                f"  completed {start_position:>2}/"
                f"{len(model_starts)} starts; "
                f"latest status={result_record['status']}; "
                f"NLL={result_record['optimized_negative_log_likelihood']!s}"
            )

optimization_total_runtime_seconds = (
    time.perf_counter() - optimization_run_start
)

HAWKES_OPTIMIZATION_RESULTS = pd.DataFrame(
    optimization_records
)

require(
    len(HAWKES_OPTIMIZATION_RESULTS)
    == len(HAWKES_OPTIMIZATION_START_LEDGER),
    "The optimizer-result ledger does not reconcile with the start ledger.",
)
require(
    HAWKES_OPTIMIZATION_RESULTS[
        "start_id"
    ].is_unique,
    "The optimizer-result ledger contains duplicate start IDs.",
)
require(
    set(
        HAWKES_OPTIMIZATION_RESULTS[
            "start_id"
        ]
    )
    == set(
        HAWKES_OPTIMIZATION_START_LEDGER[
            "start_id"
        ]
    ),
    "The optimizer-result and start ledgers contain different IDs.",
)
require(
    not HAWKES_OPTIMIZATION_RESULTS[
        "calibration_used"
    ].any(),
    "A candidate optimizer used CALIBRATION.",
)
require(
    not HAWKES_OPTIMIZATION_RESULTS[
        "validation_used"
    ].any(),
    "A candidate optimizer used VALIDATION.",
)
require(
    not HAWKES_OPTIMIZATION_RESULTS[
        "engineering_holdout_used"
    ].any(),
    "A candidate optimizer used ENGINEERING_HOLDOUT.",
)


# ------------------------------------------------------------
# Rank accepted solutions and identify material optimum clusters
# ------------------------------------------------------------

HAWKES_OPTIMIZATION_RESULTS[
    "objective_rank_within_model"
] = pd.Series(
    pd.NA,
    index=HAWKES_OPTIMIZATION_RESULTS.index,
    dtype="Int64",
)

HAWKES_OPTIMIZATION_RESULTS[
    "objective_gap_from_model_best"
] = math.nan

HAWKES_OPTIMIZATION_RESULTS[
    "objective_gap_per_event"
] = math.nan

HAWKES_OPTIMIZATION_RESULTS[
    "inside_material_optimum_cluster"
] = False

HAWKES_BEST_PARAMETERS_BY_MODEL: dict[
    str,
    DiagonalHawkesParameters,
] = {}

HAWKES_BEST_START_ID_BY_MODEL: dict[str, str] = {}

HAWKES_BEST_DEVELOPMENT_REPLAY_BY_MODEL: dict[
    str,
    DiagonalHawkesReplayResult,
] = {}

model_fit_summary_rows: list[dict[str, Any]] = []

for model_id in CANDIDATE_MODEL_IDS:
    model_mask = HAWKES_OPTIMIZATION_RESULTS[
        "model_id"
    ].eq(model_id)

    accepted_mask = (
        model_mask
        & HAWKES_OPTIMIZATION_RESULTS[
            "accepted_solution"
        ]
    )

    accepted_results = (
        HAWKES_OPTIMIZATION_RESULTS.loc[
            accepted_mask
        ]
        .sort_values(
            [
                "optimized_negative_log_likelihood",
                "start_id",
            ],
            kind="stable",
        )
    )

    require(
        len(accepted_results) >= 2,
        (
            f"{model_id} produced fewer than two accepted "
            "DEVELOPMENT solutions."
        ),
    )

    best_result = accepted_results.iloc[0]
    best_negative_log_likelihood = float(
        best_result[
            "optimized_negative_log_likelihood"
        ]
    )

    objective_gaps = (
        accepted_results[
            "optimized_negative_log_likelihood"
        ]
        - best_negative_log_likelihood
    )

    objective_gaps_per_event = (
        objective_gaps
        / EXPECTED_DEVELOPMENT_EVENT_ROWS
    )

    accepted_indices = accepted_results.index

    HAWKES_OPTIMIZATION_RESULTS.loc[
        accepted_indices,
        "objective_rank_within_model",
    ] = pd.array(
        np.arange(
            1,
            len(accepted_results) + 1,
            dtype=np.int64,
        ),
        dtype="Int64",
    )

    HAWKES_OPTIMIZATION_RESULTS.loc[
        accepted_indices,
        "objective_gap_from_model_best",
    ] = objective_gaps.to_numpy(
        dtype=np.float64
    )

    HAWKES_OPTIMIZATION_RESULTS.loc[
        accepted_indices,
        "objective_gap_per_event",
    ] = objective_gaps_per_event.to_numpy(
        dtype=np.float64
    )

    cluster_indices = accepted_results.loc[
        objective_gaps_per_event.le(
            MULTISTART_OBJECTIVE_AGREEMENT_PER_EVENT
        )
    ].index

    HAWKES_OPTIMIZATION_RESULTS.loc[
        cluster_indices,
        "inside_material_optimum_cluster",
    ] = True

    material_cluster_size = len(
        cluster_indices
    )

    require(
        material_cluster_size
        >= MINIMUM_MATERIAL_OPTIMUM_CLUSTER_SIZE,
        (
            f"{model_id} lacks repeated multi-start agreement "
            "at the material optimum."
        ),
    )

    best_parameters = DiagonalHawkesParameters(
        model_id=model_id,
        mu_buy=float(best_result["mu_buy"]),
        mu_sell=float(best_result["mu_sell"]),
        kappa_buy=float(
            best_result["kappa_buy"]
        ),
        kappa_sell=float(
            best_result["kappa_sell"]
        ),
        beta_buy=float(best_result["beta_buy"]),
        beta_sell=float(best_result["beta_sell"]),
    )

    best_validation = validate_hawkes_parameters(
        best_parameters
    )

    require(
        best_validation[
            "mathematically_admissible"
        ],
        f"{model_id} best fit is mathematically inadmissible.",
    )

    best_replay = replay_diagonal_hawkes(
        DEVELOPMENT_BATCHES_FOR_HAWKES,
        best_parameters,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
        return_replay=False,
    )

    require(
        math.isclose(
            best_replay.negative_log_likelihood,
            best_negative_log_likelihood,
            rel_tol=FLOAT_RELATIVE_TOLERANCE,
            abs_tol=1e-7,
        ),
        f"{model_id} best-fit replay does not match its optimizer objective.",
    )

    HAWKES_BEST_PARAMETERS_BY_MODEL[
        model_id
    ] = best_parameters

    HAWKES_BEST_START_ID_BY_MODEL[
        model_id
    ] = str(best_result["start_id"])

    HAWKES_BEST_DEVELOPMENT_REPLAY_BY_MODEL[
        model_id
    ] = best_replay

    parameter_count = (
        HAWKES_CANDIDATE_SPECIFICATIONS[
            model_id
        ].estimated_parameter_count
    )

    development_aic = (
        2.0 * parameter_count
        + 2.0 * best_negative_log_likelihood
    )

    development_bic = (
        parameter_count
        * math.log(
            EXPECTED_DEVELOPMENT_EVENT_ROWS
        )
        + 2.0 * best_negative_log_likelihood
    )

    model_results = (
        HAWKES_OPTIMIZATION_RESULTS.loc[
            model_mask
        ]
    )

    model_fit_summary_rows.append(
        {
            "model_id": model_id,
            "attempted_start_count": len(
                model_results
            ),
            "optimizer_success_count": int(
                model_results[
                    "optimizer_reported_success"
                ].sum()
            ),
            "accepted_solution_count": int(
                model_results[
                    "accepted_solution"
                ].sum()
            ),
            "failed_exception_count": int(
                model_results[
                    "status"
                ].eq("FAILED_EXCEPTION").sum()
            ),
            "material_optimum_cluster_size": (
                material_cluster_size
            ),
            "best_start_id": str(
                best_result["start_id"]
            ),
            "best_negative_log_likelihood": (
                best_negative_log_likelihood
            ),
            "best_log_likelihood": (
                -best_negative_log_likelihood
            ),
            "best_log_score_per_event": (
                -best_negative_log_likelihood
                / EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "best_gradient_inf_per_event": float(
                best_result[
                    "optimizer_gradient_inf_per_event"
                ]
            ),
            "mu_buy": best_parameters.mu_buy,
            "mu_sell": best_parameters.mu_sell,
            "kappa_buy": (
                best_parameters.kappa_buy
            ),
            "kappa_sell": (
                best_parameters.kappa_sell
            ),
            "beta_buy": best_parameters.beta_buy,
            "beta_sell": best_parameters.beta_sell,
            "half_life_buy_seconds": float(
                best_result[
                    "half_life_buy_seconds"
                ]
            ),
            "half_life_sell_seconds": float(
                best_result[
                    "half_life_sell_seconds"
                ]
            ),
            "spectral_radius": (
                best_validation["spectral_radius"]
            ),
            "stationarity_class": (
                best_validation[
                    "stationarity_class"
                ]
            ),
            "acceptance_margin_holds": (
                best_validation[
                    "acceptance_margin_holds"
                ]
            ),
            "development_aic": development_aic,
            "development_bic": development_bic,
            "any_best_parameter_near_numerical_boundary": bool(
                any(
                    bool(best_result[column_name])
                    for column_name in (
                        "mu_buy_near_lower_bound",
                        "mu_buy_near_upper_bound",
                        "mu_sell_near_lower_bound",
                        "mu_sell_near_upper_bound",
                        "kappa_buy_near_lower_bound",
                        "kappa_buy_near_upper_bound",
                        "kappa_sell_near_lower_bound",
                        "kappa_sell_near_upper_bound",
                        "buy_half_life_near_minimum",
                        "buy_half_life_near_maximum",
                        "sell_half_life_near_minimum",
                        "sell_half_life_near_maximum",
                    )
                )
            ),
            "total_runtime_seconds": float(
                model_results[
                    "runtime_seconds"
                ].sum()
            ),
            "status": "PASS",
        }
    )

HAWKES_MODEL_FIT_SUMMARY = pd.DataFrame(
    model_fit_summary_rows
)


# ------------------------------------------------------------
# Portable optimizer-result ledger hash
# ------------------------------------------------------------

def portable_optimizer_result_records(
    result_table: pd.DataFrame,
) -> list[dict[str, Any]]:
    """Return JSON-safe optimizer-result records for hashing."""
    portable_records: list[dict[str, Any]] = []

    ordered_results = result_table.sort_values(
        [
            "model_id",
            "start_id",
        ],
        kind="stable",
    )

    for row in ordered_results.itertuples(index=False):
        portable_records.append(
            {
                "start_id": str(row.start_id),
                "model_id": str(row.model_id),
                "accepted_solution": bool(
                    row.accepted_solution
                ),
                "optimizer_reported_success": bool(
                    row.optimizer_reported_success
                ),
                "optimized_negative_log_likelihood": (
                    None
                    if not math.isfinite(
                        float(
                            row.optimized_negative_log_likelihood
                        )
                    )
                    else float(
                        row.optimized_negative_log_likelihood
                    )
                ),
                "optimizer_gradient_inf_per_event": (
                    None
                    if not math.isfinite(
                        float(
                            row.optimizer_gradient_inf_per_event
                        )
                    )
                    else float(
                        row.optimizer_gradient_inf_per_event
                    )
                ),
                "mu_buy": (
                    None
                    if not math.isfinite(float(row.mu_buy))
                    else float(row.mu_buy)
                ),
                "mu_sell": (
                    None
                    if not math.isfinite(float(row.mu_sell))
                    else float(row.mu_sell)
                ),
                "kappa_buy": (
                    None
                    if not math.isfinite(float(row.kappa_buy))
                    else float(row.kappa_buy)
                ),
                "kappa_sell": (
                    None
                    if not math.isfinite(float(row.kappa_sell))
                    else float(row.kappa_sell)
                ),
                "beta_buy": (
                    None
                    if not math.isfinite(float(row.beta_buy))
                    else float(row.beta_buy)
                ),
                "beta_sell": (
                    None
                    if not math.isfinite(float(row.beta_sell))
                    else float(row.beta_sell)
                ),
                "spectral_radius": (
                    None
                    if not math.isfinite(
                        float(row.spectral_radius)
                    )
                    else float(row.spectral_radius)
                ),
                "inside_material_optimum_cluster": bool(
                    row.inside_material_optimum_cluster
                ),
                "status": str(row.status),
                "exception_type": (
                    None
                    if pd.isna(row.exception_type)
                    else str(row.exception_type)
                ),
            }
        )

    return portable_records


HAWKES_OPTIMIZATION_RESULTS_SHA256: Final[str] = (
    canonical_json_sha256(
        {
            "schema_version": (
                "NOTEBOOK_07_HAWKES_OPTIMIZATION_RESULTS_V1"
            ),
            "development_event_count": (
                EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "calibration_parameter_updates": 0,
            "records": portable_optimizer_result_records(
                HAWKES_OPTIMIZATION_RESULTS
            ),
        }
    )
)


# ------------------------------------------------------------
# Fit-state ledger
# ------------------------------------------------------------

HAWKES_CANDIDATE_FITTING_STARTED = True
HAWKES_CANDIDATE_FITTING_COMPLETED: bool = True
HAWKES_MULTISTART_AGREEMENT_PASSED: bool = True
HAWKES_CANDIDATE_SELECTION_COMPLETED: bool = False
CALIBRATION_USED_FOR_FITTING: bool = False
FILESYSTEM_WRITES_PERFORMED = False

optimization_summary = pd.DataFrame(
    [
        {
            "field": "candidate_fitting_started",
            "value": HAWKES_CANDIDATE_FITTING_STARTED,
        },
        {
            "field": "candidate_fitting_completed",
            "value": HAWKES_CANDIDATE_FITTING_COMPLETED,
        },
        {
            "field": "multistart_agreement_passed",
            "value": HAWKES_MULTISTART_AGREEMENT_PASSED,
        },
        {
            "field": "attempted_optimizer_starts",
            "value": len(
                HAWKES_OPTIMIZATION_RESULTS
            ),
        },
        {
            "field": "accepted_optimizer_solutions",
            "value": int(
                HAWKES_OPTIMIZATION_RESULTS[
                    "accepted_solution"
                ].sum()
            ),
        },
        {
            "field": "failed_optimizer_exceptions",
            "value": int(
                HAWKES_OPTIMIZATION_RESULTS[
                    "status"
                ].eq("FAILED_EXCEPTION").sum()
            ),
        },
        {
            "field": "optimization_runtime_seconds",
            "value": optimization_total_runtime_seconds,
        },
        {
            "field": "optimization_results_sha256",
            "value": (
                HAWKES_OPTIMIZATION_RESULTS_SHA256
            ),
        },
        {
            "field": "candidate_selection_completed",
            "value": HAWKES_CANDIDATE_SELECTION_COMPLETED,
        },
        {
            "field": "calibration_used_for_fitting",
            "value": CALIBRATION_USED_FOR_FITTING,
        },
        {
            "field": "calibration_parameter_updates",
            "value": CALIBRATION_PARAMETER_UPDATES,
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(optimization_summary)
display(HAWKES_MODEL_FIT_SUMMARY)

display(
    HAWKES_OPTIMIZATION_RESULTS.loc[
        HAWKES_OPTIMIZATION_RESULTS[
            "accepted_solution"
        ],
        [
            "model_id",
            "start_id",
            "optimized_negative_log_likelihood",
            "objective_gap_from_model_best",
            "optimizer_gradient_inf_per_event",
            "mu_buy",
            "mu_sell",
            "kappa_buy",
            "kappa_sell",
            "half_life_buy_seconds",
            "half_life_sell_seconds",
            "spectral_radius",
            "inside_material_optimum_cluster",
            "status",
        ],
    ]
    .sort_values(
        [
            "model_id",
            "optimized_negative_log_likelihood",
            "start_id",
        ],
        kind="stable",
    )
    .groupby(
        "model_id",
        observed=True,
        group_keys=False,
    )
    .head(10)
    .reset_index(drop=True)
)

print(
    "H1 and H2 were fitted from every frozen deterministic "
    "optimization start using DEVELOPMENT only. Exact gradients, "
    "objective replay, parameter admissibility, stationarity, "
    "boundary proximity, optimizer diagnostics, and repeated "
    "multi-start agreement were audited. Candidate selection has "
    "not yet been performed. CALIBRATION received zero parameter "
    "updates, protected partitions remain unopened, and no "
    "filesystem writes were performed."
)

Fitting H1_DIAGONAL_SHARED_DECAY: 25 deterministic starts.
  completed  1/25 starts; latest status=ACCEPTED; NLL=1005.4631397518557
  completed  5/25 starts; latest status=ACCEPTED; NLL=1005.463139752796
  completed 10/25 starts; latest status=ACCEPTED; NLL=1005.46313976038
  completed 15/25 starts; latest status=ACCEPTED; NLL=1005.4631397518823
  completed 20/25 starts; latest status=ACCEPTED; NLL=1005.4631397555361
  completed 25/25 starts; latest status=ACCEPTED; NLL=1005.4631397524433
Fitting H2_DIAGONAL_SEPARATE_DECAY: 41 deterministic starts.
  completed  1/41 starts; latest status=ACCEPTED; NLL=1001.0741425333005
  completed  5/41 starts; latest status=ACCEPTED; NLL=1001.0741425217569
  completed 10/41 starts; latest status=ACCEPTED; NLL=1001.0741425218023
  completed 15/41 starts; latest status=ACCEPTED; NLL=1001.0741425217996
  completed 20/41 starts; latest status=ACCEPTED; NLL=1001.0741425221403
  completed 25/41 starts; latest status=ACCEPTED; NLL=1001.0741425402385
  compl

,field,value
0,candidate_fitting_started,True
1,candidate_fitting_completed,True
2,multistart_agreement_passed,True
3,attempted_optimizer_starts,66
4,accepted_optimizer_solutions,66
5,failed_optimizer_exceptions,0
6,optimization_runtime_seconds,69.0247063
7,optimization_results_sha256,c2b14d36d2fca2e3bd5a8504c6b0d5daab840a0b1c4353...
8,candidate_selection_completed,False
9,calibration_used_for_fitting,False


,model_id,attempted_start_count,optimizer_success_count,accepted_solution_count,failed_exception_count,material_optimum_cluster_size,best_start_id,best_negative_log_likelihood,best_log_likelihood,best_log_score_per_event,best_gradient_inf_per_event,mu_buy,mu_sell,kappa_buy,kappa_sell,beta_buy,beta_sell,half_life_buy_seconds,half_life_sell_seconds,spectral_radius,stationarity_class,acceptance_margin_holds,development_aic,development_bic,any_best_parameter_near_numerical_boundary,total_runtime_seconds,status
0,H1_DIAGONAL_SHARED_DECAY,25,25,25,0,25,H1_DIAGONAL_SHARED_DECAY__START_0001,"1,005.46314","-1,005.46314",-0.1435555596,4.717983947e-09,1.536124799,1.767916532,0.1892556591,0.1126637105,70.67941658,70.67941658,0.00980691712,0.00980691712,0.1892556591,PASS_ACCEPTANCE_MARGIN,True,"2,020.92628","2,055.197463",True,18.4383812,PASS
1,H2_DIAGONAL_SEPARATE_DECAY,41,41,41,0,40,H2_DIAGONAL_SEPARATE_DECAY__START_0102,"1,001.074143","-1,001.074143",-0.1429289181,4.091591483e-09,1.548956343,1.748919526,0.1824833409,0.1221984945,84.27754365,53.12025196,0.008224577397,0.01304864256,0.1824833409,PASS_ACCEPTANCE_MARGIN,True,"2,014.148285","2,055.273705",True,47.3207483,PASS


,model_id,start_id,optimized_negative_log_likelihood,objective_gap_from_model_best,optimizer_gradient_inf_per_event,mu_buy,mu_sell,kappa_buy,kappa_sell,half_life_buy_seconds,half_life_sell_seconds,spectral_radius,inside_material_optimum_cluster,status
0,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0001,"1,005.46314",0,4.717983947e-09,1.536124799,1.767916532,0.1892556591,0.1126637105,0.00980691712,0.00980691712,0.1892556591,True,ACCEPTED
1,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0048,"1,005.46314",1.568878361e-11,1.942598185e-08,1.536124876,1.76791633,0.1892556434,0.1126636976,0.009806917436,0.009806917436,0.1892556434,True,ACCEPTED
2,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0006,"1,005.46314",1.185753717e-10,2.591511491e-08,1.536124673,1.767916668,0.1892556715,0.1126637746,0.009806913071,0.009806913071,0.1892556715,True,ACCEPTED
3,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0023,"1,005.46314",1.743956091e-10,3.86683058e-08,1.536124902,1.767916759,0.1892555867,0.1126637593,0.009806924834,0.009806924834,0.1892555867,True,ACCEPTED
4,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0064,"1,005.46314",1.804210115e-10,5.735351834e-08,1.536124327,1.767916483,0.1892556244,0.112663651,0.009806919844,0.009806919844,0.1892556244,True,ACCEPTED
5,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0021,"1,005.46314",2.03158379e-10,2.433668098e-08,1.53612485,1.767916189,0.1892556256,0.1126638251,0.00980692746,0.00980692746,0.1892556256,True,ACCEPTED
6,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0049,"1,005.46314",2.040678737e-10,3.69719425e-08,1.536124756,1.767916261,0.1892556944,0.112663605,0.009806911619,0.009806911619,0.1892556944,True,ACCEPTED
7,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0033,"1,005.46314",2.29761099e-10,4.291304399e-08,1.536124622,1.767916206,0.1892555879,0.1126636388,0.00980690794,0.00980690794,0.1892555879,True,ACCEPTED
8,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0007,"1,005.46314",4.910134521e-10,6.558737068e-08,1.536124969,1.767916972,0.1892555241,0.1126638442,0.009806917906,0.009806917906,0.1892555241,True,ACCEPTED
9,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY__START_0020,"1,005.46314",7.908056432e-10,5.962059005e-08,1.536124436,1.767917191,0.189255582,0.1126634891,0.009806904994,0.009806904994,0.189255582,True,ACCEPTED


H1 and H2 were fitted from every frozen deterministic optimization start using DEVELOPMENT only. Exact gradients, objective replay, parameter admissibility, stationarity, boundary proximity, optimizer diagnostics, and repeated multi-start agreement were audited. Candidate selection has not yet been performed. CALIBRATION received zero parameter updates, protected partitions remain unopened, and no filesystem writes were performed.


In [10]:
# ============================================================
# Finalize DEVELOPMENT fold audit and select H1 versus H2
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(globals().get("HAWKES_CANDIDATE_FITTING_COMPLETED", False)),
    "The full-DEVELOPMENT H1 and H2 fits are incomplete.",
)
require(
    bool(globals().get("HAWKES_MULTISTART_AGREEMENT_PASSED", False)),
    "Full-DEVELOPMENT multi-start agreement has not passed.",
)
require(
    "HAWKES_MODEL_FIT_SUMMARY" in globals(),
    "The full-DEVELOPMENT model-fit summary is unavailable.",
)
require(
    "HAWKES_BEST_PARAMETERS_BY_MODEL" in globals(),
    "The full-DEVELOPMENT best-parameter registry is unavailable.",
)
require(
    CALIBRATION_USED_FOR_FITTING is False,
    "CALIBRATION must not have been used for candidate fitting.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Locate the completed chronological-fold tables
#
# The prior computation completed before its final display failed.
# This cell uses the in-memory results and does not rerun any fold
# optimizer.
# ------------------------------------------------------------

def dataframe_candidates_with_columns(
    required_columns: set[str],
) -> list[tuple[str, pd.DataFrame]]:
    """Return in-memory DataFrames containing all required columns."""
    candidates: list[tuple[str, pd.DataFrame]] = []

    for object_name, object_value in globals().items():
        if not isinstance(object_value, pd.DataFrame):
            continue

        if required_columns.issubset(
            set(object_value.columns)
        ):
            candidates.append(
                (object_name, object_value)
            )

    return candidates


def choose_fold_geometry_frame() -> tuple[str, pd.DataFrame]:
    """Locate the completed DEVELOPMENT fold-geometry table."""
    required_columns = {
        "fold_id",
        "training_event_count",
        "validation_event_count",
        "training_duration_seconds",
        "validation_duration_seconds",
    }

    candidates = dataframe_candidates_with_columns(
        required_columns
    )

    require(
        candidates,
        (
            "No completed DEVELOPMENT fold-geometry DataFrame "
            "was found in memory."
        ),
    )

    ranked_candidates = sorted(
        candidates,
        key=lambda item: (
            len(item[1]) == DEVELOPMENT_CHRONOLOGICAL_FOLDS,
            "status" in item[1].columns,
            len(required_columns.intersection(
                set(item[1].columns)
            )),
        ),
        reverse=True,
    )

    return ranked_candidates[0]


VALIDATION_SCORE_COLUMN_CANDIDATES: Final[
    tuple[str, ...]
] = (
    "validation_log_score_per_event",
    "validation_log_likelihood_per_event",
    "validation_score_per_event",
    "validation_average_log_score",
    "fold_validation_log_score_per_event",
)


def choose_fold_result_frame(
) -> tuple[str, pd.DataFrame, str]:
    """Locate the completed per-fold H1/H2 result table."""
    base_candidates = dataframe_candidates_with_columns(
        {
            "fold_id",
            "model_id",
        }
    )

    scored_candidates: list[
        tuple[int, str, pd.DataFrame, str]
    ] = []

    for object_name, frame in base_candidates:
        available_score_columns = [
            column_name
            for column_name
            in VALIDATION_SCORE_COLUMN_CANDIDATES
            if column_name in frame.columns
        ]

        if not available_score_columns:
            continue

        score_column = available_score_columns[0]

        structural_score = sum(
            preferred_column in frame.columns
            for preferred_column in (
                "accepted_start_count",
                "attempted_start_count",
                "training_event_count",
                "validation_event_count",
                "any_parameter_near_numerical_boundary",
                "status",
            )
        )

        expected_row_bonus = int(
            len(frame)
            == (
                DEVELOPMENT_CHRONOLOGICAL_FOLDS
                * len(CANDIDATE_MODEL_IDS)
            )
        ) * 20

        scored_candidates.append(
            (
                expected_row_bonus + structural_score,
                object_name,
                frame,
                score_column,
            )
        )

    require(
        scored_candidates,
        (
            "No completed per-fold H1/H2 result DataFrame with a "
            "validation log-score column was found in memory."
        ),
    )

    scored_candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    _, object_name, frame, score_column = (
        scored_candidates[0]
    )

    return object_name, frame, score_column


(
    fold_geometry_source_name,
    fold_geometry_source,
) = choose_fold_geometry_frame()

(
    fold_result_source_name,
    fold_result_source,
    detected_validation_score_column,
) = choose_fold_result_frame()


# ------------------------------------------------------------
# Canonicalize fold geometry
# ------------------------------------------------------------

HAWKES_DEVELOPMENT_FOLD_GEOMETRY = (
    fold_geometry_source.copy()
)

HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
    "fold_id"
] = (
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "fold_id"
    ]
    .astype("string")
    .str.strip()
)

if (
    "fold_number"
    not in HAWKES_DEVELOPMENT_FOLD_GEOMETRY.columns
):
    extracted_fold_numbers = (
        HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
            "fold_id"
        ]
        .str.extract(
            r"(\d+)$",
            expand=False,
        )
    )

    require(
        not extracted_fold_numbers.isna().any(),
        "At least one fold ID does not end with a fold number.",
    )

    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "fold_number"
    ] = extracted_fold_numbers.astype("int64")
else:
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "fold_number"
    ] = parse_exact_int64(
        HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
            "fold_number"
        ],
        label=(
            "HAWKES_DEVELOPMENT_FOLD_GEOMETRY."
            "fold_number"
        ),
    )

for count_column in (
    "training_event_count",
    "validation_event_count",
):
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        count_column
    ] = parse_exact_int64(
        HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
            count_column
        ],
        label=(
            "HAWKES_DEVELOPMENT_FOLD_GEOMETRY."
            f"{count_column}"
        ),
    )

for duration_column in (
    "training_duration_seconds",
    "validation_duration_seconds",
):
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        duration_column
    ] = pd.to_numeric(
        HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
            duration_column
        ],
        errors="raise",
    ).astype("float64")

HAWKES_DEVELOPMENT_FOLD_GEOMETRY = (
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY
    .sort_values(
        [
            "fold_number",
            "fold_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    len(HAWKES_DEVELOPMENT_FOLD_GEOMETRY)
    == DEVELOPMENT_CHRONOLOGICAL_FOLDS,
    "The DEVELOPMENT fold-geometry table has the wrong row count.",
)
require(
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "fold_id"
    ].is_unique,
    "DEVELOPMENT fold IDs are not unique.",
)
require(
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "fold_number"
    ].tolist()
    == list(
        range(
            1,
            DEVELOPMENT_CHRONOLOGICAL_FOLDS + 1,
        )
    ),
    "DEVELOPMENT fold numbers are not sequential.",
)
require(
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "training_event_count"
    ].is_monotonic_increasing,
    "Expanding-fold training event counts are not increasing.",
)
require(
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "training_duration_seconds"
    ].is_monotonic_increasing,
    "Expanding-fold training durations are not increasing.",
)
require(
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "validation_event_count"
    ].gt(0).all(),
    "A DEVELOPMENT validation fold contains no events.",
)
require(
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        "validation_duration_seconds"
    ].gt(0.0).all(),
    "A DEVELOPMENT validation fold has nonpositive duration.",
)


# ------------------------------------------------------------
# Canonicalize per-fold model results
# ------------------------------------------------------------

HAWKES_DEVELOPMENT_FOLD_RESULTS = (
    fold_result_source.copy()
)

HAWKES_DEVELOPMENT_FOLD_RESULTS[
    "fold_id"
] = (
    HAWKES_DEVELOPMENT_FOLD_RESULTS[
        "fold_id"
    ]
    .astype("string")
    .str.strip()
)

HAWKES_DEVELOPMENT_FOLD_RESULTS[
    "model_id"
] = (
    HAWKES_DEVELOPMENT_FOLD_RESULTS[
        "model_id"
    ]
    .astype("string")
    .str.strip()
)

if (
    "fold_number"
    not in HAWKES_DEVELOPMENT_FOLD_RESULTS.columns
):
    extracted_fold_numbers = (
        HAWKES_DEVELOPMENT_FOLD_RESULTS[
            "fold_id"
        ]
        .str.extract(
            r"(\d+)$",
            expand=False,
        )
    )

    require(
        not extracted_fold_numbers.isna().any(),
        "At least one model-result fold ID lacks a fold number.",
    )

    HAWKES_DEVELOPMENT_FOLD_RESULTS[
        "fold_number"
    ] = extracted_fold_numbers.astype("int64")
else:
    HAWKES_DEVELOPMENT_FOLD_RESULTS[
        "fold_number"
    ] = parse_exact_int64(
        HAWKES_DEVELOPMENT_FOLD_RESULTS[
            "fold_number"
        ],
        label=(
            "HAWKES_DEVELOPMENT_FOLD_RESULTS."
            "fold_number"
        ),
    )

HAWKES_DEVELOPMENT_FOLD_RESULTS[
    "validation_log_score_per_event"
] = pd.to_numeric(
    HAWKES_DEVELOPMENT_FOLD_RESULTS[
        detected_validation_score_column
    ],
    errors="raise",
).astype("float64")

require(
    np.isfinite(
        HAWKES_DEVELOPMENT_FOLD_RESULTS[
            "validation_log_score_per_event"
        ].to_numpy(dtype=np.float64)
    ).all(),
    "At least one fold validation log score is non-finite.",
)

HAWKES_DEVELOPMENT_FOLD_RESULTS = (
    HAWKES_DEVELOPMENT_FOLD_RESULTS
    .sort_values(
        [
            "fold_number",
            "model_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

expected_fold_result_rows = (
    DEVELOPMENT_CHRONOLOGICAL_FOLDS
    * len(CANDIDATE_MODEL_IDS)
)

require(
    len(HAWKES_DEVELOPMENT_FOLD_RESULTS)
    == expected_fold_result_rows,
    (
        "The DEVELOPMENT fold-result table has the wrong row "
        f"count: expected={expected_fold_result_rows}; "
        f"observed={len(HAWKES_DEVELOPMENT_FOLD_RESULTS)}."
    ),
)
require(
    set(
        HAWKES_DEVELOPMENT_FOLD_RESULTS[
            "model_id"
        ]
    )
    == set(CANDIDATE_MODEL_IDS),
    "The fold-result table contains an unexpected candidate set.",
)
require(
    not HAWKES_DEVELOPMENT_FOLD_RESULTS.duplicated(
        [
            "fold_id",
            "model_id",
        ]
    ).any(),
    "A fold-model result appears more than once.",
)

results_per_fold = (
    HAWKES_DEVELOPMENT_FOLD_RESULTS.groupby(
        "fold_id",
        observed=True,
    )["model_id"]
    .nunique()
)

require(
    results_per_fold.eq(
        len(CANDIDATE_MODEL_IDS)
    ).all(),
    "At least one fold does not contain both H1 and H2.",
)

if "status" in HAWKES_DEVELOPMENT_FOLD_RESULTS.columns:
    require(
        HAWKES_DEVELOPMENT_FOLD_RESULTS[
            "status"
        ].isin(
            (
                "PASS",
                "ACCEPTED",
            )
        ).all(),
        "At least one DEVELOPMENT fold-model result did not pass.",
    )


# ------------------------------------------------------------
# Reconcile fold geometry into the result ledger
# ------------------------------------------------------------

geometry_columns = (
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        [
            "fold_id",
            "training_event_count",
            "validation_event_count",
            "training_duration_seconds",
            "validation_duration_seconds",
        ]
    ]
)

for geometry_column in (
    "training_event_count",
    "validation_event_count",
    "training_duration_seconds",
    "validation_duration_seconds",
):
    if (
        geometry_column
        in HAWKES_DEVELOPMENT_FOLD_RESULTS.columns
    ):
        merged_check = (
            HAWKES_DEVELOPMENT_FOLD_RESULTS[
                [
                    "fold_id",
                    geometry_column,
                ]
            ]
            .merge(
                geometry_columns[
                    [
                        "fold_id",
                        geometry_column,
                    ]
                ],
                on="fold_id",
                how="left",
                validate="many_to_one",
                suffixes=("_result", "_geometry"),
            )
        )

        if "count" in geometry_column:
            matches = (
                merged_check[
                    f"{geometry_column}_result"
                ]
                .astype("int64")
                .eq(
                    merged_check[
                        f"{geometry_column}_geometry"
                    ].astype("int64")
                )
            )
        else:
            matches = np.isclose(
                merged_check[
                    f"{geometry_column}_result"
                ].to_numpy(dtype=np.float64),
                merged_check[
                    f"{geometry_column}_geometry"
                ].to_numpy(dtype=np.float64),
                rtol=FLOAT_RELATIVE_TOLERANCE,
                atol=FLOAT_ABSOLUTE_TOLERANCE,
            )

        require(
            bool(np.asarray(matches).all()),
            (
                "Fold-result and fold-geometry values disagree for "
                f"{geometry_column}."
            ),
        )
    else:
        HAWKES_DEVELOPMENT_FOLD_RESULTS = (
            HAWKES_DEVELOPMENT_FOLD_RESULTS.merge(
                geometry_columns[
                    [
                        "fold_id",
                        geometry_column,
                    ]
                ],
                on="fold_id",
                how="left",
                validate="many_to_one",
            )
        )


# ------------------------------------------------------------
# Paired H2-minus-H1 validation comparison
# ------------------------------------------------------------

fold_score_pivot = (
    HAWKES_DEVELOPMENT_FOLD_RESULTS.pivot(
        index=[
            "fold_number",
            "fold_id",
        ],
        columns="model_id",
        values="validation_log_score_per_event",
    )
    .reset_index()
)

require(
    H1_MODEL_ID in fold_score_pivot.columns,
    "H1 validation scores are unavailable.",
)
require(
    H2_MODEL_ID in fold_score_pivot.columns,
    "H2 validation scores are unavailable.",
)

HAWKES_DEVELOPMENT_FOLD_COMPARISON = (
    fold_score_pivot.merge(
        HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
            [
                "fold_id",
                "training_event_count",
                "validation_event_count",
                "training_duration_seconds",
                "validation_duration_seconds",
            ]
        ],
        on="fold_id",
        how="left",
        validate="one_to_one",
    )
)

HAWKES_DEVELOPMENT_FOLD_COMPARISON = (
    HAWKES_DEVELOPMENT_FOLD_COMPARISON.rename(
        columns={
            H1_MODEL_ID: (
                "h1_validation_log_score_per_event"
            ),
            H2_MODEL_ID: (
                "h2_validation_log_score_per_event"
            ),
        }
    )
)

HAWKES_DEVELOPMENT_FOLD_COMPARISON[
    "h2_minus_h1_log_score_per_event"
] = (
    HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "h2_validation_log_score_per_event"
    ]
    - HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "h1_validation_log_score_per_event"
    ]
)

HAWKES_DEVELOPMENT_FOLD_COMPARISON[
    "fold_preferred_model"
] = np.where(
    HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "h2_minus_h1_log_score_per_event"
    ].gt(0.0),
    H2_MODEL_ID,
    H1_MODEL_ID,
)

HAWKES_DEVELOPMENT_FOLD_COMPARISON[
    "h2_validation_log_likelihood_gain"
] = (
    HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "h2_minus_h1_log_score_per_event"
    ]
    * HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "validation_event_count"
    ]
)

HAWKES_DEVELOPMENT_FOLD_COMPARISON[
    "status"
] = "PASS"

HAWKES_DEVELOPMENT_FOLD_COMPARISON = (
    HAWKES_DEVELOPMENT_FOLD_COMPARISON
    .sort_values(
        [
            "fold_number",
            "fold_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Full-DEVELOPMENT information-criterion comparison
# ------------------------------------------------------------

fit_summary_indexed = (
    HAWKES_MODEL_FIT_SUMMARY.set_index(
        "model_id",
        verify_integrity=True,
    )
)

require(
    H1_MODEL_ID in fit_summary_indexed.index,
    "The full-DEVELOPMENT H1 fit is unavailable.",
)
require(
    H2_MODEL_ID in fit_summary_indexed.index,
    "The full-DEVELOPMENT H2 fit is unavailable.",
)

h1_full_nll = float(
    fit_summary_indexed.loc[
        H1_MODEL_ID,
        "best_negative_log_likelihood",
    ]
)

h2_full_nll = float(
    fit_summary_indexed.loc[
        H2_MODEL_ID,
        "best_negative_log_likelihood",
    ]
)

h1_full_aic = float(
    fit_summary_indexed.loc[
        H1_MODEL_ID,
        "development_aic",
    ]
)

h2_full_aic = float(
    fit_summary_indexed.loc[
        H2_MODEL_ID,
        "development_aic",
    ]
)

h1_full_bic = float(
    fit_summary_indexed.loc[
        H1_MODEL_ID,
        "development_bic",
    ]
)

h2_full_bic = float(
    fit_summary_indexed.loc[
        H2_MODEL_ID,
        "development_bic",
    ]
)

H2_FULL_DEVELOPMENT_LOG_LIKELIHOOD_GAIN: Final[float] = (
    h1_full_nll - h2_full_nll
)

H2_FULL_DEVELOPMENT_AIC_GAIN: Final[float] = (
    h1_full_aic - h2_full_aic
)

H2_FULL_DEVELOPMENT_BIC_GAIN: Final[float] = (
    h1_full_bic - h2_full_bic
)


# ------------------------------------------------------------
# Freeze the simplicity-first selection rule
#
# H2 is selected only when its additional decay parameter produces
# a stable chronological validation gain and does not worsen BIC.
# ------------------------------------------------------------

H2_REQUIRED_FOLD_WIN_FRACTION: Final[float] = 0.80
H2_REQUIRED_MEAN_GAIN_PER_EVENT: Final[float] = 0.0
H2_REQUIRED_MEDIAN_GAIN_PER_EVENT: Final[float] = 0.0
H2_REQUIRE_POSITIVE_FULL_DEVELOPMENT_AIC_GAIN: Final[bool] = True
H2_REQUIRE_NONNEGATIVE_FULL_DEVELOPMENT_BIC_GAIN: Final[bool] = True

fold_gain_values = (
    HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "h2_minus_h1_log_score_per_event"
    ].to_numpy(dtype=np.float64)
)

fold_validation_weights = (
    HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "validation_event_count"
    ].to_numpy(dtype=np.float64)
)

H2_FOLD_WIN_COUNT: Final[int] = int(
    np.sum(fold_gain_values > 0.0)
)

H1_FOLD_WIN_COUNT: Final[int] = int(
    np.sum(fold_gain_values <= 0.0)
)

H2_FOLD_WIN_FRACTION: Final[float] = (
    H2_FOLD_WIN_COUNT
    / DEVELOPMENT_CHRONOLOGICAL_FOLDS
)

H2_MEAN_FOLD_GAIN_PER_EVENT: Final[float] = float(
    np.mean(fold_gain_values)
)

H2_MEDIAN_FOLD_GAIN_PER_EVENT: Final[float] = float(
    np.median(fold_gain_values)
)

H2_WEIGHTED_FOLD_GAIN_PER_EVENT: Final[float] = float(
    np.average(
        fold_gain_values,
        weights=fold_validation_weights,
    )
)

H2_WORST_FOLD_GAIN_PER_EVENT: Final[float] = float(
    np.min(fold_gain_values)
)

H2_BEST_FOLD_GAIN_PER_EVENT: Final[float] = float(
    np.max(fold_gain_values)
)

h2_selection_gates: Final[dict[str, bool]] = {
    "fold_win_fraction_gate": (
        H2_FOLD_WIN_FRACTION
        >= H2_REQUIRED_FOLD_WIN_FRACTION
    ),
    "mean_validation_gain_gate": (
        H2_MEAN_FOLD_GAIN_PER_EVENT
        > H2_REQUIRED_MEAN_GAIN_PER_EVENT
    ),
    "median_validation_gain_gate": (
        H2_MEDIAN_FOLD_GAIN_PER_EVENT
        > H2_REQUIRED_MEDIAN_GAIN_PER_EVENT
    ),
    "weighted_validation_gain_gate": (
        H2_WEIGHTED_FOLD_GAIN_PER_EVENT > 0.0
    ),
    "full_development_aic_gate": (
        H2_FULL_DEVELOPMENT_AIC_GAIN > 0.0
        if H2_REQUIRE_POSITIVE_FULL_DEVELOPMENT_AIC_GAIN
        else True
    ),
    "full_development_bic_gate": (
        H2_FULL_DEVELOPMENT_BIC_GAIN >= 0.0
        if H2_REQUIRE_NONNEGATIVE_FULL_DEVELOPMENT_BIC_GAIN
        else True
    ),
}

H2_SELECTION_AUTHORIZED: Final[bool] = bool(
    all(h2_selection_gates.values())
)

SELECTED_HAWKES_MODEL_ID: Final[str] = (
    H2_MODEL_ID
    if H2_SELECTION_AUTHORIZED
    else H1_MODEL_ID
)

REJECTED_HAWKES_MODEL_ID: Final[str] = (
    H1_MODEL_ID
    if H2_SELECTION_AUTHORIZED
    else H2_MODEL_ID
)

SELECTED_HAWKES_MODEL_REASON: Final[str] = (
    "H2_STABLE_MATERIAL_CHRONOLOGICAL_GAIN"
    if H2_SELECTION_AUTHORIZED
    else (
        "H1_SIMPLICITY_FIRST_H2_FAILED_STABLE_"
        "MATERIAL_GAIN_REQUIREMENT"
    )
)

SELECTED_HAWKES_DEVELOPMENT_PARAMETERS = (
    HAWKES_BEST_PARAMETERS_BY_MODEL[
        SELECTED_HAWKES_MODEL_ID
    ]
)

require(
    validate_hawkes_parameters(
        SELECTED_HAWKES_DEVELOPMENT_PARAMETERS
    )["mathematically_admissible"],
    "The selected DEVELOPMENT parameter set is inadmissible.",
)


# ------------------------------------------------------------
# Selection ledgers
# ------------------------------------------------------------

HAWKES_CANDIDATE_SELECTION_GATE_LEDGER = pd.DataFrame(
    [
        {
            "gate_id": gate_id,
            "required": True,
            "passed": gate_passed,
            "status": (
                "PASS"
                if gate_passed
                else "FAIL_H2_SELECTION"
            ),
        }
        for gate_id, gate_passed
        in h2_selection_gates.items()
    ]
)

HAWKES_CANDIDATE_SELECTION_SUMMARY = pd.DataFrame(
    [
        {
            "metric": "h1_fold_win_count",
            "value": H1_FOLD_WIN_COUNT,
        },
        {
            "metric": "h2_fold_win_count",
            "value": H2_FOLD_WIN_COUNT,
        },
        {
            "metric": "h2_fold_win_fraction",
            "value": H2_FOLD_WIN_FRACTION,
        },
        {
            "metric": "h2_mean_fold_gain_per_event",
            "value": H2_MEAN_FOLD_GAIN_PER_EVENT,
        },
        {
            "metric": "h2_median_fold_gain_per_event",
            "value": H2_MEDIAN_FOLD_GAIN_PER_EVENT,
        },
        {
            "metric": "h2_weighted_fold_gain_per_event",
            "value": H2_WEIGHTED_FOLD_GAIN_PER_EVENT,
        },
        {
            "metric": "h2_worst_fold_gain_per_event",
            "value": H2_WORST_FOLD_GAIN_PER_EVENT,
        },
        {
            "metric": "h2_best_fold_gain_per_event",
            "value": H2_BEST_FOLD_GAIN_PER_EVENT,
        },
        {
            "metric": (
                "h2_full_development_log_likelihood_gain"
            ),
            "value": (
                H2_FULL_DEVELOPMENT_LOG_LIKELIHOOD_GAIN
            ),
        },
        {
            "metric": "h2_full_development_aic_gain",
            "value": H2_FULL_DEVELOPMENT_AIC_GAIN,
        },
        {
            "metric": "h2_full_development_bic_gain",
            "value": H2_FULL_DEVELOPMENT_BIC_GAIN,
        },
        {
            "metric": "h2_selection_authorized",
            "value": H2_SELECTION_AUTHORIZED,
        },
        {
            "metric": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "metric": "selection_reason",
            "value": SELECTED_HAWKES_MODEL_REASON,
        },
    ]
)

selection_payload = {
    "schema_version": (
        "NOTEBOOK_07_HAWKES_CANDIDATE_SELECTION_V1"
    ),
    "selected_model_id": SELECTED_HAWKES_MODEL_ID,
    "rejected_model_id": REJECTED_HAWKES_MODEL_ID,
    "selection_reason": SELECTED_HAWKES_MODEL_REASON,
    "development_fold_count": (
        DEVELOPMENT_CHRONOLOGICAL_FOLDS
    ),
    "h1_fold_win_count": H1_FOLD_WIN_COUNT,
    "h2_fold_win_count": H2_FOLD_WIN_COUNT,
    "h2_fold_win_fraction": H2_FOLD_WIN_FRACTION,
    "h2_mean_fold_gain_per_event": (
        H2_MEAN_FOLD_GAIN_PER_EVENT
    ),
    "h2_median_fold_gain_per_event": (
        H2_MEDIAN_FOLD_GAIN_PER_EVENT
    ),
    "h2_weighted_fold_gain_per_event": (
        H2_WEIGHTED_FOLD_GAIN_PER_EVENT
    ),
    "h2_full_development_log_likelihood_gain": (
        H2_FULL_DEVELOPMENT_LOG_LIKELIHOOD_GAIN
    ),
    "h2_full_development_aic_gain": (
        H2_FULL_DEVELOPMENT_AIC_GAIN
    ),
    "h2_full_development_bic_gain": (
        H2_FULL_DEVELOPMENT_BIC_GAIN
    ),
    "h2_selection_gates": h2_selection_gates,
    "calibration_used_for_selection": False,
    "validation_content_loaded": False,
    "engineering_holdout_content_loaded": False,
}

HAWKES_CANDIDATE_SELECTION_SHA256: Final[str] = (
    canonical_json_sha256(selection_payload)
)


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

HAWKES_CHRONOLOGICAL_FOLD_AUDIT_COMPLETED: bool = True
HAWKES_CANDIDATE_SELECTION_COMPLETED = True
HAWKES_SELECTED_MODEL_REFIT_COMPLETED: bool = False
CALIBRATION_USED_FOR_SELECTION: bool = False
FILESYSTEM_WRITES_PERFORMED = False

selection_state_summary = pd.DataFrame(
    [
        {
            "field": "fold_geometry_source",
            "value": fold_geometry_source_name,
        },
        {
            "field": "fold_result_source",
            "value": fold_result_source_name,
        },
        {
            "field": "detected_validation_score_column",
            "value": detected_validation_score_column,
        },
        {
            "field": "chronological_fold_audit_completed",
            "value": (
                HAWKES_CHRONOLOGICAL_FOLD_AUDIT_COMPLETED
            ),
        },
        {
            "field": "candidate_selection_completed",
            "value": HAWKES_CANDIDATE_SELECTION_COMPLETED,
        },
        {
            "field": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "rejected_model_id",
            "value": REJECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "selection_reason",
            "value": SELECTED_HAWKES_MODEL_REASON,
        },
        {
            "field": "selected_model_refit_completed",
            "value": HAWKES_SELECTED_MODEL_REFIT_COMPLETED,
        },
        {
            "field": "candidate_selection_sha256",
            "value": HAWKES_CANDIDATE_SELECTION_SHA256,
        },
        {
            "field": "calibration_used_for_selection",
            "value": CALIBRATION_USED_FOR_SELECTION,
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)


# ------------------------------------------------------------
# Correctly ordered displays
#
# Sort before projecting columns so fold_number remains available
# to sort_values.
# ------------------------------------------------------------

fold_result_display_columns = [
    column_name
    for column_name in (
        "fold_number",
        "fold_id",
        "model_id",
        "training_event_count",
        "validation_event_count",
        "accepted_start_count",
        "attempted_start_count",
        "validation_log_score_per_event",
        "mu_buy",
        "mu_sell",
        "kappa_buy",
        "kappa_sell",
        "half_life_buy_seconds",
        "half_life_sell_seconds",
        "spectral_radius",
        "any_parameter_near_numerical_boundary",
        "status",
    )
    if column_name
    in HAWKES_DEVELOPMENT_FOLD_RESULTS.columns
]

ordered_fold_result_display = (
    HAWKES_DEVELOPMENT_FOLD_RESULTS
    .sort_values(
        [
            "fold_number",
            "model_id",
        ],
        kind="stable",
    )
    .loc[
        :,
        fold_result_display_columns,
    ]
    .reset_index(drop=True)
)

display(selection_state_summary)

display(
    HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
        [
            "fold_number",
            "fold_id",
            "training_event_count",
            "validation_event_count",
            "training_duration_seconds",
            "validation_duration_seconds",
            "status",
        ]
    ]
)

display(ordered_fold_result_display)

display(
    HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        [
            "fold_number",
            "fold_id",
            "validation_event_count",
            "h1_validation_log_score_per_event",
            "h2_validation_log_score_per_event",
            "h2_minus_h1_log_score_per_event",
            "h2_validation_log_likelihood_gain",
            "fold_preferred_model",
            "status",
        ]
    ]
)

display(HAWKES_CANDIDATE_SELECTION_SUMMARY)
display(HAWKES_CANDIDATE_SELECTION_GATE_LEDGER)

print(
    f"Chronological DEVELOPMENT candidate selection completed. "
    f"H1 won {H1_FOLD_WIN_COUNT}/"
    f"{DEVELOPMENT_CHRONOLOGICAL_FOLDS} folds; "
    f"H2 won {H2_FOLD_WIN_COUNT}/"
    f"{DEVELOPMENT_CHRONOLOGICAL_FOLDS}. "
    f"The selected model is {SELECTED_HAWKES_MODEL_ID}. "
    "H2 improved the full-DEVELOPMENT in-sample likelihood and AIC, "
    "but did not produce stable chronological validation gains and "
    "did not improve BIC. CALIBRATION was not used for selection, "
    "protected partitions remain unopened, and no filesystem writes "
    "were performed."
)

,field,value
0,fold_geometry_source,HAWKES_CHRONOLOGICAL_FOLD_CONTRACT
1,fold_result_source,HAWKES_CHRONOLOGICAL_FOLD_RESULTS
2,detected_validation_score_column,validation_log_score_per_event
3,chronological_fold_audit_completed,True
4,candidate_selection_completed,True
5,selected_model_id,H1_DIAGONAL_SHARED_DECAY
6,rejected_model_id,H2_DIAGONAL_SEPARATE_DECAY
7,selection_reason,H1_SIMPLICITY_FIRST_H2_FAILED_STABLE_MATERIAL_...
8,selected_model_refit_completed,False
9,candidate_selection_sha256,a255084307278179b0cb6e96911259e4f5dbc0846ec2f4...


,fold_number,fold_id,training_event_count,validation_event_count,training_duration_seconds,validation_duration_seconds,status
0,1,DEV_FOLD_01,3549,669,900.9297933,180.1859587,PASS
1,2,DEV_FOLD_02,4218,611,"1,081.115752",180.1859587,PASS
2,3,DEV_FOLD_03,4829,827,"1,261.301711",180.1859587,PASS
3,4,DEV_FOLD_04,5656,655,"1,441.487669",180.1859587,PASS
4,5,DEV_FOLD_05,6311,693,"1,621.673628",180.1859587,PASS


,fold_number,fold_id,model_id,training_event_count,validation_event_count,accepted_start_count,attempted_start_count,validation_log_score_per_event,mu_buy,mu_sell,kappa_buy,kappa_sell,half_life_buy_seconds,half_life_sell_seconds,spectral_radius,any_parameter_near_numerical_boundary,status
0,1,DEV_FOLD_01,H1_DIAGONAL_SHARED_DECAY,3549,669,8,8,-0.2793849725,1.584361611,1.832966744,0.1573810289,0.1097713523,0.01980360485,0.01980360485,0.1573810289,False,PASS
1,1,DEV_FOLD_01,H2_DIAGONAL_SEPARATE_DECAY,3549,669,9,9,-0.2805923402,1.553184072,1.852233724,0.1739655036,0.1004137037,0.02714082747,0.0149995766,0.1739655036,False,PASS
2,2,DEV_FOLD_02,H1_DIAGONAL_SHARED_DECAY,4218,611,8,8,-0.3934764031,1.585241958,1.805032901,0.1435132188,0.1197792323,0.01815703791,0.01815703791,0.1435132188,False,PASS
3,2,DEV_FOLD_02,H2_DIAGONAL_SEPARATE_DECAY,4218,611,9,9,-0.3939030904,1.563908469,1.819084411,0.1550394858,0.1129270385,0.02327639145,0.01494133587,0.1550394858,False,PASS
4,3,DEV_FOLD_03,H1_DIAGONAL_SHARED_DECAY,4829,827,8,8,0.2418384086,1.556799459,1.798796317,0.1421608491,0.1067618419,0.01612753122,0.01612753122,0.1421608491,False,PASS
5,3,DEV_FOLD_03,H2_DIAGONAL_SEPARATE_DECAY,4829,827,9,9,0.2377862757,1.538714116,1.810090728,0.1521263568,0.1011532014,0.02020645406,0.01346460123,0.1521263568,False,PASS
6,4,DEV_FOLD_04,H1_DIAGONAL_SHARED_DECAY,5656,655,8,8,0.003047684556,1.56295885,1.780425163,0.1710866448,0.1264597727,0.0146192073,0.0146192073,0.1710866448,False,PASS
7,4,DEV_FOLD_04,H2_DIAGONAL_SEPARATE_DECAY,5656,655,9,9,0.002861426253,1.562030543,1.78130506,0.1715789177,0.1260276483,0.01478263948,0.0144456795,0.1715789177,False,PASS
8,5,DEV_FOLD_05,H1_DIAGONAL_SHARED_DECAY,6311,693,8,8,-0.02656662558,1.546401685,1.751331267,0.1823413804,0.1245105852,0.01176009143,0.01176009143,0.1823413804,False,PASS
9,5,DEV_FOLD_05,H2_DIAGONAL_SEPARATE_DECAY,6311,693,9,9,-0.02419203858,1.549885778,1.747655588,0.180499048,0.1263480158,0.01125744875,0.01241441985,0.180499048,False,PASS


,fold_number,fold_id,validation_event_count,h1_validation_log_score_per_event,h2_validation_log_score_per_event,h2_minus_h1_log_score_per_event,h2_validation_log_likelihood_gain,fold_preferred_model,status
0,1,DEV_FOLD_01,669,-0.2793849725,-0.2805923402,-0.001207367647,-0.8077289561,H1_DIAGONAL_SHARED_DECAY,PASS
1,2,DEV_FOLD_02,611,-0.3934764031,-0.3939030904,-0.0004266872367,-0.2607059016,H1_DIAGONAL_SHARED_DECAY,PASS
2,3,DEV_FOLD_03,827,0.2418384086,0.2377862757,-0.004052132943,-3.351113944,H1_DIAGONAL_SHARED_DECAY,PASS
3,4,DEV_FOLD_04,655,0.003047684556,0.002861426253,-0.0001862583025,-0.1219991881,H1_DIAGONAL_SHARED_DECAY,PASS
4,5,DEV_FOLD_05,693,-0.02656662558,-0.02419203858,0.002374587,1.645588791,H2_DIAGONAL_SEPARATE_DECAY,PASS


,metric,value
0,h1_fold_win_count,4
1,h2_fold_win_count,1
2,h2_fold_win_fraction,0.2
3,h2_mean_fold_gain_per_event,-0.0006995718258
4,h2_median_fold_gain_per_event,-0.0004266872367
5,h2_weighted_fold_gain_per_event,-0.0008381936898
6,h2_worst_fold_gain_per_event,-0.004052132943
7,h2_best_fold_gain_per_event,0.002374587
8,h2_full_development_log_likelihood_gain,4.38899723
9,h2_full_development_aic_gain,6.77799446


,gate_id,required,passed,status
0,fold_win_fraction_gate,True,False,FAIL_H2_SELECTION
1,mean_validation_gain_gate,True,False,FAIL_H2_SELECTION
2,median_validation_gain_gate,True,False,FAIL_H2_SELECTION
3,weighted_validation_gain_gate,True,False,FAIL_H2_SELECTION
4,full_development_aic_gate,True,True,PASS
5,full_development_bic_gate,True,False,FAIL_H2_SELECTION


Chronological DEVELOPMENT candidate selection completed. H1 won 4/5 folds; H2 won 1/5. The selected model is H1_DIAGONAL_SHARED_DECAY. H2 improved the full-DEVELOPMENT in-sample likelihood and AIC, but did not produce stable chronological validation gains and did not improve BIC. CALIBRATION was not used for selection, protected partitions remain unopened, and no filesystem writes were performed.


In [11]:
# ============================================================
# Deterministic final refit and freeze of the selected Hawkes model
# ============================================================

import time


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "HAWKES_CANDIDATE_SELECTION_COMPLETED",
            False,
        )
    ),
    "Chronological DEVELOPMENT candidate selection is incomplete.",
)
require(
    bool(
        globals().get(
            "HAWKES_CHRONOLOGICAL_FOLD_AUDIT_COMPLETED",
            False,
        )
    ),
    "The chronological DEVELOPMENT fold audit is incomplete.",
)
require(
    SELECTED_HAWKES_MODEL_ID
    in HAWKES_CANDIDATE_SPECIFICATIONS,
    "The selected Hawkes model ID is not registered.",
)
require(
    SELECTED_HAWKES_MODEL_ID
    in HAWKES_BEST_PARAMETERS_BY_MODEL,
    "Selected-model DEVELOPMENT parameters are unavailable.",
)
require(
    CALIBRATION_USED_FOR_SELECTION is False,
    "CALIBRATION must not have been used for model selection.",
)
require(
    CALIBRATION_PARAMETER_UPDATES == 0,
    "The frozen contract requires zero CALIBRATION parameter updates.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Selected-model specification and prior DEVELOPMENT optimum
# ------------------------------------------------------------

SELECTED_HAWKES_SPECIFICATION: Final[
    HawkesCandidateSpecification
] = HAWKES_CANDIDATE_SPECIFICATIONS[
    SELECTED_HAWKES_MODEL_ID
]

SELECTED_HAWKES_PRIOR_PARAMETERS: Final[
    DiagonalHawkesParameters
] = HAWKES_BEST_PARAMETERS_BY_MODEL[
    SELECTED_HAWKES_MODEL_ID
]

selected_fit_summary_row = (
    HAWKES_MODEL_FIT_SUMMARY.loc[
        HAWKES_MODEL_FIT_SUMMARY[
            "model_id"
        ].eq(SELECTED_HAWKES_MODEL_ID)
    ]
)

require(
    len(selected_fit_summary_row) == 1,
    "Expected one full-DEVELOPMENT summary row for the selected model.",
)

SELECTED_PRIOR_DEVELOPMENT_NLL: Final[float] = float(
    selected_fit_summary_row.iloc[0][
        "best_negative_log_likelihood"
    ]
)

selected_prior_validation = validate_hawkes_parameters(
    SELECTED_HAWKES_PRIOR_PARAMETERS
)

require(
    selected_prior_validation[
        "mathematically_admissible"
    ],
    "The selected prior DEVELOPMENT parameters are inadmissible.",
)

if SELECTED_HAWKES_SPECIFICATION.shared_decay:
    require(
        math.isclose(
            SELECTED_HAWKES_PRIOR_PARAMETERS.beta_buy,
            SELECTED_HAWKES_PRIOR_PARAMETERS.beta_sell,
            rel_tol=FLOAT_RELATIVE_TOLERANCE,
            abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
        ),
        "The selected shared-decay model has unequal decay rates.",
    )


# ------------------------------------------------------------
# Deterministic final refinement
#
# This starts from the already selected full-DEVELOPMENT optimum.
# It does not reopen candidate selection and does not use
# CALIBRATION.
# ------------------------------------------------------------

selected_initial_optimizer_vector = (
    encode_hawkes_parameters(
        SELECTED_HAWKES_PRIOR_PARAMETERS
    )
)

selected_refit_start_time = time.perf_counter()

with warnings.catch_warnings(record=True) as selected_refit_warnings:
    warnings.simplefilter("always")

    SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT = (
        optimize.minimize(
            fun=lambda vector: (
                development_average_nll_and_gradient(
                    vector,
                    model_id=SELECTED_HAWKES_MODEL_ID,
                )
            ),
            x0=selected_initial_optimizer_vector,
            method="L-BFGS-B",
            jac=True,
            bounds=HAWKES_OPTIMIZER_BOUNDS[
                SELECTED_HAWKES_MODEL_ID
            ],
            options={
                "maxiter": MAX_OPTIMIZER_ITERATIONS,
                "maxfun": (
                    MAX_OPTIMIZER_ITERATIONS * 20
                ),
                "ftol": OPTIMIZER_FUNCTION_TOLERANCE,
                "gtol": OPTIMIZER_GRADIENT_TOLERANCE,
                "maxls": 50,
                "maxcor": 20,
            },
        )
    )

SELECTED_HAWKES_FINAL_REFIT_RUNTIME_SECONDS: Final[
    float
] = (
    time.perf_counter()
    - selected_refit_start_time
)

SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR: Final[
    np.ndarray
] = np.asarray(
    SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.x,
    dtype=np.float64,
)

require(
    SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR.ndim == 1,
    "The selected final optimizer vector is not one-dimensional.",
)
require(
    len(SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR)
    == SELECTED_HAWKES_SPECIFICATION
    .estimated_parameter_count,
    "The selected final optimizer vector has the wrong dimension.",
)
require(
    np.isfinite(
        SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR
    ).all(),
    "The selected final optimizer vector contains non-finite values.",
)

SELECTED_HAWKES_PARAMETERS: Final[
    DiagonalHawkesParameters
] = decode_hawkes_parameters(
    SELECTED_HAWKES_MODEL_ID,
    SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR,
)

(
    selected_final_average_nll,
    selected_final_average_gradient,
) = development_average_nll_and_gradient(
    SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR,
    model_id=SELECTED_HAWKES_MODEL_ID,
)

SELECTED_HAWKES_DEVELOPMENT_NLL: Final[float] = (
    selected_final_average_nll
    * EXPECTED_DEVELOPMENT_EVENT_ROWS
)

SELECTED_HAWKES_DEVELOPMENT_LOG_LIKELIHOOD: Final[
    float
] = -SELECTED_HAWKES_DEVELOPMENT_NLL

SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT: Final[
    float
] = float(
    np.linalg.norm(
        selected_final_average_gradient,
        ord=np.inf,
    )
)

SELECTED_HAWKES_FINAL_GRADIENT_L2_PER_EVENT: Final[
    float
] = float(
    np.linalg.norm(
        selected_final_average_gradient,
        ord=2,
    )
)

selected_final_validation = validate_hawkes_parameters(
    SELECTED_HAWKES_PARAMETERS
)

require(
    selected_final_validation[
        "mathematically_admissible"
    ],
    "The final selected Hawkes parameters are inadmissible.",
)
require(
    selected_final_validation[
        "stationarity_holds"
    ],
    "The final selected Hawkes model violates stationarity.",
)
require(
    SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT
    <= OPTIMIZER_ACCEPTANCE_GRADIENT_INF_PER_EVENT,
    (
        "The final selected-model gradient is too large: "
        f"{SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT}."
    ),
)

selected_refit_accepted = bool(
    SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.success
    or (
        SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT
        <= OPTIMIZER_ACCEPTANCE_GRADIENT_INF_PER_EVENT
    )
)

require(
    selected_refit_accepted,
    (
        "The final selected-model refinement was not accepted: "
        f"{SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.message}"
    ),
)


# ------------------------------------------------------------
# Exact DEVELOPMENT replay
# ------------------------------------------------------------

SELECTED_HAWKES_DEVELOPMENT_REPLAY: Final[
    DiagonalHawkesReplayResult
] = replay_diagonal_hawkes(
    DEVELOPMENT_BATCHES_FOR_HAWKES,
    SELECTED_HAWKES_PARAMETERS,
    observation_start_ns=DEVELOPMENT_START_NS,
    observation_end_exclusive_ns=(
        DEVELOPMENT_END_EXCLUSIVE_NS
    ),
    initial_state=None,
    return_replay=False,
)

require(
    SELECTED_HAWKES_DEVELOPMENT_REPLAY.event_count
    == EXPECTED_DEVELOPMENT_EVENT_ROWS,
    "The selected DEVELOPMENT replay lost events.",
)
require(
    SELECTED_HAWKES_DEVELOPMENT_REPLAY.batch_count
    == EXPECTED_DEVELOPMENT_BATCH_ROWS,
    "The selected DEVELOPMENT replay lost batches.",
)
require(
    math.isclose(
        SELECTED_HAWKES_DEVELOPMENT_REPLAY
        .negative_log_likelihood,
        SELECTED_HAWKES_DEVELOPMENT_NLL,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-7,
    ),
    (
        "The final selected optimizer objective does not match "
        "the exact DEVELOPMENT replay."
    ),
)

SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE: Final[
    DiagonalHawkesState
] = (
    SELECTED_HAWKES_DEVELOPMENT_REPLAY.final_state
)

require(
    SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
    .state_time_ns
    == DEVELOPMENT_END_EXCLUSIVE_NS,
    "The selected DEVELOPMENT state ends at the wrong boundary.",
)
require(
    DEVELOPMENT_END_EXCLUSIVE_NS
    == CALIBRATION_START_NS,
    (
        "The selected terminal DEVELOPMENT state is not expressed "
        "at the exact CALIBRATION boundary."
    ),
)


# ------------------------------------------------------------
# Refit reproducibility audit
# ------------------------------------------------------------

selected_nll_difference_from_prior = (
    SELECTED_HAWKES_DEVELOPMENT_NLL
    - SELECTED_PRIOR_DEVELOPMENT_NLL
)

selected_nll_difference_per_event = (
    selected_nll_difference_from_prior
    / EXPECTED_DEVELOPMENT_EVENT_ROWS
)

require(
    selected_nll_difference_from_prior <= 1e-5,
    (
        "The final selected-model refinement materially worsened "
        "the prior DEVELOPMENT optimum."
    ),
)
require(
    abs(selected_nll_difference_per_event) <= 1e-8,
    (
        "The prior and final selected DEVELOPMENT optima do not "
        "reconcile closely enough."
    ),
)

selected_prior_parameter_vector = np.asarray(
    [
        SELECTED_HAWKES_PRIOR_PARAMETERS.mu_buy,
        SELECTED_HAWKES_PRIOR_PARAMETERS.mu_sell,
        SELECTED_HAWKES_PRIOR_PARAMETERS.kappa_buy,
        SELECTED_HAWKES_PRIOR_PARAMETERS.kappa_sell,
        SELECTED_HAWKES_PRIOR_PARAMETERS.beta_buy,
        SELECTED_HAWKES_PRIOR_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

selected_final_parameter_vector = np.asarray(
    [
        SELECTED_HAWKES_PARAMETERS.mu_buy,
        SELECTED_HAWKES_PARAMETERS.mu_sell,
        SELECTED_HAWKES_PARAMETERS.kappa_buy,
        SELECTED_HAWKES_PARAMETERS.kappa_sell,
        SELECTED_HAWKES_PARAMETERS.beta_buy,
        SELECTED_HAWKES_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

selected_parameter_absolute_differences = np.abs(
    selected_final_parameter_vector
    - selected_prior_parameter_vector
)

selected_parameter_relative_differences = (
    selected_parameter_absolute_differences
    / np.maximum(
        np.abs(selected_prior_parameter_vector),
        np.finfo(np.float64).eps,
    )
)

SELECTED_HAWKES_MAXIMUM_PARAMETER_ABSOLUTE_DIFFERENCE: Final[
    float
] = float(
    np.max(selected_parameter_absolute_differences)
)

SELECTED_HAWKES_MAXIMUM_PARAMETER_RELATIVE_DIFFERENCE: Final[
    float
] = float(
    np.max(selected_parameter_relative_differences)
)


# ------------------------------------------------------------
# Selected-model left-edge sensitivity
# ------------------------------------------------------------

selected_left_edge_rows: list[dict[str, Any]] = []

for prefix_seconds in LEFT_EDGE_HISTORY_ONLY_SECONDS:
    prepared_replay = prepare_left_edge_replay(
        DEVELOPMENT_BATCHES_FOR_HAWKES,
        SELECTED_HAWKES_PARAMETERS,
        prefix_seconds=prefix_seconds,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
    )

    scoring_result = score_prepared_left_edge_replay(
        prepared_replay,
        SELECTED_HAWKES_PARAMETERS,
    )

    selected_left_edge_rows.append(
        {
            "prefix_seconds": float(prefix_seconds),
            "history_batch_count": (
                prepared_replay.window.history_batch_count
            ),
            "history_event_count": (
                prepared_replay.window.history_event_count
            ),
            "scoring_batch_count": (
                prepared_replay.window.scoring_batch_count
            ),
            "scoring_event_count": (
                prepared_replay.window.scoring_event_count
            ),
            "initial_buy_excitation": (
                prepared_replay
                .initial_scoring_state
                .buy_excitation
            ),
            "initial_sell_excitation": (
                prepared_replay
                .initial_scoring_state
                .sell_excitation
            ),
            "log_likelihood": (
                scoring_result.log_likelihood
            ),
            "log_score_per_event": (
                scoring_result.log_likelihood
                / scoring_result.event_count
            ),
            "terminal_buy_excitation": (
                scoring_result
                .final_state
                .buy_excitation
            ),
            "terminal_sell_excitation": (
                scoring_result
                .final_state
                .sell_excitation
            ),
            "left_censoring_recorded": True,
            "fabricated_prehistory": False,
            "status": "PASS",
        }
    )

SELECTED_HAWKES_LEFT_EDGE_SENSITIVITY = pd.DataFrame(
    selected_left_edge_rows
)

require(
    SELECTED_HAWKES_LEFT_EDGE_SENSITIVITY[
        "status"
    ].eq("PASS").all(),
    "A selected-model left-edge sensitivity replay failed.",
)


# ------------------------------------------------------------
# Derived selected-model quantities
# ------------------------------------------------------------

(
    selected_half_life_buy_seconds,
    selected_half_life_sell_seconds,
) = excitation_half_lives_seconds(
    SELECTED_HAWKES_PARAMETERS
)

(
    selected_mean_buy_intensity,
    selected_mean_sell_intensity,
) = model_implied_mean_intensities(
    SELECTED_HAWKES_PARAMETERS
)

(
    selected_exogenous_buy_share,
    selected_exogenous_sell_share,
) = model_implied_exogenous_shares(
    SELECTED_HAWKES_PARAMETERS
)

SELECTED_HAWKES_BRANCHING_MATRIX: Final[
    np.ndarray
] = integrated_kernel_matrix(
    SELECTED_HAWKES_PARAMETERS
)

require(
    np.allclose(
        SELECTED_HAWKES_BRANCHING_MATRIX[
            [0, 1],
            [1, 0],
        ],
        0.0,
        rtol=0.0,
        atol=0.0,
    ),
    "A prohibited cross-excitation channel is nonzero.",
)


# ------------------------------------------------------------
# Multiplicative guardrail audit
#
# This avoids treating a millisecond-scale half-life as near a
# boundary merely because the full allowable interval extends to
# hundreds of seconds.
# ------------------------------------------------------------

SELECTED_HAWKES_NUMERICAL_GUARDRAIL_AUDIT = pd.DataFrame(
    [
        {
            "parameter": "mu_buy_per_second",
            "value": SELECTED_HAWKES_PARAMETERS.mu_buy,
            "lower_bound": (
                MINIMUM_BASE_INTENSITY_PER_SECOND
            ),
            "upper_bound": (
                MAXIMUM_BASE_INTENSITY_PER_SECOND
            ),
            "lower_distance_ratio": (
                SELECTED_HAWKES_PARAMETERS.mu_buy
                / MINIMUM_BASE_INTENSITY_PER_SECOND
            ),
            "upper_distance_ratio": (
                MAXIMUM_BASE_INTENSITY_PER_SECOND
                / SELECTED_HAWKES_PARAMETERS.mu_buy
            ),
            "near_guardrail": False,
        },
        {
            "parameter": "mu_sell_per_second",
            "value": SELECTED_HAWKES_PARAMETERS.mu_sell,
            "lower_bound": (
                MINIMUM_BASE_INTENSITY_PER_SECOND
            ),
            "upper_bound": (
                MAXIMUM_BASE_INTENSITY_PER_SECOND
            ),
            "lower_distance_ratio": (
                SELECTED_HAWKES_PARAMETERS.mu_sell
                / MINIMUM_BASE_INTENSITY_PER_SECOND
            ),
            "upper_distance_ratio": (
                MAXIMUM_BASE_INTENSITY_PER_SECOND
                / SELECTED_HAWKES_PARAMETERS.mu_sell
            ),
            "near_guardrail": False,
        },
        {
            "parameter": "kappa_buy",
            "value": SELECTED_HAWKES_PARAMETERS.kappa_buy,
            "lower_bound": MINIMUM_OPTIMIZATION_KAPPA,
            "upper_bound": MAXIMUM_OPTIMIZATION_KAPPA,
            "lower_distance_ratio": (
                SELECTED_HAWKES_PARAMETERS.kappa_buy
                / MINIMUM_OPTIMIZATION_KAPPA
            ),
            "upper_distance_ratio": (
                MAXIMUM_OPTIMIZATION_KAPPA
                / SELECTED_HAWKES_PARAMETERS.kappa_buy
            ),
            "near_guardrail": bool(
                SELECTED_HAWKES_PARAMETERS.kappa_buy
                <= 1e-6
                or SELECTED_HAWKES_PARAMETERS.kappa_buy
                >= STATIONARITY_ACCEPTANCE_MARGIN
            ),
        },
        {
            "parameter": "kappa_sell",
            "value": SELECTED_HAWKES_PARAMETERS.kappa_sell,
            "lower_bound": MINIMUM_OPTIMIZATION_KAPPA,
            "upper_bound": MAXIMUM_OPTIMIZATION_KAPPA,
            "lower_distance_ratio": (
                SELECTED_HAWKES_PARAMETERS.kappa_sell
                / MINIMUM_OPTIMIZATION_KAPPA
            ),
            "upper_distance_ratio": (
                MAXIMUM_OPTIMIZATION_KAPPA
                / SELECTED_HAWKES_PARAMETERS.kappa_sell
            ),
            "near_guardrail": bool(
                SELECTED_HAWKES_PARAMETERS.kappa_sell
                <= 1e-6
                or SELECTED_HAWKES_PARAMETERS.kappa_sell
                >= STATIONARITY_ACCEPTANCE_MARGIN
            ),
        },
        {
            "parameter": "buy_half_life_seconds",
            "value": selected_half_life_buy_seconds,
            "lower_bound": (
                MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
            ),
            "upper_bound": (
                MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
            ),
            "lower_distance_ratio": (
                selected_half_life_buy_seconds
                / MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
            ),
            "upper_distance_ratio": (
                MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
                / selected_half_life_buy_seconds
            ),
            "near_guardrail": bool(
                selected_half_life_buy_seconds
                <= (
                    1.25
                    * MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
                )
                or selected_half_life_buy_seconds
                >= (
                    MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
                    / 1.25
                )
            ),
        },
        {
            "parameter": "sell_half_life_seconds",
            "value": selected_half_life_sell_seconds,
            "lower_bound": (
                MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
            ),
            "upper_bound": (
                MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
            ),
            "lower_distance_ratio": (
                selected_half_life_sell_seconds
                / MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
            ),
            "upper_distance_ratio": (
                MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
                / selected_half_life_sell_seconds
            ),
            "near_guardrail": bool(
                selected_half_life_sell_seconds
                <= (
                    1.25
                    * MINIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
                )
                or selected_half_life_sell_seconds
                >= (
                    MAXIMUM_OPTIMIZATION_HALF_LIFE_SECONDS
                    / 1.25
                )
            ),
        },
    ]
)

SELECTED_HAWKES_ANY_NUMERICAL_GUARDRAIL_HIT: Final[
    bool
] = bool(
    SELECTED_HAWKES_NUMERICAL_GUARDRAIL_AUDIT[
        "near_guardrail"
    ].any()
)


# ------------------------------------------------------------
# Final parameter ledger
# ------------------------------------------------------------

SELECTED_HAWKES_PARAMETER_LEDGER = pd.DataFrame(
    [
        {
            "parameter": "mu_buy_per_second",
            "value": SELECTED_HAWKES_PARAMETERS.mu_buy,
            "role": "BUY_EXOGENOUS_BASE_INTENSITY",
            "constraint": "STRICTLY_POSITIVE",
        },
        {
            "parameter": "mu_sell_per_second",
            "value": SELECTED_HAWKES_PARAMETERS.mu_sell,
            "role": "SELL_EXOGENOUS_BASE_INTENSITY",
            "constraint": "STRICTLY_POSITIVE",
        },
        {
            "parameter": "kappa_buy",
            "value": SELECTED_HAWKES_PARAMETERS.kappa_buy,
            "role": "BUY_TO_BUY_INTEGRATED_KERNEL_MASS",
            "constraint": "NONNEGATIVE_AND_BELOW_ONE",
        },
        {
            "parameter": "kappa_sell",
            "value": SELECTED_HAWKES_PARAMETERS.kappa_sell,
            "role": "SELL_TO_SELL_INTEGRATED_KERNEL_MASS",
            "constraint": "NONNEGATIVE_AND_BELOW_ONE",
        },
        {
            "parameter": "beta_buy_per_second",
            "value": SELECTED_HAWKES_PARAMETERS.beta_buy,
            "role": "BUY_EXCITATION_DECAY_RATE",
            "constraint": "STRICTLY_POSITIVE",
        },
        {
            "parameter": "beta_sell_per_second",
            "value": SELECTED_HAWKES_PARAMETERS.beta_sell,
            "role": "SELL_EXCITATION_DECAY_RATE",
            "constraint": "STRICTLY_POSITIVE",
        },
        {
            "parameter": "buy_half_life_seconds",
            "value": selected_half_life_buy_seconds,
            "role": "BUY_EXCITATION_HALF_LIFE",
            "constraint": "DERIVED_POSITIVE",
        },
        {
            "parameter": "sell_half_life_seconds",
            "value": selected_half_life_sell_seconds,
            "role": "SELL_EXCITATION_HALF_LIFE",
            "constraint": "DERIVED_POSITIVE",
        },
        {
            "parameter": "spectral_radius",
            "value": selected_final_validation[
                "spectral_radius"
            ],
            "role": "STATIONARITY_DIAGNOSTIC",
            "constraint": "STRICTLY_BELOW_ONE",
        },
        {
            "parameter": "model_implied_mean_buy_intensity",
            "value": selected_mean_buy_intensity,
            "role": "STATIONARY_MEAN_BUY_INTENSITY",
            "constraint": "DERIVED_POSITIVE",
        },
        {
            "parameter": "model_implied_mean_sell_intensity",
            "value": selected_mean_sell_intensity,
            "role": "STATIONARY_MEAN_SELL_INTENSITY",
            "constraint": "DERIVED_POSITIVE",
        },
        {
            "parameter": "model_implied_exogenous_buy_share",
            "value": selected_exogenous_buy_share,
            "role": "DESCRIPTIVE_MODEL_QUANTITY",
            "constraint": "NOT_A_CAUSAL_FRACTION",
        },
        {
            "parameter": "model_implied_exogenous_sell_share",
            "value": selected_exogenous_sell_share,
            "role": "DESCRIPTIVE_MODEL_QUANTITY",
            "constraint": "NOT_A_CAUSAL_FRACTION",
        },
    ]
)

SELECTED_HAWKES_PARAMETER_LEDGER[
    "finite"
] = np.isfinite(
    SELECTED_HAWKES_PARAMETER_LEDGER[
        "value"
    ].to_numpy(dtype=np.float64)
)

SELECTED_HAWKES_PARAMETER_LEDGER[
    "status"
] = np.where(
    SELECTED_HAWKES_PARAMETER_LEDGER[
        "finite"
    ],
    "PASS",
    "FAIL",
)

require(
    SELECTED_HAWKES_PARAMETER_LEDGER[
        "finite"
    ].all(),
    "The selected Hawkes parameter ledger contains non-finite values.",
)


# ------------------------------------------------------------
# Freeze the portable selected-model package
# ------------------------------------------------------------

SELECTED_HAWKES_MODEL_PACKAGE: Final[
    dict[str, Any]
] = {
    "artifact_type": "NOTEBOOK_07_SELECTED_HAWKES_MODEL",
    "schema_version": (
        "NOTEBOOK_07_SELECTED_HAWKES_MODEL_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "model_id": SELECTED_HAWKES_MODEL_ID,
    "model_family": AUTHORIZED_FIRST_MODEL,
    "estimator_label": PRIMARY_ESTIMATOR_LABEL,
    "selection_reason": SELECTED_HAWKES_MODEL_REASON,
    "candidate_selection_sha256": (
        HAWKES_CANDIDATE_SELECTION_SHA256
    ),
    "fit_partition": FIT_PARTITION,
    "fit_event_count": EXPECTED_DEVELOPMENT_EVENT_ROWS,
    "fit_batch_count": EXPECTED_DEVELOPMENT_BATCH_ROWS,
    "fit_start_ns": DEVELOPMENT_START_NS,
    "fit_end_exclusive_ns": (
        DEVELOPMENT_END_EXCLUSIVE_NS
    ),
    "calibration_used_for_fitting": False,
    "calibration_used_for_selection": False,
    "calibration_parameter_updates": 0,
    "authorized_channels": list(
        AUTHORIZED_CHANNELS
    ),
    "fixed_zero_channels": list(
        FIXED_ZERO_CHANNELS
    ),
    "kernel_family": AUTHORIZED_KERNEL_FAMILY,
    "shared_decay": (
        SELECTED_HAWKES_SPECIFICATION.shared_decay
    ),
    "parameters": {
        "mu_buy_per_second": (
            SELECTED_HAWKES_PARAMETERS.mu_buy
        ),
        "mu_sell_per_second": (
            SELECTED_HAWKES_PARAMETERS.mu_sell
        ),
        "kappa_buy": (
            SELECTED_HAWKES_PARAMETERS.kappa_buy
        ),
        "kappa_sell": (
            SELECTED_HAWKES_PARAMETERS.kappa_sell
        ),
        "beta_buy_per_second": (
            SELECTED_HAWKES_PARAMETERS.beta_buy
        ),
        "beta_sell_per_second": (
            SELECTED_HAWKES_PARAMETERS.beta_sell
        ),
    },
    "derived_quantities": {
        "buy_half_life_seconds": (
            selected_half_life_buy_seconds
        ),
        "sell_half_life_seconds": (
            selected_half_life_sell_seconds
        ),
        "spectral_radius": selected_final_validation[
            "spectral_radius"
        ],
        "stationarity_class": selected_final_validation[
            "stationarity_class"
        ],
        "acceptance_margin_holds": (
            selected_final_validation[
                "acceptance_margin_holds"
            ]
        ),
        "model_implied_mean_buy_intensity": (
            selected_mean_buy_intensity
        ),
        "model_implied_mean_sell_intensity": (
            selected_mean_sell_intensity
        ),
        "model_implied_exogenous_buy_share": (
            selected_exogenous_buy_share
        ),
        "model_implied_exogenous_sell_share": (
            selected_exogenous_sell_share
        ),
    },
    "development_fit": {
        "negative_log_likelihood": (
            SELECTED_HAWKES_DEVELOPMENT_NLL
        ),
        "log_likelihood": (
            SELECTED_HAWKES_DEVELOPMENT_LOG_LIKELIHOOD
        ),
        "log_score_per_event": (
            SELECTED_HAWKES_DEVELOPMENT_LOG_LIKELIHOOD
            / EXPECTED_DEVELOPMENT_EVENT_ROWS
        ),
        "gradient_inf_per_event": (
            SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT
        ),
        "gradient_l2_per_event": (
            SELECTED_HAWKES_FINAL_GRADIENT_L2_PER_EVENT
        ),
        "optimizer_success": bool(
            SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.success
        ),
        "optimizer_status_code": int(
            SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.status
        ),
        "optimizer_message": str(
            SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.message
        ),
        "optimizer_iterations": int(
            SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.nit
        ),
        "optimizer_function_evaluations": int(
            SELECTED_HAWKES_FINAL_OPTIMIZER_RESULT.nfev
        ),
    },
    "development_terminal_state": {
        "state_time_ns": (
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .state_time_ns
        ),
        "buy_excitation": (
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .buy_excitation
        ),
        "sell_excitation": (
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .sell_excitation
        ),
    },
    "left_edge_contract": {
        "zero_initial_excitation": True,
        "fabricated_prehistory": False,
        "left_censoring_recorded": True,
        "history_only_prefix_seconds": [
            float(value)
            for value in LEFT_EDGE_HISTORY_ONLY_SECONDS
        ],
    },
    "exact_time_batch_contract": {
        "strict_pre_batch_history": True,
        "within_batch_zero_lag_excitation": False,
        "batch_update_after_scoring": True,
        "timestamp_jitter": False,
        "artificial_ordering": False,
    },
    "claim_limits": {
        "hawkes_superiority_authorized": False,
        "cross_excitation_authorized": False,
        "state_dependent_hawkes_authorized": False,
        "strategy_or_quoting_authorized": False,
        "fill_simulation_authorized": False,
        "pnl_claim_authorized": False,
    },
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "future_labels_used": False,
    "market_state_features_used": False,
    "status": "FROZEN_DEVELOPMENT_SELECTED_MODEL",
}

SELECTED_HAWKES_MODEL_PACKAGE_SHA256: Final[str] = (
    canonical_json_sha256(
        SELECTED_HAWKES_MODEL_PACKAGE
    )
)


# ------------------------------------------------------------
# Reproducibility and freeze ledgers
# ------------------------------------------------------------

SELECTED_HAWKES_REFIT_REPRODUCIBILITY_AUDIT = (
    pd.DataFrame(
        [
            {
                "audit_item": "MODEL_ID_UNCHANGED",
                "expected": SELECTED_HAWKES_MODEL_ID,
                "observed": (
                    SELECTED_HAWKES_PARAMETERS.model_id
                ),
                "passed": (
                    SELECTED_HAWKES_PARAMETERS.model_id
                    == SELECTED_HAWKES_MODEL_ID
                ),
            },
            {
                "audit_item": "PRIOR_FINAL_NLL_RECONCILIATION",
                "expected": (
                    SELECTED_PRIOR_DEVELOPMENT_NLL
                ),
                "observed": (
                    SELECTED_HAWKES_DEVELOPMENT_NLL
                ),
                "passed": (
                    abs(
                        selected_nll_difference_per_event
                    )
                    <= 1e-8
                ),
            },
            {
                "audit_item": "OBJECTIVE_REPLAY_RECONCILIATION",
                "expected": (
                    SELECTED_HAWKES_DEVELOPMENT_NLL
                ),
                "observed": (
                    SELECTED_HAWKES_DEVELOPMENT_REPLAY
                    .negative_log_likelihood
                ),
                "passed": math.isclose(
                    SELECTED_HAWKES_DEVELOPMENT_REPLAY
                    .negative_log_likelihood,
                    SELECTED_HAWKES_DEVELOPMENT_NLL,
                    rel_tol=FLOAT_RELATIVE_TOLERANCE,
                    abs_tol=1e-7,
                ),
            },
            {
                "audit_item": "EXACT_GRADIENT_ACCEPTANCE",
                "expected": (
                    OPTIMIZER_ACCEPTANCE_GRADIENT_INF_PER_EVENT
                ),
                "observed": (
                    SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT
                ),
                "passed": (
                    SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT
                    <= OPTIMIZER_ACCEPTANCE_GRADIENT_INF_PER_EVENT
                ),
            },
            {
                "audit_item": "STATIONARITY",
                "expected": "SPECTRAL_RADIUS_BELOW_ONE",
                "observed": selected_final_validation[
                    "spectral_radius"
                ],
                "passed": selected_final_validation[
                    "stationarity_holds"
                ],
            },
            {
                "audit_item": "AUTHORIZED_CHANNELS_ONLY",
                "expected": (
                    "DIAGONAL_SELF_EXCITATION_ONLY"
                ),
                "observed": (
                    "DIAGONAL_SELF_EXCITATION_ONLY"
                ),
                "passed": True,
            },
            {
                "audit_item": "CALIBRATION_USED_FOR_REFIT",
                "expected": False,
                "observed": False,
                "passed": True,
            },
            {
                "audit_item": "CALIBRATION_PARAMETER_UPDATES",
                "expected": 0,
                "observed": CALIBRATION_PARAMETER_UPDATES,
                "passed": (
                    CALIBRATION_PARAMETER_UPDATES == 0
                ),
            },
        ]
    )
)

SELECTED_HAWKES_REFIT_REPRODUCIBILITY_AUDIT[
    "status"
] = np.where(
    SELECTED_HAWKES_REFIT_REPRODUCIBILITY_AUDIT[
        "passed"
    ],
    "PASS",
    "FAIL",
)

require(
    SELECTED_HAWKES_REFIT_REPRODUCIBILITY_AUDIT[
        "passed"
    ].all(),
    "The selected-model reproducibility audit failed.",
)


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

HAWKES_SELECTED_MODEL_REFIT_COMPLETED = True
HAWKES_SELECTED_MODEL_FROZEN: bool = True
HAWKES_SELECTED_MODEL_LEFT_EDGE_AUDITED: bool = True
HAWKES_SELECTED_MODEL_DEVELOPMENT_STATE_FROZEN: bool = True

CALIBRATION_USED_FOR_REFIT: bool = False
CALIBRATION_SCORING_STARTED: bool = False
CALIBRATION_PARAMETER_UPDATES_PERFORMED: int = 0

FILESYSTEM_WRITES_PERFORMED = False

selected_model_freeze_summary = pd.DataFrame(
    [
        {
            "field": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "selection_reason",
            "value": SELECTED_HAWKES_MODEL_REASON,
        },
        {
            "field": "selected_model_refit_completed",
            "value": HAWKES_SELECTED_MODEL_REFIT_COMPLETED,
        },
        {
            "field": "selected_model_frozen",
            "value": HAWKES_SELECTED_MODEL_FROZEN,
        },
        {
            "field": "estimator_label",
            "value": PRIMARY_ESTIMATOR_LABEL,
        },
        {
            "field": "development_negative_log_likelihood",
            "value": SELECTED_HAWKES_DEVELOPMENT_NLL,
        },
        {
            "field": "development_log_score_per_event",
            "value": (
                SELECTED_HAWKES_DEVELOPMENT_LOG_LIKELIHOOD
                / EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
        },
        {
            "field": "prior_minus_final_nll",
            "value": (
                SELECTED_PRIOR_DEVELOPMENT_NLL
                - SELECTED_HAWKES_DEVELOPMENT_NLL
            ),
        },
        {
            "field": "maximum_parameter_relative_difference",
            "value": (
                SELECTED_HAWKES_MAXIMUM_PARAMETER_RELATIVE_DIFFERENCE
            ),
        },
        {
            "field": "gradient_inf_per_event",
            "value": (
                SELECTED_HAWKES_FINAL_GRADIENT_INF_PER_EVENT
            ),
        },
        {
            "field": "spectral_radius",
            "value": selected_final_validation[
                "spectral_radius"
            ],
        },
        {
            "field": "stationarity_class",
            "value": selected_final_validation[
                "stationarity_class"
            ],
        },
        {
            "field": "any_numerical_guardrail_hit",
            "value": (
                SELECTED_HAWKES_ANY_NUMERICAL_GUARDRAIL_HIT
            ),
        },
        {
            "field": "development_terminal_buy_excitation",
            "value": (
                SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
                .buy_excitation
            ),
        },
        {
            "field": "development_terminal_sell_excitation",
            "value": (
                SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
                .sell_excitation
            ),
        },
        {
            "field": "selected_model_package_sha256",
            "value": (
                SELECTED_HAWKES_MODEL_PACKAGE_SHA256
            ),
        },
        {
            "field": "calibration_used_for_refit",
            "value": CALIBRATION_USED_FOR_REFIT,
        },
        {
            "field": "calibration_scoring_started",
            "value": CALIBRATION_SCORING_STARTED,
        },
        {
            "field": "calibration_parameter_updates_performed",
            "value": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(selected_model_freeze_summary)
display(SELECTED_HAWKES_PARAMETER_LEDGER)
display(SELECTED_HAWKES_NUMERICAL_GUARDRAIL_AUDIT)
display(SELECTED_HAWKES_LEFT_EDGE_SENSITIVITY)
display(SELECTED_HAWKES_REFIT_REPRODUCIBILITY_AUDIT)

print(
    f"{SELECTED_HAWKES_MODEL_ID} was deterministically refined and "
    "frozen using DEVELOPMENT only. The final objective reconciles "
    "with the previously selected optimum and with an independent "
    "strict-pre-batch replay. Parameter admissibility, stationarity, "
    "numerical guardrails, left-edge sensitivity, and the terminal "
    "DEVELOPMENT excitation state were audited. CALIBRATION scoring "
    "has not started and received zero parameter updates. Protected "
    "partitions remain unopened and no filesystem writes were "
    "performed."
)

,field,value
0,selected_model_id,H1_DIAGONAL_SHARED_DECAY
1,selection_reason,H1_SIMPLICITY_FIRST_H2_FAILED_STABLE_MATERIAL_...
2,selected_model_refit_completed,True
3,selected_model_frozen,True
4,estimator_label,STRICT_PRE_BATCH_COARSENED_TIME_QUASI_MLE
5,development_negative_log_likelihood,"1,005.46314"
6,development_log_score_per_event,-0.1435555596
7,prior_minus_final_nll,0
8,maximum_parameter_relative_difference,0
9,gradient_inf_per_event,4.717983947e-09


,parameter,value,role,constraint,finite,status
0,mu_buy_per_second,1.536124799,BUY_EXOGENOUS_BASE_INTENSITY,STRICTLY_POSITIVE,True,PASS
1,mu_sell_per_second,1.767916532,SELL_EXOGENOUS_BASE_INTENSITY,STRICTLY_POSITIVE,True,PASS
2,kappa_buy,0.1892556591,BUY_TO_BUY_INTEGRATED_KERNEL_MASS,NONNEGATIVE_AND_BELOW_ONE,True,PASS
3,kappa_sell,0.1126637105,SELL_TO_SELL_INTEGRATED_KERNEL_MASS,NONNEGATIVE_AND_BELOW_ONE,True,PASS
4,beta_buy_per_second,70.67941658,BUY_EXCITATION_DECAY_RATE,STRICTLY_POSITIVE,True,PASS
5,beta_sell_per_second,70.67941658,SELL_EXCITATION_DECAY_RATE,STRICTLY_POSITIVE,True,PASS
6,buy_half_life_seconds,0.00980691712,BUY_EXCITATION_HALF_LIFE,DERIVED_POSITIVE,True,PASS
7,sell_half_life_seconds,0.00980691712,SELL_EXCITATION_HALF_LIFE,DERIVED_POSITIVE,True,PASS
8,spectral_radius,0.1892556591,STATIONARITY_DIAGNOSTIC,STRICTLY_BELOW_ONE,True,PASS
9,model_implied_mean_buy_intensity,1.894709246,STATIONARY_MEAN_BUY_INTENSITY,DERIVED_POSITIVE,True,PASS


,parameter,value,lower_bound,upper_bound,lower_distance_ratio,upper_distance_ratio,near_guardrail
0,mu_buy_per_second,1.536124799,1e-10,500,1.536124799e+10,325.4943872,False
1,mu_sell_per_second,1.767916532,1e-10,500,1.767916532e+10,282.8187819,False
2,kappa_buy,0.1892556591,1e-10,0.999,"1,892,556,591",5.278573992,False
3,kappa_sell,0.1126637105,1e-10,0.999,"1,126,637,105",8.867096563,False
4,buy_half_life_seconds,0.00980691712,0.001,300,9.80691712,"30,590.65314",False
5,sell_half_life_seconds,0.00980691712,0.001,300,9.80691712,"30,590.65314",False


,prefix_seconds,history_batch_count,history_event_count,scoring_batch_count,scoring_event_count,initial_buy_excitation,initial_sell_excitation,log_likelihood,log_score_per_event,terminal_buy_excitation,terminal_sell_excitation,left_censoring_recorded,fabricated_prehistory,status
0,0,0,0,6859,7004,0,0,"-1,005.46314",-0.1435555596,2.541109025e-16,0.0001021398953,True,False,PASS
1,1,0,0,6859,7004,0,0,"-1,002.159098",-0.1430838233,2.541109025e-16,0.0001021398953,True,False,PASS
2,5,12,12,6847,6992,1.613458777e-08,2.97493885e-10,-993.1967942,-0.1420475964,2.541109025e-16,0.0001021398953,True,False,PASS


,audit_item,expected,observed,passed,status
0,MODEL_ID_UNCHANGED,H1_DIAGONAL_SHARED_DECAY,H1_DIAGONAL_SHARED_DECAY,True,PASS
1,PRIOR_FINAL_NLL_RECONCILIATION,"1,005.46314","1,005.46314",True,PASS
2,OBJECTIVE_REPLAY_RECONCILIATION,"1,005.46314","1,005.46314",True,PASS
3,EXACT_GRADIENT_ACCEPTANCE,1e-05,4.717983947e-09,True,PASS
4,STATIONARITY,SPECTRAL_RADIUS_BELOW_ONE,0.1892556591,True,PASS
5,AUTHORIZED_CHANNELS_ONLY,DIAGONAL_SELF_EXCITATION_ONLY,DIAGONAL_SELF_EXCITATION_ONLY,True,PASS
6,CALIBRATION_USED_FOR_REFIT,False,False,True,PASS
7,CALIBRATION_PARAMETER_UPDATES,0,0,True,PASS


H1_DIAGONAL_SHARED_DECAY was deterministically refined and frozen using DEVELOPMENT only. The final objective reconciles with the previously selected optimum and with an independent strict-pre-batch replay. Parameter admissibility, stationarity, numerical guardrails, left-edge sensitivity, and the terminal DEVELOPMENT excitation state were audited. CALIBRATION scoring has not started and received zero parameter updates. Protected partitions remain unopened and no filesystem writes were performed.


In [12]:
# ============================================================
# Chronological stability audit for the selected Hawkes model
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "HAWKES_SELECTED_MODEL_REFIT_COMPLETED",
            False,
        )
    ),
    "The selected Hawkes model has not been refitted.",
)
require(
    bool(
        globals().get(
            "HAWKES_SELECTED_MODEL_FROZEN",
            False,
        )
    ),
    "The selected Hawkes model has not been frozen.",
)
require(
    bool(
        globals().get(
            "HAWKES_CHRONOLOGICAL_FOLD_AUDIT_COMPLETED",
            False,
        )
    ),
    "The DEVELOPMENT chronological-fold audit is incomplete.",
)
require(
    "HAWKES_DEVELOPMENT_FOLD_RESULTS" in globals(),
    "The canonical DEVELOPMENT fold-result table is unavailable.",
)
require(
    "HAWKES_DEVELOPMENT_FOLD_COMPARISON" in globals(),
    "The paired H1-versus-H2 fold comparison is unavailable.",
)
require(
    CALIBRATION_SCORING_STARTED is False,
    "CALIBRATION scoring must not have started.",
)
require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION must have received zero parameter updates.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Frozen stability thresholds
#
# These are engineering stability limits, not claims that the
# parameters are invariant through time.
# ------------------------------------------------------------

MAXIMUM_FOLD_PARAMETER_COEFFICIENT_OF_VARIATION: Final[
    float
] = 0.35

MAXIMUM_ANY_FOLD_TO_FULL_DEVIATION_FACTOR: Final[
    float
] = 2.50

MAXIMUM_LATEST_FOLD_TO_FULL_DEVIATION_FACTOR: Final[
    float
] = 1.50

MAXIMUM_FIRST_TO_LAST_DEVIATION_FACTOR: Final[
    float
] = 2.00

MINIMUM_SELECTED_MODEL_FOLD_WIN_FRACTION: Final[
    float
] = 0.60

require(
    MAXIMUM_FOLD_PARAMETER_COEFFICIENT_OF_VARIATION
    > 0.0,
    "The parameter-CV threshold must be positive.",
)
require(
    MAXIMUM_ANY_FOLD_TO_FULL_DEVIATION_FACTOR
    > 1.0,
    "The maximum fold-to-full deviation factor must exceed one.",
)
require(
    MAXIMUM_LATEST_FOLD_TO_FULL_DEVIATION_FACTOR
    > 1.0,
    "The latest-fold deviation factor must exceed one.",
)
require(
    MAXIMUM_FIRST_TO_LAST_DEVIATION_FACTOR
    > 1.0,
    "The first-to-last deviation factor must exceed one.",
)
require(
    0.0
    < MINIMUM_SELECTED_MODEL_FOLD_WIN_FRACTION
    <= 1.0,
    "The minimum fold-win fraction is invalid.",
)


# ------------------------------------------------------------
# Canonical selected-model fold results
# ------------------------------------------------------------

SELECTED_HAWKES_FOLD_RESULTS = (
    HAWKES_DEVELOPMENT_FOLD_RESULTS.loc[
        HAWKES_DEVELOPMENT_FOLD_RESULTS[
            "model_id"
        ].eq(SELECTED_HAWKES_MODEL_ID)
    ]
    .sort_values(
        [
            "fold_number",
            "fold_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    len(SELECTED_HAWKES_FOLD_RESULTS)
    == DEVELOPMENT_CHRONOLOGICAL_FOLDS,
    (
        "The selected model does not have exactly one result for "
        "each DEVELOPMENT fold."
    ),
)

required_selected_fold_columns: Final[set[str]] = {
    "fold_number",
    "fold_id",
    "training_event_count",
    "validation_event_count",
    "validation_log_score_per_event",
    "mu_buy",
    "mu_sell",
    "kappa_buy",
    "kappa_sell",
    "half_life_buy_seconds",
    "half_life_sell_seconds",
    "spectral_radius",
}

missing_selected_fold_columns = (
    required_selected_fold_columns
    - set(SELECTED_HAWKES_FOLD_RESULTS.columns)
)

require(
    not missing_selected_fold_columns,
    (
        "Selected-model fold results are missing columns: "
        f"{sorted(missing_selected_fold_columns)}"
    ),
)

require(
    SELECTED_HAWKES_FOLD_RESULTS[
        "fold_number"
    ].tolist()
    == list(
        range(
            1,
            DEVELOPMENT_CHRONOLOGICAL_FOLDS + 1,
        )
    ),
    "Selected-model fold numbers are not sequential.",
)

for numeric_column in (
    "validation_log_score_per_event",
    "mu_buy",
    "mu_sell",
    "kappa_buy",
    "kappa_sell",
    "half_life_buy_seconds",
    "half_life_sell_seconds",
    "spectral_radius",
):
    numeric_values = pd.to_numeric(
        SELECTED_HAWKES_FOLD_RESULTS[
            numeric_column
        ],
        errors="raise",
    ).to_numpy(dtype=np.float64)

    require(
        np.isfinite(numeric_values).all(),
        (
            "Selected-model fold results contain non-finite values "
            f"in {numeric_column}."
        ),
    )

for positive_column in (
    "mu_buy",
    "mu_sell",
    "half_life_buy_seconds",
    "half_life_sell_seconds",
):
    require(
        SELECTED_HAWKES_FOLD_RESULTS[
            positive_column
        ].gt(0.0).all(),
        (
            "Selected-model folds contain a nonpositive value in "
            f"{positive_column}."
        ),
    )

for mass_column in (
    "kappa_buy",
    "kappa_sell",
):
    require(
        SELECTED_HAWKES_FOLD_RESULTS[
            mass_column
        ].ge(0.0).all(),
        (
            "Selected-model folds contain a negative value in "
            f"{mass_column}."
        ),
    )
    require(
        SELECTED_HAWKES_FOLD_RESULTS[
            mass_column
        ].lt(1.0).all(),
        (
            "Selected-model folds contain a nonstationary value in "
            f"{mass_column}."
        ),
    )

require(
    SELECTED_HAWKES_FOLD_RESULTS[
        "spectral_radius"
    ].lt(STATIONARITY_HARD_LIMIT).all(),
    "At least one selected-model fold violates stationarity.",
)
require(
    SELECTED_HAWKES_FOLD_RESULTS[
        "spectral_radius"
    ].le(STATIONARITY_ACCEPTANCE_MARGIN).all(),
    (
        "At least one selected-model fold violates the engineering "
        "stationarity margin."
    ),
)

if (
    "accepted_start_count"
    in SELECTED_HAWKES_FOLD_RESULTS.columns
    and "attempted_start_count"
    in SELECTED_HAWKES_FOLD_RESULTS.columns
):
    require(
        SELECTED_HAWKES_FOLD_RESULTS[
            "accepted_start_count"
        ].eq(
            SELECTED_HAWKES_FOLD_RESULTS[
                "attempted_start_count"
            ]
        ).all(),
        (
            "At least one selected-model fold has rejected "
            "optimization starts."
        ),
    )

if (
    "any_parameter_near_numerical_boundary"
    in SELECTED_HAWKES_FOLD_RESULTS.columns
):
    require(
        not SELECTED_HAWKES_FOLD_RESULTS[
            "any_parameter_near_numerical_boundary"
        ].any(),
        (
            "At least one selected-model fold reports a parameter "
            "near a numerical boundary."
        ),
    )

if SELECTED_HAWKES_SPECIFICATION.shared_decay:
    require(
        np.allclose(
            SELECTED_HAWKES_FOLD_RESULTS[
                "half_life_buy_seconds"
            ].to_numpy(dtype=np.float64),
            SELECTED_HAWKES_FOLD_RESULTS[
                "half_life_sell_seconds"
            ].to_numpy(dtype=np.float64),
            rtol=FLOAT_RELATIVE_TOLERANCE,
            atol=FLOAT_ABSOLUTE_TOLERANCE,
        ),
        "A selected H1 fold violates the shared-decay contract.",
    )


# ------------------------------------------------------------
# Full-DEVELOPMENT parameter anchor
# ------------------------------------------------------------

SELECTED_HAWKES_FULL_PARAMETER_MAP: Final[
    Mapping[str, float]
] = {
    "mu_buy": SELECTED_HAWKES_PARAMETERS.mu_buy,
    "mu_sell": SELECTED_HAWKES_PARAMETERS.mu_sell,
    "kappa_buy": SELECTED_HAWKES_PARAMETERS.kappa_buy,
    "kappa_sell": SELECTED_HAWKES_PARAMETERS.kappa_sell,
    "half_life_buy_seconds": (
        selected_half_life_buy_seconds
    ),
    "half_life_sell_seconds": (
        selected_half_life_sell_seconds
    ),
    "spectral_radius": selected_final_validation[
        "spectral_radius"
    ],
}

SELECTED_HAWKES_STABILITY_PARAMETER_COLUMNS: Final[
    tuple[str, ...]
] = (
    "mu_buy",
    "mu_sell",
    "kappa_buy",
    "kappa_sell",
    "half_life_buy_seconds",
    "half_life_sell_seconds",
    "spectral_radius",
)

for parameter_name, parameter_value in (
    SELECTED_HAWKES_FULL_PARAMETER_MAP.items()
):
    require(
        math.isfinite(parameter_value),
        f"The full-fit {parameter_name} value is non-finite.",
    )
    require(
        parameter_value > 0.0,
        f"The full-fit {parameter_name} value is nonpositive.",
    )


# ------------------------------------------------------------
# Fold-level distance from the full-DEVELOPMENT fit
# ------------------------------------------------------------

fold_distance_rows: list[dict[str, Any]] = []

for fold_row in SELECTED_HAWKES_FOLD_RESULTS.itertuples(
    index=False
):
    absolute_log_ratios: list[float] = []
    relative_differences: list[float] = []

    fold_record: dict[str, Any] = {
        "fold_number": int(fold_row.fold_number),
        "fold_id": str(fold_row.fold_id),
        "training_event_count": int(
            fold_row.training_event_count
        ),
        "validation_event_count": int(
            fold_row.validation_event_count
        ),
        "validation_log_score_per_event": float(
            fold_row.validation_log_score_per_event
        ),
    }

    for parameter_name in (
        SELECTED_HAWKES_STABILITY_PARAMETER_COLUMNS
    ):
        fold_value = float(
            getattr(fold_row, parameter_name)
        )

        full_value = float(
            SELECTED_HAWKES_FULL_PARAMETER_MAP[
                parameter_name
            ]
        )

        absolute_log_ratio = abs(
            math.log(fold_value / full_value)
        )

        relative_difference = abs(
            fold_value - full_value
        ) / abs(full_value)

        absolute_log_ratios.append(
            absolute_log_ratio
        )
        relative_differences.append(
            relative_difference
        )

        fold_record[
            f"{parameter_name}_to_full_ratio"
        ] = fold_value / full_value

    maximum_absolute_log_ratio = max(
        absolute_log_ratios
    )

    fold_record[
        "maximum_absolute_log_ratio_to_full"
    ] = maximum_absolute_log_ratio

    fold_record[
        "maximum_deviation_factor_to_full"
    ] = math.exp(
        maximum_absolute_log_ratio
    )

    fold_record[
        "maximum_relative_difference_to_full"
    ] = max(relative_differences)

    fold_record["status"] = "PASS"

    fold_distance_rows.append(fold_record)

SELECTED_HAWKES_FOLD_DISTANCE_AUDIT = pd.DataFrame(
    fold_distance_rows
)

require(
    np.isfinite(
        SELECTED_HAWKES_FOLD_DISTANCE_AUDIT[
            "maximum_deviation_factor_to_full"
        ].to_numpy(dtype=np.float64)
    ).all(),
    "Fold-to-full parameter distances are non-finite.",
)


# ------------------------------------------------------------
# Parameter-level chronological stability
# ------------------------------------------------------------

parameter_stability_rows: list[dict[str, Any]] = []

for parameter_name in (
    SELECTED_HAWKES_STABILITY_PARAMETER_COLUMNS
):
    fold_values = (
        SELECTED_HAWKES_FOLD_RESULTS[
            parameter_name
        ].to_numpy(dtype=np.float64)
    )

    full_value = float(
        SELECTED_HAWKES_FULL_PARAMETER_MAP[
            parameter_name
        ]
    )

    parameter_mean = float(
        np.mean(fold_values)
    )

    parameter_standard_deviation = float(
        np.std(
            fold_values,
            ddof=1,
        )
    )

    coefficient_of_variation = (
        parameter_standard_deviation
        / abs(parameter_mean)
    )

    fold_to_full_absolute_log_ratios = np.abs(
        np.log(fold_values / full_value)
    )

    maximum_fold_to_full_deviation_factor = float(
        np.exp(
            np.max(
                fold_to_full_absolute_log_ratios
            )
        )
    )

    latest_fold_to_full_deviation_factor = float(
        math.exp(
            abs(
                math.log(
                    fold_values[-1] / full_value
                )
            )
        )
    )

    first_to_last_deviation_factor = float(
        math.exp(
            abs(
                math.log(
                    fold_values[-1] / fold_values[0]
                )
            )
        )
    )

    coefficient_of_variation_gate = bool(
        coefficient_of_variation
        <= (
            MAXIMUM_FOLD_PARAMETER_COEFFICIENT_OF_VARIATION
        )
    )

    any_fold_to_full_gate = bool(
        maximum_fold_to_full_deviation_factor
        <= MAXIMUM_ANY_FOLD_TO_FULL_DEVIATION_FACTOR
    )

    latest_fold_to_full_gate = bool(
        latest_fold_to_full_deviation_factor
        <= MAXIMUM_LATEST_FOLD_TO_FULL_DEVIATION_FACTOR
    )

    first_to_last_gate = bool(
        first_to_last_deviation_factor
        <= MAXIMUM_FIRST_TO_LAST_DEVIATION_FACTOR
    )

    parameter_passed = bool(
        coefficient_of_variation_gate
        and any_fold_to_full_gate
        and latest_fold_to_full_gate
        and first_to_last_gate
    )

    parameter_stability_rows.append(
        {
            "parameter": parameter_name,
            "full_development_value": full_value,
            "fold_mean": parameter_mean,
            "fold_standard_deviation": (
                parameter_standard_deviation
            ),
            "fold_coefficient_of_variation": (
                coefficient_of_variation
            ),
            "fold_minimum": float(
                np.min(fold_values)
            ),
            "fold_maximum": float(
                np.max(fold_values)
            ),
            "first_fold_value": float(
                fold_values[0]
            ),
            "latest_fold_value": float(
                fold_values[-1]
            ),
            "maximum_fold_to_full_deviation_factor": (
                maximum_fold_to_full_deviation_factor
            ),
            "latest_fold_to_full_deviation_factor": (
                latest_fold_to_full_deviation_factor
            ),
            "first_to_last_deviation_factor": (
                first_to_last_deviation_factor
            ),
            "coefficient_of_variation_gate": (
                coefficient_of_variation_gate
            ),
            "any_fold_to_full_gate": (
                any_fold_to_full_gate
            ),
            "latest_fold_to_full_gate": (
                latest_fold_to_full_gate
            ),
            "first_to_last_gate": (
                first_to_last_gate
            ),
            "passed": parameter_passed,
            "status": (
                "PASS"
                if parameter_passed
                else "FAIL_CHRONOLOGICAL_STABILITY"
            ),
        }
    )

SELECTED_HAWKES_PARAMETER_STABILITY_AUDIT = (
    pd.DataFrame(parameter_stability_rows)
)

require(
    SELECTED_HAWKES_PARAMETER_STABILITY_AUDIT[
        "passed"
    ].all(),
    (
        "At least one selected-model parameter failed the "
        "chronological stability limits."
    ),
)


# ------------------------------------------------------------
# Chronological validation-score summary
#
# Score sign is not used as a stability gate because the five
# validation windows have different realized activity conditions.
# ------------------------------------------------------------

selected_fold_scores = (
    SELECTED_HAWKES_FOLD_RESULTS[
        "validation_log_score_per_event"
    ].to_numpy(dtype=np.float64)
)

selected_fold_weights = (
    SELECTED_HAWKES_FOLD_RESULTS[
        "validation_event_count"
    ].to_numpy(dtype=np.float64)
)

SELECTED_HAWKES_WEIGHTED_VALIDATION_LOG_SCORE_PER_EVENT: Final[
    float
] = float(
    np.average(
        selected_fold_scores,
        weights=selected_fold_weights,
    )
)

SELECTED_HAWKES_MEAN_VALIDATION_LOG_SCORE_PER_EVENT: Final[
    float
] = float(
    np.mean(selected_fold_scores)
)

SELECTED_HAWKES_MEDIAN_VALIDATION_LOG_SCORE_PER_EVENT: Final[
    float
] = float(
    np.median(selected_fold_scores)
)

SELECTED_HAWKES_VALIDATION_LOG_SCORE_STANDARD_DEVIATION: Final[
    float
] = float(
    np.std(
        selected_fold_scores,
        ddof=1,
    )
)

SELECTED_HAWKES_VALIDATION_LOG_SCORE_MINIMUM: Final[
    float
] = float(
    np.min(selected_fold_scores)
)

SELECTED_HAWKES_VALIDATION_LOG_SCORE_MAXIMUM: Final[
    float
] = float(
    np.max(selected_fold_scores)
)

SELECTED_HAWKES_POSITIVE_SCORE_FOLD_COUNT: Final[int] = int(
    np.sum(selected_fold_scores > 0.0)
)

SELECTED_HAWKES_NEGATIVE_SCORE_FOLD_COUNT: Final[int] = int(
    np.sum(selected_fold_scores < 0.0)
)


# ------------------------------------------------------------
# Paired candidate-selection consistency
# ------------------------------------------------------------

selected_model_fold_win_count = int(
    HAWKES_DEVELOPMENT_FOLD_COMPARISON[
        "fold_preferred_model"
    ].eq(SELECTED_HAWKES_MODEL_ID).sum()
)

SELECTED_MODEL_FOLD_WIN_FRACTION: Final[float] = (
    selected_model_fold_win_count
    / DEVELOPMENT_CHRONOLOGICAL_FOLDS
)

selected_model_fold_win_gate = bool(
    SELECTED_MODEL_FOLD_WIN_FRACTION
    >= MINIMUM_SELECTED_MODEL_FOLD_WIN_FRACTION
)

require(
    selected_model_fold_win_gate,
    (
        "The selected model does not meet the minimum paired "
        "chronological fold-win fraction."
    ),
)


# ------------------------------------------------------------
# Stability gate ledger
# ------------------------------------------------------------

all_fold_fits_admissible = bool(
    SELECTED_HAWKES_FOLD_RESULTS[
        "spectral_radius"
    ].lt(1.0).all()
)

all_fold_scores_finite = bool(
    np.isfinite(selected_fold_scores).all()
)

all_parameters_stable = bool(
    SELECTED_HAWKES_PARAMETER_STABILITY_AUDIT[
        "passed"
    ].all()
)

no_fold_boundary_flags = bool(
    (
        "any_parameter_near_numerical_boundary"
        not in SELECTED_HAWKES_FOLD_RESULTS.columns
    )
    or (
        not SELECTED_HAWKES_FOLD_RESULTS[
            "any_parameter_near_numerical_boundary"
        ].any()
    )
)

all_fold_multistarts_accepted = bool(
    (
        "accepted_start_count"
        not in SELECTED_HAWKES_FOLD_RESULTS.columns
    )
    or (
        SELECTED_HAWKES_FOLD_RESULTS[
            "accepted_start_count"
        ].eq(
            SELECTED_HAWKES_FOLD_RESULTS[
                "attempted_start_count"
            ]
        ).all()
    )
)

SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_GATE_LEDGER = (
    pd.DataFrame(
        [
            {
                "gate_id": "ALL_FOLD_SCORES_FINITE",
                "passed": all_fold_scores_finite,
            },
            {
                "gate_id": "ALL_FOLD_FITS_STATIONARY",
                "passed": all_fold_fits_admissible,
            },
            {
                "gate_id": "ALL_PARAMETER_STABILITY_LIMITS",
                "passed": all_parameters_stable,
            },
            {
                "gate_id": "NO_NUMERICAL_BOUNDARY_FLAGS",
                "passed": no_fold_boundary_flags,
            },
            {
                "gate_id": "ALL_FOLD_MULTISTARTS_ACCEPTED",
                "passed": all_fold_multistarts_accepted,
            },
            {
                "gate_id": "SELECTED_MODEL_FOLD_WIN_FRACTION",
                "passed": selected_model_fold_win_gate,
            },
            {
                "gate_id": "CALIBRATION_UNUSED",
                "passed": (
                    CALIBRATION_SCORING_STARTED is False
                    and CALIBRATION_PARAMETER_UPDATES_PERFORMED
                    == 0
                ),
            },
            {
                "gate_id": "PROTECTED_PARTITIONS_UNOPENED",
                "passed": (
                    not any(
                        PROTECTED_PARTITION_CONTENT_LOADED[
                            partition
                        ]
                        for partition in PROTECTED_PARTITIONS
                    )
                ),
            },
        ]
    )
)

SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_GATE_LEDGER[
    "status"
] = np.where(
    SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_GATE_LEDGER[
        "passed"
    ],
    "PASS",
    "FAIL",
)

SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_PASSED: Final[
    bool
] = bool(
    SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_GATE_LEDGER[
        "passed"
    ].all()
)

require(
    SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_PASSED,
    "The selected model failed chronological stability.",
)


# ------------------------------------------------------------
# Portable stability package and hash
# ------------------------------------------------------------

chronological_stability_payload = {
    "schema_version": (
        "NOTEBOOK_07_SELECTED_HAWKES_"
        "CHRONOLOGICAL_STABILITY_V1"
    ),
    "model_id": SELECTED_HAWKES_MODEL_ID,
    "development_fold_count": (
        DEVELOPMENT_CHRONOLOGICAL_FOLDS
    ),
    "fold_win_count": selected_model_fold_win_count,
    "fold_win_fraction": (
        SELECTED_MODEL_FOLD_WIN_FRACTION
    ),
    "weighted_validation_log_score_per_event": (
        SELECTED_HAWKES_WEIGHTED_VALIDATION_LOG_SCORE_PER_EVENT
    ),
    "mean_validation_log_score_per_event": (
        SELECTED_HAWKES_MEAN_VALIDATION_LOG_SCORE_PER_EVENT
    ),
    "median_validation_log_score_per_event": (
        SELECTED_HAWKES_MEDIAN_VALIDATION_LOG_SCORE_PER_EVENT
    ),
    "validation_log_score_standard_deviation": (
        SELECTED_HAWKES_VALIDATION_LOG_SCORE_STANDARD_DEVIATION
    ),
    "parameter_stability": [
        {
            "parameter": str(row.parameter),
            "full_development_value": float(
                row.full_development_value
            ),
            "fold_coefficient_of_variation": float(
                row.fold_coefficient_of_variation
            ),
            "maximum_fold_to_full_deviation_factor": float(
                row.maximum_fold_to_full_deviation_factor
            ),
            "latest_fold_to_full_deviation_factor": float(
                row.latest_fold_to_full_deviation_factor
            ),
            "first_to_last_deviation_factor": float(
                row.first_to_last_deviation_factor
            ),
            "passed": bool(row.passed),
        }
        for row in (
            SELECTED_HAWKES_PARAMETER_STABILITY_AUDIT
            .itertuples(index=False)
        )
    ],
    "gates": {
        str(row.gate_id): bool(row.passed)
        for row in (
            SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_GATE_LEDGER
            .itertuples(index=False)
        )
    },
    "calibration_used": False,
    "calibration_parameter_updates": 0,
    "validation_content_loaded": False,
    "engineering_holdout_content_loaded": False,
    "status": "PASS",
}

SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_SHA256: Final[
    str
] = canonical_json_sha256(
    chronological_stability_payload
)


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

HAWKES_SELECTED_MODEL_CHRONOLOGICAL_STABILITY_COMPLETED: bool = (
    True
)

HAWKES_SELECTED_MODEL_CHRONOLOGICALLY_STABLE: bool = (
    SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_PASSED
)

LOCKED_CALIBRATION_REPLAY_AUTHORIZED: bool = (
    HAWKES_SELECTED_MODEL_CHRONOLOGICALLY_STABLE
)

CALIBRATION_SCORING_STARTED = False
CALIBRATION_PARAMETER_UPDATES_PERFORMED = 0
FILESYSTEM_WRITES_PERFORMED = False

chronological_stability_summary = pd.DataFrame(
    [
        {
            "field": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "chronological_stability_completed",
            "value": (
                HAWKES_SELECTED_MODEL_CHRONOLOGICAL_STABILITY_COMPLETED
            ),
        },
        {
            "field": "chronological_stability_passed",
            "value": (
                HAWKES_SELECTED_MODEL_CHRONOLOGICALLY_STABLE
            ),
        },
        {
            "field": "selected_model_fold_win_count",
            "value": selected_model_fold_win_count,
        },
        {
            "field": "selected_model_fold_win_fraction",
            "value": (
                SELECTED_MODEL_FOLD_WIN_FRACTION
            ),
        },
        {
            "field": "weighted_validation_log_score_per_event",
            "value": (
                SELECTED_HAWKES_WEIGHTED_VALIDATION_LOG_SCORE_PER_EVENT
            ),
        },
        {
            "field": "mean_validation_log_score_per_event",
            "value": (
                SELECTED_HAWKES_MEAN_VALIDATION_LOG_SCORE_PER_EVENT
            ),
        },
        {
            "field": "validation_log_score_standard_deviation",
            "value": (
                SELECTED_HAWKES_VALIDATION_LOG_SCORE_STANDARD_DEVIATION
            ),
        },
        {
            "field": "minimum_validation_log_score_per_event",
            "value": (
                SELECTED_HAWKES_VALIDATION_LOG_SCORE_MINIMUM
            ),
        },
        {
            "field": "maximum_validation_log_score_per_event",
            "value": (
                SELECTED_HAWKES_VALIDATION_LOG_SCORE_MAXIMUM
            ),
        },
        {
            "field": "positive_score_fold_count",
            "value": (
                SELECTED_HAWKES_POSITIVE_SCORE_FOLD_COUNT
            ),
        },
        {
            "field": "negative_score_fold_count",
            "value": (
                SELECTED_HAWKES_NEGATIVE_SCORE_FOLD_COUNT
            ),
        },
        {
            "field": "locked_calibration_replay_authorized",
            "value": (
                LOCKED_CALIBRATION_REPLAY_AUTHORIZED
            ),
        },
        {
            "field": "chronological_stability_sha256",
            "value": (
                SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_SHA256
            ),
        },
        {
            "field": "calibration_scoring_started",
            "value": CALIBRATION_SCORING_STARTED,
        },
        {
            "field": "calibration_parameter_updates_performed",
            "value": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

selected_fold_display_columns = [
    column_name
    for column_name in (
        "fold_number",
        "fold_id",
        "training_event_count",
        "validation_event_count",
        "validation_log_score_per_event",
        "mu_buy",
        "mu_sell",
        "kappa_buy",
        "kappa_sell",
        "half_life_buy_seconds",
        "half_life_sell_seconds",
        "spectral_radius",
        "accepted_start_count",
        "attempted_start_count",
        "any_parameter_near_numerical_boundary",
        "status",
    )
    if column_name
    in SELECTED_HAWKES_FOLD_RESULTS.columns
]

display(chronological_stability_summary)

display(
    SELECTED_HAWKES_FOLD_RESULTS
    .sort_values(
        [
            "fold_number",
            "fold_id",
        ],
        kind="stable",
    )
    .loc[
        :,
        selected_fold_display_columns,
    ]
    .reset_index(drop=True)
)

display(
    SELECTED_HAWKES_PARAMETER_STABILITY_AUDIT[
        [
            "parameter",
            "full_development_value",
            "fold_mean",
            "fold_coefficient_of_variation",
            "maximum_fold_to_full_deviation_factor",
            "latest_fold_to_full_deviation_factor",
            "first_to_last_deviation_factor",
            "passed",
            "status",
        ]
    ]
)

display(
    SELECTED_HAWKES_FOLD_DISTANCE_AUDIT[
        [
            "fold_number",
            "fold_id",
            "validation_log_score_per_event",
            "maximum_deviation_factor_to_full",
            "maximum_relative_difference_to_full",
            "status",
        ]
    ]
)

display(
    SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_GATE_LEDGER
)

print(
    f"{SELECTED_HAWKES_MODEL_ID} passed the selected-model "
    "chronological stability audit across all expanding "
    "DEVELOPMENT folds. All fold fits were finite, stationary, "
    "away from numerical guardrails, and consistent with the "
    "full-DEVELOPMENT parameter scale under the frozen engineering "
    "limits. Variation in validation log score was retained as an "
    "observed chronological feature rather than hidden or averaged "
    "away. Locked CALIBRATION replay is now authorized with the "
    "frozen parameters and inherited terminal DEVELOPMENT state. "
    "CALIBRATION scoring has not yet started, parameter updates "
    "remain zero, protected partitions remain unopened, and no "
    "filesystem writes were performed."
)

,field,value
0,selected_model_id,H1_DIAGONAL_SHARED_DECAY
1,chronological_stability_completed,True
2,chronological_stability_passed,True
3,selected_model_fold_win_count,4
4,selected_model_fold_win_fraction,0.8
5,weighted_validation_log_score_per_event,-0.07054607905
6,mean_validation_log_score_per_event,-0.09090838161
7,validation_log_score_standard_deviation,0.2503791442
8,minimum_validation_log_score_per_event,-0.3934764031
9,maximum_validation_log_score_per_event,0.2418384086


,fold_number,fold_id,training_event_count,validation_event_count,validation_log_score_per_event,mu_buy,mu_sell,kappa_buy,kappa_sell,half_life_buy_seconds,half_life_sell_seconds,spectral_radius,accepted_start_count,attempted_start_count,any_parameter_near_numerical_boundary,status
0,1,DEV_FOLD_01,3549,669,-0.2793849725,1.584361611,1.832966744,0.1573810289,0.1097713523,0.01980360485,0.01980360485,0.1573810289,8,8,False,PASS
1,2,DEV_FOLD_02,4218,611,-0.3934764031,1.585241958,1.805032901,0.1435132188,0.1197792323,0.01815703791,0.01815703791,0.1435132188,8,8,False,PASS
2,3,DEV_FOLD_03,4829,827,0.2418384086,1.556799459,1.798796317,0.1421608491,0.1067618419,0.01612753122,0.01612753122,0.1421608491,8,8,False,PASS
3,4,DEV_FOLD_04,5656,655,0.003047684556,1.56295885,1.780425163,0.1710866448,0.1264597727,0.0146192073,0.0146192073,0.1710866448,8,8,False,PASS
4,5,DEV_FOLD_05,6311,693,-0.02656662558,1.546401685,1.751331267,0.1823413804,0.1245105852,0.01176009143,0.01176009143,0.1823413804,8,8,False,PASS


,parameter,full_development_value,fold_mean,fold_coefficient_of_variation,maximum_fold_to_full_deviation_factor,latest_fold_to_full_deviation_factor,first_to_last_deviation_factor,passed,status
0,mu_buy,1.536124799,1.567152713,0.01095392131,1.031974719,1.006690138,1.024547261,True,PASS
1,mu_sell,1.767916532,1.793710478,0.01688144061,1.036794844,1.00947009,1.046613384,True,PASS
2,kappa_buy,0.1892556591,0.1592966244,0.1094754929,1.331278339,1.037919416,1.158598223,True,PASS
3,kappa_sell,0.1126637105,0.1174565569,0.07490817428,1.122453469,1.105152534,1.134272126,True,PASS
4,half_life_buy_seconds,0.00980691712,0.01609349454,0.1939632085,2.01935069,1.199162926,1.683966912,True,PASS
5,half_life_sell_seconds,0.00980691712,0.01609349454,0.1939632085,2.01935069,1.199162926,1.683966912,True,PASS
6,spectral_radius,0.1892556591,0.1592966244,0.1094754929,1.331278339,1.037919416,1.158598223,True,PASS


,fold_number,fold_id,validation_log_score_per_event,maximum_deviation_factor_to_full,maximum_relative_difference_to_full,status
0,1,DEV_FOLD_01,-0.2793849725,2.01935069,1.01935069,PASS
1,2,DEV_FOLD_02,-0.3934764031,1.851452163,0.8514521629,PASS
2,3,DEV_FOLD_03,0.2418384086,1.644505712,0.6445057118,PASS
3,4,DEV_FOLD_04,0.003047684556,1.490703666,0.4907036658,PASS
4,5,DEV_FOLD_05,-0.02656662558,1.199162926,0.1991629261,PASS


,gate_id,passed,status
0,ALL_FOLD_SCORES_FINITE,True,PASS
1,ALL_FOLD_FITS_STATIONARY,True,PASS
2,ALL_PARAMETER_STABILITY_LIMITS,True,PASS
3,NO_NUMERICAL_BOUNDARY_FLAGS,True,PASS
4,ALL_FOLD_MULTISTARTS_ACCEPTED,True,PASS
5,SELECTED_MODEL_FOLD_WIN_FRACTION,True,PASS
6,CALIBRATION_UNUSED,True,PASS
7,PROTECTED_PARTITIONS_UNOPENED,True,PASS


H1_DIAGONAL_SHARED_DECAY passed the selected-model chronological stability audit across all expanding DEVELOPMENT folds. All fold fits were finite, stationary, away from numerical guardrails, and consistent with the full-DEVELOPMENT parameter scale under the frozen engineering limits. Variation in validation log score was retained as an observed chronological feature rather than hidden or averaged away. Locked CALIBRATION replay is now authorized with the frozen parameters and inherited terminal DEVELOPMENT state. CALIBRATION scoring has not yet started, parameter updates remain zero, protected partitions remain unopened, and no filesystem writes were performed.


In [13]:
# ============================================================
# Locked CALIBRATION replay with inherited DEVELOPMENT history
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "LOCKED_CALIBRATION_REPLAY_AUTHORIZED",
            False,
        )
    ),
    "Locked CALIBRATION replay has not been authorized.",
)
require(
    bool(
        globals().get(
            "HAWKES_SELECTED_MODEL_CHRONOLOGICALLY_STABLE",
            False,
        )
    ),
    "The selected Hawkes model has not passed chronological stability.",
)
require(
    bool(
        globals().get(
            "HAWKES_SELECTED_MODEL_FROZEN",
            False,
        )
    ),
    "The selected Hawkes model is not frozen.",
)
require(
    SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE.state_time_ns
    == CALIBRATION_START_NS,
    (
        "The terminal DEVELOPMENT state is not expressed at the "
        "exact CALIBRATION boundary."
    ),
)
require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION already contains parameter updates.",
)
require(
    CALIBRATION_USED_FOR_SELECTION is False,
    "CALIBRATION must not have been used for model selection.",
)
require(
    CALIBRATION_USED_FOR_REFIT is False,
    "CALIBRATION must not have been used for model refitting.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Freeze pre-replay parameter and package identity
# ------------------------------------------------------------

CALIBRATION_PARAMETER_SNAPSHOT_BEFORE: Final[
    np.ndarray
] = np.asarray(
    [
        SELECTED_HAWKES_PARAMETERS.mu_buy,
        SELECTED_HAWKES_PARAMETERS.mu_sell,
        SELECTED_HAWKES_PARAMETERS.kappa_buy,
        SELECTED_HAWKES_PARAMETERS.kappa_sell,
        SELECTED_HAWKES_PARAMETERS.beta_buy,
        SELECTED_HAWKES_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

CALIBRATION_OPTIMIZER_VECTOR_SNAPSHOT_BEFORE: Final[
    np.ndarray
] = (
    SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR.copy()
)

SELECTED_MODEL_PACKAGE_SHA256_BEFORE_CALIBRATION: Final[
    str
] = canonical_json_sha256(
    SELECTED_HAWKES_MODEL_PACKAGE
)

require(
    SELECTED_MODEL_PACKAGE_SHA256_BEFORE_CALIBRATION
    == SELECTED_HAWKES_MODEL_PACKAGE_SHA256,
    "The selected-model package changed before CALIBRATION replay.",
)


# ------------------------------------------------------------
# Locked replay
#
# No optimizer, fit, parameter update, or model-selection function
# is called below.
# ------------------------------------------------------------

CALIBRATION_SCORING_STARTED = True

LOCKED_CALIBRATION_HAWKES_RESULT: Final[
    DiagonalHawkesReplayResult
] = replay_diagonal_hawkes(
    CALIBRATION_BATCHES_FOR_HAWKES,
    SELECTED_HAWKES_PARAMETERS,
    observation_start_ns=CALIBRATION_START_NS,
    observation_end_exclusive_ns=(
        CALIBRATION_END_EXCLUSIVE_NS
    ),
    initial_state=(
        SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
    ),
    return_replay=True,
)

require(
    LOCKED_CALIBRATION_HAWKES_RESULT.event_count
    == EXPECTED_CALIBRATION_EVENT_ROWS,
    "The locked CALIBRATION replay lost events.",
)
require(
    LOCKED_CALIBRATION_HAWKES_RESULT.batch_count
    == EXPECTED_CALIBRATION_BATCH_ROWS,
    "The locked CALIBRATION replay lost exact-time batches.",
)
require(
    LOCKED_CALIBRATION_HAWKES_RESULT
    .observation_start_ns
    == CALIBRATION_START_NS,
    "The locked CALIBRATION replay began at the wrong boundary.",
)
require(
    LOCKED_CALIBRATION_HAWKES_RESULT
    .observation_end_exclusive_ns
    == CALIBRATION_END_EXCLUSIVE_NS,
    "The locked CALIBRATION replay ended at the wrong boundary.",
)
require(
    math.isfinite(
        LOCKED_CALIBRATION_HAWKES_RESULT.log_likelihood
    ),
    "The locked CALIBRATION log likelihood is non-finite.",
)
require(
    LOCKED_CALIBRATION_HAWKES_RESULT
    .final_state
    .state_time_ns
    == CALIBRATION_END_EXCLUSIVE_NS,
    "The locked CALIBRATION terminal state has the wrong time.",
)


# ------------------------------------------------------------
# Reconcile inherited state at the first CALIBRATION batch
# ------------------------------------------------------------

require(
    not LOCKED_CALIBRATION_HAWKES_RESULT.replay.empty,
    "The locked CALIBRATION replay table is empty.",
)

calibration_first_batch_time_ns: Final[int] = int(
    LOCKED_CALIBRATION_HAWKES_RESULT.replay[
        "event_time_ns"
    ].iloc[0]
)

calibration_boundary_to_first_batch_seconds: Final[
    float
] = (
    calibration_first_batch_time_ns
    - CALIBRATION_START_NS
) / NANOSECONDS_PER_SECOND

require(
    calibration_boundary_to_first_batch_seconds
    >= 0.0,
    "The first CALIBRATION batch precedes the boundary.",
)

expected_first_pre_buy_excitation: Final[float] = (
    SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
    .buy_excitation
    * math.exp(
        -SELECTED_HAWKES_PARAMETERS.beta_buy
        * calibration_boundary_to_first_batch_seconds
    )
)

expected_first_pre_sell_excitation: Final[float] = (
    SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
    .sell_excitation
    * math.exp(
        -SELECTED_HAWKES_PARAMETERS.beta_sell
        * calibration_boundary_to_first_batch_seconds
    )
)

observed_first_pre_buy_excitation: Final[float] = float(
    LOCKED_CALIBRATION_HAWKES_RESULT.replay[
        "pre_buy_excitation"
    ].iloc[0]
)

observed_first_pre_sell_excitation: Final[float] = float(
    LOCKED_CALIBRATION_HAWKES_RESULT.replay[
        "pre_sell_excitation"
    ].iloc[0]
)

require(
    math.isclose(
        observed_first_pre_buy_excitation,
        expected_first_pre_buy_excitation,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "The first CALIBRATION BUY excitation does not match the "
        "decayed terminal DEVELOPMENT state."
    ),
)
require(
    math.isclose(
        observed_first_pre_sell_excitation,
        expected_first_pre_sell_excitation,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "The first CALIBRATION SELL excitation does not match the "
        "decayed terminal DEVELOPMENT state."
    ),
)


# ------------------------------------------------------------
# Construct the identified CALIBRATION replay table
# ------------------------------------------------------------

calibration_batch_identity = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            "event_partition"
        ].eq(LOCKED_EVALUATION_PARTITION),
        [
            "primary_event_batch_id",
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_partition",
            "event_time_ns",
            "relative_batch_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "mixed_side_batch_flag",
            "simultaneous_batch_required_flag",
        ],
    ]
    .copy()
    .sort_values(
        [
            "event_time_ns",
            "primary_event_batch_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    calibration_batch_identity[
        "event_time_ns"
    ].is_unique,
    "CALIBRATION exact-time batch timestamps are not unique.",
)

LOCKED_CALIBRATION_HAWKES_REPLAY = (
    calibration_batch_identity.merge(
        LOCKED_CALIBRATION_HAWKES_RESULT.replay,
        on=[
            "event_time_ns",
            "buy_event_count",
            "sell_event_count",
        ],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

require(
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "_merge"
    ].eq("both").all(),
    (
        "The locked Hawkes replay does not reconcile with the "
        "authoritative CALIBRATION batch table."
    ),
)

LOCKED_CALIBRATION_HAWKES_REPLAY = (
    LOCKED_CALIBRATION_HAWKES_REPLAY.drop(
        columns="_merge"
    )
    .sort_values(
        [
            "event_time_ns",
            "primary_event_batch_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    len(LOCKED_CALIBRATION_HAWKES_REPLAY)
    == EXPECTED_CALIBRATION_BATCH_ROWS,
    "The identified CALIBRATION replay has the wrong row count.",
)
require(
    int(
        LOCKED_CALIBRATION_HAWKES_REPLAY[
            "batch_event_count"
        ].sum()
    )
    == EXPECTED_CALIBRATION_EVENT_ROWS,
    "The identified CALIBRATION replay does not conserve events.",
)
require(
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "strict_pre_batch_scoring"
    ].all(),
    "A CALIBRATION batch violated strict pre-batch scoring.",
)
require(
    not LOCKED_CALIBRATION_HAWKES_REPLAY[
        "zero_lag_within_batch_excitation"
    ].any(),
    "A CALIBRATION batch used prohibited zero-lag excitation.",
)

LOCKED_CALIBRATION_HAWKES_REPLAY[
    "buy_excitation_share_pre"
] = (
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "pre_buy_excitation"
    ]
    / LOCKED_CALIBRATION_HAWKES_REPLAY[
        "lambda_buy_pre"
    ]
)

LOCKED_CALIBRATION_HAWKES_REPLAY[
    "sell_excitation_share_pre"
] = (
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "pre_sell_excitation"
    ]
    / LOCKED_CALIBRATION_HAWKES_REPLAY[
        "lambda_sell_pre"
    ]
)

for intensity_column in (
    "lambda_buy_pre",
    "lambda_sell_pre",
):
    require(
        np.isfinite(
            LOCKED_CALIBRATION_HAWKES_REPLAY[
                intensity_column
            ].to_numpy(dtype=np.float64)
        ).all(),
        (
            "The locked CALIBRATION replay contains non-finite "
            f"values in {intensity_column}."
        ),
    )
    require(
        LOCKED_CALIBRATION_HAWKES_REPLAY[
            intensity_column
        ].gt(0.0).all(),
        (
            "The locked CALIBRATION replay contains a nonpositive "
            f"value in {intensity_column}."
        ),
    )

for excitation_share_column in (
    "buy_excitation_share_pre",
    "sell_excitation_share_pre",
):
    require(
        LOCKED_CALIBRATION_HAWKES_REPLAY[
            excitation_share_column
        ].between(
            0.0,
            1.0,
            inclusive="both",
        ).all(),
        (
            "A CALIBRATION pre-batch excitation share lies outside "
            f"[0, 1] in {excitation_share_column}."
        ),
    )


# ------------------------------------------------------------
# Side-specific locked scores
# ------------------------------------------------------------

calibration_buy_log_event_term: Final[float] = float(
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "log_event_term_buy"
    ].sum()
)

calibration_sell_log_event_term: Final[float] = float(
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "log_event_term_sell"
    ].sum()
)

calibration_buy_log_likelihood: Final[float] = (
    calibration_buy_log_event_term
    - LOCKED_CALIBRATION_HAWKES_RESULT.compensator_buy
)

calibration_sell_log_likelihood: Final[float] = (
    calibration_sell_log_event_term
    - LOCKED_CALIBRATION_HAWKES_RESULT.compensator_sell
)

require(
    math.isclose(
        calibration_buy_log_likelihood
        + calibration_sell_log_likelihood,
        LOCKED_CALIBRATION_HAWKES_RESULT.log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-8,
    ),
    "The side-specific CALIBRATION scores do not sum to the total.",
)

CALIBRATION_BUY_EVENT_COUNT: Final[int] = int(
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "buy_event_count"
    ].sum()
)

CALIBRATION_SELL_EVENT_COUNT: Final[int] = int(
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        "sell_event_count"
    ].sum()
)

require(
    CALIBRATION_BUY_EVENT_COUNT
    + CALIBRATION_SELL_EVENT_COUNT
    == EXPECTED_CALIBRATION_EVENT_ROWS,
    "CALIBRATION side counts do not conserve total events.",
)

LOCKED_CALIBRATION_SIDE_SCORE_SUMMARY = pd.DataFrame(
    [
        {
            "event_side": "BUY",
            "event_count": CALIBRATION_BUY_EVENT_COUNT,
            "log_event_term": (
                calibration_buy_log_event_term
            ),
            "compensator": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .compensator_buy
            ),
            "log_likelihood": (
                calibration_buy_log_likelihood
            ),
            "log_score_per_event": (
                calibration_buy_log_likelihood
                / CALIBRATION_BUY_EVENT_COUNT
            ),
        },
        {
            "event_side": "SELL",
            "event_count": CALIBRATION_SELL_EVENT_COUNT,
            "log_event_term": (
                calibration_sell_log_event_term
            ),
            "compensator": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .compensator_sell
            ),
            "log_likelihood": (
                calibration_sell_log_likelihood
            ),
            "log_score_per_event": (
                calibration_sell_log_likelihood
                / CALIBRATION_SELL_EVENT_COUNT
            ),
        },
    ]
)

LOCKED_CALIBRATION_SIDE_SCORE_SUMMARY[
    "status"
] = "PASS"


# ------------------------------------------------------------
# Continuous replay equivalence
# ------------------------------------------------------------

SELECTED_HAWKES_CONTINUOUS_ANALYTICAL_REPLAY: Final[
    DiagonalHawkesReplayResult
] = replay_diagonal_hawkes(
    ANALYTICAL_BATCHES_FOR_HAWKES,
    SELECTED_HAWKES_PARAMETERS,
    observation_start_ns=DEVELOPMENT_START_NS,
    observation_end_exclusive_ns=(
        CALIBRATION_END_EXCLUSIVE_NS
    ),
    initial_state=None,
    return_replay=False,
)

selected_split_log_likelihood: Final[float] = (
    SELECTED_HAWKES_DEVELOPMENT_REPLAY.log_likelihood
    + LOCKED_CALIBRATION_HAWKES_RESULT.log_likelihood
)

selected_split_compensator_buy: Final[float] = (
    SELECTED_HAWKES_DEVELOPMENT_REPLAY.compensator_buy
    + LOCKED_CALIBRATION_HAWKES_RESULT.compensator_buy
)

selected_split_compensator_sell: Final[float] = (
    SELECTED_HAWKES_DEVELOPMENT_REPLAY.compensator_sell
    + LOCKED_CALIBRATION_HAWKES_RESULT.compensator_sell
)

require(
    math.isclose(
        selected_split_log_likelihood,
        SELECTED_HAWKES_CONTINUOUS_ANALYTICAL_REPLAY
        .log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-8,
    ),
    (
        "The selected split DEVELOPMENT/CALIBRATION likelihood "
        "does not match one continuous replay."
    ),
)
require(
    math.isclose(
        selected_split_compensator_buy,
        SELECTED_HAWKES_CONTINUOUS_ANALYTICAL_REPLAY
        .compensator_buy,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-8,
    ),
    (
        "The selected split BUY compensator does not match the "
        "continuous replay."
    ),
)
require(
    math.isclose(
        selected_split_compensator_sell,
        SELECTED_HAWKES_CONTINUOUS_ANALYTICAL_REPLAY
        .compensator_sell,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-8,
    ),
    (
        "The selected split SELL compensator does not match the "
        "continuous replay."
    ),
)
require(
    math.isclose(
        LOCKED_CALIBRATION_HAWKES_RESULT
        .final_state
        .buy_excitation,
        SELECTED_HAWKES_CONTINUOUS_ANALYTICAL_REPLAY
        .final_state
        .buy_excitation,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "The selected split BUY terminal state does not match the "
        "continuous replay."
    ),
)
require(
    math.isclose(
        LOCKED_CALIBRATION_HAWKES_RESULT
        .final_state
        .sell_excitation,
        SELECTED_HAWKES_CONTINUOUS_ANALYTICAL_REPLAY
        .final_state
        .sell_excitation,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    ),
    (
        "The selected split SELL terminal state does not match the "
        "continuous replay."
    ),
)


# ------------------------------------------------------------
# Reset replay for boundary audit only
#
# A reset may be numerically almost identical because the selected
# excitation half-life is short and the boundary-to-first-event gap
# is long. Numerical equality does not authorize a reset.
# ------------------------------------------------------------

CALIBRATION_RESET_REPLAY_SELECTED_MODEL_AUDIT_ONLY: Final[
    DiagonalHawkesReplayResult
] = replay_diagonal_hawkes(
    CALIBRATION_BATCHES_FOR_HAWKES,
    SELECTED_HAWKES_PARAMETERS,
    observation_start_ns=CALIBRATION_START_NS,
    observation_end_exclusive_ns=(
        CALIBRATION_END_EXCLUSIVE_NS
    ),
    initial_state=None,
    return_replay=False,
)

CALIBRATION_RESET_MINUS_INHERITED_LOG_LIKELIHOOD: Final[
    float
] = (
    CALIBRATION_RESET_REPLAY_SELECTED_MODEL_AUDIT_ONLY
    .log_likelihood
    - LOCKED_CALIBRATION_HAWKES_RESULT.log_likelihood
)

CALIBRATION_RESET_NUMERICALLY_EQUIVALENT: Final[bool] = (
    math.isclose(
        CALIBRATION_RESET_REPLAY_SELECTED_MODEL_AUDIT_ONLY
        .log_likelihood,
        LOCKED_CALIBRATION_HAWKES_RESULT.log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
    )
)


# ------------------------------------------------------------
# Verify zero parameter mutation
# ------------------------------------------------------------

CALIBRATION_PARAMETER_SNAPSHOT_AFTER: Final[
    np.ndarray
] = np.asarray(
    [
        SELECTED_HAWKES_PARAMETERS.mu_buy,
        SELECTED_HAWKES_PARAMETERS.mu_sell,
        SELECTED_HAWKES_PARAMETERS.kappa_buy,
        SELECTED_HAWKES_PARAMETERS.kappa_sell,
        SELECTED_HAWKES_PARAMETERS.beta_buy,
        SELECTED_HAWKES_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

CALIBRATION_OPTIMIZER_VECTOR_SNAPSHOT_AFTER: Final[
    np.ndarray
] = (
    SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR.copy()
)

SELECTED_MODEL_PACKAGE_SHA256_AFTER_CALIBRATION: Final[
    str
] = canonical_json_sha256(
    SELECTED_HAWKES_MODEL_PACKAGE
)

CALIBRATION_NATURAL_PARAMETERS_UNCHANGED: Final[
    bool
] = bool(
    np.array_equal(
        CALIBRATION_PARAMETER_SNAPSHOT_BEFORE,
        CALIBRATION_PARAMETER_SNAPSHOT_AFTER,
    )
)

CALIBRATION_OPTIMIZER_VECTOR_UNCHANGED: Final[
    bool
] = bool(
    np.array_equal(
        CALIBRATION_OPTIMIZER_VECTOR_SNAPSHOT_BEFORE,
        CALIBRATION_OPTIMIZER_VECTOR_SNAPSHOT_AFTER,
    )
)

CALIBRATION_MODEL_PACKAGE_UNCHANGED: Final[bool] = (
    SELECTED_MODEL_PACKAGE_SHA256_BEFORE_CALIBRATION
    == SELECTED_MODEL_PACKAGE_SHA256_AFTER_CALIBRATION
    == SELECTED_HAWKES_MODEL_PACKAGE_SHA256
)

require(
    CALIBRATION_NATURAL_PARAMETERS_UNCHANGED,
    "Natural Hawkes parameters changed during CALIBRATION replay.",
)
require(
    CALIBRATION_OPTIMIZER_VECTOR_UNCHANGED,
    "The optimizer vector changed during CALIBRATION replay.",
)
require(
    CALIBRATION_MODEL_PACKAGE_UNCHANGED,
    "The selected-model package changed during CALIBRATION replay.",
)

CALIBRATION_PARAMETER_UPDATES_PERFORMED = 0


# ------------------------------------------------------------
# Intensity distribution summary
# ------------------------------------------------------------

def finite_quantile(
    values: pd.Series,
    probability: float,
) -> float:
    """Return one finite empirical quantile."""
    numeric_values = values.to_numpy(dtype=np.float64)

    require(
        np.isfinite(numeric_values).all(),
        "Quantile input contains non-finite values.",
    )

    return float(
        np.quantile(
            numeric_values,
            probability,
            method="linear",
        )
    )


calibration_intensity_summary_rows: list[
    dict[str, Any]
] = []

for event_side, intensity_column, share_column in (
    (
        "BUY",
        "lambda_buy_pre",
        "buy_excitation_share_pre",
    ),
    (
        "SELL",
        "lambda_sell_pre",
        "sell_excitation_share_pre",
    ),
):
    calibration_intensity_summary_rows.append(
        {
            "event_side": event_side,
            "minimum_intensity_per_second": float(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    intensity_column
                ].min()
            ),
            "p25_intensity_per_second": finite_quantile(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    intensity_column
                ],
                0.25,
            ),
            "median_intensity_per_second": finite_quantile(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    intensity_column
                ],
                0.50,
            ),
            "p75_intensity_per_second": finite_quantile(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    intensity_column
                ],
                0.75,
            ),
            "p95_intensity_per_second": finite_quantile(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    intensity_column
                ],
                0.95,
            ),
            "maximum_intensity_per_second": float(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    intensity_column
                ].max()
            ),
            "mean_pre_batch_excitation_share": float(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    share_column
                ].mean()
            ),
            "p95_pre_batch_excitation_share": finite_quantile(
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    share_column
                ],
                0.95,
            ),
            "status": "PASS",
        }
    )

LOCKED_CALIBRATION_INTENSITY_SUMMARY = pd.DataFrame(
    calibration_intensity_summary_rows
)


# ------------------------------------------------------------
# Replay hash
# ------------------------------------------------------------

def portable_calibration_replay_records(
    replay_table: pd.DataFrame,
) -> list[dict[str, Any]]:
    """Return portable exact-time CALIBRATION replay records."""
    portable_records: list[dict[str, Any]] = []

    for row in replay_table.itertuples(index=False):
        portable_records.append(
            {
                "primary_event_batch_id": str(
                    row.primary_event_batch_id
                ),
                "primary_event_batch_number": int(
                    row.primary_event_batch_number
                ),
                "partition_batch_index": int(
                    row.partition_batch_index
                ),
                "event_time_ns": int(row.event_time_ns),
                "buy_event_count": int(
                    row.buy_event_count
                ),
                "sell_event_count": int(
                    row.sell_event_count
                ),
                "lambda_buy_pre": float(
                    row.lambda_buy_pre
                ),
                "lambda_sell_pre": float(
                    row.lambda_sell_pre
                ),
                "pre_buy_excitation": float(
                    row.pre_buy_excitation
                ),
                "pre_sell_excitation": float(
                    row.pre_sell_excitation
                ),
                "post_buy_excitation": float(
                    row.post_buy_excitation
                ),
                "post_sell_excitation": float(
                    row.post_sell_excitation
                ),
                "interval_compensator_buy": float(
                    row.interval_compensator_buy
                ),
                "interval_compensator_sell": float(
                    row.interval_compensator_sell
                ),
                "log_event_term_buy": float(
                    row.log_event_term_buy
                ),
                "log_event_term_sell": float(
                    row.log_event_term_sell
                ),
                "strict_pre_batch_scoring": bool(
                    row.strict_pre_batch_scoring
                ),
                "zero_lag_within_batch_excitation": bool(
                    row.zero_lag_within_batch_excitation
                ),
            }
        )

    return portable_records


LOCKED_CALIBRATION_HAWKES_REPLAY_SHA256: Final[str] = (
    canonical_json_sha256(
        {
            "schema_version": (
                "NOTEBOOK_07_LOCKED_CALIBRATION_"
                "HAWKES_REPLAY_V1"
            ),
            "model_id": SELECTED_HAWKES_MODEL_ID,
            "selected_model_package_sha256": (
                SELECTED_HAWKES_MODEL_PACKAGE_SHA256
            ),
            "records": portable_calibration_replay_records(
                LOCKED_CALIBRATION_HAWKES_REPLAY
            ),
        }
    )
)


# ------------------------------------------------------------
# Locked CALIBRATION package
# ------------------------------------------------------------

LOCKED_CALIBRATION_HAWKES_PACKAGE: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_LOCKED_CALIBRATION_HAWKES_EVALUATION"
    ),
    "schema_version": (
        "NOTEBOOK_07_LOCKED_CALIBRATION_"
        "HAWKES_EVALUATION_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "model_id": SELECTED_HAWKES_MODEL_ID,
    "selected_model_package_sha256": (
        SELECTED_HAWKES_MODEL_PACKAGE_SHA256
    ),
    "chronological_stability_sha256": (
        SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_SHA256
    ),
    "replay_sha256": (
        LOCKED_CALIBRATION_HAWKES_REPLAY_SHA256
    ),
    "evaluation_partition": (
        LOCKED_EVALUATION_PARTITION
    ),
    "observation_start_ns": CALIBRATION_START_NS,
    "observation_end_exclusive_ns": (
        CALIBRATION_END_EXCLUSIVE_NS
    ),
    "event_count": EXPECTED_CALIBRATION_EVENT_ROWS,
    "batch_count": EXPECTED_CALIBRATION_BATCH_ROWS,
    "initial_state_source": (
        "TERMINAL_DEVELOPMENT_STATE"
    ),
    "initial_state": {
        "state_time_ns": (
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .state_time_ns
        ),
        "buy_excitation": (
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .buy_excitation
        ),
        "sell_excitation": (
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .sell_excitation
        ),
    },
    "locked_score": {
        "log_event_term": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .log_event_term
        ),
        "compensator_buy": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .compensator_buy
        ),
        "compensator_sell": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .compensator_sell
        ),
        "total_compensator": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .total_compensator
        ),
        "log_likelihood": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .log_likelihood
        ),
        "negative_log_likelihood": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .negative_log_likelihood
        ),
        "log_score_per_event": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .log_likelihood
            / EXPECTED_CALIBRATION_EVENT_ROWS
        ),
        "log_score_per_second": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .log_likelihood
            / CALIBRATION_DURATION_SECONDS
        ),
    },
    "terminal_state": {
        "state_time_ns": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .final_state
            .state_time_ns
        ),
        "buy_excitation": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .final_state
            .buy_excitation
        ),
        "sell_excitation": (
            LOCKED_CALIBRATION_HAWKES_RESULT
            .final_state
            .sell_excitation
        ),
    },
    "continuous_replay_reconciled": True,
    "natural_parameters_unchanged": (
        CALIBRATION_NATURAL_PARAMETERS_UNCHANGED
    ),
    "optimizer_vector_unchanged": (
        CALIBRATION_OPTIMIZER_VECTOR_UNCHANGED
    ),
    "model_package_unchanged": (
        CALIBRATION_MODEL_PACKAGE_UNCHANGED
    ),
    "parameter_updates": 0,
    "reset_replay_authorized": False,
    "reset_minus_inherited_log_likelihood": (
        CALIBRATION_RESET_MINUS_INHERITED_LOG_LIKELIHOOD
    ),
    "reset_numerically_equivalent": (
        CALIBRATION_RESET_NUMERICALLY_EQUIVALENT
    ),
    "claim_limits": {
        "hawkes_superiority_authorized": False,
        "baseline_superiority_authorized": False,
        "cross_excitation_authorized": False,
        "state_dependent_hawkes_authorized": False,
        "strategy_or_quoting_authorized": False,
        "pnl_claim_authorized": False,
    },
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "status": "PASS_LOCKED_CALIBRATION_REPLAY",
}

LOCKED_CALIBRATION_HAWKES_PACKAGE_SHA256: Final[
    str
] = canonical_json_sha256(
    LOCKED_CALIBRATION_HAWKES_PACKAGE
)


# ------------------------------------------------------------
# Gate ledger
# ------------------------------------------------------------

LOCKED_CALIBRATION_GATE_LEDGER = pd.DataFrame(
    [
        {
            "gate_id": "FROZEN_PARAMETERS_USED",
            "passed": (
                CALIBRATION_NATURAL_PARAMETERS_UNCHANGED
            ),
        },
        {
            "gate_id": "ZERO_PARAMETER_UPDATES",
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "gate_id": "TERMINAL_DEVELOPMENT_STATE_INHERITED",
            "passed": True,
        },
        {
            "gate_id": "FIRST_BATCH_STATE_RECONCILIATION",
            "passed": True,
        },
        {
            "gate_id": "CALIBRATION_EVENT_CONSERVATION",
            "passed": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .event_count
                == EXPECTED_CALIBRATION_EVENT_ROWS
            ),
        },
        {
            "gate_id": "CALIBRATION_BATCH_CONSERVATION",
            "passed": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .batch_count
                == EXPECTED_CALIBRATION_BATCH_ROWS
            ),
        },
        {
            "gate_id": "STRICT_PRE_BATCH_SCORING",
            "passed": (
                LOCKED_CALIBRATION_HAWKES_REPLAY[
                    "strict_pre_batch_scoring"
                ].all()
            ),
        },
        {
            "gate_id": "ZERO_LAG_EXCITATION_FORBIDDEN",
            "passed": (
                not LOCKED_CALIBRATION_HAWKES_REPLAY[
                    "zero_lag_within_batch_excitation"
                ].any()
            ),
        },
        {
            "gate_id": "SPLIT_CONTINUOUS_REPLAY_RECONCILIATION",
            "passed": True,
        },
        {
            "gate_id": "PROTECTED_PARTITIONS_UNOPENED",
            "passed": (
                not any(
                    PROTECTED_PARTITION_CONTENT_LOADED[
                        partition
                    ]
                    for partition in PROTECTED_PARTITIONS
                )
            ),
        },
    ]
)

LOCKED_CALIBRATION_GATE_LEDGER[
    "status"
] = np.where(
    LOCKED_CALIBRATION_GATE_LEDGER[
        "passed"
    ],
    "PASS",
    "FAIL",
)

require(
    LOCKED_CALIBRATION_GATE_LEDGER[
        "passed"
    ].all(),
    "The locked CALIBRATION replay failed at least one gate.",
)


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

CALIBRATION_SCORING_COMPLETED: bool = True
LOCKED_CALIBRATION_REPLAY_COMPLETED: bool = True
LOCKED_CALIBRATION_REPLAY_PASSED: bool = True

HAWKES_SUPERIORITY_CLAIM_AUTHORIZED: bool = False
BASELINE_SUPERIORITY_CLAIM_AUTHORIZED: bool = False

TIMESTAMP_COARSENING_SENSITIVITY_AUTHORIZED: bool = (
    LOCKED_CALIBRATION_REPLAY_PASSED
)

FILESYSTEM_WRITES_PERFORMED = False

locked_calibration_summary = pd.DataFrame(
    [
        {
            "field": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "calibration_scoring_completed",
            "value": CALIBRATION_SCORING_COMPLETED,
        },
        {
            "field": "locked_calibration_replay_passed",
            "value": LOCKED_CALIBRATION_REPLAY_PASSED,
        },
        {
            "field": "calibration_event_count",
            "value": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .event_count
            ),
        },
        {
            "field": "calibration_batch_count",
            "value": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .batch_count
            ),
        },
        {
            "field": "calibration_log_likelihood",
            "value": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .log_likelihood
            ),
        },
        {
            "field": "calibration_log_score_per_event",
            "value": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .log_likelihood
                / EXPECTED_CALIBRATION_EVENT_ROWS
            ),
        },
        {
            "field": "calibration_log_score_per_second",
            "value": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .log_likelihood
                / CALIBRATION_DURATION_SECONDS
            ),
        },
        {
            "field": "development_log_score_per_event",
            "value": (
                SELECTED_HAWKES_DEVELOPMENT_LOG_LIKELIHOOD
                / EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
        },
        {
            "field": (
                "calibration_minus_development_log_score_per_event"
            ),
            "value": (
                (
                    LOCKED_CALIBRATION_HAWKES_RESULT
                    .log_likelihood
                    / EXPECTED_CALIBRATION_EVENT_ROWS
                )
                - (
                    SELECTED_HAWKES_DEVELOPMENT_LOG_LIKELIHOOD
                    / EXPECTED_DEVELOPMENT_EVENT_ROWS
                )
            ),
        },
        {
            "field": "boundary_to_first_batch_seconds",
            "value": (
                calibration_boundary_to_first_batch_seconds
            ),
        },
        {
            "field": "first_pre_buy_excitation",
            "value": (
                observed_first_pre_buy_excitation
            ),
        },
        {
            "field": "first_pre_sell_excitation",
            "value": (
                observed_first_pre_sell_excitation
            ),
        },
        {
            "field": (
                "reset_minus_inherited_log_likelihood"
            ),
            "value": (
                CALIBRATION_RESET_MINUS_INHERITED_LOG_LIKELIHOOD
            ),
        },
        {
            "field": "reset_numerically_equivalent",
            "value": (
                CALIBRATION_RESET_NUMERICALLY_EQUIVALENT
            ),
        },
        {
            "field": "parameter_updates_performed",
            "value": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
            ),
        },
        {
            "field": "selected_model_package_unchanged",
            "value": CALIBRATION_MODEL_PACKAGE_UNCHANGED,
        },
        {
            "field": "calibration_replay_sha256",
            "value": (
                LOCKED_CALIBRATION_HAWKES_REPLAY_SHA256
            ),
        },
        {
            "field": "calibration_package_sha256",
            "value": (
                LOCKED_CALIBRATION_HAWKES_PACKAGE_SHA256
            ),
        },
        {
            "field": "hawkes_superiority_claim_authorized",
            "value": (
                HAWKES_SUPERIORITY_CLAIM_AUTHORIZED
            ),
        },
        {
            "field": "timestamp_coarsening_sensitivity_authorized",
            "value": (
                TIMESTAMP_COARSENING_SENSITIVITY_AUTHORIZED
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(locked_calibration_summary)
display(LOCKED_CALIBRATION_SIDE_SCORE_SUMMARY)
display(LOCKED_CALIBRATION_INTENSITY_SUMMARY)
display(LOCKED_CALIBRATION_GATE_LEDGER)

display(
    LOCKED_CALIBRATION_HAWKES_REPLAY[
        [
            "primary_event_batch_number",
            "event_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "lambda_buy_pre",
            "lambda_sell_pre",
            "pre_buy_excitation",
            "pre_sell_excitation",
            "buy_excitation_share_pre",
            "sell_excitation_share_pre",
            "mixed_side_batch_flag",
            "simultaneous_batch_required_flag",
        ]
    ].head(10)
)

print(
    f"{SELECTED_HAWKES_MODEL_ID} completed locked CALIBRATION "
    "replay using the frozen DEVELOPMENT parameters and inherited "
    "terminal DEVELOPMENT excitation state. All CALIBRATION events "
    "and exact-time batches were conserved, split replay matched one "
    "continuous analytical replay, and parameter identity remained "
    "unchanged with zero updates. The reset replay was computed for "
    "audit only and remains unauthorized even when rapid excitation "
    "decay makes it numerically similar. No Hawkes-superiority claim "
    "is authorized by this result. Protected partitions remain "
    "unopened and no filesystem writes were performed."
)

,field,value
0,selected_model_id,H1_DIAGONAL_SHARED_DECAY
1,calibration_scoring_completed,True
2,locked_calibration_replay_passed,True
3,calibration_event_count,2493
4,calibration_batch_count,2400
5,calibration_log_likelihood,-452.7379259
6,calibration_log_score_per_event,-0.1816036606
7,calibration_log_score_per_second,-0.6285414992
8,development_log_score_per_event,-0.1435555596
9,calibration_minus_development_log_score_per_event,-0.03804810096


,event_side,event_count,log_event_term,compensator,log_likelihood,log_score_per_event,status
0,BUY,1230,"1,346.220544","1,339.253892",6.966651678,0.005663944454,PASS
1,SELL,1263,956.0185167,"1,415.723094",-459.7045776,-0.3639782878,PASS


,event_side,minimum_intensity_per_second,p25_intensity_per_second,median_intensity_per_second,p75_intensity_per_second,p95_intensity_per_second,maximum_intensity_per_second,mean_pre_batch_excitation_share,p95_pre_batch_excitation_share,status
0,BUY,1.536124799,1.536124799,1.536124799,1.540677088,21.70498418,103.849428,0.1289200186,0.9292270771,PASS
1,SELL,1.767916532,1.767916532,1.767916532,1.768253726,6.019856694,39.77249046,0.07107570533,0.7063190684,PASS


,gate_id,passed,status
0,FROZEN_PARAMETERS_USED,True,PASS
1,ZERO_PARAMETER_UPDATES,True,PASS
2,TERMINAL_DEVELOPMENT_STATE_INHERITED,True,PASS
3,FIRST_BATCH_STATE_RECONCILIATION,True,PASS
4,CALIBRATION_EVENT_CONSERVATION,True,PASS
5,CALIBRATION_BATCH_CONSERVATION,True,PASS
6,STRICT_PRE_BATCH_SCORING,True,PASS
7,ZERO_LAG_EXCITATION_FORBIDDEN,True,PASS
8,SPLIT_CONTINUOUS_REPLAY_RECONCILIATION,True,PASS
9,PROTECTED_PARTITIONS_UNOPENED,True,PASS


,primary_event_batch_number,event_time_ns,batch_event_count,buy_event_count,sell_event_count,lambda_buy_pre,lambda_sell_pre,pre_buy_excitation,pre_sell_excitation,buy_excitation_share_pre,sell_excitation_share_pre,mixed_side_batch_flag,simultaneous_batch_required_flag
0,6860,1783667270546982000,1,1,0,1.536124799,1.767916532,8.688204585e-52,3.492224448e-40,5.655923654e-52,1.975333329e-40,False,False
1,6861,1783667270994015000,1,0,1,1.536124799,1.767916532,2.537228857e-13,6.62399445e-54,1.651707504e-13,3.746780083e-54,False,False
2,6862,1783667271146401800,1,0,1,1.536124799,1.768083818,5.330195039e-18,0.0001672863344,3.469897135e-18,9.461448188e-05,False,False
3,6863,1783667271213810600,1,1,0,1.536124799,1.8358255,4.545525577e-20,0.06790896868,2.959086124e-20,0.03699097145,False,False
4,6864,1783667271564793500,1,1,0,1.536124799,1.767916532,2.252593538e-10,1.143584178e-12,1.466413106e-10,6.468541688e-13,False,False
5,6865,1783667271574882100,1,0,1,8.092523979,1.767916532,6.55639918,5.605207506e-13,0.8101797656,3.170515918e-13,False,False
6,6866,1783667272720052300,1,1,0,1.536124799,1.767916532,4.622603972e-35,5.614334797e-35,3.009263294e-35,3.175678657e-35,False,False
7,6867,1783667273472754900,1,1,0,1.536124799,1.767916532,1.051049644e-22,4.411433186e-58,6.842215195e-23,2.49527232e-58,False,False
8,6868,1783667273575643200,1,1,0,1.545416932,1.767916532,0.009292133394,3.064455442e-61,0.006012703239,1.733371111e-61,False,False
9,6869,1783667273849563700,1,0,1,1.536124851,1.767916532,5.229585096e-08,1.197228734e-69,3.404401076e-08,6.771975443e-70,False,False


H1_DIAGONAL_SHARED_DECAY completed locked CALIBRATION replay using the frozen DEVELOPMENT parameters and inherited terminal DEVELOPMENT excitation state. All CALIBRATION events and exact-time batches were conserved, split replay matched one continuous analytical replay, and parameter identity remained unchanged with zero updates. The reset replay was computed for audit only and remains unauthorized even when rapid excitation decay makes it numerically similar. No Hawkes-superiority claim is authorized by this result. Protected partitions remain unopened and no filesystem writes were performed.


In [14]:
# ============================================================
# Exact 1 ms timestamp-coarsening sensitivity
# ============================================================

import time


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "LOCKED_CALIBRATION_REPLAY_PASSED",
            False,
        )
    ),
    "The primary locked CALIBRATION replay has not passed.",
)
require(
    bool(
        globals().get(
            "TIMESTAMP_COARSENING_SENSITIVITY_AUTHORIZED",
            False,
        )
    ),
    "Timestamp-coarsening sensitivity has not been authorized.",
)
require(
    bool(
        globals().get(
            "HAWKES_SELECTED_MODEL_FROZEN",
            False,
        )
    ),
    "The primary selected Hawkes model is not frozen.",
)
require(
    SELECTED_HAWKES_MODEL_ID
    == H1_MODEL_ID,
    (
        "The timestamp-coarsening sensitivity must preserve the "
        "selected model family."
    ),
)
require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "Primary CALIBRATION parameter updates must remain zero.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Frozen coarsening contract
#
# This is a sensitivity analysis only. It does not replace the
# Notebook 04 event_time_ns authority or the primary fit.
# ------------------------------------------------------------

TIMESTAMP_COARSENING_WIDTH_NS: Final[int] = (
    NANOSECONDS_PER_MILLISECOND
)

TIMESTAMP_COARSENING_WIDTH_SECONDS: Final[float] = (
    TIMESTAMP_COARSENING_WIDTH_NS
    / NANOSECONDS_PER_SECOND
)

TIMESTAMP_COARSENING_METHOD: Final[str] = (
    "PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID"
)

TIMESTAMP_COARSENING_ESTIMATOR_LABEL: Final[str] = (
    "STRICT_PRE_BATCH_1MS_COARSENED_SENSITIVITY_QUASI_MLE"
)

TIMESTAMP_COARSENING_ROLE: Final[str] = (
    "SENSITIVITY_ONLY_NOT_PRIMARY_AUTHORITY"
)

COARSENED_SENSITIVITY_PARAMETER_UPDATES_ON_CALIBRATION: Final[
    int
] = 0

require(
    TIMESTAMP_COARSENING_WIDTH_NS == 1_000_000,
    "The timestamp-coarsening width must be exactly one millisecond.",
)
require(
    DEVELOPMENT_END_EXCLUSIVE_NS
    == CALIBRATION_START_NS,
    "DEVELOPMENT and CALIBRATION are not exactly contiguous.",
)


# ------------------------------------------------------------
# Preserve the primary selected-model identity
# ------------------------------------------------------------

PRIMARY_SELECTED_PARAMETERS_BEFORE_COARSENING: Final[
    np.ndarray
] = np.asarray(
    [
        SELECTED_HAWKES_PARAMETERS.mu_buy,
        SELECTED_HAWKES_PARAMETERS.mu_sell,
        SELECTED_HAWKES_PARAMETERS.kappa_buy,
        SELECTED_HAWKES_PARAMETERS.kappa_sell,
        SELECTED_HAWKES_PARAMETERS.beta_buy,
        SELECTED_HAWKES_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

PRIMARY_SELECTED_MODEL_PACKAGE_SHA256_BEFORE_COARSENING: Final[
    str
] = canonical_json_sha256(
    SELECTED_HAWKES_MODEL_PACKAGE
)

require(
    PRIMARY_SELECTED_MODEL_PACKAGE_SHA256_BEFORE_COARSENING
    == SELECTED_HAWKES_MODEL_PACKAGE_SHA256,
    "The selected-model package changed before coarsening.",
)


# ------------------------------------------------------------
# Exact integer-nanosecond coarsening
# ------------------------------------------------------------

def coarsen_hawkes_batches_to_exact_grid(
    batch_frame: pd.DataFrame,
    *,
    observation_start_ns: int,
    observation_end_exclusive_ns: int,
    grid_width_ns: int,
) -> pd.DataFrame:
    """
    Floor exact batch timestamps to a partition-relative grid.

    All source batches mapped to one grid timestamp are aggregated
    before scoring. No jitter, artificial ordering, event removal,
    or within-grid zero-lag excitation is introduced.
    """
    require(
        isinstance(grid_width_ns, int),
        "The grid width must be an integer number of nanoseconds.",
    )
    require(
        grid_width_ns > 0,
        "The timestamp-coarsening grid width must be positive.",
    )

    start_ns = int(observation_start_ns)
    end_ns = int(observation_end_exclusive_ns)

    validated = validate_hawkes_batch_input(
        batch_frame,
        observation_start_ns=start_ns,
        observation_end_exclusive_ns=end_ns,
    )

    source = validated.copy()

    source["original_event_time_ns"] = (
        source["event_time_ns"].astype("int64")
    )

    relative_time_ns = (
        source["original_event_time_ns"].to_numpy(
            dtype=np.int64
        )
        - np.int64(start_ns)
    )

    require(
        np.all(relative_time_ns >= 0),
        "A source batch precedes the coarsening origin.",
    )
    require(
        np.all(
            relative_time_ns
            < np.int64(end_ns - start_ns)
        ),
        "A source batch lies outside the coarsening interval.",
    )

    coarsened_relative_time_ns = (
        relative_time_ns
        // np.int64(grid_width_ns)
    ) * np.int64(grid_width_ns)

    coarsened_event_time_ns = (
        np.int64(start_ns)
        + coarsened_relative_time_ns
    )

    require(
        np.all(
            coarsened_event_time_ns
            <= source[
                "original_event_time_ns"
            ].to_numpy(dtype=np.int64)
        ),
        "Floor coarsening moved an event forward in time.",
    )
    require(
        np.all(coarsened_event_time_ns >= start_ns),
        "Floor coarsening moved an event before the interval.",
    )
    require(
        np.all(coarsened_event_time_ns < end_ns),
        "Floor coarsening moved an event outside the interval.",
    )

    source["coarsened_event_time_ns"] = (
        coarsened_event_time_ns.astype(np.int64)
    )

    coarsened = (
        source.groupby(
            "coarsened_event_time_ns",
            sort=True,
            observed=True,
            dropna=False,
        )
        .agg(
            buy_event_count=(
                "buy_event_count",
                "sum",
            ),
            sell_event_count=(
                "sell_event_count",
                "sum",
            ),
            source_exact_batch_count=(
                "original_event_time_ns",
                "size",
            ),
            first_source_event_time_ns=(
                "original_event_time_ns",
                "min",
            ),
            last_source_event_time_ns=(
                "original_event_time_ns",
                "max",
            ),
        )
        .reset_index()
        .rename(
            columns={
                "coarsened_event_time_ns": (
                    "event_time_ns"
                )
            }
        )
    )

    for integer_column in (
        "event_time_ns",
        "buy_event_count",
        "sell_event_count",
        "source_exact_batch_count",
        "first_source_event_time_ns",
        "last_source_event_time_ns",
    ):
        coarsened[integer_column] = parse_exact_int64(
            coarsened[integer_column],
            label=(
                "COARSENED_HAWKES_BATCHES."
                f"{integer_column}"
            ),
        )

    coarsened["batch_event_count"] = (
        coarsened["buy_event_count"]
        + coarsened["sell_event_count"]
    )

    coarsened["source_time_span_ns"] = (
        coarsened["last_source_event_time_ns"]
        - coarsened["first_source_event_time_ns"]
    )

    coarsened["mixed_side_batch_flag"] = (
        coarsened["buy_event_count"].gt(0)
        & coarsened["sell_event_count"].gt(0)
    )

    coarsened[
        "simultaneous_batch_required_flag"
    ] = coarsened["batch_event_count"].gt(1)

    coarsened["strict_pre_batch_scoring"] = True
    coarsened[
        "zero_lag_within_coarsened_batch_excitation"
    ] = False

    coarsened = coarsened.sort_values(
        "event_time_ns",
        kind="stable",
    ).reset_index(drop=True)

    require(
        coarsened["event_time_ns"].is_unique,
        "Coarsened batch timestamps are not unique.",
    )
    require(
        coarsened["event_time_ns"].is_monotonic_increasing,
        "Coarsened batch timestamps are not increasing.",
    )
    require(
        int(
            coarsened[
                [
                    "buy_event_count",
                    "sell_event_count",
                ]
            ].to_numpy(dtype=np.int64).sum()
        )
        == int(
            validated[
                [
                    "buy_event_count",
                    "sell_event_count",
                ]
            ].to_numpy(dtype=np.int64).sum()
        ),
        "Timestamp coarsening did not conserve event count.",
    )
    require(
        int(
            coarsened[
                "source_exact_batch_count"
            ].sum()
        )
        == len(validated),
        "Timestamp coarsening did not conserve source batches.",
    )
    require(
        not coarsened[
            "zero_lag_within_coarsened_batch_excitation"
        ].any(),
        "A coarsened batch permits prohibited zero-lag excitation.",
    )

    return coarsened


# ------------------------------------------------------------
# Construct DEVELOPMENT and CALIBRATION 1 ms batch interfaces
# ------------------------------------------------------------

COARSENED_DEVELOPMENT_BATCHES = (
    coarsen_hawkes_batches_to_exact_grid(
        DEVELOPMENT_BATCHES_FOR_HAWKES,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
        grid_width_ns=(
            TIMESTAMP_COARSENING_WIDTH_NS
        ),
    )
)

COARSENED_CALIBRATION_BATCHES = (
    coarsen_hawkes_batches_to_exact_grid(
        CALIBRATION_BATCHES_FOR_HAWKES,
        observation_start_ns=CALIBRATION_START_NS,
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        grid_width_ns=(
            TIMESTAMP_COARSENING_WIDTH_NS
        ),
    )
)

COARSENED_ANALYTICAL_BATCHES = (
    pd.concat(
        [
            COARSENED_DEVELOPMENT_BATCHES[
                [
                    "event_time_ns",
                    "buy_event_count",
                    "sell_event_count",
                ]
            ],
            COARSENED_CALIBRATION_BATCHES[
                [
                    "event_time_ns",
                    "buy_event_count",
                    "sell_event_count",
                ]
            ],
        ],
        axis=0,
        ignore_index=True,
    )
    .sort_values(
        "event_time_ns",
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    batch_event_count(
        COARSENED_DEVELOPMENT_BATCHES
    )
    == EXPECTED_DEVELOPMENT_EVENT_ROWS,
    "Coarsened DEVELOPMENT lost events.",
)
require(
    batch_event_count(
        COARSENED_CALIBRATION_BATCHES
    )
    == EXPECTED_CALIBRATION_EVENT_ROWS,
    "Coarsened CALIBRATION lost events.",
)
require(
    batch_event_count(
        COARSENED_ANALYTICAL_BATCHES
    )
    == EXPECTED_ANALYTICAL_EVENT_ROWS,
    "The combined coarsened interface lost events.",
)


# ------------------------------------------------------------
# Batch-geometry audit
# ------------------------------------------------------------

def coarsened_batch_geometry_record(
    *,
    partition: str,
    exact_batches: pd.DataFrame,
    coarsened_batches: pd.DataFrame,
) -> dict[str, Any]:
    """Return one partition-level coarsening geometry record."""
    exact_event_count = batch_event_count(
        exact_batches
    )

    coarsened_event_count = batch_event_count(
        coarsened_batches
    )

    merged_source_batch_count = int(
        coarsened_batches[
            "source_exact_batch_count"
        ].gt(1).sum()
    )

    source_batches_inside_merged_groups = int(
        coarsened_batches.loc[
            coarsened_batches[
                "source_exact_batch_count"
            ].gt(1),
            "source_exact_batch_count",
        ].sum()
    )

    return {
        "event_partition": partition,
        "grid_width_ns": (
            TIMESTAMP_COARSENING_WIDTH_NS
        ),
        "exact_batch_count": len(exact_batches),
        "coarsened_batch_count": len(
            coarsened_batches
        ),
        "batch_count_reduction": (
            len(exact_batches)
            - len(coarsened_batches)
        ),
        "exact_event_count": exact_event_count,
        "coarsened_event_count": coarsened_event_count,
        "coarsened_groups_merging_source_batches": (
            merged_source_batch_count
        ),
        "source_batches_inside_merged_groups": (
            source_batches_inside_merged_groups
        ),
        "coarsened_simultaneous_batch_count": int(
            coarsened_batches[
                "simultaneous_batch_required_flag"
            ].sum()
        ),
        "coarsened_mixed_side_batch_count": int(
            coarsened_batches[
                "mixed_side_batch_flag"
            ].sum()
        ),
        "maximum_events_in_coarsened_batch": int(
            coarsened_batches[
                "batch_event_count"
            ].max()
        ),
        "maximum_source_batches_merged": int(
            coarsened_batches[
                "source_exact_batch_count"
            ].max()
        ),
        "maximum_source_time_span_ns": int(
            coarsened_batches[
                "source_time_span_ns"
            ].max()
        ),
        "event_count_conserved": (
            exact_event_count
            == coarsened_event_count
        ),
        "status": "PASS",
    }


TIMESTAMP_COARSENING_BATCH_GEOMETRY = pd.DataFrame(
    [
        coarsened_batch_geometry_record(
            partition=FIT_PARTITION,
            exact_batches=(
                DEVELOPMENT_BATCHES_FOR_HAWKES
            ),
            coarsened_batches=(
                COARSENED_DEVELOPMENT_BATCHES
            ),
        ),
        coarsened_batch_geometry_record(
            partition=LOCKED_EVALUATION_PARTITION,
            exact_batches=(
                CALIBRATION_BATCHES_FOR_HAWKES
            ),
            coarsened_batches=(
                COARSENED_CALIBRATION_BATCHES
            ),
        ),
    ]
)

require(
    TIMESTAMP_COARSENING_BATCH_GEOMETRY[
        "event_count_conserved"
    ].all(),
    "At least one partition failed coarsened event conservation.",
)


# ------------------------------------------------------------
# Fixed-primary-parameter replay on the coarsened interface
#
# This isolates sensitivity to the timestamp interface without
# changing parameters.
# ------------------------------------------------------------

COARSENED_FIXED_PRIMARY_DEVELOPMENT_REPLAY = (
    replay_diagonal_hawkes(
        COARSENED_DEVELOPMENT_BATCHES[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ]
        ],
        SELECTED_HAWKES_PARAMETERS,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
        initial_state=None,
        return_replay=False,
    )
)

COARSENED_FIXED_PRIMARY_CALIBRATION_REPLAY = (
    replay_diagonal_hawkes(
        COARSENED_CALIBRATION_BATCHES[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ]
        ],
        SELECTED_HAWKES_PARAMETERS,
        observation_start_ns=CALIBRATION_START_NS,
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        initial_state=(
            COARSENED_FIXED_PRIMARY_DEVELOPMENT_REPLAY
            .final_state
        ),
        return_replay=False,
    )
)

COARSENED_FIXED_PRIMARY_CONTINUOUS_REPLAY = (
    replay_diagonal_hawkes(
        COARSENED_ANALYTICAL_BATCHES[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ]
        ],
        SELECTED_HAWKES_PARAMETERS,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        initial_state=None,
        return_replay=False,
    )
)

require(
    math.isclose(
        (
            COARSENED_FIXED_PRIMARY_DEVELOPMENT_REPLAY
            .log_likelihood
            + COARSENED_FIXED_PRIMARY_CALIBRATION_REPLAY
            .log_likelihood
        ),
        COARSENED_FIXED_PRIMARY_CONTINUOUS_REPLAY
        .log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-8,
    ),
    (
        "Fixed-primary split replay does not match the continuous "
        "coarsened replay."
    ),
)


# ------------------------------------------------------------
# Coarsened DEVELOPMENT objective
# ------------------------------------------------------------

def coarsened_development_average_nll_and_gradient(
    optimizer_vector: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Return selected-model average NLL and gradient on coarsened DEV."""
    total_nll, total_gradient = (
        negative_log_likelihood_and_gradient(
            optimizer_vector,
            model_id=SELECTED_HAWKES_MODEL_ID,
            batch_frame=(
                COARSENED_DEVELOPMENT_BATCHES[
                    [
                        "event_time_ns",
                        "buy_event_count",
                        "sell_event_count",
                    ]
                ]
            ),
            observation_start_ns=(
                DEVELOPMENT_START_NS
            ),
            observation_end_exclusive_ns=(
                DEVELOPMENT_END_EXCLUSIVE_NS
            ),
        )
    )

    return (
        total_nll / EXPECTED_DEVELOPMENT_EVENT_ROWS,
        total_gradient / EXPECTED_DEVELOPMENT_EVENT_ROWS,
    )


# ------------------------------------------------------------
# Deterministic selected-family multi-start sensitivity fit
#
# All starts come from the already frozen H1 optimization ledger.
# CALIBRATION is not inspected during optimization or selection.
# ------------------------------------------------------------

coarsened_start_ledger = (
    HAWKES_OPTIMIZATION_START_LEDGER.loc[
        HAWKES_OPTIMIZATION_START_LEDGER[
            "model_id"
        ].eq(SELECTED_HAWKES_MODEL_ID)
    ]
    .sort_values(
        [
            "initial_negative_log_likelihood",
            "start_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    len(coarsened_start_ledger) >= 2,
    "Too few deterministic starts are available for sensitivity fitting.",
)

coarsened_fit_records: list[dict[str, Any]] = []

coarsened_fit_start_time = time.perf_counter()

for start_position, start_row in (
    coarsened_start_ledger.iterrows()
):
    start_id = str(start_row["start_id"])

    initial_vector = np.asarray(
        start_row["optimizer_vector"],
        dtype=np.float64,
    )

    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always")

        optimizer_result = optimize.minimize(
            fun=(
                coarsened_development_average_nll_and_gradient
            ),
            x0=initial_vector,
            method="L-BFGS-B",
            jac=True,
            bounds=HAWKES_OPTIMIZER_BOUNDS[
                SELECTED_HAWKES_MODEL_ID
            ],
            options={
                "maxiter": MAX_OPTIMIZER_ITERATIONS,
                "maxfun": (
                    MAX_OPTIMIZER_ITERATIONS * 20
                ),
                "ftol": OPTIMIZER_FUNCTION_TOLERANCE,
                "gtol": OPTIMIZER_GRADIENT_TOLERANCE,
                "maxls": 50,
                "maxcor": 20,
            },
        )

    optimized_vector = np.asarray(
        optimizer_result.x,
        dtype=np.float64,
    )

    coarsened_parameters = decode_hawkes_parameters(
        SELECTED_HAWKES_MODEL_ID,
        optimized_vector,
    )

    (
        average_nll,
        average_gradient,
    ) = coarsened_development_average_nll_and_gradient(
        optimized_vector
    )

    total_nll = (
        average_nll
        * EXPECTED_DEVELOPMENT_EVENT_ROWS
    )

    gradient_inf_per_event = float(
        np.linalg.norm(
            average_gradient,
            ord=np.inf,
        )
    )

    parameter_validation = validate_hawkes_parameters(
        coarsened_parameters
    )

    replay_result = replay_diagonal_hawkes(
        COARSENED_DEVELOPMENT_BATCHES[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ]
        ],
        coarsened_parameters,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
        initial_state=None,
        return_replay=False,
    )

    replay_matches = math.isclose(
        replay_result.negative_log_likelihood,
        total_nll,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-7,
    )

    accepted = bool(
        np.isfinite(optimized_vector).all()
        and math.isfinite(total_nll)
        and math.isfinite(
            gradient_inf_per_event
        )
        and parameter_validation[
            "mathematically_admissible"
        ]
        and replay_matches
        and (
            bool(optimizer_result.success)
            or (
                gradient_inf_per_event
                <= (
                    OPTIMIZER_ACCEPTANCE_GRADIENT_INF_PER_EVENT
                )
            )
        )
    )

    (
        half_life_buy_seconds,
        half_life_sell_seconds,
    ) = excitation_half_lives_seconds(
        coarsened_parameters
    )

    coarsened_fit_records.append(
        {
            "start_id": start_id,
            "start_position": int(
                start_position + 1
            ),
            "optimized_negative_log_likelihood": float(
                total_nll
            ),
            "optimized_log_likelihood": float(
                -total_nll
            ),
            "optimized_log_score_per_event": float(
                -total_nll
                / EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "gradient_inf_per_event": (
                gradient_inf_per_event
            ),
            "optimizer_success": bool(
                optimizer_result.success
            ),
            "optimizer_status_code": int(
                optimizer_result.status
            ),
            "optimizer_message": str(
                optimizer_result.message
            ),
            "optimizer_iterations": int(
                optimizer_result.nit
            ),
            "optimizer_function_evaluations": int(
                optimizer_result.nfev
            ),
            "warning_count": len(
                caught_warnings
            ),
            "mu_buy": coarsened_parameters.mu_buy,
            "mu_sell": coarsened_parameters.mu_sell,
            "kappa_buy": (
                coarsened_parameters.kappa_buy
            ),
            "kappa_sell": (
                coarsened_parameters.kappa_sell
            ),
            "beta_buy": coarsened_parameters.beta_buy,
            "beta_sell": coarsened_parameters.beta_sell,
            "half_life_buy_seconds": (
                half_life_buy_seconds
            ),
            "half_life_sell_seconds": (
                half_life_sell_seconds
            ),
            "spectral_radius": (
                parameter_validation[
                    "spectral_radius"
                ]
            ),
            "acceptance_margin_holds": (
                parameter_validation[
                    "acceptance_margin_holds"
                ]
            ),
            "replay_matches": replay_matches,
            "accepted_solution": accepted,
            "calibration_used": False,
            "status": (
                "ACCEPTED"
                if accepted
                else "REJECTED"
            ),
            "optimized_vector": tuple(
                float(value)
                for value in optimized_vector
            ),
        }
    )

COARSENED_SENSITIVITY_FIT_RUNTIME_SECONDS: Final[
    float
] = (
    time.perf_counter()
    - coarsened_fit_start_time
)

TIMESTAMP_COARSENING_MULTISTART_RESULTS = pd.DataFrame(
    coarsened_fit_records
)

require(
    len(TIMESTAMP_COARSENING_MULTISTART_RESULTS)
    == len(coarsened_start_ledger),
    "The coarsened multi-start result ledger has the wrong size.",
)
require(
    TIMESTAMP_COARSENING_MULTISTART_RESULTS[
        "start_id"
    ].is_unique,
    "The coarsened multi-start result IDs are not unique.",
)
require(
    not TIMESTAMP_COARSENING_MULTISTART_RESULTS[
        "calibration_used"
    ].any(),
    "CALIBRATION was used during coarsened sensitivity fitting.",
)

accepted_coarsened_fits = (
    TIMESTAMP_COARSENING_MULTISTART_RESULTS.loc[
        TIMESTAMP_COARSENING_MULTISTART_RESULTS[
            "accepted_solution"
        ]
    ]
    .sort_values(
        [
            "optimized_negative_log_likelihood",
            "start_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    len(accepted_coarsened_fits) >= 2,
    (
        "The coarsened sensitivity fit produced fewer than two "
        "accepted deterministic solutions."
    ),
)

best_coarsened_fit = accepted_coarsened_fits.iloc[0]

COARSENED_SENSITIVITY_BEST_NLL: Final[float] = float(
    best_coarsened_fit[
        "optimized_negative_log_likelihood"
    ]
)

TIMESTAMP_COARSENING_MULTISTART_RESULTS[
    "objective_gap_from_best"
] = (
    TIMESTAMP_COARSENING_MULTISTART_RESULTS[
        "optimized_negative_log_likelihood"
    ]
    - COARSENED_SENSITIVITY_BEST_NLL
)

TIMESTAMP_COARSENING_MULTISTART_RESULTS[
    "objective_gap_per_event"
] = (
    TIMESTAMP_COARSENING_MULTISTART_RESULTS[
        "objective_gap_from_best"
    ]
    / EXPECTED_DEVELOPMENT_EVENT_ROWS
)

TIMESTAMP_COARSENING_MULTISTART_RESULTS[
    "inside_material_optimum_cluster"
] = (
    TIMESTAMP_COARSENING_MULTISTART_RESULTS[
        "accepted_solution"
    ]
    & TIMESTAMP_COARSENING_MULTISTART_RESULTS[
        "objective_gap_per_event"
    ].le(
        MULTISTART_OBJECTIVE_AGREEMENT_PER_EVENT
    )
)

COARSENED_SENSITIVITY_OPTIMUM_CLUSTER_SIZE: Final[
    int
] = int(
    TIMESTAMP_COARSENING_MULTISTART_RESULTS[
        "inside_material_optimum_cluster"
    ].sum()
)

require(
    COARSENED_SENSITIVITY_OPTIMUM_CLUSTER_SIZE
    >= MINIMUM_MATERIAL_OPTIMUM_CLUSTER_SIZE,
    (
        "The coarsened sensitivity fit lacks repeated multi-start "
        "agreement."
    ),
)


# ------------------------------------------------------------
# Freeze the best coarsened sensitivity parameters
# ------------------------------------------------------------

COARSENED_SENSITIVITY_PARAMETERS: Final[
    DiagonalHawkesParameters
] = DiagonalHawkesParameters(
    model_id=SELECTED_HAWKES_MODEL_ID,
    mu_buy=float(best_coarsened_fit["mu_buy"]),
    mu_sell=float(best_coarsened_fit["mu_sell"]),
    kappa_buy=float(
        best_coarsened_fit["kappa_buy"]
    ),
    kappa_sell=float(
        best_coarsened_fit["kappa_sell"]
    ),
    beta_buy=float(best_coarsened_fit["beta_buy"]),
    beta_sell=float(
        best_coarsened_fit["beta_sell"]
    ),
)

coarsened_parameter_validation = (
    validate_hawkes_parameters(
        COARSENED_SENSITIVITY_PARAMETERS
    )
)

require(
    coarsened_parameter_validation[
        "mathematically_admissible"
    ],
    "The best coarsened sensitivity parameters are inadmissible.",
)
require(
    coarsened_parameter_validation[
        "acceptance_margin_holds"
    ],
    (
        "The best coarsened sensitivity parameters violate the "
        "engineering stationarity margin."
    ),
)

if SELECTED_HAWKES_SPECIFICATION.shared_decay:
    require(
        math.isclose(
            COARSENED_SENSITIVITY_PARAMETERS.beta_buy,
            COARSENED_SENSITIVITY_PARAMETERS.beta_sell,
            rel_tol=FLOAT_RELATIVE_TOLERANCE,
            abs_tol=FLOAT_ABSOLUTE_TOLERANCE,
        ),
        "The coarsened H1 sensitivity fit violates shared decay.",
    )


# ------------------------------------------------------------
# Coarsened-fit DEVELOPMENT and locked CALIBRATION replays
# ------------------------------------------------------------

COARSENED_REFIT_DEVELOPMENT_REPLAY = (
    replay_diagonal_hawkes(
        COARSENED_DEVELOPMENT_BATCHES[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ]
        ],
        COARSENED_SENSITIVITY_PARAMETERS,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
        initial_state=None,
        return_replay=False,
    )
)

COARSENED_REFIT_CALIBRATION_REPLAY = (
    replay_diagonal_hawkes(
        COARSENED_CALIBRATION_BATCHES[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ]
        ],
        COARSENED_SENSITIVITY_PARAMETERS,
        observation_start_ns=CALIBRATION_START_NS,
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        initial_state=(
            COARSENED_REFIT_DEVELOPMENT_REPLAY
            .final_state
        ),
        return_replay=False,
    )
)

COARSENED_REFIT_CONTINUOUS_REPLAY = (
    replay_diagonal_hawkes(
        COARSENED_ANALYTICAL_BATCHES[
            [
                "event_time_ns",
                "buy_event_count",
                "sell_event_count",
            ]
        ],
        COARSENED_SENSITIVITY_PARAMETERS,
        observation_start_ns=DEVELOPMENT_START_NS,
        observation_end_exclusive_ns=(
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        initial_state=None,
        return_replay=False,
    )
)

require(
    math.isclose(
        (
            COARSENED_REFIT_DEVELOPMENT_REPLAY
            .log_likelihood
            + COARSENED_REFIT_CALIBRATION_REPLAY
            .log_likelihood
        ),
        COARSENED_REFIT_CONTINUOUS_REPLAY
        .log_likelihood,
        rel_tol=FLOAT_RELATIVE_TOLERANCE,
        abs_tol=1e-8,
    ),
    (
        "The coarsened-refit split replay does not match its "
        "continuous replay."
    ),
)
require(
    COARSENED_SENSITIVITY_PARAMETER_UPDATES_ON_CALIBRATION
    == 0,
    (
        "The coarsened sensitivity CALIBRATION evaluation must "
        "receive zero parameter updates."
    ),
)


# ------------------------------------------------------------
# Exact versus 1 ms parameter comparison
# ------------------------------------------------------------

(
    coarsened_half_life_buy_seconds,
    coarsened_half_life_sell_seconds,
) = excitation_half_lives_seconds(
    COARSENED_SENSITIVITY_PARAMETERS
)

exact_parameter_map: Final[Mapping[str, float]] = {
    "mu_buy_per_second": (
        SELECTED_HAWKES_PARAMETERS.mu_buy
    ),
    "mu_sell_per_second": (
        SELECTED_HAWKES_PARAMETERS.mu_sell
    ),
    "kappa_buy": (
        SELECTED_HAWKES_PARAMETERS.kappa_buy
    ),
    "kappa_sell": (
        SELECTED_HAWKES_PARAMETERS.kappa_sell
    ),
    "beta_buy_per_second": (
        SELECTED_HAWKES_PARAMETERS.beta_buy
    ),
    "beta_sell_per_second": (
        SELECTED_HAWKES_PARAMETERS.beta_sell
    ),
    "buy_half_life_seconds": (
        selected_half_life_buy_seconds
    ),
    "sell_half_life_seconds": (
        selected_half_life_sell_seconds
    ),
    "spectral_radius": selected_final_validation[
        "spectral_radius"
    ],
}

coarsened_parameter_map: Final[
    Mapping[str, float]
] = {
    "mu_buy_per_second": (
        COARSENED_SENSITIVITY_PARAMETERS.mu_buy
    ),
    "mu_sell_per_second": (
        COARSENED_SENSITIVITY_PARAMETERS.mu_sell
    ),
    "kappa_buy": (
        COARSENED_SENSITIVITY_PARAMETERS.kappa_buy
    ),
    "kappa_sell": (
        COARSENED_SENSITIVITY_PARAMETERS.kappa_sell
    ),
    "beta_buy_per_second": (
        COARSENED_SENSITIVITY_PARAMETERS.beta_buy
    ),
    "beta_sell_per_second": (
        COARSENED_SENSITIVITY_PARAMETERS.beta_sell
    ),
    "buy_half_life_seconds": (
        coarsened_half_life_buy_seconds
    ),
    "sell_half_life_seconds": (
        coarsened_half_life_sell_seconds
    ),
    "spectral_radius": (
        coarsened_parameter_validation[
            "spectral_radius"
        ]
    ),
}

parameter_comparison_rows: list[dict[str, Any]] = []

for parameter_name in exact_parameter_map:
    exact_value = float(
        exact_parameter_map[parameter_name]
    )

    coarsened_value = float(
        coarsened_parameter_map[
            parameter_name
        ]
    )

    require(
        exact_value > 0.0,
        f"The exact {parameter_name} value is nonpositive.",
    )
    require(
        coarsened_value > 0.0,
        f"The coarsened {parameter_name} value is nonpositive.",
    )

    parameter_comparison_rows.append(
        {
            "parameter": parameter_name,
            "exact_primary_value": exact_value,
            "coarsened_sensitivity_value": (
                coarsened_value
            ),
            "coarsened_to_exact_ratio": (
                coarsened_value / exact_value
            ),
            "absolute_difference": (
                coarsened_value - exact_value
            ),
            "absolute_relative_difference": abs(
                coarsened_value - exact_value
            ) / exact_value,
            "absolute_log_ratio": abs(
                math.log(
                    coarsened_value / exact_value
                )
            ),
            "status": "DESCRIPTIVE",
        }
    )

TIMESTAMP_COARSENING_PARAMETER_COMPARISON = (
    pd.DataFrame(parameter_comparison_rows)
)

TIMESTAMP_COARSENING_MAXIMUM_PARAMETER_ABSOLUTE_LOG_RATIO: Final[
    float
] = float(
    TIMESTAMP_COARSENING_PARAMETER_COMPARISON[
        "absolute_log_ratio"
    ].max()
)

TIMESTAMP_COARSENING_MAXIMUM_PARAMETER_DEVIATION_FACTOR: Final[
    float
] = math.exp(
    TIMESTAMP_COARSENING_MAXIMUM_PARAMETER_ABSOLUTE_LOG_RATIO
)


# ------------------------------------------------------------
# Score comparison
# ------------------------------------------------------------

primary_exact_development_score_per_event = (
    SELECTED_HAWKES_DEVELOPMENT_REPLAY
    .log_likelihood
    / EXPECTED_DEVELOPMENT_EVENT_ROWS
)

primary_exact_calibration_score_per_event = (
    LOCKED_CALIBRATION_HAWKES_RESULT
    .log_likelihood
    / EXPECTED_CALIBRATION_EVENT_ROWS
)

coarsened_fixed_development_score_per_event = (
    COARSENED_FIXED_PRIMARY_DEVELOPMENT_REPLAY
    .log_likelihood
    / EXPECTED_DEVELOPMENT_EVENT_ROWS
)

coarsened_fixed_calibration_score_per_event = (
    COARSENED_FIXED_PRIMARY_CALIBRATION_REPLAY
    .log_likelihood
    / EXPECTED_CALIBRATION_EVENT_ROWS
)

coarsened_refit_development_score_per_event = (
    COARSENED_REFIT_DEVELOPMENT_REPLAY
    .log_likelihood
    / EXPECTED_DEVELOPMENT_EVENT_ROWS
)

coarsened_refit_calibration_score_per_event = (
    COARSENED_REFIT_CALIBRATION_REPLAY
    .log_likelihood
    / EXPECTED_CALIBRATION_EVENT_ROWS
)

TIMESTAMP_COARSENING_SCORE_COMPARISON = pd.DataFrame(
    [
        {
            "evaluation": (
                "PRIMARY_EXACT_TIMESTAMPS_PRIMARY_PARAMETERS"
            ),
            "development_log_score_per_event": (
                primary_exact_development_score_per_event
            ),
            "calibration_log_score_per_event": (
                primary_exact_calibration_score_per_event
            ),
            "calibration_minus_development": (
                primary_exact_calibration_score_per_event
                - primary_exact_development_score_per_event
            ),
            "parameters_fit_on": "EXACT_DEVELOPMENT",
            "timestamp_interface": "NOTEBOOK_04_EVENT_TIME_NS",
            "primary_authority": True,
        },
        {
            "evaluation": (
                "COARSENED_TIMESTAMPS_PRIMARY_PARAMETERS"
            ),
            "development_log_score_per_event": (
                coarsened_fixed_development_score_per_event
            ),
            "calibration_log_score_per_event": (
                coarsened_fixed_calibration_score_per_event
            ),
            "calibration_minus_development": (
                coarsened_fixed_calibration_score_per_event
                - coarsened_fixed_development_score_per_event
            ),
            "parameters_fit_on": "EXACT_DEVELOPMENT",
            "timestamp_interface": (
                TIMESTAMP_COARSENING_METHOD
            ),
            "primary_authority": False,
        },
        {
            "evaluation": (
                "COARSENED_TIMESTAMPS_COARSENED_DEV_REFIT"
            ),
            "development_log_score_per_event": (
                coarsened_refit_development_score_per_event
            ),
            "calibration_log_score_per_event": (
                coarsened_refit_calibration_score_per_event
            ),
            "calibration_minus_development": (
                coarsened_refit_calibration_score_per_event
                - coarsened_refit_development_score_per_event
            ),
            "parameters_fit_on": (
                "COARSENED_DEVELOPMENT_ONLY"
            ),
            "timestamp_interface": (
                TIMESTAMP_COARSENING_METHOD
            ),
            "primary_authority": False,
        },
    ]
)

TIMESTAMP_COARSENING_SCORE_COMPARISON[
    "status"
] = "PASS"

COARSENED_FIXED_MINUS_EXACT_CALIBRATION_SCORE_PER_EVENT: Final[
    float
] = (
    coarsened_fixed_calibration_score_per_event
    - primary_exact_calibration_score_per_event
)

COARSENED_REFIT_MINUS_EXACT_CALIBRATION_SCORE_PER_EVENT: Final[
    float
] = (
    coarsened_refit_calibration_score_per_event
    - primary_exact_calibration_score_per_event
)

COARSENED_REFIT_MINUS_FIXED_CALIBRATION_SCORE_PER_EVENT: Final[
    float
] = (
    coarsened_refit_calibration_score_per_event
    - coarsened_fixed_calibration_score_per_event
)


# ------------------------------------------------------------
# Confirm the primary selected model was not mutated
# ------------------------------------------------------------

PRIMARY_SELECTED_PARAMETERS_AFTER_COARSENING: Final[
    np.ndarray
] = np.asarray(
    [
        SELECTED_HAWKES_PARAMETERS.mu_buy,
        SELECTED_HAWKES_PARAMETERS.mu_sell,
        SELECTED_HAWKES_PARAMETERS.kappa_buy,
        SELECTED_HAWKES_PARAMETERS.kappa_sell,
        SELECTED_HAWKES_PARAMETERS.beta_buy,
        SELECTED_HAWKES_PARAMETERS.beta_sell,
    ],
    dtype=np.float64,
)

PRIMARY_SELECTED_MODEL_PACKAGE_SHA256_AFTER_COARSENING: Final[
    str
] = canonical_json_sha256(
    SELECTED_HAWKES_MODEL_PACKAGE
)

PRIMARY_SELECTED_PARAMETERS_UNCHANGED_AFTER_COARSENING: Final[
    bool
] = bool(
    np.array_equal(
        PRIMARY_SELECTED_PARAMETERS_BEFORE_COARSENING,
        PRIMARY_SELECTED_PARAMETERS_AFTER_COARSENING,
    )
)

PRIMARY_SELECTED_MODEL_PACKAGE_UNCHANGED_AFTER_COARSENING: Final[
    bool
] = (
    PRIMARY_SELECTED_MODEL_PACKAGE_SHA256_BEFORE_COARSENING
    == PRIMARY_SELECTED_MODEL_PACKAGE_SHA256_AFTER_COARSENING
    == SELECTED_HAWKES_MODEL_PACKAGE_SHA256
)

require(
    PRIMARY_SELECTED_PARAMETERS_UNCHANGED_AFTER_COARSENING,
    "The primary selected parameters changed during sensitivity analysis.",
)
require(
    PRIMARY_SELECTED_MODEL_PACKAGE_UNCHANGED_AFTER_COARSENING,
    "The primary selected-model package changed during sensitivity analysis.",
)


# ------------------------------------------------------------
# Portable sensitivity package
# ------------------------------------------------------------

TIMESTAMP_COARSENING_SENSITIVITY_PACKAGE: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_TIMESTAMP_COARSENING_SENSITIVITY"
    ),
    "schema_version": (
        "NOTEBOOK_07_TIMESTAMP_COARSENING_SENSITIVITY_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V01_RUN_ID,
    "role": TIMESTAMP_COARSENING_ROLE,
    "primary_model_id": SELECTED_HAWKES_MODEL_ID,
    "primary_model_package_sha256": (
        SELECTED_HAWKES_MODEL_PACKAGE_SHA256
    ),
    "primary_calibration_package_sha256": (
        LOCKED_CALIBRATION_HAWKES_PACKAGE_SHA256
    ),
    "coarsening_contract": {
        "width_ns": TIMESTAMP_COARSENING_WIDTH_NS,
        "width_seconds": (
            TIMESTAMP_COARSENING_WIDTH_SECONDS
        ),
        "method": TIMESTAMP_COARSENING_METHOD,
        "partition_relative_origin": True,
        "floor_operation": True,
        "timestamp_jitter": False,
        "artificial_ordering": False,
        "event_removal": False,
        "within_batch_zero_lag_excitation": False,
    },
    "coarsened_development": {
        "exact_batch_count": len(
            DEVELOPMENT_BATCHES_FOR_HAWKES
        ),
        "coarsened_batch_count": len(
            COARSENED_DEVELOPMENT_BATCHES
        ),
        "event_count": (
            EXPECTED_DEVELOPMENT_EVENT_ROWS
        ),
    },
    "coarsened_calibration": {
        "exact_batch_count": len(
            CALIBRATION_BATCHES_FOR_HAWKES
        ),
        "coarsened_batch_count": len(
            COARSENED_CALIBRATION_BATCHES
        ),
        "event_count": (
            EXPECTED_CALIBRATION_EVENT_ROWS
        ),
        "parameter_updates": 0,
    },
    "coarsened_sensitivity_parameters": {
        "mu_buy_per_second": (
            COARSENED_SENSITIVITY_PARAMETERS.mu_buy
        ),
        "mu_sell_per_second": (
            COARSENED_SENSITIVITY_PARAMETERS.mu_sell
        ),
        "kappa_buy": (
            COARSENED_SENSITIVITY_PARAMETERS.kappa_buy
        ),
        "kappa_sell": (
            COARSENED_SENSITIVITY_PARAMETERS.kappa_sell
        ),
        "beta_buy_per_second": (
            COARSENED_SENSITIVITY_PARAMETERS.beta_buy
        ),
        "beta_sell_per_second": (
            COARSENED_SENSITIVITY_PARAMETERS.beta_sell
        ),
        "spectral_radius": (
            coarsened_parameter_validation[
                "spectral_radius"
            ]
        ),
    },
    "multi_start": {
        "attempted_start_count": len(
            TIMESTAMP_COARSENING_MULTISTART_RESULTS
        ),
        "accepted_start_count": int(
            TIMESTAMP_COARSENING_MULTISTART_RESULTS[
                "accepted_solution"
            ].sum()
        ),
        "material_optimum_cluster_size": (
            COARSENED_SENSITIVITY_OPTIMUM_CLUSTER_SIZE
        ),
        "runtime_seconds": (
            COARSENED_SENSITIVITY_FIT_RUNTIME_SECONDS
        ),
    },
    "score_comparison": {
        "primary_exact_development_log_score_per_event": (
            primary_exact_development_score_per_event
        ),
        "primary_exact_calibration_log_score_per_event": (
            primary_exact_calibration_score_per_event
        ),
        "coarsened_fixed_development_log_score_per_event": (
            coarsened_fixed_development_score_per_event
        ),
        "coarsened_fixed_calibration_log_score_per_event": (
            coarsened_fixed_calibration_score_per_event
        ),
        "coarsened_refit_development_log_score_per_event": (
            coarsened_refit_development_score_per_event
        ),
        "coarsened_refit_calibration_log_score_per_event": (
            coarsened_refit_calibration_score_per_event
        ),
    },
    "parameter_sensitivity": {
        "maximum_absolute_log_ratio": (
            TIMESTAMP_COARSENING_MAXIMUM_PARAMETER_ABSOLUTE_LOG_RATIO
        ),
        "maximum_deviation_factor": (
            TIMESTAMP_COARSENING_MAXIMUM_PARAMETER_DEVIATION_FACTOR
        ),
    },
    "primary_model_unchanged": True,
    "primary_authority_replaced": False,
    "calibration_used_for_sensitivity_fit": False,
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "claim_limits": {
        "hawkes_superiority_authorized": False,
        "coarsened_fit_replaces_primary_fit": False,
        "cross_excitation_authorized": False,
        "state_dependent_hawkes_authorized": False,
        "strategy_or_quoting_authorized": False,
        "pnl_claim_authorized": False,
    },
    "status": "PASS_SENSITIVITY_ONLY",
}

TIMESTAMP_COARSENING_SENSITIVITY_SHA256: Final[str] = (
    canonical_json_sha256(
        TIMESTAMP_COARSENING_SENSITIVITY_PACKAGE
    )
)


# ------------------------------------------------------------
# Gate ledger
# ------------------------------------------------------------

TIMESTAMP_COARSENING_GATE_LEDGER = pd.DataFrame(
    [
        {
            "gate_id": "EXACT_INTEGER_1MS_GRID",
            "passed": (
                TIMESTAMP_COARSENING_WIDTH_NS
                == NANOSECONDS_PER_MILLISECOND
            ),
        },
        {
            "gate_id": "DEVELOPMENT_EVENT_CONSERVATION",
            "passed": (
                batch_event_count(
                    COARSENED_DEVELOPMENT_BATCHES
                )
                == EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
        },
        {
            "gate_id": "CALIBRATION_EVENT_CONSERVATION",
            "passed": (
                batch_event_count(
                    COARSENED_CALIBRATION_BATCHES
                )
                == EXPECTED_CALIBRATION_EVENT_ROWS
            ),
        },
        {
            "gate_id": "NO_FORWARD_TIMESTAMP_MOVEMENT",
            "passed": True,
        },
        {
            "gate_id": "STRICT_PRE_BATCH_COARSENED_SCORING",
            "passed": True,
        },
        {
            "gate_id": "NO_WITHIN_BATCH_ZERO_LAG_EXCITATION",
            "passed": True,
        },
        {
            "gate_id": "COARSENED_MULTISTART_AGREEMENT",
            "passed": (
                COARSENED_SENSITIVITY_OPTIMUM_CLUSTER_SIZE
                >= MINIMUM_MATERIAL_OPTIMUM_CLUSTER_SIZE
            ),
        },
        {
            "gate_id": "COARSENED_PARAMETERS_ADMISSIBLE",
            "passed": (
                coarsened_parameter_validation[
                    "mathematically_admissible"
                ]
            ),
        },
        {
            "gate_id": "COARSENED_STATIONARITY_MARGIN",
            "passed": (
                coarsened_parameter_validation[
                    "acceptance_margin_holds"
                ]
            ),
        },
        {
            "gate_id": "COARSENED_CALIBRATION_ZERO_UPDATES",
            "passed": (
                COARSENED_SENSITIVITY_PARAMETER_UPDATES_ON_CALIBRATION
                == 0
            ),
        },
        {
            "gate_id": "PRIMARY_PARAMETERS_UNCHANGED",
            "passed": (
                PRIMARY_SELECTED_PARAMETERS_UNCHANGED_AFTER_COARSENING
            ),
        },
        {
            "gate_id": "PRIMARY_MODEL_PACKAGE_UNCHANGED",
            "passed": (
                PRIMARY_SELECTED_MODEL_PACKAGE_UNCHANGED_AFTER_COARSENING
            ),
        },
        {
            "gate_id": "PROTECTED_PARTITIONS_UNOPENED",
            "passed": (
                not any(
                    PROTECTED_PARTITION_CONTENT_LOADED[
                        partition
                    ]
                    for partition in PROTECTED_PARTITIONS
                )
            ),
        },
    ]
)

TIMESTAMP_COARSENING_GATE_LEDGER[
    "status"
] = np.where(
    TIMESTAMP_COARSENING_GATE_LEDGER[
        "passed"
    ],
    "PASS",
    "FAIL",
)

require(
    TIMESTAMP_COARSENING_GATE_LEDGER[
        "passed"
    ].all(),
    "The timestamp-coarsening sensitivity failed at least one gate.",
)


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

TIMESTAMP_COARSENING_SENSITIVITY_COMPLETED: bool = True
TIMESTAMP_COARSENING_SENSITIVITY_PASSED: bool = True
TIMESTAMP_COARSENING_PRIMARY_MODEL_REPLACED: bool = False

HAWKES_PARAMETER_LEDGER_AUTHORIZED: bool = True
V00_HAWKES_RECONCILIATION_AUTHORIZED: bool = True

HAWKES_SUPERIORITY_CLAIM_AUTHORIZED = False
BASELINE_SUPERIORITY_CLAIM_AUTHORIZED = False

FILESYSTEM_WRITES_PERFORMED = False

timestamp_coarsening_summary = pd.DataFrame(
    [
        {
            "field": "sensitivity_role",
            "value": TIMESTAMP_COARSENING_ROLE,
        },
        {
            "field": "coarsening_method",
            "value": TIMESTAMP_COARSENING_METHOD,
        },
        {
            "field": "coarsening_width_ns",
            "value": TIMESTAMP_COARSENING_WIDTH_NS,
        },
        {
            "field": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "development_exact_batch_count",
            "value": len(
                DEVELOPMENT_BATCHES_FOR_HAWKES
            ),
        },
        {
            "field": "development_coarsened_batch_count",
            "value": len(
                COARSENED_DEVELOPMENT_BATCHES
            ),
        },
        {
            "field": "calibration_exact_batch_count",
            "value": len(
                CALIBRATION_BATCHES_FOR_HAWKES
            ),
        },
        {
            "field": "calibration_coarsened_batch_count",
            "value": len(
                COARSENED_CALIBRATION_BATCHES
            ),
        },
        {
            "field": "attempted_sensitivity_starts",
            "value": len(
                TIMESTAMP_COARSENING_MULTISTART_RESULTS
            ),
        },
        {
            "field": "accepted_sensitivity_starts",
            "value": int(
                TIMESTAMP_COARSENING_MULTISTART_RESULTS[
                    "accepted_solution"
                ].sum()
            ),
        },
        {
            "field": "material_optimum_cluster_size",
            "value": (
                COARSENED_SENSITIVITY_OPTIMUM_CLUSTER_SIZE
            ),
        },
        {
            "field": (
                "maximum_parameter_deviation_factor"
            ),
            "value": (
                TIMESTAMP_COARSENING_MAXIMUM_PARAMETER_DEVIATION_FACTOR
            ),
        },
        {
            "field": (
                "coarsened_fixed_minus_exact_calibration_score_per_event"
            ),
            "value": (
                COARSENED_FIXED_MINUS_EXACT_CALIBRATION_SCORE_PER_EVENT
            ),
        },
        {
            "field": (
                "coarsened_refit_minus_exact_calibration_score_per_event"
            ),
            "value": (
                COARSENED_REFIT_MINUS_EXACT_CALIBRATION_SCORE_PER_EVENT
            ),
        },
        {
            "field": (
                "coarsened_refit_minus_fixed_calibration_score_per_event"
            ),
            "value": (
                COARSENED_REFIT_MINUS_FIXED_CALIBRATION_SCORE_PER_EVENT
            ),
        },
        {
            "field": "primary_model_replaced",
            "value": (
                TIMESTAMP_COARSENING_PRIMARY_MODEL_REPLACED
            ),
        },
        {
            "field": "primary_parameters_unchanged",
            "value": (
                PRIMARY_SELECTED_PARAMETERS_UNCHANGED_AFTER_COARSENING
            ),
        },
        {
            "field": "calibration_parameter_updates",
            "value": (
                COARSENED_SENSITIVITY_PARAMETER_UPDATES_ON_CALIBRATION
            ),
        },
        {
            "field": "sensitivity_sha256",
            "value": (
                TIMESTAMP_COARSENING_SENSITIVITY_SHA256
            ),
        },
        {
            "field": "hawkes_superiority_claim_authorized",
            "value": (
                HAWKES_SUPERIORITY_CLAIM_AUTHORIZED
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": "engineering_holdout_content_loaded",
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(timestamp_coarsening_summary)
display(TIMESTAMP_COARSENING_BATCH_GEOMETRY)
display(TIMESTAMP_COARSENING_PARAMETER_COMPARISON)
display(TIMESTAMP_COARSENING_SCORE_COMPARISON)

display(
    TIMESTAMP_COARSENING_MULTISTART_RESULTS.loc[
        TIMESTAMP_COARSENING_MULTISTART_RESULTS[
            "accepted_solution"
        ],
        [
            "start_id",
            "optimized_negative_log_likelihood",
            "objective_gap_from_best",
            "gradient_inf_per_event",
            "mu_buy",
            "mu_sell",
            "kappa_buy",
            "kappa_sell",
            "half_life_buy_seconds",
            "half_life_sell_seconds",
            "spectral_radius",
            "inside_material_optimum_cluster",
            "status",
        ],
    ]
    .sort_values(
        [
            "optimized_negative_log_likelihood",
            "start_id",
        ],
        kind="stable",
    )
    .head(10)
    .reset_index(drop=True)
)

display(TIMESTAMP_COARSENING_GATE_LEDGER)

print(
    "The selected H1 model completed an exact integer-nanosecond "
    "one-millisecond timestamp-coarsening sensitivity. DEVELOPMENT "
    "and CALIBRATION event counts were conserved, newly tied events "
    "were scored against strict pre-batch history, and no timestamp "
    "was moved forward. The primary frozen parameters were replayed "
    "on the coarsened interface, and a separate sensitivity-only H1 "
    "fit was estimated on coarsened DEVELOPMENT and evaluated on "
    "coarsened CALIBRATION with zero updates. The sensitivity fit "
    "does not replace the Notebook 04 timestamp authority or the "
    "primary selected model. Hawkes-superiority claims remain "
    "unauthorized, protected partitions remain unopened, and no "
    "filesystem writes were performed."
)

,field,value
0,sensitivity_role,SENSITIVITY_ONLY_NOT_PRIMARY_AUTHORITY
1,coarsening_method,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID
2,coarsening_width_ns,1000000
3,selected_model_id,H1_DIAGONAL_SHARED_DECAY
4,development_exact_batch_count,6859
5,development_coarsened_batch_count,6854
6,calibration_exact_batch_count,2400
7,calibration_coarsened_batch_count,2396
8,attempted_sensitivity_starts,25
9,accepted_sensitivity_starts,25


,event_partition,grid_width_ns,exact_batch_count,coarsened_batch_count,batch_count_reduction,exact_event_count,coarsened_event_count,coarsened_groups_merging_source_batches,source_batches_inside_merged_groups,coarsened_simultaneous_batch_count,coarsened_mixed_side_batch_count,maximum_events_in_coarsened_batch,maximum_source_batches_merged,maximum_source_time_span_ns,event_count_conserved,status
0,DEVELOPMENT,1000000,6859,6854,5,7004,7004,5,10,116,26,6,2,851900,True,PASS
1,CALIBRATION,1000000,2400,2396,4,2493,2493,4,8,69,6,5,2,786300,True,PASS


,parameter,exact_primary_value,coarsened_sensitivity_value,coarsened_to_exact_ratio,absolute_difference,absolute_relative_difference,absolute_log_ratio,status
0,mu_buy_per_second,1.536124799,1.53582699,0.9998061296,-0.0002978091324,0.0001938704021,0.0001938891974,DESCRIPTIVE
1,mu_sell_per_second,1.767916532,1.767683086,0.9998679544,-0.0002334456014,0.0001320456013,0.0001320543201,DESCRIPTIVE
2,kappa_buy,0.1892556591,0.1894127962,1.00083029,0.0001571370643,0.0008302899106,0.0008299454106,DESCRIPTIVE
3,kappa_sell,0.1126637105,0.1127808561,1.001039782,0.0001171456606,0.001039781666,0.001039241468,DESCRIPTIVE
4,beta_buy_per_second,70.67941658,69.81434436,0.9877606202,-0.8650722218,0.01223937977,0.01231489781,DESCRIPTIVE
5,beta_sell_per_second,70.67941658,69.81434436,0.9877606202,-0.8650722218,0.01223937977,0.01231489781,DESCRIPTIVE
6,buy_half_life_seconds,0.00980691712,0.009928435007,1.012391038,0.0001215178866,0.0123910384,0.01231489781,DESCRIPTIVE
7,sell_half_life_seconds,0.00980691712,0.009928435007,1.012391038,0.0001215178866,0.0123910384,0.01231489781,DESCRIPTIVE
8,spectral_radius,0.1892556591,0.1894127962,1.00083029,0.0001571370643,0.0008302899106,0.0008299454106,DESCRIPTIVE


,evaluation,development_log_score_per_event,calibration_log_score_per_event,calibration_minus_development,parameters_fit_on,timestamp_interface,primary_authority,status
0,PRIMARY_EXACT_TIMESTAMPS_PRIMARY_PARAMETERS,-0.1435555596,-0.1816036606,-0.03804810096,EXACT_DEVELOPMENT,NOTEBOOK_04_EVENT_TIME_NS,True,PASS
1,COARSENED_TIMESTAMPS_PRIMARY_PARAMETERS,-0.1442835127,-0.1843181629,-0.04003465016,EXACT_DEVELOPMENT,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID,False,PASS
2,COARSENED_TIMESTAMPS_COARSENED_DEV_REFIT,-0.1442812976,-0.1845162595,-0.04023496195,COARSENED_DEVELOPMENT_ONLY,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID,False,PASS


,start_id,optimized_negative_log_likelihood,objective_gap_from_best,gradient_inf_per_event,mu_buy,mu_sell,kappa_buy,kappa_sell,half_life_buy_seconds,half_life_sell_seconds,spectral_radius,inside_material_optimum_cluster,status
0,H1_DIAGONAL_SHARED_DECAY__START_0054,"1,010.546208",0,1.072682556e-08,1.53582699,1.767683086,0.1894127962,0.1127808561,0.009928435007,0.009928435007,0.1894127962,True,ACCEPTED
1,H1_DIAGONAL_SHARED_DECAY__START_0006,"1,010.546208",7.048583939e-12,8.051564717e-09,1.535826957,1.767683015,0.1894128486,0.1127808771,0.009928438856,0.009928438856,0.1894128486,True,ACCEPTED
2,H1_DIAGONAL_SHARED_DECAY__START_0048,"1,010.546208",3.990408004e-11,2.361231892e-08,1.535826874,1.767683272,0.1894128469,0.1127808931,0.009928440241,0.009928440241,0.1894128469,True,ACCEPTED
3,H1_DIAGONAL_SHARED_DECAY__START_0027,"1,010.546208",4.206412996e-11,2.952950531e-08,1.535827201,1.767683221,0.1894128768,0.1127808342,0.009928437732,0.009928437732,0.1894128768,True,ACCEPTED
4,H1_DIAGONAL_SHARED_DECAY__START_0080,"1,010.546208",4.877165338e-11,2.847147955e-08,1.535827019,1.767682902,0.1894128678,0.1127808116,0.009928433843,0.009928433843,0.1894128678,True,ACCEPTED
5,H1_DIAGONAL_SHARED_DECAY__START_0049,"1,010.546208",7.560174708e-11,2.57494846e-08,1.535826961,1.767682954,0.1894128546,0.112780775,0.0099284323,0.0099284323,0.1894128546,True,ACCEPTED
6,H1_DIAGONAL_SHARED_DECAY__START_0001,"1,010.546208",1.034550223e-10,3.065904164e-08,1.535826977,1.767682837,0.189412758,0.112780838,0.00992844044,0.00992844044,0.189412758,True,ACCEPTED
7,H1_DIAGONAL_SHARED_DECAY__START_0039,"1,010.546208",1.728039933e-10,3.071750967e-08,1.53582682,1.767683128,0.1894127437,0.1127809035,0.009928429423,0.009928429423,0.1894127437,True,ACCEPTED
8,H1_DIAGONAL_SHARED_DECAY__START_0033,"1,010.546208",3.209379429e-10,6.429596987e-08,1.535826562,1.767682839,0.1894127307,0.1127807715,0.00992842873,0.00992842873,0.1894127307,True,ACCEPTED
9,H1_DIAGONAL_SHARED_DECAY__START_0047,"1,010.546208",5.550191418e-10,9.7622076e-08,1.535826144,1.767682954,0.1894130152,0.1127808716,0.009928438643,0.009928438643,0.1894130152,True,ACCEPTED


,gate_id,passed,status
0,EXACT_INTEGER_1MS_GRID,True,PASS
1,DEVELOPMENT_EVENT_CONSERVATION,True,PASS
2,CALIBRATION_EVENT_CONSERVATION,True,PASS
3,NO_FORWARD_TIMESTAMP_MOVEMENT,True,PASS
4,STRICT_PRE_BATCH_COARSENED_SCORING,True,PASS
5,NO_WITHIN_BATCH_ZERO_LAG_EXCITATION,True,PASS
6,COARSENED_MULTISTART_AGREEMENT,True,PASS
7,COARSENED_PARAMETERS_ADMISSIBLE,True,PASS
8,COARSENED_STATIONARITY_MARGIN,True,PASS
9,COARSENED_CALIBRATION_ZERO_UPDATES,True,PASS


The selected H1 model completed an exact integer-nanosecond one-millisecond timestamp-coarsening sensitivity. DEVELOPMENT and CALIBRATION event counts were conserved, newly tied events were scored against strict pre-batch history, and no timestamp was moved forward. The primary frozen parameters were replayed on the coarsened interface, and a separate sensitivity-only H1 fit was estimated on coarsened DEVELOPMENT and evaluated on coarsened CALIBRATION with zero updates. The sensitivity fit does not replace the Notebook 04 timestamp authority or the primary selected model. Hawkes-superiority claims remain unauthorized, protected partitions remain unopened, and no filesystem writes were performed.


In [15]:
# ============================================================
# Freeze the final Hawkes parameter, selection, and evaluation ledger
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "HAWKES_SELECTED_MODEL_FROZEN",
            False,
        )
    ),
    "The selected primary Hawkes model is not frozen.",
)
require(
    bool(
        globals().get(
            "LOCKED_CALIBRATION_REPLAY_PASSED",
            False,
        )
    ),
    "The locked CALIBRATION replay has not passed.",
)
require(
    bool(
        globals().get(
            "TIMESTAMP_COARSENING_SENSITIVITY_PASSED",
            False,
        )
    ),
    "The timestamp-coarsening sensitivity has not passed.",
)
require(
    bool(
        globals().get(
            "HAWKES_SELECTED_MODEL_CHRONOLOGICALLY_STABLE",
            False,
        )
    ),
    "The selected model has not passed chronological stability.",
)
require(
    SELECTED_HAWKES_MODEL_ID == H1_MODEL_ID,
    "The frozen selected model is not the authorized H1 candidate.",
)
require(
    TIMESTAMP_COARSENING_PRIMARY_MODEL_REPLACED is False,
    "The sensitivity model must not replace the primary model.",
)
require(
    PRIMARY_SELECTED_PARAMETERS_UNCHANGED_AFTER_COARSENING,
    "The primary parameters changed during coarsening sensitivity.",
)
require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "Primary CALIBRATION parameter updates must remain zero.",
)
require(
    COARSENED_SENSITIVITY_PARAMETER_UPDATES_ON_CALIBRATION
    == 0,
    "Coarsened CALIBRATION parameter updates must remain zero.",
)
require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)
require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)
require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized at this stage.",
)


# ------------------------------------------------------------
# Primary and sensitivity-derived parameter quantities
# ------------------------------------------------------------

PRIMARY_ALPHA_BUY_TO_BUY_PER_SECOND: Final[float] = (
    SELECTED_HAWKES_PARAMETERS.kappa_buy
    * SELECTED_HAWKES_PARAMETERS.beta_buy
)

PRIMARY_ALPHA_SELL_TO_SELL_PER_SECOND: Final[float] = (
    SELECTED_HAWKES_PARAMETERS.kappa_sell
    * SELECTED_HAWKES_PARAMETERS.beta_sell
)

COARSENED_ALPHA_BUY_TO_BUY_PER_SECOND: Final[float] = (
    COARSENED_SENSITIVITY_PARAMETERS.kappa_buy
    * COARSENED_SENSITIVITY_PARAMETERS.beta_buy
)

COARSENED_ALPHA_SELL_TO_SELL_PER_SECOND: Final[float] = (
    COARSENED_SENSITIVITY_PARAMETERS.kappa_sell
    * COARSENED_SENSITIVITY_PARAMETERS.beta_sell
)

(
    COARSENED_MODEL_IMPLIED_MEAN_BUY_INTENSITY,
    COARSENED_MODEL_IMPLIED_MEAN_SELL_INTENSITY,
) = model_implied_mean_intensities(
    COARSENED_SENSITIVITY_PARAMETERS
)

(
    COARSENED_MODEL_IMPLIED_EXOGENOUS_BUY_SHARE,
    COARSENED_MODEL_IMPLIED_EXOGENOUS_SELL_SHARE,
) = model_implied_exogenous_shares(
    COARSENED_SENSITIVITY_PARAMETERS
)

PRIMARY_BRANCHING_MATRIX: Final[np.ndarray] = (
    integrated_kernel_matrix(
        SELECTED_HAWKES_PARAMETERS
    )
)

PRIMARY_KERNEL_AMPLITUDE_MATRIX_PER_SECOND: Final[
    np.ndarray
] = np.asarray(
    [
        [
            PRIMARY_ALPHA_BUY_TO_BUY_PER_SECOND,
            0.0,
        ],
        [
            0.0,
            PRIMARY_ALPHA_SELL_TO_SELL_PER_SECOND,
        ],
    ],
    dtype=np.float64,
)

COARSENED_BRANCHING_MATRIX: Final[np.ndarray] = (
    integrated_kernel_matrix(
        COARSENED_SENSITIVITY_PARAMETERS
    )
)

COARSENED_KERNEL_AMPLITUDE_MATRIX_PER_SECOND: Final[
    np.ndarray
] = np.asarray(
    [
        [
            COARSENED_ALPHA_BUY_TO_BUY_PER_SECOND,
            0.0,
        ],
        [
            0.0,
            COARSENED_ALPHA_SELL_TO_SELL_PER_SECOND,
        ],
    ],
    dtype=np.float64,
)

require(
    np.allclose(
        PRIMARY_BRANCHING_MATRIX[
            [0, 1],
            [1, 0],
        ],
        0.0,
        rtol=0.0,
        atol=0.0,
    ),
    "The primary branching matrix contains cross-excitation.",
)
require(
    np.allclose(
        PRIMARY_KERNEL_AMPLITUDE_MATRIX_PER_SECOND[
            [0, 1],
            [1, 0],
        ],
        0.0,
        rtol=0.0,
        atol=0.0,
    ),
    "The primary amplitude matrix contains cross-excitation.",
)
require(
    np.isfinite(
        PRIMARY_KERNEL_AMPLITUDE_MATRIX_PER_SECOND
    ).all(),
    "The primary kernel-amplitude matrix is non-finite.",
)
require(
    np.isfinite(
        COARSENED_KERNEL_AMPLITUDE_MATRIX_PER_SECOND
    ).all(),
    "The coarsened kernel-amplitude matrix is non-finite.",
)


# ------------------------------------------------------------
# Final primary-versus-sensitivity parameter ledger
# ------------------------------------------------------------

def parameter_comparison_record(
    *,
    parameter: str,
    symbol: str,
    primary_value: float,
    sensitivity_value: float,
    unit: str,
    role: str,
    constraint: str,
    interpretation_limit: str,
) -> dict[str, Any]:
    """Return one final primary/sensitivity parameter record."""
    require(
        math.isfinite(primary_value),
        f"The primary {parameter} value is non-finite.",
    )
    require(
        math.isfinite(sensitivity_value),
        f"The sensitivity {parameter} value is non-finite.",
    )

    if primary_value == 0.0:
        require(
            sensitivity_value == 0.0,
            (
                f"The fixed-zero parameter {parameter} changed "
                "under sensitivity analysis."
            ),
        )

        sensitivity_to_primary_ratio = math.nan
        absolute_relative_difference = 0.0
        absolute_log_ratio = math.nan
    else:
        sensitivity_to_primary_ratio = (
            sensitivity_value / primary_value
        )

        absolute_relative_difference = (
            abs(
                sensitivity_value - primary_value
            )
            / abs(primary_value)
        )

        require(
            sensitivity_to_primary_ratio > 0.0,
            (
                f"The sensitivity-to-primary ratio for "
                f"{parameter} is nonpositive."
            ),
        )

        absolute_log_ratio = abs(
            math.log(
                sensitivity_to_primary_ratio
            )
        )

    return {
        "parameter": parameter,
        "symbol": symbol,
        "primary_exact_value": float(
            primary_value
        ),
        "coarsened_sensitivity_value": float(
            sensitivity_value
        ),
        "coarsened_to_primary_ratio": (
            sensitivity_to_primary_ratio
        ),
        "absolute_difference": float(
            sensitivity_value - primary_value
        ),
        "absolute_relative_difference": float(
            absolute_relative_difference
        ),
        "absolute_log_ratio": absolute_log_ratio,
        "unit": unit,
        "role": role,
        "constraint": constraint,
        "primary_authority": (
            "NOTEBOOK_04_EVENT_TIME_NS"
        ),
        "sensitivity_authority": (
            "SECONDARY_1MS_COARSENING_ONLY"
        ),
        "interpretation_limit": (
            interpretation_limit
        ),
        "status": "PASS",
    }


NOTEBOOK07_FINAL_PARAMETER_LEDGER = pd.DataFrame(
    [
        parameter_comparison_record(
            parameter="mu_buy_per_second",
            symbol="mu_BUY",
            primary_value=(
                SELECTED_HAWKES_PARAMETERS.mu_buy
            ),
            sensitivity_value=(
                COARSENED_SENSITIVITY_PARAMETERS
                .mu_buy
            ),
            unit="EVENTS_PER_SECOND",
            role="BUY_BASE_INTENSITY",
            constraint="STRICTLY_POSITIVE",
            interpretation_limit=(
                "MODEL_PARAMETER_NOT_A_CAUSAL_"
                "EXOGENOUS_EVENT_COUNT"
            ),
        ),
        parameter_comparison_record(
            parameter="mu_sell_per_second",
            symbol="mu_SELL",
            primary_value=(
                SELECTED_HAWKES_PARAMETERS.mu_sell
            ),
            sensitivity_value=(
                COARSENED_SENSITIVITY_PARAMETERS
                .mu_sell
            ),
            unit="EVENTS_PER_SECOND",
            role="SELL_BASE_INTENSITY",
            constraint="STRICTLY_POSITIVE",
            interpretation_limit=(
                "MODEL_PARAMETER_NOT_A_CAUSAL_"
                "EXOGENOUS_EVENT_COUNT"
            ),
        ),
        parameter_comparison_record(
            parameter="kappa_buy_to_buy",
            symbol="kappa_BUY_BUY",
            primary_value=(
                SELECTED_HAWKES_PARAMETERS.kappa_buy
            ),
            sensitivity_value=(
                COARSENED_SENSITIVITY_PARAMETERS
                .kappa_buy
            ),
            unit="DIMENSIONLESS",
            role="BUY_SELF_EXCITATION_INTEGRATED_MASS",
            constraint="NONNEGATIVE_AND_BELOW_ONE",
            interpretation_limit=(
                "BRANCHING_INTERPRETATION_IS_MODEL_"
                "IMPLIED_NOT_PROVEN_CAUSALITY"
            ),
        ),
        parameter_comparison_record(
            parameter="kappa_sell_to_sell",
            symbol="kappa_SELL_SELL",
            primary_value=(
                SELECTED_HAWKES_PARAMETERS.kappa_sell
            ),
            sensitivity_value=(
                COARSENED_SENSITIVITY_PARAMETERS
                .kappa_sell
            ),
            unit="DIMENSIONLESS",
            role="SELL_SELF_EXCITATION_INTEGRATED_MASS",
            constraint="NONNEGATIVE_AND_BELOW_ONE",
            interpretation_limit=(
                "BRANCHING_INTERPRETATION_IS_MODEL_"
                "IMPLIED_NOT_PROVEN_CAUSALITY"
            ),
        ),
        parameter_comparison_record(
            parameter="kappa_buy_to_sell",
            symbol="kappa_SELL_BUY",
            primary_value=0.0,
            sensitivity_value=0.0,
            unit="DIMENSIONLESS",
            role="BUY_SOURCE_TO_SELL_TARGET_CROSS_MASS",
            constraint="FIXED_TO_ZERO",
            interpretation_limit=(
                "NOT_ESTIMATED_NOT_AUTHORIZED"
            ),
        ),
        parameter_comparison_record(
            parameter="kappa_sell_to_buy",
            symbol="kappa_BUY_SELL",
            primary_value=0.0,
            sensitivity_value=0.0,
            unit="DIMENSIONLESS",
            role="SELL_SOURCE_TO_BUY_TARGET_CROSS_MASS",
            constraint="FIXED_TO_ZERO",
            interpretation_limit=(
                "NOT_ESTIMATED_NOT_AUTHORIZED"
            ),
        ),
        parameter_comparison_record(
            parameter="beta_buy_per_second",
            symbol="beta_BUY",
            primary_value=(
                SELECTED_HAWKES_PARAMETERS.beta_buy
            ),
            sensitivity_value=(
                COARSENED_SENSITIVITY_PARAMETERS
                .beta_buy
            ),
            unit="PER_SECOND",
            role="BUY_EXCITATION_DECAY_RATE",
            constraint="STRICTLY_POSITIVE",
            interpretation_limit=(
                "SHARED_WITH_SELL_UNDER_SELECTED_H1"
            ),
        ),
        parameter_comparison_record(
            parameter="beta_sell_per_second",
            symbol="beta_SELL",
            primary_value=(
                SELECTED_HAWKES_PARAMETERS.beta_sell
            ),
            sensitivity_value=(
                COARSENED_SENSITIVITY_PARAMETERS
                .beta_sell
            ),
            unit="PER_SECOND",
            role="SELL_EXCITATION_DECAY_RATE",
            constraint="STRICTLY_POSITIVE",
            interpretation_limit=(
                "SHARED_WITH_BUY_UNDER_SELECTED_H1"
            ),
        ),
        parameter_comparison_record(
            parameter="alpha_buy_to_buy_per_second",
            symbol="alpha_BUY_BUY",
            primary_value=(
                PRIMARY_ALPHA_BUY_TO_BUY_PER_SECOND
            ),
            sensitivity_value=(
                COARSENED_ALPHA_BUY_TO_BUY_PER_SECOND
            ),
            unit="PER_SECOND",
            role="BUY_SELF_EXCITATION_KERNEL_AMPLITUDE",
            constraint="NONNEGATIVE",
            interpretation_limit=(
                "AMPLITUDE_EQUALS_KAPPA_TIMES_BETA"
            ),
        ),
        parameter_comparison_record(
            parameter="alpha_sell_to_sell_per_second",
            symbol="alpha_SELL_SELL",
            primary_value=(
                PRIMARY_ALPHA_SELL_TO_SELL_PER_SECOND
            ),
            sensitivity_value=(
                COARSENED_ALPHA_SELL_TO_SELL_PER_SECOND
            ),
            unit="PER_SECOND",
            role="SELL_SELF_EXCITATION_KERNEL_AMPLITUDE",
            constraint="NONNEGATIVE",
            interpretation_limit=(
                "AMPLITUDE_EQUALS_KAPPA_TIMES_BETA"
            ),
        ),
        parameter_comparison_record(
            parameter="alpha_buy_to_sell_per_second",
            symbol="alpha_SELL_BUY",
            primary_value=0.0,
            sensitivity_value=0.0,
            unit="PER_SECOND",
            role="BUY_SOURCE_TO_SELL_TARGET_AMPLITUDE",
            constraint="FIXED_TO_ZERO",
            interpretation_limit=(
                "NOT_ESTIMATED_NOT_AUTHORIZED"
            ),
        ),
        parameter_comparison_record(
            parameter="alpha_sell_to_buy_per_second",
            symbol="alpha_BUY_SELL",
            primary_value=0.0,
            sensitivity_value=0.0,
            unit="PER_SECOND",
            role="SELL_SOURCE_TO_BUY_TARGET_AMPLITUDE",
            constraint="FIXED_TO_ZERO",
            interpretation_limit=(
                "NOT_ESTIMATED_NOT_AUTHORIZED"
            ),
        ),
        parameter_comparison_record(
            parameter="buy_half_life_seconds",
            symbol="t_half_BUY",
            primary_value=(
                selected_half_life_buy_seconds
            ),
            sensitivity_value=(
                coarsened_half_life_buy_seconds
            ),
            unit="SECONDS",
            role="BUY_EXCITATION_HALF_LIFE",
            constraint="DERIVED_STRICTLY_POSITIVE",
            interpretation_limit=(
                "TIMESTAMP_INTERFACE_SENSITIVE"
            ),
        ),
        parameter_comparison_record(
            parameter="sell_half_life_seconds",
            symbol="t_half_SELL",
            primary_value=(
                selected_half_life_sell_seconds
            ),
            sensitivity_value=(
                coarsened_half_life_sell_seconds
            ),
            unit="SECONDS",
            role="SELL_EXCITATION_HALF_LIFE",
            constraint="DERIVED_STRICTLY_POSITIVE",
            interpretation_limit=(
                "TIMESTAMP_INTERFACE_SENSITIVE"
            ),
        ),
        parameter_comparison_record(
            parameter="spectral_radius",
            symbol="rho_K",
            primary_value=float(
                selected_final_validation[
                    "spectral_radius"
                ]
            ),
            sensitivity_value=float(
                coarsened_parameter_validation[
                    "spectral_radius"
                ]
            ),
            unit="DIMENSIONLESS",
            role="STATIONARITY_DIAGNOSTIC",
            constraint="STRICTLY_BELOW_ONE",
            interpretation_limit=(
                "DIAGONAL_MATRIX_MAXIMUM_KAPPA"
            ),
        ),
        parameter_comparison_record(
            parameter="model_implied_mean_buy_intensity",
            symbol="Lambda_bar_BUY",
            primary_value=(
                selected_mean_buy_intensity
            ),
            sensitivity_value=(
                COARSENED_MODEL_IMPLIED_MEAN_BUY_INTENSITY
            ),
            unit="EVENTS_PER_SECOND",
            role="STATIONARY_MODEL_MEAN_BUY_INTENSITY",
            constraint="DERIVED_STRICTLY_POSITIVE",
            interpretation_limit=(
                "MODEL_IMPLIED_STATIONARY_QUANTITY"
            ),
        ),
        parameter_comparison_record(
            parameter="model_implied_mean_sell_intensity",
            symbol="Lambda_bar_SELL",
            primary_value=(
                selected_mean_sell_intensity
            ),
            sensitivity_value=(
                COARSENED_MODEL_IMPLIED_MEAN_SELL_INTENSITY
            ),
            unit="EVENTS_PER_SECOND",
            role="STATIONARY_MODEL_MEAN_SELL_INTENSITY",
            constraint="DERIVED_STRICTLY_POSITIVE",
            interpretation_limit=(
                "MODEL_IMPLIED_STATIONARY_QUANTITY"
            ),
        ),
        parameter_comparison_record(
            parameter="model_implied_exogenous_buy_share",
            symbol="one_minus_kappa_BUY",
            primary_value=(
                selected_exogenous_buy_share
            ),
            sensitivity_value=(
                COARSENED_MODEL_IMPLIED_EXOGENOUS_BUY_SHARE
            ),
            unit="DIMENSIONLESS",
            role="MODEL_IMPLIED_BUY_EXOGENOUS_SHARE",
            constraint="DERIVED_IN_ZERO_ONE",
            interpretation_limit=(
                "NOT_AN_EMPIRICALLY_IDENTIFIED_CAUSAL_SHARE"
            ),
        ),
        parameter_comparison_record(
            parameter="model_implied_exogenous_sell_share",
            symbol="one_minus_kappa_SELL",
            primary_value=(
                selected_exogenous_sell_share
            ),
            sensitivity_value=(
                COARSENED_MODEL_IMPLIED_EXOGENOUS_SELL_SHARE
            ),
            unit="DIMENSIONLESS",
            role="MODEL_IMPLIED_SELL_EXOGENOUS_SHARE",
            constraint="DERIVED_IN_ZERO_ONE",
            interpretation_limit=(
                "NOT_AN_EMPIRICALLY_IDENTIFIED_CAUSAL_SHARE"
            ),
        ),
    ]
)

require(
    NOTEBOOK07_FINAL_PARAMETER_LEDGER[
        "status"
    ].eq("PASS").all(),
    "The final Hawkes parameter ledger contains a failed row.",
)

require(
    NOTEBOOK07_FINAL_PARAMETER_LEDGER.loc[
        NOTEBOOK07_FINAL_PARAMETER_LEDGER[
            "parameter"
        ].isin(
            (
                "kappa_buy_to_sell",
                "kappa_sell_to_buy",
                "alpha_buy_to_sell_per_second",
                "alpha_sell_to_buy_per_second",
            )
        ),
        [
            "primary_exact_value",
            "coarsened_sensitivity_value",
        ],
    ].eq(0.0).all().all(),
    "A prohibited cross-excitation parameter is nonzero.",
)


# ------------------------------------------------------------
# Candidate-selection ledger
# ------------------------------------------------------------

model_fit_summary_index = (
    HAWKES_MODEL_FIT_SUMMARY.set_index(
        "model_id",
        verify_integrity=True,
    )
)

fold_win_count_map: Final[Mapping[str, int]] = {
    H1_MODEL_ID: H1_FOLD_WIN_COUNT,
    H2_MODEL_ID: H2_FOLD_WIN_COUNT,
}

candidate_selection_rows: list[dict[str, Any]] = []

for model_id in CANDIDATE_MODEL_IDS:
    fit_row = model_fit_summary_index.loc[
        model_id
    ]

    candidate_selection_rows.append(
        {
            "model_id": model_id,
            "estimated_parameter_count": int(
                HAWKES_CANDIDATE_SPECIFICATIONS[
                    model_id
                ].estimated_parameter_count
            ),
            "shared_decay": bool(
                HAWKES_CANDIDATE_SPECIFICATIONS[
                    model_id
                ].shared_decay
            ),
            "development_negative_log_likelihood": float(
                fit_row[
                    "best_negative_log_likelihood"
                ]
            ),
            "development_log_likelihood": float(
                fit_row[
                    "best_log_likelihood"
                ]
            ),
            "development_log_score_per_event": float(
                fit_row[
                    "best_log_score_per_event"
                ]
            ),
            "development_aic": float(
                fit_row["development_aic"]
            ),
            "development_bic": float(
                fit_row["development_bic"]
            ),
            "development_fold_win_count": int(
                fold_win_count_map[model_id]
            ),
            "development_fold_win_fraction": (
                fold_win_count_map[model_id]
                / DEVELOPMENT_CHRONOLOGICAL_FOLDS
            ),
            "accepted_full_fit_start_count": int(
                fit_row[
                    "accepted_solution_count"
                ]
            ),
            "full_fit_optimum_cluster_size": int(
                fit_row[
                    "material_optimum_cluster_size"
                ]
            ),
            "spectral_radius": float(
                fit_row["spectral_radius"]
            ),
            "selected": (
                model_id
                == SELECTED_HAWKES_MODEL_ID
            ),
            "selection_outcome": (
                "SELECTED_SIMPLICITY_FIRST"
                if (
                    model_id
                    == SELECTED_HAWKES_MODEL_ID
                )
                else (
                    "REJECTED_NO_STABLE_"
                    "CHRONOLOGICAL_GAIN"
                )
            ),
            "calibration_used_for_selection": False,
            "status": "PASS",
        }
    )

NOTEBOOK07_CANDIDATE_SELECTION_LEDGER = (
    pd.DataFrame(candidate_selection_rows)
)

require(
    NOTEBOOK07_CANDIDATE_SELECTION_LEDGER[
        "selected"
    ].sum() == 1,
    "The candidate-selection ledger must contain one selected model.",
)
require(
    NOTEBOOK07_CANDIDATE_SELECTION_LEDGER.loc[
        NOTEBOOK07_CANDIDATE_SELECTION_LEDGER[
            "selected"
        ],
        "model_id",
    ].iloc[0]
    == SELECTED_HAWKES_MODEL_ID,
    "The candidate-selection ledger identifies the wrong model.",
)


# ------------------------------------------------------------
# Chronological parameter-range ledger
# ------------------------------------------------------------

chronological_range_rows: list[dict[str, Any]] = []

for stability_row in (
    SELECTED_HAWKES_PARAMETER_STABILITY_AUDIT
    .itertuples(index=False)
):
    chronological_range_rows.append(
        {
            "parameter": str(
                stability_row.parameter
            ),
            "full_development_value": float(
                stability_row.full_development_value
            ),
            "minimum_expanding_fold_value": float(
                stability_row.fold_minimum
            ),
            "maximum_expanding_fold_value": float(
                stability_row.fold_maximum
            ),
            "expanding_fold_mean": float(
                stability_row.fold_mean
            ),
            "expanding_fold_standard_deviation": float(
                stability_row.fold_standard_deviation
            ),
            "coefficient_of_variation": float(
                stability_row
                .fold_coefficient_of_variation
            ),
            "maximum_fold_to_full_deviation_factor": float(
                stability_row
                .maximum_fold_to_full_deviation_factor
            ),
            "latest_fold_to_full_deviation_factor": float(
                stability_row
                .latest_fold_to_full_deviation_factor
            ),
            "first_to_last_deviation_factor": float(
                stability_row
                .first_to_last_deviation_factor
            ),
            "passed": bool(
                stability_row.passed
            ),
            "status": str(
                stability_row.status
            ),
        }
    )

NOTEBOOK07_CHRONOLOGICAL_PARAMETER_LEDGER = (
    pd.DataFrame(chronological_range_rows)
)

require(
    NOTEBOOK07_CHRONOLOGICAL_PARAMETER_LEDGER[
        "passed"
    ].all(),
    "The chronological parameter ledger contains a failed parameter.",
)


# ------------------------------------------------------------
# Evaluation ledger
# ------------------------------------------------------------

DEVELOPMENT_PRIMARY_LOG_SCORE_PER_EVENT: Final[
    float
] = (
    SELECTED_HAWKES_DEVELOPMENT_REPLAY
    .log_likelihood
    / EXPECTED_DEVELOPMENT_EVENT_ROWS
)

CALIBRATION_PRIMARY_LOG_SCORE_PER_EVENT: Final[
    float
] = (
    LOCKED_CALIBRATION_HAWKES_RESULT
    .log_likelihood
    / EXPECTED_CALIBRATION_EVENT_ROWS
)

TOTAL_CHRONOLOGICAL_VALIDATION_EVENT_COUNT: Final[
    int
] = int(
    SELECTED_HAWKES_FOLD_RESULTS[
        "validation_event_count"
    ].sum()
)

CHRONOLOGICAL_VALIDATION_AGGREGATE_LOG_SCORE: Final[
    float
] = (
    SELECTED_HAWKES_WEIGHTED_VALIDATION_LOG_SCORE_PER_EVENT
    * TOTAL_CHRONOLOGICAL_VALIDATION_EVENT_COUNT
)

NOTEBOOK07_EVALUATION_LEDGER = pd.DataFrame(
    [
        {
            "evaluation_id": (
                "DEVELOPMENT_PRIMARY_EXACT"
            ),
            "evaluation_role": (
                "FINAL_PARAMETER_FIT_AND_REPLAY"
            ),
            "partition": FIT_PARTITION,
            "timestamp_interface": (
                "NOTEBOOK_04_EVENT_TIME_NS"
            ),
            "parameter_source": (
                "EXACT_DEVELOPMENT_SELECTED_MODEL"
            ),
            "event_count": (
                EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "batch_count": (
                EXPECTED_DEVELOPMENT_BATCH_ROWS
            ),
            "duration_seconds": (
                DEVELOPMENT_DURATION_SECONDS
            ),
            "log_likelihood": (
                SELECTED_HAWKES_DEVELOPMENT_REPLAY
                .log_likelihood
            ),
            "log_score_per_event": (
                DEVELOPMENT_PRIMARY_LOG_SCORE_PER_EVENT
            ),
            "parameter_updates": (
                "DEVELOPMENT_FIT"
            ),
            "primary_authority": True,
            "directly_comparable_to_primary": True,
            "status": "PASS",
        },
        {
            "evaluation_id": (
                "DEVELOPMENT_EXPANDING_FOLD_VALIDATION"
            ),
            "evaluation_role": (
                "MODEL_SELECTION_AND_STABILITY_DIAGNOSTIC"
            ),
            "partition": FIT_PARTITION,
            "timestamp_interface": (
                "NOTEBOOK_04_EVENT_TIME_NS"
            ),
            "parameter_source": (
                "EACH_FOLD_TRAINING_PREFIX"
            ),
            "event_count": (
                TOTAL_CHRONOLOGICAL_VALIDATION_EVENT_COUNT
            ),
            "batch_count": pd.NA,
            "duration_seconds": float(
                HAWKES_DEVELOPMENT_FOLD_GEOMETRY[
                    "validation_duration_seconds"
                ].sum()
            ),
            "log_likelihood": (
                CHRONOLOGICAL_VALIDATION_AGGREGATE_LOG_SCORE
            ),
            "log_score_per_event": (
                SELECTED_HAWKES_WEIGHTED_VALIDATION_LOG_SCORE_PER_EVENT
            ),
            "parameter_updates": (
                "TRAINING_PREFIX_ONLY"
            ),
            "primary_authority": False,
            "directly_comparable_to_primary": False,
            "status": "PASS",
        },
        {
            "evaluation_id": (
                "CALIBRATION_PRIMARY_EXACT_LOCKED"
            ),
            "evaluation_role": (
                "LOCKED_EVALUATION"
            ),
            "partition": (
                LOCKED_EVALUATION_PARTITION
            ),
            "timestamp_interface": (
                "NOTEBOOK_04_EVENT_TIME_NS"
            ),
            "parameter_source": (
                "FROZEN_EXACT_DEVELOPMENT_MODEL"
            ),
            "event_count": (
                EXPECTED_CALIBRATION_EVENT_ROWS
            ),
            "batch_count": (
                EXPECTED_CALIBRATION_BATCH_ROWS
            ),
            "duration_seconds": (
                CALIBRATION_DURATION_SECONDS
            ),
            "log_likelihood": (
                LOCKED_CALIBRATION_HAWKES_RESULT
                .log_likelihood
            ),
            "log_score_per_event": (
                CALIBRATION_PRIMARY_LOG_SCORE_PER_EVENT
            ),
            "parameter_updates": 0,
            "primary_authority": True,
            "directly_comparable_to_primary": True,
            "status": "PASS",
        },
        {
            "evaluation_id": (
                "DEVELOPMENT_1MS_PRIMARY_PARAMETERS"
            ),
            "evaluation_role": (
                "TIMESTAMP_INTERFACE_SENSITIVITY"
            ),
            "partition": FIT_PARTITION,
            "timestamp_interface": (
                TIMESTAMP_COARSENING_METHOD
            ),
            "parameter_source": (
                "FROZEN_EXACT_DEVELOPMENT_MODEL"
            ),
            "event_count": (
                EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "batch_count": len(
                COARSENED_DEVELOPMENT_BATCHES
            ),
            "duration_seconds": (
                DEVELOPMENT_DURATION_SECONDS
            ),
            "log_likelihood": (
                COARSENED_FIXED_PRIMARY_DEVELOPMENT_REPLAY
                .log_likelihood
            ),
            "log_score_per_event": (
                COARSENED_FIXED_PRIMARY_DEVELOPMENT_REPLAY
                .log_likelihood
                / EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "parameter_updates": 0,
            "primary_authority": False,
            "directly_comparable_to_primary": (
                "DESCRIPTIVE_INTERFACE_COMPARISON"
            ),
            "status": "PASS",
        },
        {
            "evaluation_id": (
                "CALIBRATION_1MS_PRIMARY_PARAMETERS"
            ),
            "evaluation_role": (
                "TIMESTAMP_INTERFACE_SENSITIVITY"
            ),
            "partition": (
                LOCKED_EVALUATION_PARTITION
            ),
            "timestamp_interface": (
                TIMESTAMP_COARSENING_METHOD
            ),
            "parameter_source": (
                "FROZEN_EXACT_DEVELOPMENT_MODEL"
            ),
            "event_count": (
                EXPECTED_CALIBRATION_EVENT_ROWS
            ),
            "batch_count": len(
                COARSENED_CALIBRATION_BATCHES
            ),
            "duration_seconds": (
                CALIBRATION_DURATION_SECONDS
            ),
            "log_likelihood": (
                COARSENED_FIXED_PRIMARY_CALIBRATION_REPLAY
                .log_likelihood
            ),
            "log_score_per_event": (
                COARSENED_FIXED_PRIMARY_CALIBRATION_REPLAY
                .log_likelihood
                / EXPECTED_CALIBRATION_EVENT_ROWS
            ),
            "parameter_updates": 0,
            "primary_authority": False,
            "directly_comparable_to_primary": (
                "DESCRIPTIVE_INTERFACE_COMPARISON"
            ),
            "status": "PASS",
        },
        {
            "evaluation_id": (
                "DEVELOPMENT_1MS_SENSITIVITY_REFIT"
            ),
            "evaluation_role": (
                "TIMESTAMP_INTERFACE_SENSITIVITY"
            ),
            "partition": FIT_PARTITION,
            "timestamp_interface": (
                TIMESTAMP_COARSENING_METHOD
            ),
            "parameter_source": (
                "COARSENED_DEVELOPMENT_ONLY"
            ),
            "event_count": (
                EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "batch_count": len(
                COARSENED_DEVELOPMENT_BATCHES
            ),
            "duration_seconds": (
                DEVELOPMENT_DURATION_SECONDS
            ),
            "log_likelihood": (
                COARSENED_REFIT_DEVELOPMENT_REPLAY
                .log_likelihood
            ),
            "log_score_per_event": (
                COARSENED_REFIT_DEVELOPMENT_REPLAY
                .log_likelihood
                / EXPECTED_DEVELOPMENT_EVENT_ROWS
            ),
            "parameter_updates": (
                "DEVELOPMENT_SENSITIVITY_FIT"
            ),
            "primary_authority": False,
            "directly_comparable_to_primary": (
                "DESCRIPTIVE_INTERFACE_COMPARISON"
            ),
            "status": "PASS",
        },
        {
            "evaluation_id": (
                "CALIBRATION_1MS_SENSITIVITY_REFIT_LOCKED"
            ),
            "evaluation_role": (
                "TIMESTAMP_INTERFACE_SENSITIVITY"
            ),
            "partition": (
                LOCKED_EVALUATION_PARTITION
            ),
            "timestamp_interface": (
                TIMESTAMP_COARSENING_METHOD
            ),
            "parameter_source": (
                "COARSENED_DEVELOPMENT_ONLY"
            ),
            "event_count": (
                EXPECTED_CALIBRATION_EVENT_ROWS
            ),
            "batch_count": len(
                COARSENED_CALIBRATION_BATCHES
            ),
            "duration_seconds": (
                CALIBRATION_DURATION_SECONDS
            ),
            "log_likelihood": (
                COARSENED_REFIT_CALIBRATION_REPLAY
                .log_likelihood
            ),
            "log_score_per_event": (
                COARSENED_REFIT_CALIBRATION_REPLAY
                .log_likelihood
                / EXPECTED_CALIBRATION_EVENT_ROWS
            ),
            "parameter_updates": 0,
            "primary_authority": False,
            "directly_comparable_to_primary": (
                "DESCRIPTIVE_INTERFACE_COMPARISON"
            ),
            "status": "PASS",
        },
    ]
)

require(
    NOTEBOOK07_EVALUATION_LEDGER[
        "status"
    ].eq("PASS").all(),
    "The final evaluation ledger contains a failed row.",
)

finite_evaluation_columns = (
    NOTEBOOK07_EVALUATION_LEDGER[
        [
            "event_count",
            "duration_seconds",
            "log_likelihood",
            "log_score_per_event",
        ]
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

require(
    np.isfinite(
        finite_evaluation_columns.to_numpy(
            dtype=np.float64
        )
    ).all(),
    "The final evaluation ledger contains non-finite values.",
)


# ------------------------------------------------------------
# Locked CALIBRATION side asymmetry ledger
#
# This records fitted-score behavior only. It does not authorize
# directional trading or price-movement interpretation.
# ------------------------------------------------------------

NOTEBOOK07_CALIBRATION_SIDE_LEDGER = (
    LOCKED_CALIBRATION_SIDE_SCORE_SUMMARY.copy()
)

NOTEBOOK07_CALIBRATION_SIDE_LEDGER[
    "interpretation_limit"
] = (
    "ARRIVAL_SCORE_ONLY_NOT_DIRECTIONAL_PRICE_SIGNAL"
)

require(
    NOTEBOOK07_CALIBRATION_SIDE_LEDGER[
        "status"
    ].eq("PASS").all(),
    "The CALIBRATION side ledger contains a failed row.",
)


# ------------------------------------------------------------
# Authority and claim ledger
# ------------------------------------------------------------

NOTEBOOK07_AUTHORITY_AND_CLAIM_LEDGER = pd.DataFrame(
    [
        {
            "item": (
                "PRIMARY_SELECTED_MODEL"
            ),
            "state": SELECTED_HAWKES_MODEL_ID,
            "authorized": True,
            "authority_scope": (
                "RESTRICTED_DIAGONAL_EXPONENTIAL_HAWKES"
            ),
        },
        {
            "item": (
                "PRIMARY_TIMESTAMP_INTERFACE"
            ),
            "state": (
                "NOTEBOOK_04_EVENT_TIME_NS"
            ),
            "authorized": True,
            "authority_scope": (
                "STRICT_PRE_BATCH_EXACT_TIME"
            ),
        },
        {
            "item": (
                "PRIMARY_ESTIMATOR_LABEL"
            ),
            "state": PRIMARY_ESTIMATOR_LABEL,
            "authorized": True,
            "authority_scope": (
                "COARSENED_TIME_QUASI_MLE"
            ),
        },
        {
            "item": (
                "BUY_TO_BUY_SELF_EXCITATION"
            ),
            "state": "FITTED",
            "authorized": True,
            "authority_scope": (
                "INTEGRATED_MASS_AND_SHARED_DECAY"
            ),
        },
        {
            "item": (
                "SELL_TO_SELL_SELF_EXCITATION"
            ),
            "state": "FITTED",
            "authorized": True,
            "authority_scope": (
                "INTEGRATED_MASS_AND_SHARED_DECAY"
            ),
        },
        {
            "item": "BUY_TO_SELL_CROSS_EXCITATION",
            "state": "FIXED_ZERO",
            "authorized": False,
            "authority_scope": "NOT_ESTIMATED",
        },
        {
            "item": "SELL_TO_BUY_CROSS_EXCITATION",
            "state": "FIXED_ZERO",
            "authorized": False,
            "authority_scope": "NOT_ESTIMATED",
        },
        {
            "item": "LOCKED_CALIBRATION_REPLAY",
            "state": "PASS",
            "authorized": True,
            "authority_scope": (
                "ZERO_PARAMETER_UPDATES_WITH_"
                "INHERITED_DEVELOPMENT_STATE"
            ),
        },
        {
            "item": (
                "ONE_MILLISECOND_COARSENING_SENSITIVITY"
            ),
            "state": "PASS_SENSITIVITY_ONLY",
            "authorized": True,
            "authority_scope": (
                "SECONDARY_DOES_NOT_REPLACE_PRIMARY"
            ),
        },
        {
            "item": "HAWKES_SUPERIORITY_CLAIM",
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": (
                "REQUIRES_NOTEBOOK_08_DIAGNOSTICS"
            ),
        },
        {
            "item": (
                "SIMPLE_BASELINE_SUPERIORITY_CLAIM"
            ),
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": (
                "REQUIRES_PAIRED_DIAGNOSTIC_COMPARISON"
            ),
        },
        {
            "item": (
                "STATE_DEPENDENT_HAWKES"
            ),
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": "OUT_OF_SCOPE",
        },
        {
            "item": (
                "UNRESTRICTED_BIVARIATE_HAWKES"
            ),
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": "OUT_OF_SCOPE",
        },
        {
            "item": "MARKET_MAKING_OR_QUOTING",
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": "V0_1_NON_GOAL",
        },
        {
            "item": "FILL_OR_QUEUE_SIMULATION",
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": "V0_1_NON_GOAL",
        },
        {
            "item": "PNL_OR_PERFORMANCE_CLAIM",
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": "V0_1_NON_GOAL",
        },
        {
            "item": "LIVE_TRADING_OR_DEPLOYMENT",
            "state": "NOT_AUTHORIZED",
            "authorized": False,
            "authority_scope": "V0_1_NON_GOAL",
        },
        {
            "item": (
                "NEXT_AUTHORIZED_OPERATION"
            ),
            "state": (
                "V0_0_HAWKES_REFERENCE_RECONCILIATION"
            ),
            "authorized": True,
            "authority_scope": (
                "READ_ONLY_REFERENCE_COMPARISON"
            ),
        },
    ]
)

NOTEBOOK07_AUTHORITY_AND_CLAIM_LEDGER[
    "status"
] = "PASS"


# ------------------------------------------------------------
# JSON-safe table conversion
# ------------------------------------------------------------

def json_safe_value(
    value: Any,
) -> Any:
    """Convert common pandas and NumPy values to canonical JSON types."""
    if value is None:
        return None

    if value is pd.NA:
        return None

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, bool):
        return value

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, int):
        return value

    if isinstance(value, np.floating):
        value = float(value)

    if isinstance(value, float):
        return value if math.isfinite(value) else None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.ndarray):
        return [
            json_safe_value(item)
            for item in value.tolist()
        ]

    if isinstance(value, tuple):
        return [
            json_safe_value(item)
            for item in value
        ]

    if isinstance(value, list):
        return [
            json_safe_value(item)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            str(key): json_safe_value(item)
            for key, item in value.items()
        }

    try:
        missing = pd.isna(value)
    except Exception:
        missing = False

    if isinstance(missing, (bool, np.bool_)) and missing:
        return None

    return str(value)


def portable_frame_records(
    frame: pd.DataFrame,
) -> list[dict[str, Any]]:
    """Return JSON-safe records preserving current row order."""
    return [
        {
            str(key): json_safe_value(value)
            for key, value in record.items()
        }
        for record in frame.to_dict(
            orient="records"
        )
    ]


# ------------------------------------------------------------
# Final parameter-and-evaluation package
# ------------------------------------------------------------

NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_HAWKES_PARAMETER_AND_EVALUATION_LEDGER"
    ),
    "schema_version": (
        "NOTEBOOK_07_HAWKES_PARAMETER_AND_EVALUATION_LEDGER_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
    "upstream_terminal_status": (
        UPSTREAM_REQUIRED_TERMINAL_STATUS
    ),
    "model_contract_sha256": (
        HAWKES_MODEL_CONTRACT_SHA256
    ),
    "candidate_registry_sha256": (
        HAWKES_CANDIDATE_REGISTRY_SHA256
    ),
    "optimization_results_sha256": (
        HAWKES_OPTIMIZATION_RESULTS_SHA256
    ),
    "candidate_selection_sha256": (
        HAWKES_CANDIDATE_SELECTION_SHA256
    ),
    "selected_model_package_sha256": (
        SELECTED_HAWKES_MODEL_PACKAGE_SHA256
    ),
    "chronological_stability_sha256": (
        SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_SHA256
    ),
    "locked_calibration_package_sha256": (
        LOCKED_CALIBRATION_HAWKES_PACKAGE_SHA256
    ),
    "timestamp_coarsening_sensitivity_sha256": (
        TIMESTAMP_COARSENING_SENSITIVITY_SHA256
    ),
    "selected_model": {
        "model_id": SELECTED_HAWKES_MODEL_ID,
        "selection_reason": (
            SELECTED_HAWKES_MODEL_REASON
        ),
        "model_family": AUTHORIZED_FIRST_MODEL,
        "kernel_family": AUTHORIZED_KERNEL_FAMILY,
        "shared_decay": True,
        "authorized_channels": list(
            AUTHORIZED_CHANNELS
        ),
        "fixed_zero_channels": list(
            FIXED_ZERO_CHANNELS
        ),
        "estimator_label": (
            PRIMARY_ESTIMATOR_LABEL
        ),
        "primary_timestamp_interface": (
            "NOTEBOOK_04_EVENT_TIME_NS"
        ),
    },
    "primary_branching_matrix": (
        PRIMARY_BRANCHING_MATRIX.tolist()
    ),
    "primary_kernel_amplitude_matrix_per_second": (
        PRIMARY_KERNEL_AMPLITUDE_MATRIX_PER_SECOND
        .tolist()
    ),
    "parameter_ledger": portable_frame_records(
        NOTEBOOK07_FINAL_PARAMETER_LEDGER
    ),
    "candidate_selection_ledger": (
        portable_frame_records(
            NOTEBOOK07_CANDIDATE_SELECTION_LEDGER
        )
    ),
    "chronological_parameter_ledger": (
        portable_frame_records(
            NOTEBOOK07_CHRONOLOGICAL_PARAMETER_LEDGER
        )
    ),
    "evaluation_ledger": portable_frame_records(
        NOTEBOOK07_EVALUATION_LEDGER
    ),
    "calibration_side_ledger": (
        portable_frame_records(
            NOTEBOOK07_CALIBRATION_SIDE_LEDGER
        )
    ),
    "authority_and_claim_ledger": (
        portable_frame_records(
            NOTEBOOK07_AUTHORITY_AND_CLAIM_LEDGER
        )
    ),
    "partition_access": {
        "development_loaded": True,
        "calibration_loaded": True,
        "validation_loaded": False,
        "engineering_holdout_loaded": False,
    },
    "parameter_updates": {
        "locked_calibration": 0,
        "coarsened_sensitivity_calibration": 0,
    },
    "claim_limits": {
        "hawkes_superiority_authorized": False,
        "simple_baseline_superiority_authorized": False,
        "cross_excitation_authorized": False,
        "state_dependent_hawkes_authorized": False,
        "market_making_authorized": False,
        "quoting_authorized": False,
        "fill_simulation_authorized": False,
        "pnl_claim_authorized": False,
        "live_trading_authorized": False,
    },
    "next_authorized_operation": (
        "V0_0_HAWKES_REFERENCE_RECONCILIATION"
    ),
    "status": "PASS_PARAMETER_LEDGER_FROZEN",
}

NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE
)


# ------------------------------------------------------------
# Freeze audit
# ------------------------------------------------------------

NOTEBOOK07_PARAMETER_LEDGER_GATE_LEDGER = pd.DataFrame(
    [
        {
            "gate_id": "SELECTED_MODEL_FROZEN",
            "passed": HAWKES_SELECTED_MODEL_FROZEN,
        },
        {
            "gate_id": "PRIMARY_MODEL_IS_H1",
            "passed": (
                SELECTED_HAWKES_MODEL_ID
                == H1_MODEL_ID
            ),
        },
        {
            "gate_id": "PRIMARY_PARAMETERS_FINITE",
            "passed": (
                NOTEBOOK07_FINAL_PARAMETER_LEDGER[
                    "primary_exact_value"
                ]
                .map(math.isfinite)
                .all()
            ),
        },
        {
            "gate_id": "SENSITIVITY_PARAMETERS_FINITE",
            "passed": (
                NOTEBOOK07_FINAL_PARAMETER_LEDGER[
                    "coarsened_sensitivity_value"
                ]
                .map(math.isfinite)
                .all()
            ),
        },
        {
            "gate_id": "CROSS_EXCITATION_FIXED_ZERO",
            "passed": (
                np.allclose(
                    PRIMARY_BRANCHING_MATRIX[
                        [0, 1],
                        [1, 0],
                    ],
                    0.0,
                    rtol=0.0,
                    atol=0.0,
                )
            ),
        },
        {
            "gate_id": "PRIMARY_STATIONARITY",
            "passed": (
                selected_final_validation[
                    "stationarity_holds"
                ]
            ),
        },
        {
            "gate_id": "PRIMARY_STATIONARITY_MARGIN",
            "passed": (
                selected_final_validation[
                    "acceptance_margin_holds"
                ]
            ),
        },
        {
            "gate_id": (
                "CHRONOLOGICAL_PARAMETER_STABILITY"
            ),
            "passed": (
                NOTEBOOK07_CHRONOLOGICAL_PARAMETER_LEDGER[
                    "passed"
                ].all()
            ),
        },
        {
            "gate_id": "LOCKED_CALIBRATION_PASS",
            "passed": (
                LOCKED_CALIBRATION_REPLAY_PASSED
            ),
        },
        {
            "gate_id": (
                "TIMESTAMP_COARSENING_SENSITIVITY_PASS"
            ),
            "passed": (
                TIMESTAMP_COARSENING_SENSITIVITY_PASSED
            ),
        },
        {
            "gate_id": (
                "SENSITIVITY_DOES_NOT_REPLACE_PRIMARY"
            ),
            "passed": (
                not TIMESTAMP_COARSENING_PRIMARY_MODEL_REPLACED
            ),
        },
        {
            "gate_id": "CALIBRATION_ZERO_UPDATES",
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "gate_id": (
                "PROTECTED_PARTITIONS_UNOPENED"
            ),
            "passed": (
                not any(
                    PROTECTED_PARTITION_CONTENT_LOADED[
                        partition
                    ]
                    for partition in PROTECTED_PARTITIONS
                )
            ),
        },
        {
            "gate_id": "FILESYSTEM_UNCHANGED",
            "passed": (
                FILESYSTEM_WRITES_PERFORMED
                is False
            ),
        },
    ]
)

NOTEBOOK07_PARAMETER_LEDGER_GATE_LEDGER[
    "status"
] = np.where(
    NOTEBOOK07_PARAMETER_LEDGER_GATE_LEDGER[
        "passed"
    ],
    "PASS",
    "FAIL",
)

require(
    NOTEBOOK07_PARAMETER_LEDGER_GATE_LEDGER[
        "passed"
    ].all(),
    "The final Hawkes parameter ledger failed at least one gate.",
)


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

HAWKES_PARAMETER_LEDGER_COMPLETED: bool = True
HAWKES_PARAMETER_LEDGER_FROZEN: bool = True
HAWKES_EVALUATION_LEDGER_FROZEN: bool = True

V00_HAWKES_RECONCILIATION_AUTHORIZED = True
V00_HAWKES_RECONCILIATION_STARTED: bool = False

HAWKES_SUPERIORITY_CLAIM_AUTHORIZED = False
BASELINE_SUPERIORITY_CLAIM_AUTHORIZED = False

FILESYSTEM_WRITES_PERFORMED = False

parameter_ledger_summary = pd.DataFrame(
    [
        {
            "field": "parameter_ledger_completed",
            "value": HAWKES_PARAMETER_LEDGER_COMPLETED,
        },
        {
            "field": "parameter_ledger_frozen",
            "value": HAWKES_PARAMETER_LEDGER_FROZEN,
        },
        {
            "field": "evaluation_ledger_frozen",
            "value": HAWKES_EVALUATION_LEDGER_FROZEN,
        },
        {
            "field": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "primary_parameter_count",
            "value": (
                SELECTED_HAWKES_SPECIFICATION
                .estimated_parameter_count
            ),
        },
        {
            "field": "primary_spectral_radius",
            "value": (
                selected_final_validation[
                    "spectral_radius"
                ]
            ),
        },
        {
            "field": (
                "maximum_coarsening_parameter_deviation_factor"
            ),
            "value": (
                TIMESTAMP_COARSENING_MAXIMUM_PARAMETER_DEVIATION_FACTOR
            ),
        },
        {
            "field": (
                "development_primary_log_score_per_event"
            ),
            "value": (
                DEVELOPMENT_PRIMARY_LOG_SCORE_PER_EVENT
            ),
        },
        {
            "field": (
                "chronological_validation_weighted_log_score_per_event"
            ),
            "value": (
                SELECTED_HAWKES_WEIGHTED_VALIDATION_LOG_SCORE_PER_EVENT
            ),
        },
        {
            "field": (
                "calibration_primary_log_score_per_event"
            ),
            "value": (
                CALIBRATION_PRIMARY_LOG_SCORE_PER_EVENT
            ),
        },
        {
            "field": (
                "parameter_and_evaluation_package_sha256"
            ),
            "value": (
                NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256
            ),
        },
        {
            "field": (
                "v0_0_hawkes_reconciliation_authorized"
            ),
            "value": (
                V00_HAWKES_RECONCILIATION_AUTHORIZED
            ),
        },
        {
            "field": (
                "v0_0_hawkes_reconciliation_started"
            ),
            "value": (
                V00_HAWKES_RECONCILIATION_STARTED
            ),
        },
        {
            "field": (
                "hawkes_superiority_claim_authorized"
            ),
            "value": (
                HAWKES_SUPERIORITY_CLAIM_AUTHORIZED
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": (
                "engineering_holdout_content_loaded"
            ),
            "value": False,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(parameter_ledger_summary)

display(
    NOTEBOOK07_FINAL_PARAMETER_LEDGER[
        [
            "parameter",
            "symbol",
            "primary_exact_value",
            "coarsened_sensitivity_value",
            "coarsened_to_primary_ratio",
            "absolute_relative_difference",
            "unit",
            "role",
            "constraint",
            "status",
        ]
    ]
)

display(
    NOTEBOOK07_CANDIDATE_SELECTION_LEDGER
)

display(
    NOTEBOOK07_CHRONOLOGICAL_PARAMETER_LEDGER
)

display(
    NOTEBOOK07_EVALUATION_LEDGER[
        [
            "evaluation_id",
            "evaluation_role",
            "partition",
            "timestamp_interface",
            "parameter_source",
            "event_count",
            "batch_count",
            "log_likelihood",
            "log_score_per_event",
            "parameter_updates",
            "primary_authority",
            "status",
        ]
    ]
)

display(NOTEBOOK07_CALIBRATION_SIDE_LEDGER)
display(NOTEBOOK07_AUTHORITY_AND_CLAIM_LEDGER)
display(NOTEBOOK07_PARAMETER_LEDGER_GATE_LEDGER)

print(
    "The final Notebook 07 Hawkes parameter, candidate-selection, "
    "chronological-stability, locked-evaluation, and timestamp-"
    "sensitivity ledgers are frozen. The primary authority remains "
    "the H1 diagonal shared-decay model fitted on exact Notebook 04 "
    "event_time_ns timestamps. The one-millisecond model remains a "
    "secondary sensitivity result and differs only modestly from the "
    "primary parameters. Cross-excitation remains fixed to zero, "
    "CALIBRATION parameter updates remain zero, and no Hawkes-"
    "superiority or strategy claim is authorized. Read-only V0.0 "
    "Hawkes reference reconciliation is now authorized. Protected "
    "partitions remain unopened and no filesystem writes were "
    "performed."
)

,field,value
0,parameter_ledger_completed,True
1,parameter_ledger_frozen,True
2,evaluation_ledger_frozen,True
3,selected_model_id,H1_DIAGONAL_SHARED_DECAY
4,primary_parameter_count,5
5,primary_spectral_radius,0.1892556591
6,maximum_coarsening_parameter_deviation_factor,1.012391038
7,development_primary_log_score_per_event,-0.1435555596
8,chronological_validation_weighted_log_score_pe...,-0.07054607905
9,calibration_primary_log_score_per_event,-0.1816036606


,parameter,symbol,primary_exact_value,coarsened_sensitivity_value,coarsened_to_primary_ratio,absolute_relative_difference,unit,role,constraint,status
0,mu_buy_per_second,mu_BUY,1.536124799,1.53582699,0.9998061296,0.0001938704021,EVENTS_PER_SECOND,BUY_BASE_INTENSITY,STRICTLY_POSITIVE,PASS
1,mu_sell_per_second,mu_SELL,1.767916532,1.767683086,0.9998679544,0.0001320456013,EVENTS_PER_SECOND,SELL_BASE_INTENSITY,STRICTLY_POSITIVE,PASS
2,kappa_buy_to_buy,kappa_BUY_BUY,0.1892556591,0.1894127962,1.00083029,0.0008302899106,DIMENSIONLESS,BUY_SELF_EXCITATION_INTEGRATED_MASS,NONNEGATIVE_AND_BELOW_ONE,PASS
3,kappa_sell_to_sell,kappa_SELL_SELL,0.1126637105,0.1127808561,1.001039782,0.001039781666,DIMENSIONLESS,SELL_SELF_EXCITATION_INTEGRATED_MASS,NONNEGATIVE_AND_BELOW_ONE,PASS
4,kappa_buy_to_sell,kappa_SELL_BUY,0,0,NaN,0,DIMENSIONLESS,BUY_SOURCE_TO_SELL_TARGET_CROSS_MASS,FIXED_TO_ZERO,PASS
5,kappa_sell_to_buy,kappa_BUY_SELL,0,0,NaN,0,DIMENSIONLESS,SELL_SOURCE_TO_BUY_TARGET_CROSS_MASS,FIXED_TO_ZERO,PASS
6,beta_buy_per_second,beta_BUY,70.67941658,69.81434436,0.9877606202,0.01223937977,PER_SECOND,BUY_EXCITATION_DECAY_RATE,STRICTLY_POSITIVE,PASS
7,beta_sell_per_second,beta_SELL,70.67941658,69.81434436,0.9877606202,0.01223937977,PER_SECOND,SELL_EXCITATION_DECAY_RATE,STRICTLY_POSITIVE,PASS
8,alpha_buy_to_buy_per_second,alpha_BUY_BUY,13.37647957,13.22373018,0.9885807479,0.0114192521,PER_SECOND,BUY_SELF_EXCITATION_KERNEL_AMPLITUDE,NONNEGATIVE,PASS
9,alpha_sell_to_sell_per_second,alpha_SELL_SELL,7.963005327,7.873721528,0.9887876756,0.01121232439,PER_SECOND,SELL_SELF_EXCITATION_KERNEL_AMPLITUDE,NONNEGATIVE,PASS


,model_id,estimated_parameter_count,shared_decay,development_negative_log_likelihood,development_log_likelihood,development_log_score_per_event,development_aic,development_bic,development_fold_win_count,development_fold_win_fraction,accepted_full_fit_start_count,full_fit_optimum_cluster_size,spectral_radius,selected,selection_outcome,calibration_used_for_selection,status
0,H1_DIAGONAL_SHARED_DECAY,5,True,"1,005.46314","-1,005.46314",-0.1435555596,"2,020.92628","2,055.197463",4,0.8,25,25,0.1892556591,True,SELECTED_SIMPLICITY_FIRST,False,PASS
1,H2_DIAGONAL_SEPARATE_DECAY,6,False,"1,001.074143","-1,001.074143",-0.1429289181,"2,014.148285","2,055.273705",1,0.2,41,40,0.1824833409,False,REJECTED_NO_STABLE_CHRONOLOGICAL_GAIN,False,PASS


,parameter,full_development_value,minimum_expanding_fold_value,maximum_expanding_fold_value,expanding_fold_mean,expanding_fold_standard_deviation,coefficient_of_variation,maximum_fold_to_full_deviation_factor,latest_fold_to_full_deviation_factor,first_to_last_deviation_factor,passed,status
0,mu_buy,1.536124799,1.546401685,1.585241958,1.567152713,0.0171664675,0.01095392131,1.031974719,1.006690138,1.024547261,True,PASS
1,mu_sell,1.767916532,1.751331267,1.832966744,1.793710478,0.03028041692,0.01688144061,1.036794844,1.00947009,1.046613384,True,PASS
2,kappa_buy,0.1892556591,0.1421608491,0.1823413804,0.1592966244,0.01743907648,0.1094754929,1.331278339,1.037919416,1.158598223,True,PASS
3,kappa_sell,0.1126637105,0.1067618419,0.1264597727,0.1174565569,0.008798456232,0.07490817428,1.122453469,1.105152534,1.134272126,True,PASS
4,half_life_buy_seconds,0.00980691712,0.01176009143,0.01980360485,0.01609349454,0.003121545837,0.1939632085,2.01935069,1.199162926,1.683966912,True,PASS
5,half_life_sell_seconds,0.00980691712,0.01176009143,0.01980360485,0.01609349454,0.003121545837,0.1939632085,2.01935069,1.199162926,1.683966912,True,PASS
6,spectral_radius,0.1892556591,0.1421608491,0.1823413804,0.1592966244,0.01743907648,0.1094754929,1.331278339,1.037919416,1.158598223,True,PASS


,evaluation_id,evaluation_role,partition,timestamp_interface,parameter_source,event_count,batch_count,log_likelihood,log_score_per_event,parameter_updates,primary_authority,status
0,DEVELOPMENT_PRIMARY_EXACT,FINAL_PARAMETER_FIT_AND_REPLAY,DEVELOPMENT,NOTEBOOK_04_EVENT_TIME_NS,EXACT_DEVELOPMENT_SELECTED_MODEL,7004,6859,"-1,005.46314",-0.1435555596,DEVELOPMENT_FIT,True,PASS
1,DEVELOPMENT_EXPANDING_FOLD_VALIDATION,MODEL_SELECTION_AND_STABILITY_DIAGNOSTIC,DEVELOPMENT,NOTEBOOK_04_EVENT_TIME_NS,EACH_FOLD_TRAINING_PREFIX,3455,<NA>,-243.7367031,-0.07054607905,TRAINING_PREFIX_ONLY,False,PASS
2,CALIBRATION_PRIMARY_EXACT_LOCKED,LOCKED_EVALUATION,CALIBRATION,NOTEBOOK_04_EVENT_TIME_NS,FROZEN_EXACT_DEVELOPMENT_MODEL,2493,2400,-452.7379259,-0.1816036606,0,True,PASS
3,DEVELOPMENT_1MS_PRIMARY_PARAMETERS,TIMESTAMP_INTERFACE_SENSITIVITY,DEVELOPMENT,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID,FROZEN_EXACT_DEVELOPMENT_MODEL,7004,6854,"-1,010.561723",-0.1442835127,0,False,PASS
4,CALIBRATION_1MS_PRIMARY_PARAMETERS,TIMESTAMP_INTERFACE_SENSITIVITY,CALIBRATION,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID,FROZEN_EXACT_DEVELOPMENT_MODEL,2493,2396,-459.50518,-0.1843181629,0,False,PASS
5,DEVELOPMENT_1MS_SENSITIVITY_REFIT,TIMESTAMP_INTERFACE_SENSITIVITY,DEVELOPMENT,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID,COARSENED_DEVELOPMENT_ONLY,7004,6854,"-1,010.546208",-0.1442812976,DEVELOPMENT_SENSITIVITY_FIT,False,PASS
6,CALIBRATION_1MS_SENSITIVITY_REFIT_LOCKED,TIMESTAMP_INTERFACE_SENSITIVITY,CALIBRATION,PARTITION_RELATIVE_FLOOR_TO_EXACT_1MS_GRID,COARSENED_DEVELOPMENT_ONLY,2493,2396,-459.999035,-0.1845162595,0,False,PASS


,event_side,event_count,log_event_term,compensator,log_likelihood,log_score_per_event,status,interpretation_limit
0,BUY,1230,"1,346.220544","1,339.253892",6.966651678,0.005663944454,PASS,ARRIVAL_SCORE_ONLY_NOT_DIRECTIONAL_PRICE_SIGNAL
1,SELL,1263,956.0185167,"1,415.723094",-459.7045776,-0.3639782878,PASS,ARRIVAL_SCORE_ONLY_NOT_DIRECTIONAL_PRICE_SIGNAL


,item,state,authorized,authority_scope,status
0,PRIMARY_SELECTED_MODEL,H1_DIAGONAL_SHARED_DECAY,True,RESTRICTED_DIAGONAL_EXPONENTIAL_HAWKES,PASS
1,PRIMARY_TIMESTAMP_INTERFACE,NOTEBOOK_04_EVENT_TIME_NS,True,STRICT_PRE_BATCH_EXACT_TIME,PASS
2,PRIMARY_ESTIMATOR_LABEL,STRICT_PRE_BATCH_COARSENED_TIME_QUASI_MLE,True,COARSENED_TIME_QUASI_MLE,PASS
3,BUY_TO_BUY_SELF_EXCITATION,FITTED,True,INTEGRATED_MASS_AND_SHARED_DECAY,PASS
4,SELL_TO_SELL_SELF_EXCITATION,FITTED,True,INTEGRATED_MASS_AND_SHARED_DECAY,PASS
5,BUY_TO_SELL_CROSS_EXCITATION,FIXED_ZERO,False,NOT_ESTIMATED,PASS
6,SELL_TO_BUY_CROSS_EXCITATION,FIXED_ZERO,False,NOT_ESTIMATED,PASS
7,LOCKED_CALIBRATION_REPLAY,PASS,True,ZERO_PARAMETER_UPDATES_WITH_INHERITED_DEVELOPM...,PASS
8,ONE_MILLISECOND_COARSENING_SENSITIVITY,PASS_SENSITIVITY_ONLY,True,SECONDARY_DOES_NOT_REPLACE_PRIMARY,PASS
9,HAWKES_SUPERIORITY_CLAIM,NOT_AUTHORIZED,False,REQUIRES_NOTEBOOK_08_DIAGNOSTICS,PASS


,gate_id,passed,status
0,SELECTED_MODEL_FROZEN,True,PASS
1,PRIMARY_MODEL_IS_H1,True,PASS
2,PRIMARY_PARAMETERS_FINITE,True,PASS
3,SENSITIVITY_PARAMETERS_FINITE,True,PASS
4,CROSS_EXCITATION_FIXED_ZERO,True,PASS
5,PRIMARY_STATIONARITY,True,PASS
6,PRIMARY_STATIONARITY_MARGIN,True,PASS
7,CHRONOLOGICAL_PARAMETER_STABILITY,True,PASS
8,LOCKED_CALIBRATION_PASS,True,PASS
9,TIMESTAMP_COARSENING_SENSITIVITY_PASS,True,PASS


The final Notebook 07 Hawkes parameter, candidate-selection, chronological-stability, locked-evaluation, and timestamp-sensitivity ledgers are frozen. The primary authority remains the H1 diagonal shared-decay model fitted on exact Notebook 04 event_time_ns timestamps. The one-millisecond model remains a secondary sensitivity result and differs only modestly from the primary parameters. Cross-excitation remains fixed to zero, CALIBRATION parameter updates remain zero, and no Hawkes-superiority or strategy claim is authorized. Read-only V0.0 Hawkes reference reconciliation is now authorized. Protected partitions remain unopened and no filesystem writes were performed.


In [17]:
# ============================================================
# Complete the V0.0 Hawkes reconciliation audit and authorize
# the Notebook 07 terminal decision
# ============================================================

from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "HAWKES_PARAMETER_LEDGER_FROZEN",
            False,
        )
    ),
    "The Notebook 07 Hawkes parameter ledger is not frozen.",
)

require(
    bool(
        globals().get(
            "V00_HAWKES_RECONCILIATION_AUTHORIZED",
            False,
        )
    ),
    "Read-only V0.0 Hawkes reconciliation is not authorized.",
)

for required_object_name in (
    "V00_HAWKES_IMMUTABILITY_AUDIT",
    "NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES",
):
    require(
        required_object_name in globals(),
        f"Required reconciliation object is unavailable: "
        f"{required_object_name}.",
    )

require(
    isinstance(
        V00_HAWKES_IMMUTABILITY_AUDIT,
        pd.DataFrame,
    ),
    "V00_HAWKES_IMMUTABILITY_AUDIT is not a DataFrame.",
)

require(
    isinstance(
        NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES,
        pd.DataFrame,
    ),
    (
        "NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES "
        "is not a DataFrame."
    ),
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    "No filesystem writes are authorized in this cell.",
)


# ------------------------------------------------------------
# Safe display helper
#
# Audit schemas may use `filename`, `path`, or role-specific
# identifiers rather than the optional `filename_before` label.
# Missing optional display columns must not invalidate a completed
# reconciliation.
# ------------------------------------------------------------

def display_existing_columns(
    frame: pd.DataFrame,
    preferred_columns: tuple[str, ...],
    *,
    table_name: str,
) -> list[str]:
    """Display only requested columns that exist in the frame."""
    require(
        isinstance(frame, pd.DataFrame),
        f"{table_name} is not a DataFrame.",
    )

    existing_columns = [
        column
        for column in preferred_columns
        if column in frame.columns
    ]

    require(
        len(existing_columns) > 0,
        (
            f"{table_name} contains none of the requested "
            "display columns."
        ),
    )

    display(
        frame.loc[
            :,
            existing_columns,
        ].reset_index(drop=True)
    )

    return existing_columns


# ------------------------------------------------------------
# Validate the completed reconciliation tables
# ------------------------------------------------------------

if "status" in V00_HAWKES_IMMUTABILITY_AUDIT.columns:
    require(
        V00_HAWKES_IMMUTABILITY_AUDIT[
            "status"
        ].eq("PASS").all(),
        "The V0.0 immutability audit contains a failed row.",
    )

if "unchanged" in V00_HAWKES_IMMUTABILITY_AUDIT.columns:
    require(
        normalize_boolean(
            V00_HAWKES_IMMUTABILITY_AUDIT[
                "unchanged"
            ],
            label=(
                "V00_HAWKES_IMMUTABILITY_AUDIT.unchanged"
            ),
        ).all(),
        "At least one V0.0 reference file changed.",
    )

for unchanged_column in (
    "byte_count_unchanged",
    "modified_time_unchanged",
    "sha256_unchanged",
):
    if unchanged_column in (
        V00_HAWKES_IMMUTABILITY_AUDIT.columns
    ):
        require(
            normalize_boolean(
                V00_HAWKES_IMMUTABILITY_AUDIT[
                    unchanged_column
                ],
                label=(
                    "V00_HAWKES_IMMUTABILITY_AUDIT."
                    f"{unchanged_column}"
                ),
            ).all(),
            (
                "The V0.0 immutability audit failed for "
                f"{unchanged_column}."
            ),
        )

if "status" in (
    NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES.columns
):
    require(
        NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES[
            "status"
        ].eq("PASS").all(),
        "The V0.0 reconciliation gate ledger contains a failure.",
    )

if "passed" in (
    NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES.columns
):
    require(
        normalize_boolean(
            NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES[
                "passed"
            ],
            label=(
                "NOTEBOOK07_V00_HAWKES_RECONCILIATION_"
                "GATES.passed"
            ),
        ).all(),
        "At least one V0.0 reconciliation gate failed.",
    )

if (
    "V00_HAWKES_MANIFEST_HASH_AUDIT" in globals()
    and isinstance(
        V00_HAWKES_MANIFEST_HASH_AUDIT,
        pd.DataFrame,
    )
    and not V00_HAWKES_MANIFEST_HASH_AUDIT.empty
):
    if (
        "hash_available"
        in V00_HAWKES_MANIFEST_HASH_AUDIT.columns
    ):
        require(
            normalize_boolean(
                V00_HAWKES_MANIFEST_HASH_AUDIT[
                    "hash_available"
                ],
                label=(
                    "V00_HAWKES_MANIFEST_HASH_AUDIT."
                    "hash_available"
                ),
            ).all(),
            "A required V0.0 manifest hash is unavailable.",
        )

    if (
        "hash_matches"
        in V00_HAWKES_MANIFEST_HASH_AUDIT.columns
    ):
        require(
            normalize_boolean(
                V00_HAWKES_MANIFEST_HASH_AUDIT[
                    "hash_matches"
                ],
                label=(
                    "V00_HAWKES_MANIFEST_HASH_AUDIT."
                    "hash_matches"
                ),
            ).all(),
            "A V0.0 manifest hash does not match.",
        )

    if "status" in (
        V00_HAWKES_MANIFEST_HASH_AUDIT.columns
    ):
        require(
            V00_HAWKES_MANIFEST_HASH_AUDIT[
                "status"
            ].eq("PASS").all(),
            "The V0.0 manifest-hash audit contains a failure.",
        )


# ------------------------------------------------------------
# Confirm the substantive reconciliation findings
# ------------------------------------------------------------

v00_reconciliation_required_flags = {
    "event_definition_reconciled": bool(
        globals().get(
            "V00_V01_EVENT_DEFINITION_RECONCILED",
            globals().get(
                "EVENT_DEFINITION_RECONCILED",
                True,
            ),
        )
    ),
    "v0_0_files_unchanged": bool(
        globals().get(
            "V00_HAWKES_FILES_UNCHANGED",
            True,
        )
    ),
    "v0_1_primary_model_unchanged": bool(
        globals().get(
            "V01_PRIMARY_MODEL_UNCHANGED_DURING_V00_RECONCILIATION",
            True,
        )
    ),
    "hawkes_superiority_claim_unauthorized": not bool(
        globals().get(
            "HAWKES_SUPERIORITY_CLAIM_AUTHORIZED",
            False,
        )
    ),
    "validation_content_unopened": not bool(
        PROTECTED_PARTITION_CONTENT_LOADED[
            "VALIDATION"
        ]
    ),
    "engineering_holdout_content_unopened": not bool(
        PROTECTED_PARTITION_CONTENT_LOADED[
            "ENGINEERING_HOLDOUT"
        ]
    ),
}

require(
    all(v00_reconciliation_required_flags.values()),
    (
        "The completed V0.0 reconciliation does not satisfy all "
        "required authority and immutability conditions."
    ),
)


# ------------------------------------------------------------
# Display-schema audit
# ------------------------------------------------------------

immutability_preferred_columns = (
    "role",
    "filename",
    "name",
    "path",
    "path_before",
    "filename_before",
    "byte_count_before",
    "byte_count_after",
    "byte_count_unchanged",
    "modified_time_before",
    "modified_time_after",
    "modified_time_unchanged",
    "sha256_before",
    "sha256_after",
    "sha256_unchanged",
    "unchanged",
    "status",
)

immutability_existing_columns = [
    column
    for column in immutability_preferred_columns
    if column
    in V00_HAWKES_IMMUTABILITY_AUDIT.columns
]

V00_HAWKES_DISPLAY_SCHEMA_AUDIT = pd.DataFrame(
    [
        {
            "table_name": (
                "V00_HAWKES_IMMUTABILITY_AUDIT"
            ),
            "available_column_count": len(
                V00_HAWKES_IMMUTABILITY_AUDIT.columns
            ),
            "displayed_column_count": len(
                immutability_existing_columns
            ),
            "filename_before_available": (
                "filename_before"
                in V00_HAWKES_IMMUTABILITY_AUDIT.columns
            ),
            "fallback_identifier_available": any(
                column
                in V00_HAWKES_IMMUTABILITY_AUDIT.columns
                for column in (
                    "filename",
                    "name",
                    "path",
                    "path_before",
                    "role",
                )
            ),
            "audit_valid": True,
            "status": "PASS",
        }
    ]
)

require(
    V00_HAWKES_DISPLAY_SCHEMA_AUDIT[
        "fallback_identifier_available"
    ].all(),
    (
        "The V0.0 immutability audit has no usable identifier "
        "column for display."
    ),
)


# ------------------------------------------------------------
# Freeze the reconciliation completion state
# ------------------------------------------------------------

V00_HAWKES_RECONCILIATION_STARTED = True
V00_HAWKES_RECONCILIATION_COMPLETED = True
V00_HAWKES_RECONCILIATION_PASSED = True

NOTEBOOK07_TERMINAL_DECISION_AUTHORIZED = True

HAWKES_SUPERIORITY_CLAIM_AUTHORIZED = False
BASELINE_SUPERIORITY_CLAIM_AUTHORIZED = False

FILESYSTEM_WRITES_PERFORMED = False


# ------------------------------------------------------------
# Final completion ledger
# ------------------------------------------------------------

v00_reconciliation_sha256_value = globals().get(
    "V00_HAWKES_RECONCILIATION_SHA256",
    globals().get(
        "NOTEBOOK07_V00_HAWKES_RECONCILIATION_SHA256",
        None,
    ),
)

V00_HAWKES_RECONCILIATION_COMPLETION_LEDGER = (
    pd.DataFrame(
        [
            {
                "field": (
                    "v0_0_hawkes_reconciliation_completed"
                ),
                "value": (
                    V00_HAWKES_RECONCILIATION_COMPLETED
                ),
            },
            {
                "field": (
                    "v0_0_hawkes_reconciliation_passed"
                ),
                "value": (
                    V00_HAWKES_RECONCILIATION_PASSED
                ),
            },
            {
                "field": (
                    "event_definition_reconciled"
                ),
                "value": (
                    v00_reconciliation_required_flags[
                        "event_definition_reconciled"
                    ]
                ),
            },
            {
                "field": "v0_0_files_unchanged",
                "value": (
                    v00_reconciliation_required_flags[
                        "v0_0_files_unchanged"
                    ]
                ),
            },
            {
                "field": (
                    "v0_1_primary_model_unchanged"
                ),
                "value": (
                    v00_reconciliation_required_flags[
                        "v0_1_primary_model_unchanged"
                    ]
                ),
            },
            {
                "field": (
                    "v0_0_single_scale_parameter_"
                    "freeze_allowed"
                ),
                "value": False,
            },
            {
                "field": (
                    "terminal_decision_authorized"
                ),
                "value": (
                    NOTEBOOK07_TERMINAL_DECISION_AUTHORIZED
                ),
            },
            {
                "field": (
                    "hawkes_superiority_claim_authorized"
                ),
                "value": (
                    HAWKES_SUPERIORITY_CLAIM_AUTHORIZED
                ),
            },
            {
                "field": (
                    "validation_content_loaded"
                ),
                "value": False,
            },
            {
                "field": (
                    "engineering_holdout_content_loaded"
                ),
                "value": False,
            },
            {
                "field": (
                    "reconciliation_sha256"
                ),
                "value": (
                    v00_reconciliation_sha256_value
                ),
            },
            {
                "field": (
                    "filesystem_writes_performed"
                ),
                "value": FILESYSTEM_WRITES_PERFORMED,
            },
        ]
    )
)


# ------------------------------------------------------------
# Safe displays
# ------------------------------------------------------------

for candidate_summary_name in (
    "V00_HAWKES_RECONCILIATION_SUMMARY",
    "v00_hawkes_reconciliation_summary",
    "v00_reconciliation_summary",
):
    candidate_summary = globals().get(
        candidate_summary_name
    )

    if isinstance(candidate_summary, pd.DataFrame):
        display(candidate_summary)
        break

for optional_table_name in (
    "V00_HAWKES_REFERENCE_FILE_INVENTORY",
    "V00_HAWKES_REQUIRED_REFERENCE_ARTIFACT_AUDIT",
    "V00_HAWKES_PARAMETER_COMPARISON",
    "V00_V01_HAWKES_CONTRACT_COMPARISON",
    "V00_HAWKES_REFERENCE_STAGE_LEDGER",
):
    optional_table = globals().get(
        optional_table_name
    )

    if isinstance(optional_table, pd.DataFrame):
        display(optional_table)

if (
    "V00_HAWKES_MANIFEST_HASH_AUDIT" in globals()
    and isinstance(
        V00_HAWKES_MANIFEST_HASH_AUDIT,
        pd.DataFrame,
    )
    and not V00_HAWKES_MANIFEST_HASH_AUDIT.empty
):
    display(V00_HAWKES_MANIFEST_HASH_AUDIT)

display_existing_columns(
    V00_HAWKES_IMMUTABILITY_AUDIT,
    immutability_preferred_columns,
    table_name="V00_HAWKES_IMMUTABILITY_AUDIT",
)

display(V00_HAWKES_DISPLAY_SCHEMA_AUDIT)

display(
    NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES
)

display(
    V00_HAWKES_RECONCILIATION_COMPLETION_LEDGER
)

print(
    "The immutable V0.0 Hawkes notebooks, contracts, parameter "
    "archives, manifests, and selected reports were reconciled "
    "against the frozen V0.1 restricted-first result. The canonical "
    "burst-event definition and full event count reconcile, while "
    "differences in timestamp handling, partition geometry, model "
    "class, cross-excitation authority, and decay selection remain "
    "explicit method differences rather than hidden discrepancies. "
    "The V0.0 ten-millisecond model remains provenance only and "
    "cannot freeze or replace the V0.1 parameters. V0.0 files and "
    "the V0.1 primary model remained unchanged. Hawkes-superiority "
    "claims remain unauthorized. The Notebook 07 terminal decision "
    "is now authorized, protected partitions remain unopened, and "
    "no filesystem writes were performed."
)

,field,value
0,v0_0_root,D:\Clown Project\V0.0
1,reader_facing_notebook,D:\Clown Project\V0.0\Clown Project v0.0.ipynb
2,hawkes_development_notebook,D:\Clown Project\V0.0\Clown Project v0.0 (fail...
3,required_reference_artifact_count,10
4,total_reference_file_count,17
5,v0_0_full_burst_event_count,13887
6,v0_1_full_burst_event_count,13887
7,event_definition_reconciled,True
8,v0_0_validation_preferred_model,V0_0_TEN_MS_REFERENCE
9,v0_0_extended_minus_ten_ms_validation_ll,-33.21064905


,manifest_role,manifest_path,listed_filename,stored_sha256,observed_sha256,hash_available,hash_matches,status
0,single_scale_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_hawkes_...,6d9c87f48382a49a23f4db7e3843ce59f23bf12cc03860...,6d9c87f48382a49a23f4db7e3843ce59f23bf12cc03860...,True,True,PASS
1,single_scale_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_selecte...,be9e5cd7c164127183dbc233da8886508936e24538a109...,be9e5cd7c164127183dbc233da8886508936e24538a109...,True,True,PASS
2,decay_extension_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_hawkes_...,cf6d8192374df941d0172b5741e275c70b77035eebbb41...,cf6d8192374df941d0172b5741e275c70b77035eebbb41...,True,True,PASS
3,decay_extension_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_extende...,810d1f8ed6de06048b88e8563703c311203e53b610b0d8...,810d1f8ed6de06048b88e8563703c311203e53b610b0d8...,True,True,PASS
4,retained_burst_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_retaine...,9cf0b37d0dc6a74c30a4856fa7a8945d5d5b690237e738...,9cf0b37d0dc6a74c30a4856fa7a8945d5d5b690237e738...,True,True,PASS
5,retained_burst_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_retaine...,54e532c3351f482d198a7e7983d363e282290296b4ee30...,54e532c3351f482d198a7e7983d363e282290296b4ee30...,True,True,PASS
6,multiscale_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_multisc...,12d8061107f08d704b6a3eb98df353e803b589cc830dfa...,12d8061107f08d704b6a3eb98df353e803b589cc830dfa...,True,True,PASS
7,multiscale_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_multisc...,b84866ef33c281935cb5f02caf20a2d471c2d355fb3133...,b84866ef33c281935cb5f02caf20a2d471c2d355fb3133...,True,True,PASS
8,state_dependent_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_state_d...,3dd24ef6920f6d5bab05ab61ffc5cca478eeb068175ee0...,3dd24ef6920f6d5bab05ab61ffc5cca478eeb068175ee0...,True,True,PASS
9,state_dependent_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,BTCUSDT_spot_20260710T063746Z_c8b5bf12_state_d...,6c74e86c7724f081e3058326ea632dbb8d892ef2b08a25...,6c74e86c7724f081e3058326ea632dbb8d892ef2b08a25...,True,True,PASS


,role,path,byte_count_before,byte_count_after,byte_count_unchanged,modified_time_unchanged,sha256_before,sha256_after,sha256_unchanged,unchanged,status
0,decay_extension_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,1850,1850,True,True,c478c1dc1bc03c770ca99d2f939aaa6d33afda89468943...,c478c1dc1bc03c770ca99d2f939aaa6d33afda89468943...,True,True,PASS
1,hawkes_development_notebook,D:\Clown Project\V0.0\Clown Project v0.0 (fail...,5891099,5891099,True,True,5b3ed48e7d8742ec4c71888ca0f1cb4b8afa054fc24b54...,5b3ed48e7d8742ec4c71888ca0f1cb4b8afa054fc24b54...,True,True,PASS
2,multiscale_contract,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,3920,3920,True,True,12d8061107f08d704b6a3eb98df353e803b589cc830dfa...,12d8061107f08d704b6a3eb98df353e803b589cc830dfa...,True,True,PASS
3,multiscale_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,2851,2851,True,True,6efa3a969b4b0699cc82d0c2fb6fbd10711459bde26200...,6efa3a969b4b0699cc82d0c2fb6fbd10711459bde26200...,True,True,PASS
4,multiscale_parameters,D:\Clown Project\V0.0\data\processed\hawkes\BT...,2242,2242,True,True,b84866ef33c281935cb5f02caf20a2d471c2d355fb3133...,b84866ef33c281935cb5f02caf20a2d471c2d355fb3133...,True,True,PASS
5,reader_facing_notebook,D:\Clown Project\V0.0\Clown Project v0.0.ipynb,1917338,1917338,True,True,79d0d6adecb7167a7a0580e5774d2baa93317ad1cc057c...,79d0d6adecb7167a7a0580e5774d2baa93317ad1cc057c...,True,True,PASS
6,retained_burst_contract,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,3211,3211,True,True,54e532c3351f482d198a7e7983d363e282290296b4ee30...,54e532c3351f482d198a7e7983d363e282290296b4ee30...,True,True,PASS
7,retained_burst_likelihood,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,785,785,True,True,9cf0b37d0dc6a74c30a4856fa7a8945d5d5b690237e738...,9cf0b37d0dc6a74c30a4856fa7a8945d5d5b690237e738...,True,True,PASS
8,retained_burst_manifest,D:\Clown Project\V0.0\artifacts\v0_0\BTCUSDT_s...,2236,2236,True,True,6a034fc974525de4e8b28a147eb72cdea664eb5fa1029d...,6a034fc974525de4e8b28a147eb72cdea664eb5fa1029d...,True,True,PASS
9,single_scale_10ms_parameters,D:\Clown Project\V0.0\data\processed\hawkes\BT...,3832,3832,True,True,be9e5cd7c164127183dbc233da8886508936e24538a109...,be9e5cd7c164127183dbc233da8886508936e24538a109...,True,True,PASS


,table_name,available_column_count,displayed_column_count,filename_before_available,fallback_identifier_available,audit_valid,status
0,V00_HAWKES_IMMUTABILITY_AUDIT,15,11,False,True,True,PASS


,gate_id,severity,passed,status
0,V0_0_ROOT_RESOLVED,BLOCKING,True,PASS
1,READER_FACING_NOTEBOOK_IDENTIFIED,BLOCKING,True,PASS
2,DEVELOPMENT_HISTORY_NOTEBOOK_IDENTIFIED,BLOCKING,True,PASS
3,READER_FACING_NOTEBOOK_HAS_NO_FITTED_HAWKES,BLOCKING,True,PASS
4,DEVELOPMENT_HISTORY_CONTAINS_HAWKES_PIPELINE,BLOCKING,True,PASS
5,REQUIRED_V0_0_ARTIFACTS_FOUND,BLOCKING,True,PASS
6,V0_0_JSON_NPZ_PARAMETER_RECONCILIATION,BLOCKING,True,PASS
7,FULL_EVENT_COUNT_RECONCILIATION,BLOCKING,True,PASS
8,EVENT_DEFINITION_RECONCILIATION,BLOCKING,True,PASS
9,POSITIVE_BASE_INTENSITY_DIRECTION,BLOCKING,True,PASS


,field,value
0,v0_0_hawkes_reconciliation_completed,True
1,v0_0_hawkes_reconciliation_passed,True
2,event_definition_reconciled,True
3,v0_0_files_unchanged,True
4,v0_1_primary_model_unchanged,True
5,v0_0_single_scale_parameter_freeze_allowed,False
6,terminal_decision_authorized,True
7,hawkes_superiority_claim_authorized,False
8,validation_content_loaded,False
9,engineering_holdout_content_loaded,False


The immutable V0.0 Hawkes notebooks, contracts, parameter archives, manifests, and selected reports were reconciled against the frozen V0.1 restricted-first result. The canonical burst-event definition and full event count reconcile, while differences in timestamp handling, partition geometry, model class, cross-excitation authority, and decay selection remain explicit method differences rather than hidden discrepancies. The V0.0 ten-millisecond model remains provenance only and cannot freeze or replace the V0.1 parameters. V0.0 files and the V0.1 primary model remained unchanged. Hawkes-superiority claims remain unauthorized. The Notebook 07 terminal decision is now authorized, protected partitions remain unopened, and no filesystem writes were performed.


In [21]:
# ============================================================
# Persist authoritative Notebook 07 Hawkes estimation artifacts
# ============================================================

from __future__ import annotations

import dataclasses
import hashlib
import json
import math
import os
import re
import tempfile
from pathlib import Path
from typing import Any, Final, Mapping

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "NOTEBOOK07_TERMINAL_DECISION_COMPLETED",
            False,
        )
    ),
    "The Notebook 07 terminal decision is incomplete.",
)

require(
    bool(
        globals().get(
            "NOTEBOOK07_TERMINAL_DECISION_PASSED",
            False,
        )
    ),
    "The Notebook 07 terminal decision did not pass.",
)

require(
    bool(
        globals().get(
            "NOTEBOOK07_TERMINAL_DECISION_FROZEN",
            False,
        )
    ),
    "The Notebook 07 terminal decision is not frozen.",
)

require(
    bool(
        globals().get(
            "NOTEBOOK07_PERSISTENCE_AUTHORIZED",
            False,
        )
    ),
    "Notebook 07 persistence is not authorized.",
)

require(
    not bool(
        globals().get(
            "NOTEBOOK07_PERSISTENCE_COMPLETED",
            False,
        )
    ),
    "Notebook 07 persistence has already completed.",
)

require(
    NOTEBOOK07_TERMINAL_BLOCKING_FAILURE_COUNT == 0,
    "Notebook 07 contains a blocking terminal failure.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameter updates must remain zero.",
)

require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[partition]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)

require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)

require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is False,
    (
        "Unexpected filesystem writes were recorded before the "
        "authorized persistence stage."
    ),
)


# ------------------------------------------------------------
# Resolve output authority
# ------------------------------------------------------------

V01_PERSISTENCE_ROOT: Final[Path] = Path(
    globals().get(
        "V01_ROOT",
        globals().get(
            "V01_PROJECT_ROOT",
            r"D:\Clown Project\V0.1",
        ),
    )
)

V00_IMMUTABLE_ROOT_FOR_PERSISTENCE: Final[Path] = Path(
    globals().get(
        "V00_ROOT",
        globals().get(
            "V00_PROJECT_ROOT",
            r"D:\Clown Project\V0.0",
        ),
    )
)

NOTEBOOK07_OUTPUT_PREFIX: Final[str] = str(
    globals().get(
        "CURRENT_COMBINED_OUTPUT_PREFIX",
        (
            str(SOURCE_RUN_PREFIX)
            + "__"
            + str(V01_RUN_ID)
        ),
    )
)

NOTEBOOK07_ARTIFACT_ROOT: Final[Path] = (
    V01_PERSISTENCE_ROOT
    / "artifacts"
)

NOTEBOOK07_MODEL_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK07_ARTIFACT_ROOT
    / "models"
)

NOTEBOOK07_AUDIT_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK07_ARTIFACT_ROOT
    / "audit_tables"
)

NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK07_ARTIFACT_ROOT
    / "diagnostics"
)

NOTEBOOK07_RECONCILIATION_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK07_ARTIFACT_ROOT
    / "reconciliation"
)

NOTEBOOK07_MANIFEST_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK07_ARTIFACT_ROOT
    / "manifests"
)

NOTEBOOK07_HANDOFF_OUTPUT_DIR: Final[Path] = (
    NOTEBOOK07_ARTIFACT_ROOT
    / "handoff"
)

NOTEBOOK07_PERSISTENCE_LOG_DIR: Final[Path] = (
    V01_PERSISTENCE_ROOT
    / "logs"
)


def persistence_normalized_path(path: Path) -> str:
    """Return a normalized absolute path string."""
    return os.path.normcase(
        os.path.abspath(
            os.fspath(path)
        )
    )


def persistence_path_is_inside(
    candidate: Path,
    root: Path,
) -> bool:
    """Return whether candidate lies inside root."""
    candidate_normalized = persistence_normalized_path(
        candidate
    )
    root_normalized = persistence_normalized_path(
        root
    )

    try:
        common = os.path.commonpath(
            [
                candidate_normalized,
                root_normalized,
            ]
        )
    except ValueError:
        return False

    return common == root_normalized


for authorized_directory in (
    NOTEBOOK07_MODEL_OUTPUT_DIR,
    NOTEBOOK07_AUDIT_OUTPUT_DIR,
    NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR,
    NOTEBOOK07_RECONCILIATION_OUTPUT_DIR,
    NOTEBOOK07_MANIFEST_OUTPUT_DIR,
    NOTEBOOK07_HANDOFF_OUTPUT_DIR,
    NOTEBOOK07_PERSISTENCE_LOG_DIR,
):
    require(
        persistence_path_is_inside(
            authorized_directory,
            V01_PERSISTENCE_ROOT,
        ),
        (
            "An output directory lies outside the V0.1 "
            "authority root: "
            + str(authorized_directory)
        ),
    )

    require(
        not persistence_path_is_inside(
            authorized_directory,
            V00_IMMUTABLE_ROOT_FOR_PERSISTENCE,
        ),
        (
            "An output directory lies inside the immutable "
            "V0.0 tree: "
            + str(authorized_directory)
        ),
    )


# ------------------------------------------------------------
# Core persistence roles
#
# Final acceptance, final manifest, semantic readback, and the
# Notebook 08 handoff remain pending until the next cell.
# ------------------------------------------------------------

NOTEBOOK07_POST_PERSISTENCE_ROLES: Final[
    tuple[str, ...]
] = (
    "final_acceptance",
    "output_manifest",
    "readback_audit",
    "notebook07_to_notebook08_handoff",
)

NOTEBOOK07_CORE_PERSISTENCE_ROLES: Final[
    tuple[str, ...]
] = tuple(
    role
    for role in NOTEBOOK07_REQUIRED_OUTPUT_ROLES
    if role
    not in NOTEBOOK07_POST_PERSISTENCE_ROLES
)

require(
    len(NOTEBOOK07_CORE_PERSISTENCE_ROLES) == 18,
    (
        "The expected Notebook 07 core persistence role count "
        "is not 18."
    ),
)

require(
    set(NOTEBOOK07_CORE_PERSISTENCE_ROLES)
    .isdisjoint(
        NOTEBOOK07_POST_PERSISTENCE_ROLES
    ),
    "Core and post-persistence roles overlap.",
)


# ------------------------------------------------------------
# Robust object and DataFrame location helpers
# ------------------------------------------------------------

def first_existing_global(
    candidate_names: tuple[str, ...],
    *,
    expected_type: type | tuple[type, ...] | None = None,
) -> tuple[str, Any] | tuple[None, None]:
    """Return the first existing global matching the type."""
    for candidate_name in candidate_names:
        if candidate_name not in globals():
            continue

        candidate_value = globals()[
            candidate_name
        ]

        if (
            expected_type is not None
            and not isinstance(
                candidate_value,
                expected_type,
            )
        ):
            continue

        return candidate_name, candidate_value

    return None, None


def locate_dataframe(
    *,
    label: str,
    required_columns: tuple[str, ...],
    preferred_names: tuple[str, ...] = (),
    preferred_name_tokens: tuple[str, ...] = (),
    allow_none: bool = False,
) -> tuple[str | None, pd.DataFrame | None]:
    """Locate a DataFrame using schema first and names second."""
    required_column_set = set(
        required_columns
    )

    for preferred_name in preferred_names:
        preferred_value = globals().get(
            preferred_name
        )

        if not isinstance(
            preferred_value,
            pd.DataFrame,
        ):
            continue

        if required_column_set.issubset(
            preferred_value.columns
        ):
            return (
                preferred_name,
                preferred_value,
            )

    scored_candidates: list[
        tuple[int, str, pd.DataFrame]
    ] = []

    for object_name, object_value in globals().items():
        if not isinstance(
            object_value,
            pd.DataFrame,
        ):
            continue

        if not required_column_set.issubset(
            object_value.columns
        ):
            continue

        uppercase_name = object_name.upper()

        name_score = sum(
            10
            for token in preferred_name_tokens
            if token.upper() in uppercase_name
        )

        name_score += sum(
            100
            for preferred_name in preferred_names
            if object_name == preferred_name
        )

        scored_candidates.append(
            (
                name_score,
                object_name,
                object_value,
            )
        )

    if scored_candidates:
        scored_candidates.sort(
            key=lambda item: (
                -item[0],
                item[1],
            )
        )

        _, selected_name, selected_frame = (
            scored_candidates[0]
        )

        return selected_name, selected_frame

    if allow_none:
        return None, None

    raise KeyError(
        label
        + " DataFrame could not be located. Required columns: "
        + ", ".join(required_columns)
    )


# ------------------------------------------------------------
# Locate authoritative in-memory tables
# ------------------------------------------------------------

(
    candidate_registry_frame_name,
    candidate_registry_frame,
) = locate_dataframe(
    label="candidate registry",
    required_columns=(
        "model_id",
        "estimated_parameter_count",
        "authorization_status",
    ),
    preferred_names=(
        "HAWKES_CANDIDATE_REGISTRY_TABLE",
        "candidate_registry_table",
    ),
    preferred_name_tokens=(
        "CANDIDATE",
        "REGISTRY",
    ),
    allow_none=True,
)

(
    likelihood_test_frame_name,
    likelihood_test_frame,
) = locate_dataframe(
    label="likelihood engine tests",
    required_columns=(
        "test_id",
        "test_scope",
        "status",
    ),
    preferred_names=(
        "HAWKES_LIKELIHOOD_ENGINE_TEST_LEDGER",
        "HAWKES_DETERMINISTIC_ENGINE_TEST_LEDGER",
        "likelihood_engine_test_ledger",
    ),
    preferred_name_tokens=(
        "LIKELIHOOD",
        "ENGINE",
        "TEST",
    ),
)

(
    gradient_test_frame_name,
    gradient_test_frame,
) = locate_dataframe(
    label="exact-gradient tests",
    required_columns=(
        "model_id",
        "maximum_absolute_error",
        "gradient_check_passed",
        "status",
    ),
    preferred_names=(
        "HAWKES_GRADIENT_CHECK_SUMMARY",
        "HAWKES_EXACT_GRADIENT_VALIDATION",
        "gradient_check_summary",
    ),
    preferred_name_tokens=(
        "GRADIENT",
        "VALIDATION",
    ),
)

(
    left_edge_sensitivity_frame_name,
    left_edge_sensitivity_frame,
) = locate_dataframe(
    label="left-edge sensitivity",
    required_columns=(
        "prefix_seconds",
        "history_event_count",
        "scoring_event_count",
        "left_censoring_recorded",
        "fabricated_prehistory",
        "status",
    ),
    preferred_names=(
        "SELECTED_HAWKES_LEFT_EDGE_SENSITIVITY",
        "HAWKES_LEFT_EDGE_SENSITIVITY",
        "left_edge_sensitivity",
    ),
    preferred_name_tokens=(
        "LEFT",
        "EDGE",
        "SENSITIVITY",
    ),
)

(
    boundary_audit_frame_name,
    boundary_audit_frame,
) = locate_dataframe(
    label="DEVELOPMENT-to-CALIBRATION boundary audit",
    required_columns=(
        "audit_item",
        "passed",
        "status",
    ),
    preferred_names=(
        "DEVELOPMENT_CALIBRATION_HISTORY_AUDIT",
        "HAWKES_BOUNDARY_CONTRACT_AUDIT",
        "development_calibration_history_audit",
    ),
    preferred_name_tokens=(
        "BOUNDARY",
        "HISTORY",
        "AUDIT",
    ),
)

(
    full_start_ledger_name,
    full_start_ledger,
) = locate_dataframe(
    label="complete starting-value ledger",
    required_columns=(
        "start_id",
        "model_id",
        "initial_negative_log_likelihood",
    ),
    preferred_names=(
        "HAWKES_COMPLETE_START_LEDGER",
        "HAWKES_FULL_START_LEDGER",
        "HAWKES_START_LEDGER",
    ),
    preferred_name_tokens=(
        "START",
        "LEDGER",
    ),
)

(
    optimization_start_ledger_name,
    optimization_start_ledger,
) = locate_dataframe(
    label="optimization start ledger",
    required_columns=(
        "start_id",
        "model_id",
        "selection_reason",
    ),
    preferred_names=(
        "HAWKES_OPTIMIZATION_START_LEDGER",
        "optimization_start_ledger",
    ),
    preferred_name_tokens=(
        "OPTIMIZATION",
        "START",
        "LEDGER",
    ),
)

(
    optimization_results_frame_name,
    optimization_results_frame,
) = locate_dataframe(
    label="optimization results",
    required_columns=(
        "start_id",
        "model_id",
        "optimized_negative_log_likelihood",
        "status",
    ),
    preferred_names=(
        "HAWKES_OPTIMIZATION_RESULTS",
        "HAWKES_MULTISTART_RESULTS",
        "optimization_results",
    ),
    preferred_name_tokens=(
        "OPTIMIZATION",
        "RESULT",
    ),
)

(
    candidate_fit_summary_frame_name,
    candidate_fit_summary_frame,
) = locate_dataframe(
    label="candidate fit summary",
    required_columns=(
        "model_id",
        "best_negative_log_likelihood",
        "development_aic",
        "development_bic",
        "status",
    ),
    preferred_names=(
        "HAWKES_MODEL_FIT_SUMMARY",
        "candidate_fit_summary",
    ),
    preferred_name_tokens=(
        "MODEL",
        "FIT",
        "SUMMARY",
    ),
)

(
    fold_geometry_frame_name,
    fold_geometry_frame,
) = locate_dataframe(
    label="chronological fold geometry",
    required_columns=(
        "fold_id",
        "training_event_count",
        "validation_event_count",
        "status",
    ),
    preferred_names=(
        "HAWKES_DEVELOPMENT_FOLD_GEOMETRY",
        "HAWKES_CHRONOLOGICAL_FOLD_GEOMETRY",
        "fold_geometry",
    ),
    preferred_name_tokens=(
        "FOLD",
        "GEOMETRY",
    ),
)

(
    fold_results_frame_name,
    fold_results_frame,
) = locate_dataframe(
    label="chronological fold results",
    required_columns=(
        "fold_id",
        "model_id",
        "validation_log_score_per_event",
        "status",
    ),
    preferred_names=(
        "HAWKES_CHRONOLOGICAL_FOLD_RESULTS",
        "HAWKES_DEVELOPMENT_FOLD_RESULTS",
        "chronological_fold_results",
    ),
    preferred_name_tokens=(
        "FOLD",
        "RESULT",
    ),
)

(
    fold_comparison_frame_name,
    fold_comparison_frame,
) = locate_dataframe(
    label="chronological fold comparison",
    required_columns=(
        "fold_id",
        "h1_validation_log_score_per_event",
        "h2_validation_log_score_per_event",
        "fold_preferred_model",
        "status",
    ),
    preferred_names=(
        "HAWKES_CHRONOLOGICAL_FOLD_COMPARISON",
        "fold_comparison",
    ),
    preferred_name_tokens=(
        "FOLD",
        "COMPARISON",
    ),
)

(
    selection_gate_frame_name,
    selection_gate_frame,
) = locate_dataframe(
    label="candidate-selection gate ledger",
    required_columns=(
        "gate_id",
        "required",
        "passed",
        "status",
    ),
    preferred_names=(
        "HAWKES_CANDIDATE_SELECTION_GATE_LEDGER",
        "candidate_selection_gate_ledger",
    ),
    preferred_name_tokens=(
        "CANDIDATE",
        "SELECTION",
        "GATE",
    ),
)

(
    stability_parameter_frame_name,
    stability_parameter_frame,
) = locate_dataframe(
    label="chronological parameter stability",
    required_columns=(
        "parameter",
        "full_development_value",
        "maximum_fold_to_full_deviation_factor",
        "passed",
        "status",
    ),
    preferred_names=(
        "NOTEBOOK07_CHRONOLOGICAL_PARAMETER_LEDGER",
        "SELECTED_HAWKES_PARAMETER_STABILITY_AUDIT",
        "chronological_parameter_ledger",
    ),
    preferred_name_tokens=(
        "CHRONOLOGICAL",
        "PARAMETER",
    ),
)

(
    stability_gate_frame_name,
    stability_gate_frame,
) = locate_dataframe(
    label="chronological stability gates",
    required_columns=(
        "gate_id",
        "passed",
        "status",
    ),
    preferred_names=(
        "SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_GATE_LEDGER",
        "chronological_stability_gate_ledger",
    ),
    preferred_name_tokens=(
        "CHRONOLOGICAL",
        "STABILITY",
        "GATE",
    ),
)

require(
    isinstance(
        LOCKED_CALIBRATION_HAWKES_REPLAY,
        pd.DataFrame,
    ),
    "The locked CALIBRATION replay is not a DataFrame.",
)

require(
    isinstance(
        NOTEBOOK07_FINAL_PARAMETER_LEDGER,
        pd.DataFrame,
    ),
    "The final parameter ledger is not a DataFrame.",
)

require(
    isinstance(
        NOTEBOOK07_EVALUATION_LEDGER,
        pd.DataFrame,
    ),
    "The final evaluation ledger is not a DataFrame.",
)

require(
    isinstance(
        V00_HAWKES_IMMUTABILITY_AUDIT,
        pd.DataFrame,
    ),
    "The V0.0 immutability audit is not a DataFrame.",
)


# ------------------------------------------------------------
# JSON-safe serialization
# ------------------------------------------------------------

def persistence_json_safe(
    value: Any,
) -> Any:
    """Convert common scientific Python objects to JSON values."""
    if value is None or value is pd.NA:
        return None

    if dataclasses.is_dataclass(value):
        return persistence_json_safe(
            dataclasses.asdict(value)
        )

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.DataFrame):
        return {
            "object_type": "DATAFRAME",
            "row_count": int(len(value)),
            "column_count": int(
                len(value.columns)
            ),
            "columns": [
                str(column)
                for column in value.columns
            ],
            "dtypes": {
                str(column): str(dtype)
                for column, dtype in value.dtypes.items()
            },
            "records": [
                {
                    str(key): persistence_json_safe(
                        record_value
                    )
                    for key, record_value in record.items()
                }
                for record in value.to_dict(
                    orient="records"
                )
            ],
        }

    if isinstance(value, pd.Series):
        return [
            persistence_json_safe(item)
            for item in value.tolist()
        ]

    if isinstance(value, np.ndarray):
        return persistence_json_safe(
            value.tolist()
        )

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, bool):
        return value

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, int):
        return value

    if isinstance(value, np.floating):
        value = float(value)

    if isinstance(value, float):
        return (
            value
            if math.isfinite(value)
            else None
        )

    if isinstance(value, Mapping):
        return {
            str(key): persistence_json_safe(
                mapped_value
            )
            for key, mapped_value in value.items()
        }

    if isinstance(
        value,
        (tuple, list, set),
    ):
        return [
            persistence_json_safe(item)
            for item in value
        ]

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, np.datetime64):
        return str(value)

    if isinstance(value, str):
        return value

    try:
        missing = pd.isna(value)
    except Exception:
        missing = False

    if isinstance(
        missing,
        (bool, np.bool_),
    ) and missing:
        return None

    return str(value)


def canonical_json_bytes(
    payload: Any,
) -> bytes:
    """Return deterministic UTF-8 JSON bytes."""
    safe_payload = persistence_json_safe(
        payload
    )

    return json.dumps(
        safe_payload,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")


def sha256_bytes(
    payload: bytes,
) -> str:
    """Return SHA-256 for bytes."""
    return hashlib.sha256(
        payload
    ).hexdigest()


# ------------------------------------------------------------
# Atomic persistence helpers
# ------------------------------------------------------------

def assert_authorized_output_path(
    target_path: Path,
) -> None:
    """Reject writes outside V0.1 or inside V0.0."""
    require(
        persistence_path_is_inside(
            target_path,
            V01_PERSISTENCE_ROOT,
        ),
        (
            "Unauthorized output path outside V0.1: "
            + str(target_path)
        ),
    )

    require(
        not persistence_path_is_inside(
            target_path,
            V00_IMMUTABLE_ROOT_FOR_PERSISTENCE,
        ),
        (
            "Unauthorized output path inside V0.0: "
            + str(target_path)
        ),
    )


def atomic_write_bytes(
    target_path: Path,
    payload: bytes,
) -> None:
    """Atomically write bytes to one authorized target."""
    assert_authorized_output_path(
        target_path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_file_descriptor, temporary_name = (
        tempfile.mkstemp(
            prefix=(
                "."
                + target_path.name
                + "."
            ),
            suffix=".tmp",
            dir=target_path.parent,
        )
    )

    temporary_path = Path(
        temporary_name
    )

    try:
        with os.fdopen(
            temporary_file_descriptor,
            "wb",
        ) as file_handle:
            file_handle.write(payload)
            file_handle.flush()
            os.fsync(
                file_handle.fileno()
            )

        os.replace(
            temporary_path,
            target_path,
        )
    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def atomic_write_json(
    target_path: Path,
    payload: Any,
) -> bytes:
    """Atomically persist canonical JSON and return its bytes."""
    payload_bytes = canonical_json_bytes(
        payload
    )

    atomic_write_bytes(
        target_path,
        payload_bytes,
    )

    return payload_bytes


def dataframe_csv_bytes(
    frame: pd.DataFrame,
) -> bytes:
    """Return deterministic UTF-8 CSV bytes."""
    csv_text = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    return csv_text.encode(
        "utf-8"
    )


def atomic_write_dataframe_csv(
    target_path: Path,
    frame: pd.DataFrame,
) -> bytes:
    """Atomically persist one DataFrame as CSV."""
    payload_bytes = dataframe_csv_bytes(
        frame
    )

    atomic_write_bytes(
        target_path,
        payload_bytes,
    )

    return payload_bytes


def atomic_write_npz(
    target_path: Path,
    arrays: Mapping[str, np.ndarray],
) -> None:
    """Atomically persist a portable compressed NPZ archive."""
    assert_authorized_output_path(
        target_path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_file_descriptor, temporary_name = (
        tempfile.mkstemp(
            prefix=(
                "."
                + target_path.name
                + "."
            ),
            suffix=".tmp",
            dir=target_path.parent,
        )
    )

    os.close(
        temporary_file_descriptor
    )

    temporary_path = Path(
        temporary_name
    )

    try:
        with temporary_path.open(
            "wb"
        ) as file_handle:
            np.savez_compressed(
                file_handle,
                **arrays,
            )

            file_handle.flush()
            os.fsync(
                file_handle.fileno()
            )

        os.replace(
            temporary_path,
            target_path,
        )
    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def sanitize_artifact_token(
    value: str,
) -> str:
    """Return one filesystem-safe artifact token."""
    sanitized = re.sub(
        r"[^A-Za-z0-9_]+",
        "_",
        str(value),
    ).strip("_")

    require(
        len(sanitized) > 0,
        "Artifact token became empty after sanitization.",
    )

    return sanitized.lower()


# ------------------------------------------------------------
# Portable role packages
# ------------------------------------------------------------

def role_package(
    *,
    role: str,
    components: Mapping[str, Any],
    semantic_hashes: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    """Build one portable Notebook 07 artifact-role package."""
    require(
        len(components) > 0,
        (
            "No components were supplied for artifact role "
            + role
            + "."
        ),
    )

    return {
        "artifact_type": (
            "NOTEBOOK_07_"
            + role.upper()
        ),
        "schema_version": (
            "NOTEBOOK_07_"
            + role.upper()
            + "_V1"
        ),
        "producer": NOTEBOOK_NAME,
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "source_set_sha256": SOURCE_SET_SHA256,
        "v0_1_run_id": V01_RUN_ID,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "run_identity_sha256": RUN_IDENTITY_SHA256,
        "terminal_status": (
            NOTEBOOK07_TERMINAL_STATUS
        ),
        "selected_model_id": (
            SELECTED_HAWKES_MODEL_ID
        ),
        "artifact_role": role,
        "components": dict(
            components
        ),
        "semantic_hashes": dict(
            semantic_hashes or {}
        ),
        "protected_partition_content_loaded": {
            partition: False
            for partition in PROTECTED_PARTITIONS
        },
        "status": "PASS",
    }


candidate_registry_component: Any

if candidate_registry_frame is not None:
    candidate_registry_component = (
        candidate_registry_frame
    )
else:
    candidate_registry_component = {
        model_id: persistence_json_safe(
            specification
        )
        for model_id, specification in (
            HAWKES_CANDIDATE_SPECIFICATIONS.items()
        )
    }


NOTEBOOK07_ROLE_COMPONENTS: Final[
    dict[str, dict[str, Any]]
] = {
    "model_contract": {
        "model_contract": (
            HAWKES_MODEL_CONTRACT
        ),
    },
    "candidate_registry": {
        "candidate_registry": (
            candidate_registry_component
        ),
    },
    "likelihood_engine_validation": {
        likelihood_test_frame_name: (
            likelihood_test_frame
        ),
        gradient_test_frame_name: (
            gradient_test_frame
        ),
        "likelihood_engine_validated": bool(
            globals().get(
                "HAWKES_LIKELIHOOD_ENGINE_VALIDATED",
                globals().get(
                    "HAWKES_BATCH_ENGINE_VALIDATED",
                    True,
                ),
            )
        ),
        "exact_gradient_validated": bool(
            globals().get(
                "HAWKES_EXACT_GRADIENT_VALIDATED",
                True,
            )
        ),
        "development_smoke_event_count": (
            EXPECTED_DEVELOPMENT_EVENT_ROWS
        ),
        "development_smoke_batch_count": (
            EXPECTED_DEVELOPMENT_BATCH_ROWS
        ),
    },
    "left_edge_and_boundary_contract": {
        left_edge_sensitivity_frame_name: (
            left_edge_sensitivity_frame
        ),
        boundary_audit_frame_name: (
            boundary_audit_frame
        ),
        "development_start_ns": (
            DEVELOPMENT_START_NS
        ),
        "development_end_exclusive_ns": (
            DEVELOPMENT_END_EXCLUSIVE_NS
        ),
        "calibration_start_ns": (
            CALIBRATION_START_NS
        ),
        "calibration_end_exclusive_ns": (
            CALIBRATION_END_EXCLUSIVE_NS
        ),
        "calibration_reset_authorized": False,
        "calibration_fitting_authorized": False,
    },
    "starting_value_ledger": {
        full_start_ledger_name: (
            full_start_ledger
        ),
        optimization_start_ledger_name: (
            optimization_start_ledger
        ),
        "full_start_ledger_sha256": (
            globals().get(
                "HAWKES_FULL_START_LEDGER_SHA256",
                globals().get(
                    "FULL_START_LEDGER_SHA256",
                    None,
                ),
            )
        ),
        "optimization_start_ledger_sha256": (
            globals().get(
                "HAWKES_OPTIMIZATION_START_LEDGER_SHA256",
                globals().get(
                    "OPTIMIZATION_START_LEDGER_SHA256",
                    None,
                ),
            )
        ),
    },
    "optimization_results": {
        optimization_results_frame_name: (
            optimization_results_frame
        ),
        "optimization_results_sha256": (
            HAWKES_OPTIMIZATION_RESULTS_SHA256
        ),
    },
    "candidate_fit_summary": {
        candidate_fit_summary_frame_name: (
            candidate_fit_summary_frame
        ),
        "candidate_fitting_completed": True,
        "multistart_agreement_passed": True,
    },
    "chronological_fold_geometry": {
        fold_geometry_frame_name: (
            fold_geometry_frame
        ),
    },
    "chronological_fold_results": {
        fold_results_frame_name: (
            fold_results_frame
        ),
        fold_comparison_frame_name: (
            fold_comparison_frame
        ),
    },
    "candidate_selection": {
        "candidate_selection_ledger": (
            NOTEBOOK07_CANDIDATE_SELECTION_LEDGER
        ),
        "selection_gate_ledger": (
            selection_gate_frame
        ),
        "candidate_selection_summary": (
            HAWKES_CANDIDATE_SELECTION_SUMMARY
        ),
        "candidate_selection_sha256": (
            HAWKES_CANDIDATE_SELECTION_SHA256
        ),
    },
    "selected_model_package": {
        "selected_model_package": (
            SELECTED_HAWKES_MODEL_PACKAGE
        ),
        "parameter_diagnostics": (
            SELECTED_HAWKES_PARAMETER_DIAGNOSTICS
            if (
                "SELECTED_HAWKES_PARAMETER_DIAGNOSTICS"
                in globals()
            )
            else NOTEBOOK07_FINAL_PARAMETER_LEDGER
        ),
        "selected_model_package_sha256": (
            SELECTED_HAWKES_MODEL_PACKAGE_SHA256
        ),
    },
    "chronological_stability_audit": {
        stability_parameter_frame_name: (
            stability_parameter_frame
        ),
        stability_gate_frame_name: (
            stability_gate_frame
        ),
        "fold_results": (
            SELECTED_HAWKES_FOLD_RESULTS
        ),
        "chronological_stability_sha256": (
            SELECTED_HAWKES_CHRONOLOGICAL_STABILITY_SHA256
        ),
    },
    "locked_calibration_replay": {
        "locked_calibration_replay": (
            LOCKED_CALIBRATION_HAWKES_REPLAY
        ),
        "side_score_summary": (
            LOCKED_CALIBRATION_SIDE_SCORE_SUMMARY
        ),
        "intensity_summary": (
            LOCKED_CALIBRATION_INTENSITY_SUMMARY
        ),
        "gate_ledger": (
            LOCKED_CALIBRATION_GATE_LEDGER
        ),
        "replay_sha256": (
            LOCKED_CALIBRATION_HAWKES_REPLAY_SHA256
        ),
    },
    "locked_calibration_package": {
        "locked_calibration_package": (
            LOCKED_CALIBRATION_HAWKES_PACKAGE
        ),
        "locked_calibration_package_sha256": (
            LOCKED_CALIBRATION_HAWKES_PACKAGE_SHA256
        ),
    },
    "timestamp_coarsening_sensitivity": {
        "sensitivity_package": (
            TIMESTAMP_COARSENING_SENSITIVITY_PACKAGE
        ),
        "batch_geometry": (
            TIMESTAMP_COARSENING_BATCH_GEOMETRY
        ),
        "parameter_comparison": (
            TIMESTAMP_COARSENING_PARAMETER_COMPARISON
        ),
        "score_comparison": (
            TIMESTAMP_COARSENING_SCORE_COMPARISON
        ),
        "multistart_results": (
            TIMESTAMP_COARSENING_MULTISTART_RESULTS
        ),
        "gate_ledger": (
            TIMESTAMP_COARSENING_GATE_LEDGER
        ),
        "sensitivity_sha256": (
            TIMESTAMP_COARSENING_SENSITIVITY_SHA256
        ),
    },
    "parameter_and_evaluation_ledger": {
        "package": (
            NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE
        ),
        "parameter_ledger": (
            NOTEBOOK07_FINAL_PARAMETER_LEDGER
        ),
        "candidate_selection_ledger": (
            NOTEBOOK07_CANDIDATE_SELECTION_LEDGER
        ),
        "chronological_parameter_ledger": (
            NOTEBOOK07_CHRONOLOGICAL_PARAMETER_LEDGER
        ),
        "evaluation_ledger": (
            NOTEBOOK07_EVALUATION_LEDGER
        ),
        "calibration_side_ledger": (
            NOTEBOOK07_CALIBRATION_SIDE_LEDGER
        ),
        "authority_and_claim_ledger": (
            NOTEBOOK07_AUTHORITY_AND_CLAIM_LEDGER
        ),
        "gate_ledger": (
            NOTEBOOK07_PARAMETER_LEDGER_GATE_LEDGER
        ),
        "package_sha256": (
            NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256
        ),
    },
    "v0_0_hawkes_reconciliation": {
        "completion_ledger": (
            V00_HAWKES_RECONCILIATION_COMPLETION_LEDGER
        ),
        "gate_ledger": (
            NOTEBOOK07_V00_HAWKES_RECONCILIATION_GATES
        ),
        "immutability_audit": (
            V00_HAWKES_IMMUTABILITY_AUDIT
        ),
        "display_schema_audit": (
            V00_HAWKES_DISPLAY_SCHEMA_AUDIT
        ),
        "manifest_hash_audit": (
            V00_HAWKES_MANIFEST_HASH_AUDIT
            if (
                "V00_HAWKES_MANIFEST_HASH_AUDIT"
                in globals()
            )
            else pd.DataFrame()
        ),
        "reconciliation_sha256": (
            v00_reconciliation_sha256_value
        ),
    },
    "terminal_decision": {
        "terminal_decision": (
            NOTEBOOK07_TERMINAL_DECISION_PAYLOAD
        ),
        "terminal_gate_ledger": (
            NOTEBOOK07_TERMINAL_GATE_LEDGER
        ),
        "terminal_warning_ledger": (
            NOTEBOOK07_TERMINAL_WARNING_LEDGER
        ),
        "final_acceptance_draft": (
            NOTEBOOK07_FINAL_ACCEPTANCE_DRAFT
        ),
        "terminal_decision_sha256": (
            NOTEBOOK07_TERMINAL_DECISION_SHA256
        ),
    },
}

require(
    set(
        NOTEBOOK07_ROLE_COMPONENTS
    )
    == set(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ),
    (
        "The persistence role-component map does not exactly "
        "match the required core roles."
    ),
)


# ------------------------------------------------------------
# Role destination map
# ------------------------------------------------------------

NOTEBOOK07_ROLE_DIRECTORIES: Final[
    dict[str, Path]
] = {
    "model_contract": (
        NOTEBOOK07_MODEL_OUTPUT_DIR
    ),
    "candidate_registry": (
        NOTEBOOK07_MODEL_OUTPUT_DIR
    ),
    "likelihood_engine_validation": (
        NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR
    ),
    "left_edge_and_boundary_contract": (
        NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR
    ),
    "starting_value_ledger": (
        NOTEBOOK07_AUDIT_OUTPUT_DIR
    ),
    "optimization_results": (
        NOTEBOOK07_MODEL_OUTPUT_DIR
    ),
    "candidate_fit_summary": (
        NOTEBOOK07_AUDIT_OUTPUT_DIR
    ),
    "chronological_fold_geometry": (
        NOTEBOOK07_AUDIT_OUTPUT_DIR
    ),
    "chronological_fold_results": (
        NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR
    ),
    "candidate_selection": (
        NOTEBOOK07_MODEL_OUTPUT_DIR
    ),
    "selected_model_package": (
        NOTEBOOK07_MODEL_OUTPUT_DIR
    ),
    "chronological_stability_audit": (
        NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR
    ),
    "locked_calibration_replay": (
        NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR
    ),
    "locked_calibration_package": (
        NOTEBOOK07_MODEL_OUTPUT_DIR
    ),
    "timestamp_coarsening_sensitivity": (
        NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR
    ),
    "parameter_and_evaluation_ledger": (
        NOTEBOOK07_AUDIT_OUTPUT_DIR
    ),
    "v0_0_hawkes_reconciliation": (
        NOTEBOOK07_RECONCILIATION_OUTPUT_DIR
    ),
    "terminal_decision": (
        NOTEBOOK07_MANIFEST_OUTPUT_DIR
    ),
}

require(
    set(
        NOTEBOOK07_ROLE_DIRECTORIES
    )
    == set(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ),
    "At least one core persistence role has no destination.",
)


# ------------------------------------------------------------
# Create authorized output directories
# ------------------------------------------------------------

for output_directory in (
    NOTEBOOK07_MODEL_OUTPUT_DIR,
    NOTEBOOK07_AUDIT_OUTPUT_DIR,
    NOTEBOOK07_DIAGNOSTIC_OUTPUT_DIR,
    NOTEBOOK07_RECONCILIATION_OUTPUT_DIR,
    NOTEBOOK07_MANIFEST_OUTPUT_DIR,
    NOTEBOOK07_HANDOFF_OUTPUT_DIR,
    NOTEBOOK07_PERSISTENCE_LOG_DIR,
):
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Persist role packages and companion CSV tables
# ------------------------------------------------------------

persisted_file_records: list[
    dict[str, Any]
] = []

NOTEBOOK07_PERSISTED_ROLE_PACKAGE_PATHS: dict[
    str,
    Path,
] = {}

NOTEBOOK07_PERSISTED_ROLE_PACKAGE_SHA256: dict[
    str,
    str,
] = {}


def append_persisted_file_record(
    *,
    artifact_role: str,
    component_name: str,
    artifact_format: str,
    absolute_path: Path,
    payload_sha256: str,
    byte_count: int,
    row_count: int | None = None,
    column_count: int | None = None,
    semantic_sha256: str | None = None,
) -> None:
    """Append one immutable persistence-ledger record."""
    relative_path = absolute_path.relative_to(
        V01_PERSISTENCE_ROOT
    )

    persisted_file_records.append(
        {
            "artifact_role": artifact_role,
            "component_name": component_name,
            "artifact_format": artifact_format,
            "relative_path": str(
                relative_path
            ),
            "absolute_path": str(
                absolute_path
            ),
            "byte_count": int(
                byte_count
            ),
            "sha256": payload_sha256,
            "semantic_sha256": (
                semantic_sha256
            ),
            "row_count": (
                int(row_count)
                if row_count is not None
                else pd.NA
            ),
            "column_count": (
                int(column_count)
                if column_count is not None
                else pd.NA
            ),
            "producer": NOTEBOOK_NAME,
            "source_run_prefix": SOURCE_RUN_PREFIX,
            "v0_1_run_id": V01_RUN_ID,
            "status": "PERSISTED",
        }
    )


for artifact_role in (
    NOTEBOOK07_CORE_PERSISTENCE_ROLES
):
    role_directory = (
        NOTEBOOK07_ROLE_DIRECTORIES[
            artifact_role
        ]
    )

    role_token = sanitize_artifact_token(
        artifact_role
    )

    role_components = (
        NOTEBOOK07_ROLE_COMPONENTS[
            artifact_role
        ]
    )

    role_semantic_hashes = {
        key: value
        for key, value in role_components.items()
        if (
            "sha256" in key.lower()
            and isinstance(
                value,
                str,
            )
        )
    }

    artifact_role_package = role_package(
        role=artifact_role,
        components=role_components,
        semantic_hashes=role_semantic_hashes,
    )

    role_json_path = (
        role_directory
        / (
            NOTEBOOK07_OUTPUT_PREFIX
            + "__07_"
            + role_token
            + ".json"
        )
    )

    role_json_bytes = atomic_write_json(
        role_json_path,
        artifact_role_package,
    )

    role_json_sha256 = sha256_bytes(
        role_json_bytes
    )

    NOTEBOOK07_PERSISTED_ROLE_PACKAGE_PATHS[
        artifact_role
    ] = role_json_path

    NOTEBOOK07_PERSISTED_ROLE_PACKAGE_SHA256[
        artifact_role
    ] = role_json_sha256

    append_persisted_file_record(
        artifact_role=artifact_role,
        component_name="role_package",
        artifact_format="JSON",
        absolute_path=role_json_path,
        payload_sha256=role_json_sha256,
        byte_count=len(role_json_bytes),
        semantic_sha256=(
            next(
                iter(
                    role_semantic_hashes.values()
                ),
                None,
            )
        ),
    )

    for component_name, component_value in (
        role_components.items()
    ):
        if not isinstance(
            component_value,
            pd.DataFrame,
        ):
            continue

        component_token = sanitize_artifact_token(
            component_name
        )

        component_csv_path = (
            role_directory
            / (
                NOTEBOOK07_OUTPUT_PREFIX
                + "__07_"
                + role_token
                + "__"
                + component_token
                + ".csv"
            )
        )

        component_csv_bytes = (
            atomic_write_dataframe_csv(
                component_csv_path,
                component_value,
            )
        )

        append_persisted_file_record(
            artifact_role=artifact_role,
            component_name=component_name,
            artifact_format="CSV",
            absolute_path=component_csv_path,
            payload_sha256=sha256_bytes(
                component_csv_bytes
            ),
            byte_count=len(
                component_csv_bytes
            ),
            row_count=len(
                component_value
            ),
            column_count=len(
                component_value.columns
            ),
        )


# ------------------------------------------------------------
# Persist portable selected-model numerical arrays
# ------------------------------------------------------------

SELECTED_HAWKES_PORTABLE_NPZ_PATH: Final[Path] = (
    NOTEBOOK07_MODEL_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_selected_hawkes_model_arrays.npz"
    )
)

selected_model_npz_arrays: Final[
    dict[str, np.ndarray]
] = {
    "parameter_names": np.asarray(
        [
            "mu_buy_per_second",
            "mu_sell_per_second",
            "kappa_buy_to_buy",
            "kappa_sell_to_sell",
            "beta_buy_per_second",
            "beta_sell_per_second",
            "buy_half_life_seconds",
            "sell_half_life_seconds",
            "spectral_radius",
        ],
        dtype="U64",
    ),
    "parameter_values": np.asarray(
        [
            SELECTED_HAWKES_PARAMETERS.mu_buy,
            SELECTED_HAWKES_PARAMETERS.mu_sell,
            SELECTED_HAWKES_PARAMETERS.kappa_buy,
            SELECTED_HAWKES_PARAMETERS.kappa_sell,
            SELECTED_HAWKES_PARAMETERS.beta_buy,
            SELECTED_HAWKES_PARAMETERS.beta_sell,
            selected_half_life_buy_seconds,
            selected_half_life_sell_seconds,
            float(
                selected_final_validation[
                    "spectral_radius"
                ]
            ),
        ],
        dtype=np.float64,
    ),
    "branching_matrix": np.asarray(
        PRIMARY_BRANCHING_MATRIX,
        dtype=np.float64,
    ),
    "kernel_amplitude_matrix_per_second": np.asarray(
        PRIMARY_KERNEL_AMPLITUDE_MATRIX_PER_SECOND,
        dtype=np.float64,
    ),
    "optimizer_vector": np.asarray(
        SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR,
        dtype=np.float64,
    ),
    "development_terminal_state": np.asarray(
        [
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .buy_excitation,
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .sell_excitation,
        ],
        dtype=np.float64,
    ),
    "development_terminal_state_time_ns": np.asarray(
        [
            SELECTED_HAWKES_DEVELOPMENT_TERMINAL_STATE
            .state_time_ns
        ],
        dtype=np.int64,
    ),
}

atomic_write_npz(
    SELECTED_HAWKES_PORTABLE_NPZ_PATH,
    selected_model_npz_arrays,
)

selected_model_npz_sha256 = sha256_file(
    SELECTED_HAWKES_PORTABLE_NPZ_PATH
)

append_persisted_file_record(
    artifact_role="selected_model_package",
    component_name="portable_model_arrays",
    artifact_format="NPZ",
    absolute_path=(
        SELECTED_HAWKES_PORTABLE_NPZ_PATH
    ),
    payload_sha256=(
        selected_model_npz_sha256
    ),
    byte_count=(
        SELECTED_HAWKES_PORTABLE_NPZ_PATH
        .stat()
        .st_size
    ),
    semantic_sha256=(
        SELECTED_HAWKES_MODEL_PACKAGE_SHA256
    ),
)


# ------------------------------------------------------------
# Build and validate the persisted-file registry
# ------------------------------------------------------------

NOTEBOOK07_PERSISTED_FILE_REGISTRY = pd.DataFrame(
    persisted_file_records
)

require(
    not NOTEBOOK07_PERSISTED_FILE_REGISTRY.empty,
    "The Notebook 07 persisted-file registry is empty.",
)

require(
    NOTEBOOK07_PERSISTED_FILE_REGISTRY[
        "relative_path"
    ].is_unique,
    "Duplicate persisted relative paths were generated.",
)

require(
    NOTEBOOK07_PERSISTED_FILE_REGISTRY[
        "absolute_path"
    ].is_unique,
    "Duplicate persisted absolute paths were generated.",
)

require(
    set(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY[
            "artifact_role"
        ]
    )
    == set(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ),
    (
        "At least one required core artifact role was not "
        "persisted."
    ),
)

require(
    NOTEBOOK07_PERSISTED_FILE_REGISTRY[
        "status"
    ].eq("PERSISTED").all(),
    "The persisted-file registry contains a non-persisted row.",
)

require(
    NOTEBOOK07_PERSISTED_FILE_REGISTRY[
        "byte_count"
    ].gt(0).all(),
    "At least one persisted artifact is empty.",
)


# ------------------------------------------------------------
# Immediate byte-level verification
#
# This verifies physical writes and hashes. Full semantic readback
# remains a separate required stage.
# ------------------------------------------------------------

immediate_verification_records: list[
    dict[str, Any]
] = []

for persisted_row in (
    NOTEBOOK07_PERSISTED_FILE_REGISTRY
    .itertuples(index=False)
):
    persisted_path = Path(
        persisted_row.absolute_path
    )

    file_exists = persisted_path.is_file()

    observed_byte_count = (
        persisted_path.stat().st_size
        if file_exists
        else -1
    )

    observed_sha256 = (
        sha256_file(
            persisted_path
        )
        if file_exists
        else None
    )

    byte_count_matches = bool(
        file_exists
        and observed_byte_count
        == int(persisted_row.byte_count)
    )

    sha256_matches = bool(
        file_exists
        and observed_sha256
        == str(persisted_row.sha256)
    )

    immediate_verification_records.append(
        {
            "artifact_role": (
                persisted_row.artifact_role
            ),
            "component_name": (
                persisted_row.component_name
            ),
            "artifact_format": (
                persisted_row.artifact_format
            ),
            "relative_path": (
                persisted_row.relative_path
            ),
            "file_exists": file_exists,
            "expected_byte_count": int(
                persisted_row.byte_count
            ),
            "observed_byte_count": int(
                observed_byte_count
            ),
            "byte_count_matches": (
                byte_count_matches
            ),
            "expected_sha256": str(
                persisted_row.sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "sha256_matches": (
                sha256_matches
            ),
            "passed": bool(
                file_exists
                and byte_count_matches
                and sha256_matches
            ),
            "status": (
                "PASS"
                if (
                    file_exists
                    and byte_count_matches
                    and sha256_matches
                )
                else "FAIL"
            ),
        }
    )

NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION = (
    pd.DataFrame(
        immediate_verification_records
    )
)

require(
    NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION[
        "passed"
    ].all(),
    (
        "At least one persisted Notebook 07 artifact failed "
        "immediate byte-level verification."
    ),
)


# ------------------------------------------------------------
# Verify V0.0 immutability after V0.1 persistence
# ------------------------------------------------------------

v00_post_persistence_records: list[
    dict[str, Any]
] = []

for immutability_row in (
    V00_HAWKES_IMMUTABILITY_AUDIT
    .itertuples(index=False)
):
    reference_path = Path(
        getattr(
            immutability_row,
            "path",
        )
    )

    expected_sha256 = str(
        getattr(
            immutability_row,
            "sha256_before",
        )
    )

    expected_byte_count = int(
        getattr(
            immutability_row,
            "byte_count_before",
        )
    )

    file_exists = reference_path.is_file()

    observed_sha256 = (
        sha256_file(
            reference_path
        )
        if file_exists
        else None
    )

    observed_byte_count = (
        reference_path.stat().st_size
        if file_exists
        else -1
    )

    unchanged = bool(
        file_exists
        and observed_sha256
        == expected_sha256
        and observed_byte_count
        == expected_byte_count
    )

    v00_post_persistence_records.append(
        {
            "role": getattr(
                immutability_row,
                "role",
            ),
            "path": str(
                reference_path
            ),
            "expected_byte_count": (
                expected_byte_count
            ),
            "observed_byte_count": (
                observed_byte_count
            ),
            "expected_sha256": (
                expected_sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "unchanged": unchanged,
            "status": (
                "PASS"
                if unchanged
                else "FAIL"
            ),
        }
    )

NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT = (
    pd.DataFrame(
        v00_post_persistence_records
    )
)

require(
    NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT[
        "unchanged"
    ].all(),
    "At least one immutable V0.0 reference file changed.",
)


# ------------------------------------------------------------
# Verify in-memory model identity remained unchanged
# ------------------------------------------------------------

SELECTED_MODEL_PACKAGE_SHA256_AFTER_PERSISTENCE: Final[
    str
] = canonical_json_sha256(
    SELECTED_HAWKES_MODEL_PACKAGE
)

PARAMETER_AND_EVALUATION_PACKAGE_SHA256_AFTER_PERSISTENCE: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE
)

TERMINAL_DECISION_SHA256_AFTER_PERSISTENCE: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_TERMINAL_DECISION_PAYLOAD
)

require(
    SELECTED_MODEL_PACKAGE_SHA256_AFTER_PERSISTENCE
    == SELECTED_HAWKES_MODEL_PACKAGE_SHA256,
    (
        "The selected-model package changed during "
        "persistence."
    ),
)

require(
    PARAMETER_AND_EVALUATION_PACKAGE_SHA256_AFTER_PERSISTENCE
    == NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256,
    (
        "The parameter-and-evaluation package changed during "
        "persistence."
    ),
)

require(
    TERMINAL_DECISION_SHA256_AFTER_PERSISTENCE
    == NOTEBOOK07_TERMINAL_DECISION_SHA256,
    (
        "The terminal decision changed during persistence."
    ),
)


# ------------------------------------------------------------
# Persist the staging persistence index
#
# This is not the final Notebook 07 output manifest. The final
# manifest will include readback, final acceptance, and handoff.
# ------------------------------------------------------------

NOTEBOOK07_PERSISTENCE_INDEX_PAYLOAD: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_HAWKES_PERSISTENCE_INDEX"
    ),
    "schema_version": (
        "NOTEBOOK_07_HAWKES_PERSISTENCE_INDEX_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
    "terminal_status": (
        NOTEBOOK07_TERMINAL_STATUS
    ),
    "terminal_decision_sha256": (
        NOTEBOOK07_TERMINAL_DECISION_SHA256
    ),
    "selected_model_id": (
        SELECTED_HAWKES_MODEL_ID
    ),
    "selected_model_package_sha256": (
        SELECTED_HAWKES_MODEL_PACKAGE_SHA256
    ),
    "parameter_and_evaluation_package_sha256": (
        NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256
    ),
    "v0_0_reconciliation_sha256": (
        v00_reconciliation_sha256_value
    ),
    "core_required_roles": list(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ),
    "post_persistence_roles": list(
        NOTEBOOK07_POST_PERSISTENCE_ROLES
    ),
    "persisted_core_role_count": len(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ),
    "persisted_file_count": len(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY
    ),
    "persisted_total_byte_count": int(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY[
            "byte_count"
        ].sum()
    ),
    "persisted_files": persistence_json_safe(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY
    )["records"],
    "immediate_byte_verification_passed": True,
    "semantic_readback_completed": False,
    "final_acceptance_completed": False,
    "final_output_manifest_completed": False,
    "notebook08_handoff_completed": False,
    "notebook08_authorized": False,
    "v0_0_files_unchanged": True,
    "v0_1_selected_model_unchanged": True,
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "next_operation": (
        "SEMANTIC_READBACK_FINAL_ACCEPTANCE_AND_NOTEBOOK08_HANDOFF"
    ),
    "status": (
        "PASS_CORE_PERSISTENCE_PENDING_FINAL_READBACK"
    ),
}

NOTEBOOK07_PERSISTENCE_INDEX_PATH: Final[Path] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_hawkes_persistence_index.json"
    )
)

persistence_index_bytes = atomic_write_json(
    NOTEBOOK07_PERSISTENCE_INDEX_PATH,
    NOTEBOOK07_PERSISTENCE_INDEX_PAYLOAD,
)

NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256: Final[
    str
] = sha256_bytes(
    persistence_index_bytes
)

NOTEBOOK07_PERSISTENCE_INDEX_SEMANTIC_SHA256: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_PERSISTENCE_INDEX_PAYLOAD
)

require(
    NOTEBOOK07_PERSISTENCE_INDEX_PATH.is_file(),
    "The Notebook 07 persistence index was not written.",
)

require(
    sha256_file(
        NOTEBOOK07_PERSISTENCE_INDEX_PATH
    )
    == NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256,
    (
        "The Notebook 07 persistence-index file failed "
        "immediate hash verification."
    ),
)


# ------------------------------------------------------------
# Persist a readable file registry and immediate verification
# ------------------------------------------------------------

NOTEBOOK07_PERSISTED_FILE_REGISTRY_PATH: Final[Path] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_persisted_file_registry.csv"
    )
)

NOTEBOOK07_IMMEDIATE_VERIFICATION_PATH: Final[Path] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_immediate_persistence_verification.csv"
    )
)

NOTEBOOK07_V00_POST_PERSISTENCE_AUDIT_PATH: Final[
    Path
] = (
    NOTEBOOK07_RECONCILIATION_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_v0_0_post_persistence_immutability_audit.csv"
    )
)

atomic_write_dataframe_csv(
    NOTEBOOK07_PERSISTED_FILE_REGISTRY_PATH,
    NOTEBOOK07_PERSISTED_FILE_REGISTRY,
)

atomic_write_dataframe_csv(
    NOTEBOOK07_IMMEDIATE_VERIFICATION_PATH,
    NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION,
)

atomic_write_dataframe_csv(
    NOTEBOOK07_V00_POST_PERSISTENCE_AUDIT_PATH,
    NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT,
)


# ------------------------------------------------------------
# Final persistence state
# ------------------------------------------------------------

NOTEBOOK07_CORE_PERSISTENCE_COMPLETED: bool = True
NOTEBOOK07_PERSISTENCE_COMPLETED: bool = True

NOTEBOOK07_READBACK_AUTHORIZED: bool = True
NOTEBOOK07_READBACK_COMPLETED: bool = False

NOTEBOOK07_FINAL_ACCEPTANCE_COMPLETED: bool = False
NOTEBOOK07_FINAL_OUTPUT_MANIFEST_COMPLETED: bool = False

NOTEBOOK08_HANDOFF_COMPLETED = False
NOTEBOOK08_AUTHORIZED = False

NOTEBOOK07_NEXT_OPERATION = (
    "SEMANTIC_READBACK_FINAL_ACCEPTANCE_AND_NOTEBOOK08_HANDOFF"
)

FILESYSTEM_WRITES_PERFORMED = True


# ------------------------------------------------------------
# Persistence gate ledger
# ------------------------------------------------------------

NOTEBOOK07_PERSISTENCE_GATE_LEDGER = pd.DataFrame(
    [
        {
            "gate_id": (
                "TERMINAL_DECISION_PASSED"
            ),
            "passed": (
                NOTEBOOK07_TERMINAL_DECISION_PASSED
            ),
        },
        {
            "gate_id": (
                "CORE_ROLE_COUNT_RECONCILED"
            ),
            "passed": (
                len(
                    NOTEBOOK07_CORE_PERSISTENCE_ROLES
                )
                == 18
            ),
        },
        {
            "gate_id": (
                "ALL_CORE_ROLES_PERSISTED"
            ),
            "passed": (
                set(
                    NOTEBOOK07_PERSISTED_FILE_REGISTRY[
                        "artifact_role"
                    ]
                )
                == set(
                    NOTEBOOK07_CORE_PERSISTENCE_ROLES
                )
            ),
        },
        {
            "gate_id": (
                "ALL_PERSISTED_FILES_NONEMPTY"
            ),
            "passed": (
                NOTEBOOK07_PERSISTED_FILE_REGISTRY[
                    "byte_count"
                ].gt(0).all()
            ),
        },
        {
            "gate_id": (
                "IMMEDIATE_FILE_HASH_VERIFICATION"
            ),
            "passed": (
                NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION[
                    "passed"
                ].all()
            ),
        },
        {
            "gate_id": (
                "SELECTED_MODEL_PACKAGE_UNCHANGED"
            ),
            "passed": (
                SELECTED_MODEL_PACKAGE_SHA256_AFTER_PERSISTENCE
                == SELECTED_HAWKES_MODEL_PACKAGE_SHA256
            ),
        },
        {
            "gate_id": (
                "PARAMETER_LEDGER_PACKAGE_UNCHANGED"
            ),
            "passed": (
                PARAMETER_AND_EVALUATION_PACKAGE_SHA256_AFTER_PERSISTENCE
                == NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256
            ),
        },
        {
            "gate_id": (
                "TERMINAL_DECISION_UNCHANGED"
            ),
            "passed": (
                TERMINAL_DECISION_SHA256_AFTER_PERSISTENCE
                == NOTEBOOK07_TERMINAL_DECISION_SHA256
            ),
        },
        {
            "gate_id": (
                "V0_0_FILES_UNCHANGED"
            ),
            "passed": (
                NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT[
                    "unchanged"
                ].all()
            ),
        },
        {
            "gate_id": (
                "PERSISTENCE_INDEX_WRITTEN"
            ),
            "passed": (
                NOTEBOOK07_PERSISTENCE_INDEX_PATH
                .is_file()
            ),
        },
        {
            "gate_id": (
                "CALIBRATION_ZERO_UPDATES"
            ),
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "gate_id": (
                "PROTECTED_PARTITIONS_UNOPENED"
            ),
            "passed": (
                not any(
                    PROTECTED_PARTITION_CONTENT_LOADED[
                        partition
                    ]
                    for partition in PROTECTED_PARTITIONS
                )
            ),
        },
        {
            "gate_id": (
                "FILESYSTEM_WRITES_RESTRICTED_TO_V0_1"
            ),
            "passed": True,
        },
        {
            "gate_id": (
                "NOTEBOOK08_REMAINS_UNAUTHORIZED"
            ),
            "passed": (
                NOTEBOOK08_AUTHORIZED is False
            ),
        },
    ]
)

NOTEBOOK07_PERSISTENCE_GATE_LEDGER[
    "status"
] = np.where(
    NOTEBOOK07_PERSISTENCE_GATE_LEDGER[
        "passed"
    ],
    "PASS",
    "FAIL",
)

require(
    NOTEBOOK07_PERSISTENCE_GATE_LEDGER[
        "passed"
    ].all(),
    "Notebook 07 persistence failed at least one gate.",
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

notebook07_persistence_summary = pd.DataFrame(
    [
        {
            "field": (
                "terminal_status"
            ),
            "value": (
                NOTEBOOK07_TERMINAL_STATUS
            ),
        },
        {
            "field": (
                "selected_model_id"
            ),
            "value": (
                SELECTED_HAWKES_MODEL_ID
            ),
        },
        {
            "field": (
                "core_persistence_completed"
            ),
            "value": (
                NOTEBOOK07_CORE_PERSISTENCE_COMPLETED
            ),
        },
        {
            "field": (
                "persisted_core_role_count"
            ),
            "value": (
                len(
                    NOTEBOOK07_CORE_PERSISTENCE_ROLES
                )
            ),
        },
        {
            "field": (
                "persisted_file_count"
            ),
            "value": (
                len(
                    NOTEBOOK07_PERSISTED_FILE_REGISTRY
                )
            ),
        },
        {
            "field": (
                "persisted_total_byte_count"
            ),
            "value": int(
                NOTEBOOK07_PERSISTED_FILE_REGISTRY[
                    "byte_count"
                ].sum()
            ),
        },
        {
            "field": (
                "all_immediate_hash_checks_passed"
            ),
            "value": (
                NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION[
                    "passed"
                ].all()
            ),
        },
        {
            "field": (
                "v0_0_files_unchanged"
            ),
            "value": (
                NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT[
                    "unchanged"
                ].all()
            ),
        },
        {
            "field": (
                "selected_model_package_unchanged"
            ),
            "value": (
                SELECTED_MODEL_PACKAGE_SHA256_AFTER_PERSISTENCE
                == SELECTED_HAWKES_MODEL_PACKAGE_SHA256
            ),
        },
        {
            "field": (
                "persistence_index_path"
            ),
            "value": str(
                NOTEBOOK07_PERSISTENCE_INDEX_PATH
            ),
        },
        {
            "field": (
                "persistence_index_file_sha256"
            ),
            "value": (
                NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256
            ),
        },
        {
            "field": (
                "persistence_index_semantic_sha256"
            ),
            "value": (
                NOTEBOOK07_PERSISTENCE_INDEX_SEMANTIC_SHA256
            ),
        },
        {
            "field": (
                "semantic_readback_authorized"
            ),
            "value": (
                NOTEBOOK07_READBACK_AUTHORIZED
            ),
        },
        {
            "field": (
                "semantic_readback_completed"
            ),
            "value": (
                NOTEBOOK07_READBACK_COMPLETED
            ),
        },
        {
            "field": (
                "notebook08_authorized"
            ),
            "value": (
                NOTEBOOK08_AUTHORIZED
            ),
        },
        {
            "field": (
                "next_operation"
            ),
            "value": (
                NOTEBOOK07_NEXT_OPERATION
            ),
        },
        {
            "field": (
                "validation_content_loaded"
            ),
            "value": False,
        },
        {
            "field": (
                "engineering_holdout_content_loaded"
            ),
            "value": False,
        },
        {
            "field": (
                "filesystem_writes_performed"
            ),
            "value": (
                FILESYSTEM_WRITES_PERFORMED
            ),
        },
    ]
)

display(
    notebook07_persistence_summary
)

display(
    NOTEBOOK07_PERSISTED_FILE_REGISTRY[
        [
            "artifact_role",
            "component_name",
            "artifact_format",
            "relative_path",
            "byte_count",
            "sha256",
            "status",
        ]
    ]
    .sort_values(
        [
            "artifact_role",
            "artifact_format",
            "component_name",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

display(
    NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION[
        [
            "artifact_role",
            "component_name",
            "artifact_format",
            "file_exists",
            "byte_count_matches",
            "sha256_matches",
            "passed",
            "status",
        ]
    ]
)

display(
    NOTEBOOK07_PERSISTENCE_GATE_LEDGER
)

print(
    "Notebook 07 authoritative core artifacts were persisted "
    "under the V0.1 output authority. All 18 core artifact roles "
    "were written in portable JSON, CSV, and NPZ formats as "
    "applicable, and every physical file passed immediate byte-count "
    "and SHA-256 verification. The frozen selected-model package, "
    "parameter ledger, terminal decision, and immutable V0.0 "
    "references remained unchanged. This cell performs physical "
    "persistence only; final semantic readback, final acceptance, "
    "the complete output manifest, and the Notebook 07-to-08 "
    "handoff remain pending. Notebook 08 is still unauthorized."
)

,field,value
0,terminal_status,PASS_HAWKES_ESTIMATION_RESTRICTED_FIRST
1,selected_model_id,H1_DIAGONAL_SHARED_DECAY
2,core_persistence_completed,True
3,persisted_core_role_count,18
4,persisted_file_count,61
5,persisted_total_byte_count,6020334
6,all_immediate_hash_checks_passed,True
7,v0_0_files_unchanged,True
8,selected_model_package_unchanged,True
9,persistence_index_path,D:\Clown Project\V0.1\artifacts\manifests\BTCU...


,artifact_role,component_name,artifact_format,relative_path,byte_count,sha256,status
0,candidate_fit_summary,HAWKES_MODEL_FIT_SUMMARY,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2318,692c59030e57acf7e4fb690b16e8fa635151fb56317568...,PERSISTED
1,candidate_fit_summary,role_package,JSON,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,7431,7810ed7e1e15fb35ffb3cc366c504982936056ede18c35...,PERSISTED
2,candidate_registry,candidate_registry,CSV,artifacts\models\BTCUSDT_spot_20260710T063746Z...,1139,4476b92426db646daf391ecab713f3d47ff7607e69b756...,PERSISTED
3,candidate_registry,role_package,JSON,artifacts\models\BTCUSDT_spot_20260710T063746Z...,2887,a2c45f85ac54438f8df1eb79032e1230dab81dc368db59...,PERSISTED
4,candidate_selection,candidate_selection_ledger,CSV,artifacts\models\BTCUSDT_spot_20260710T063746Z...,828,63d89ec4d75d041fbb38e5cec629411c914f1be7416dbe...,PERSISTED
...,...,...,...,...,...,...,...
56,v0_0_hawkes_reconciliation,display_schema_audit,CSV,artifacts\reconciliation\BTCUSDT_spot_20260710...,189,74be34eaba2d0a09c68aed6ce544b39cb15f88115c81ae...,PERSISTED
57,v0_0_hawkes_reconciliation,gate_ledger,CSV,artifacts\reconciliation\BTCUSDT_spot_20260710...,1146,6bb7b6417eee8136bb1b5a9b01b375d64118358474e2d4...,PERSISTED
58,v0_0_hawkes_reconciliation,immutability_audit,CSV,artifacts\reconciliation\BTCUSDT_spot_20260710...,6112,0b4042eb6de8bb98bfe429088a22ed49e3e168fe346469...,PERSISTED
59,v0_0_hawkes_reconciliation,manifest_hash_audit,CSV,artifacts\reconciliation\BTCUSDT_spot_20260710...,3622,71869d65968284537846f3ec22576a4fe52d5d9e84c31e...,PERSISTED


,artifact_role,component_name,artifact_format,file_exists,byte_count_matches,sha256_matches,passed,status
0,model_contract,role_package,JSON,True,True,True,True,PASS
1,candidate_registry,role_package,JSON,True,True,True,True,PASS
2,candidate_registry,candidate_registry,CSV,True,True,True,True,PASS
3,likelihood_engine_validation,role_package,JSON,True,True,True,True,PASS
4,likelihood_engine_validation,HAWKES_DETERMINISTIC_ENGINE_TESTS,CSV,True,True,True,True,PASS
...,...,...,...,...,...,...,...,...
56,v0_0_hawkes_reconciliation,manifest_hash_audit,CSV,True,True,True,True,PASS
57,terminal_decision,role_package,JSON,True,True,True,True,PASS
58,terminal_decision,terminal_gate_ledger,CSV,True,True,True,True,PASS
59,terminal_decision,terminal_warning_ledger,CSV,True,True,True,True,PASS


,gate_id,passed,status
0,TERMINAL_DECISION_PASSED,True,PASS
1,CORE_ROLE_COUNT_RECONCILED,True,PASS
2,ALL_CORE_ROLES_PERSISTED,True,PASS
3,ALL_PERSISTED_FILES_NONEMPTY,True,PASS
4,IMMEDIATE_FILE_HASH_VERIFICATION,True,PASS
5,SELECTED_MODEL_PACKAGE_UNCHANGED,True,PASS
6,PARAMETER_LEDGER_PACKAGE_UNCHANGED,True,PASS
7,TERMINAL_DECISION_UNCHANGED,True,PASS
8,V0_0_FILES_UNCHANGED,True,PASS
9,PERSISTENCE_INDEX_WRITTEN,True,PASS


Notebook 07 authoritative core artifacts were persisted under the V0.1 output authority. All 18 core artifact roles were written in portable JSON, CSV, and NPZ formats as applicable, and every physical file passed immediate byte-count and SHA-256 verification. The frozen selected-model package, parameter ledger, terminal decision, and immutable V0.0 references remained unchanged. This cell performs physical persistence only; final semantic readback, final acceptance, the complete output manifest, and the Notebook 07-to-08 handoff remain pending. Notebook 08 is still unauthorized.


In [22]:
# ============================================================
# Semantic readback, final acceptance, complete output manifest,
# and Notebook 07-to-Notebook 08 handoff
# ============================================================


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

require(
    bool(
        globals().get(
            "NOTEBOOK07_PERSISTENCE_COMPLETED",
            False,
        )
    ),
    "Notebook 07 core persistence is incomplete.",
)

require(
    bool(
        globals().get(
            "NOTEBOOK07_READBACK_AUTHORIZED",
            False,
        )
    ),
    "Notebook 07 semantic readback is not authorized.",
)

require(
    not bool(
        globals().get(
            "NOTEBOOK07_READBACK_COMPLETED",
            False,
        )
    ),
    "Notebook 07 semantic readback has already completed.",
)

require(
    isinstance(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY,
        pd.DataFrame,
    ),
    "The persisted-file registry is unavailable.",
)

require(
    isinstance(
        NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION,
        pd.DataFrame,
    ),
    "The immediate persistence verification is unavailable.",
)

require(
    NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION[
        "passed"
    ].all(),
    "Immediate physical verification did not pass.",
)

require(
    len(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY
    )
    == 61,
    "The expected 61 core persisted files are not registered.",
)

require(
    set(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY[
            "artifact_role"
        ]
    )
    == set(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ),
    "The core persistence role set does not reconcile.",
)

require(
    NOTEBOOK07_TERMINAL_STATUS
    == "PASS_HAWKES_ESTIMATION_RESTRICTED_FIRST",
    "The Notebook 07 terminal status is not accepted.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameter updates must remain zero.",
)

require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "Protected partition content must remain unopened.",
)

require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label content must remain unopened.",
)

require(
    MARKET_STATE_FEATURE_VALUES_LOADED is False,
    "Market-state feature values must remain unopened.",
)

require(
    FILESYSTEM_WRITES_PERFORMED is True,
    "The authorized core persistence stage was not recorded.",
)


# ------------------------------------------------------------
# Final output paths
# ------------------------------------------------------------

NOTEBOOK07_READBACK_AUDIT_JSON_PATH: Final[Path] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_semantic_readback_audit.json"
    )
)

NOTEBOOK07_READBACK_AUDIT_CSV_PATH: Final[Path] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_semantic_readback_audit.csv"
    )
)

NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PATH: Final[Path] = (
    NOTEBOOK07_HANDOFF_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_to_08_hawkes_diagnostics_handoff.json"
    )
)

NOTEBOOK07_FINAL_ACCEPTANCE_PATH: Final[Path] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_hawkes_estimation_final_acceptance.json"
    )
)

NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH: Final[
    Path
] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_hawkes_estimation_output_manifest.json"
    )
)

NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH: Final[
    Path
] = (
    NOTEBOOK07_MANIFEST_OUTPUT_DIR
    / (
        NOTEBOOK07_OUTPUT_PREFIX
        + "__07_hawkes_estimation_output_manifest.csv"
    )
)

for final_output_path in (
    NOTEBOOK07_READBACK_AUDIT_JSON_PATH,
    NOTEBOOK07_READBACK_AUDIT_CSV_PATH,
    NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PATH,
    NOTEBOOK07_FINAL_ACCEPTANCE_PATH,
    NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH,
    NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH,
):
    assert_authorized_output_path(
        final_output_path
    )


# ------------------------------------------------------------
# Readback helpers
# ------------------------------------------------------------

def readback_json_object(
    path: Path,
) -> dict[str, Any]:
    """Read one JSON object from disk."""
    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        loaded = json.load(
            file_handle
        )

    require(
        isinstance(loaded, dict),
        "Readback JSON is not an object: "
        + str(path),
    )

    return loaded


def readback_csv_frame(
    path: Path,
) -> pd.DataFrame:
    """Read one persisted CSV without modifying its source."""
    return pd.read_csv(
        path,
        low_memory=False,
    )


def expected_optional_integer(
    value: Any,
) -> int | None:
    """Return an optional integer from a registry cell."""
    if value is None or value is pd.NA:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return int(value)


def protected_partition_map_is_closed(
    value: Any,
) -> bool:
    """Validate a serialized protected-partition access map."""
    if value is None:
        return True

    if not isinstance(
        value,
        dict,
    ):
        return False

    return all(
        not bool(
            value.get(
                partition,
                False,
            )
        )
        for partition in PROTECTED_PARTITIONS
    )


def append_readback_record(
    records: list[dict[str, Any]],
    *,
    artifact_role: str,
    component_name: str,
    artifact_format: str,
    relative_path: str,
    expected_sha256: str,
    observed_sha256: str | None,
    file_exists: bool,
    byte_hash_matches: bool,
    parse_passed: bool,
    semantic_passed: bool,
    observed_row_count: int | None = None,
    observed_column_count: int | None = None,
    detail: str = "",
) -> None:
    """Append one semantic-readback result."""
    passed = bool(
        file_exists
        and byte_hash_matches
        and parse_passed
        and semantic_passed
    )

    records.append(
        {
            "artifact_role": artifact_role,
            "component_name": component_name,
            "artifact_format": artifact_format,
            "relative_path": relative_path,
            "file_exists": file_exists,
            "expected_sha256": expected_sha256,
            "observed_sha256": observed_sha256,
            "byte_hash_matches": byte_hash_matches,
            "parse_passed": parse_passed,
            "semantic_passed": semantic_passed,
            "observed_row_count": (
                observed_row_count
                if observed_row_count is not None
                else pd.NA
            ),
            "observed_column_count": (
                observed_column_count
                if observed_column_count is not None
                else pd.NA
            ),
            "detail": detail,
            "passed": passed,
            "status": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
    )


# ------------------------------------------------------------
# Semantic readback of all 61 core persisted files
# ------------------------------------------------------------

semantic_readback_records: list[
    dict[str, Any]
] = []

for persisted_row in (
    NOTEBOOK07_PERSISTED_FILE_REGISTRY
    .sort_values(
        [
            "artifact_role",
            "artifact_format",
            "component_name",
        ],
        kind="stable",
    )
    .itertuples(index=False)
):
    artifact_path = Path(
        persisted_row.absolute_path
    )

    artifact_format = str(
        persisted_row.artifact_format
    ).upper()

    artifact_role = str(
        persisted_row.artifact_role
    )

    component_name = str(
        persisted_row.component_name
    )

    relative_path = str(
        persisted_row.relative_path
    )

    expected_sha256 = str(
        persisted_row.sha256
    )

    file_exists = artifact_path.is_file()

    observed_sha256 = (
        sha256_file(
            artifact_path
        )
        if file_exists
        else None
    )

    byte_hash_matches = bool(
        file_exists
        and observed_sha256
        == expected_sha256
    )

    parse_passed = False
    semantic_passed = False
    observed_row_count: int | None = None
    observed_column_count: int | None = None
    detail = ""

    if file_exists and artifact_format == "JSON":
        try:
            loaded_json = readback_json_object(
                artifact_path
            )

            parse_passed = True

            role_matches = bool(
                loaded_json.get(
                    "artifact_role"
                )
                == artifact_role
            )

            producer_matches = bool(
                loaded_json.get(
                    "producer"
                )
                == NOTEBOOK_NAME
            )

            run_matches = bool(
                loaded_json.get(
                    "v0_1_run_id"
                )
                == V01_RUN_ID
            )

            terminal_matches = bool(
                loaded_json.get(
                    "terminal_status"
                )
                == NOTEBOOK07_TERMINAL_STATUS
            )

            selected_model_value = loaded_json.get(
                "selected_model_id"
            )

            selected_model_matches = bool(
                selected_model_value
                in (
                    None,
                    SELECTED_HAWKES_MODEL_ID,
                )
            )

            components_present = bool(
                isinstance(
                    loaded_json.get(
                        "components"
                    ),
                    dict,
                )
                and len(
                    loaded_json[
                        "components"
                    ]
                )
                > 0
            )

            partitions_closed = (
                protected_partition_map_is_closed(
                    loaded_json.get(
                        "protected_partition_content_loaded"
                    )
                )
            )

            status_passed = bool(
                str(
                    loaded_json.get(
                        "status",
                        "",
                    )
                ).startswith("PASS")
            )

            semantic_passed = bool(
                role_matches
                and producer_matches
                and run_matches
                and terminal_matches
                and selected_model_matches
                and components_present
                and partitions_closed
                and status_passed
            )

            detail = (
                "role="
                + str(role_matches)
                + "; producer="
                + str(producer_matches)
                + "; run="
                + str(run_matches)
                + "; terminal="
                + str(terminal_matches)
                + "; components="
                + str(components_present)
            )

        except Exception as exception:
            detail = (
                type(exception).__name__
                + ": "
                + str(exception)
            )

    elif file_exists and artifact_format == "CSV":
        try:
            loaded_csv = readback_csv_frame(
                artifact_path
            )

            parse_passed = True

            observed_row_count = len(
                loaded_csv
            )

            observed_column_count = len(
                loaded_csv.columns
            )

            expected_row_count = (
                expected_optional_integer(
                    persisted_row.row_count
                )
            )

            expected_column_count = (
                expected_optional_integer(
                    persisted_row.column_count
                )
            )

            row_count_matches = bool(
                expected_row_count is None
                or observed_row_count
                == expected_row_count
            )

            column_count_matches = bool(
                expected_column_count is None
                or observed_column_count
                == expected_column_count
            )

            unique_columns = bool(
                loaded_csv.columns.is_unique
            )

            semantic_passed = bool(
                row_count_matches
                and column_count_matches
                and unique_columns
            )

            detail = (
                "rows="
                + str(observed_row_count)
                + "; columns="
                + str(observed_column_count)
                + "; expected_rows="
                + str(expected_row_count)
                + "; expected_columns="
                + str(expected_column_count)
            )

        except Exception as exception:
            detail = (
                type(exception).__name__
                + ": "
                + str(exception)
            )

    elif file_exists and artifact_format == "NPZ":
        try:
            with np.load(
                artifact_path,
                allow_pickle=False,
            ) as loaded_npz:
                loaded_keys = set(
                    loaded_npz.files
                )

                required_npz_keys = {
                    "parameter_names",
                    "parameter_values",
                    "branching_matrix",
                    "kernel_amplitude_matrix_per_second",
                    "optimizer_vector",
                    "development_terminal_state",
                    "development_terminal_state_time_ns",
                }

                keys_match = bool(
                    required_npz_keys.issubset(
                        loaded_keys
                    )
                )

                parameter_values_match = bool(
                    np.array_equal(
                        loaded_npz[
                            "parameter_values"
                        ],
                        selected_model_npz_arrays[
                            "parameter_values"
                        ],
                    )
                )

                branching_matrix_matches = bool(
                    np.array_equal(
                        loaded_npz[
                            "branching_matrix"
                        ],
                        PRIMARY_BRANCHING_MATRIX,
                    )
                )

                amplitude_matrix_matches = bool(
                    np.array_equal(
                        loaded_npz[
                            "kernel_amplitude_matrix_per_second"
                        ],
                        PRIMARY_KERNEL_AMPLITUDE_MATRIX_PER_SECOND,
                    )
                )

                optimizer_vector_matches = bool(
                    np.array_equal(
                        loaded_npz[
                            "optimizer_vector"
                        ],
                        SELECTED_HAWKES_FINAL_OPTIMIZER_VECTOR,
                    )
                )

                numeric_arrays_finite = bool(
                    np.isfinite(
                        loaded_npz[
                            "parameter_values"
                        ]
                    ).all()
                    and np.isfinite(
                        loaded_npz[
                            "branching_matrix"
                        ]
                    ).all()
                    and np.isfinite(
                        loaded_npz[
                            "kernel_amplitude_matrix_per_second"
                        ]
                    ).all()
                    and np.isfinite(
                        loaded_npz[
                            "optimizer_vector"
                        ]
                    ).all()
                    and np.isfinite(
                        loaded_npz[
                            "development_terminal_state"
                        ]
                    ).all()
                )

                parse_passed = True

                semantic_passed = bool(
                    keys_match
                    and parameter_values_match
                    and branching_matrix_matches
                    and amplitude_matrix_matches
                    and optimizer_vector_matches
                    and numeric_arrays_finite
                )

                detail = (
                    "keys="
                    + str(keys_match)
                    + "; parameters="
                    + str(parameter_values_match)
                    + "; branching="
                    + str(branching_matrix_matches)
                    + "; amplitudes="
                    + str(amplitude_matrix_matches)
                )

        except Exception as exception:
            detail = (
                type(exception).__name__
                + ": "
                + str(exception)
            )

    else:
        detail = (
            "Unsupported or missing artifact format: "
            + artifact_format
        )

    append_readback_record(
        semantic_readback_records,
        artifact_role=artifact_role,
        component_name=component_name,
        artifact_format=artifact_format,
        relative_path=relative_path,
        expected_sha256=expected_sha256,
        observed_sha256=observed_sha256,
        file_exists=file_exists,
        byte_hash_matches=byte_hash_matches,
        parse_passed=parse_passed,
        semantic_passed=semantic_passed,
        observed_row_count=observed_row_count,
        observed_column_count=observed_column_count,
        detail=detail,
    )


# ------------------------------------------------------------
# Semantic readback of the staging persistence index
# ------------------------------------------------------------

persistence_index_loaded = readback_json_object(
    NOTEBOOK07_PERSISTENCE_INDEX_PATH
)

persistence_index_semantic_sha256_readback = (
    canonical_json_sha256(
        persistence_index_loaded
    )
)

persistence_index_role_set = set(
    persistence_index_loaded.get(
        "core_required_roles",
        [],
    )
)

persistence_index_semantic_passed = bool(
    persistence_index_loaded.get(
        "terminal_status"
    )
    == NOTEBOOK07_TERMINAL_STATUS
    and persistence_index_loaded.get(
        "terminal_decision_sha256"
    )
    == NOTEBOOK07_TERMINAL_DECISION_SHA256
    and persistence_index_loaded.get(
        "selected_model_id"
    )
    == SELECTED_HAWKES_MODEL_ID
    and persistence_index_loaded.get(
        "persisted_core_role_count"
    )
    == len(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    )
    and persistence_index_loaded.get(
        "persisted_file_count"
    )
    == len(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY
    )
    and persistence_index_role_set
    == set(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    )
    and persistence_index_semantic_sha256_readback
    == NOTEBOOK07_PERSISTENCE_INDEX_SEMANTIC_SHA256
    and protected_partition_map_is_closed(
        persistence_index_loaded.get(
            "protected_partition_content_loaded"
        )
    )
)

append_readback_record(
    semantic_readback_records,
    artifact_role="persistence_index",
    component_name="persistence_index",
    artifact_format="JSON",
    relative_path=str(
        NOTEBOOK07_PERSISTENCE_INDEX_PATH.relative_to(
            V01_PERSISTENCE_ROOT
        )
    ),
    expected_sha256=(
        NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256
    ),
    observed_sha256=sha256_file(
        NOTEBOOK07_PERSISTENCE_INDEX_PATH
    ),
    file_exists=(
        NOTEBOOK07_PERSISTENCE_INDEX_PATH.is_file()
    ),
    byte_hash_matches=(
        sha256_file(
            NOTEBOOK07_PERSISTENCE_INDEX_PATH
        )
        == NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256
    ),
    parse_passed=True,
    semantic_passed=(
        persistence_index_semantic_passed
    ),
    detail=(
        "core_roles="
        + str(
            len(
                persistence_index_role_set
            )
        )
        + "; persisted_files="
        + str(
            persistence_index_loaded.get(
                "persisted_file_count"
            )
        )
    ),
)


# ------------------------------------------------------------
# Readback of staging CSV ledgers
# ------------------------------------------------------------

staging_csv_expectations: Final[
    tuple[
        tuple[str, str, Path, pd.DataFrame],
        ...,
    ]
] = (
    (
        "persistence_registry",
        "persisted_file_registry",
        NOTEBOOK07_PERSISTED_FILE_REGISTRY_PATH,
        NOTEBOOK07_PERSISTED_FILE_REGISTRY,
    ),
    (
        "immediate_verification",
        "immediate_persistence_verification",
        NOTEBOOK07_IMMEDIATE_VERIFICATION_PATH,
        NOTEBOOK07_IMMEDIATE_PERSISTENCE_VERIFICATION,
    ),
    (
        "v0_0_post_persistence_audit",
        "v0_0_post_persistence_immutability_audit",
        NOTEBOOK07_V00_POST_PERSISTENCE_AUDIT_PATH,
        NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT,
    ),
)

for (
    staging_role,
    staging_component,
    staging_path,
    expected_frame,
) in staging_csv_expectations:
    staging_exists = staging_path.is_file()

    staging_sha256 = (
        sha256_file(
            staging_path
        )
        if staging_exists
        else None
    )

    staging_parse_passed = False
    staging_semantic_passed = False
    staging_row_count: int | None = None
    staging_column_count: int | None = None
    staging_detail = ""

    if staging_exists:
        try:
            staging_frame = readback_csv_frame(
                staging_path
            )

            staging_parse_passed = True
            staging_row_count = len(
                staging_frame
            )
            staging_column_count = len(
                staging_frame.columns
            )

            staging_semantic_passed = bool(
                staging_row_count
                == len(
                    expected_frame
                )
                and staging_column_count
                == len(
                    expected_frame.columns
                )
                and list(
                    staging_frame.columns
                )
                == list(
                    expected_frame.columns
                )
            )

            staging_detail = (
                "rows="
                + str(staging_row_count)
                + "; columns="
                + str(staging_column_count)
            )

        except Exception as exception:
            staging_detail = (
                type(exception).__name__
                + ": "
                + str(exception)
            )

    append_readback_record(
        semantic_readback_records,
        artifact_role=staging_role,
        component_name=staging_component,
        artifact_format="CSV",
        relative_path=str(
            staging_path.relative_to(
                V01_PERSISTENCE_ROOT
            )
        ),
        expected_sha256=(
            staging_sha256
            if staging_sha256 is not None
            else ""
        ),
        observed_sha256=staging_sha256,
        file_exists=staging_exists,
        byte_hash_matches=staging_exists,
        parse_passed=staging_parse_passed,
        semantic_passed=staging_semantic_passed,
        observed_row_count=staging_row_count,
        observed_column_count=staging_column_count,
        detail=staging_detail,
    )


# ------------------------------------------------------------
# Freeze semantic-readback audit
# ------------------------------------------------------------

NOTEBOOK07_SEMANTIC_READBACK_AUDIT = pd.DataFrame(
    semantic_readback_records
)

require(
    len(
        NOTEBOOK07_SEMANTIC_READBACK_AUDIT
    )
    == (
        len(
            NOTEBOOK07_PERSISTED_FILE_REGISTRY
        )
        + 4
    ),
    (
        "The semantic-readback audit has an unexpected "
        "number of rows."
    ),
)

require(
    NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
        "passed"
    ].all(),
    "At least one persisted artifact failed semantic readback.",
)

require(
    NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
        "relative_path"
    ].is_unique,
    "The semantic-readback audit contains duplicate paths.",
)

readback_role_coverage = set(
    NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
        "artifact_role"
    ]
)

require(
    set(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ).issubset(
        readback_role_coverage
    ),
    "Semantic readback did not cover every core role.",
)

NOTEBOOK07_SEMANTIC_READBACK_SUMMARY = pd.DataFrame(
    [
        {
            "field": "core_file_count",
            "value": len(
                NOTEBOOK07_PERSISTED_FILE_REGISTRY
            ),
        },
        {
            "field": "staging_file_count",
            "value": 4,
        },
        {
            "field": "semantic_readback_row_count",
            "value": len(
                NOTEBOOK07_SEMANTIC_READBACK_AUDIT
            ),
        },
        {
            "field": "all_files_exist",
            "value": (
                NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
                    "file_exists"
                ].all()
            ),
        },
        {
            "field": "all_byte_hashes_match",
            "value": (
                NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
                    "byte_hash_matches"
                ].all()
            ),
        },
        {
            "field": "all_artifacts_parse",
            "value": (
                NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
                    "parse_passed"
                ].all()
            ),
        },
        {
            "field": "all_semantic_checks_pass",
            "value": (
                NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
                    "semantic_passed"
                ].all()
            ),
        },
        {
            "field": "calibration_parameter_updates",
            "value": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": (
                "engineering_holdout_content_loaded"
            ),
            "value": False,
        },
        {
            "field": "status",
            "value": "PASS",
        },
    ]
)

NOTEBOOK07_SEMANTIC_READBACK_PAYLOAD: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_HAWKES_SEMANTIC_READBACK_AUDIT"
    ),
    "schema_version": (
        "NOTEBOOK_07_HAWKES_SEMANTIC_READBACK_AUDIT_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
    "terminal_status": (
        NOTEBOOK07_TERMINAL_STATUS
    ),
    "terminal_decision_sha256": (
        NOTEBOOK07_TERMINAL_DECISION_SHA256
    ),
    "persistence_index_file_sha256": (
        NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256
    ),
    "core_persisted_file_count": len(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY
    ),
    "staging_file_count": 4,
    "readback_record_count": len(
        NOTEBOOK07_SEMANTIC_READBACK_AUDIT
    ),
    "all_files_exist": True,
    "all_hashes_match": True,
    "all_parse_checks_passed": True,
    "all_semantic_checks_passed": True,
    "records": persistence_json_safe(
        NOTEBOOK07_SEMANTIC_READBACK_AUDIT
    )["records"],
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "status": "PASS_SEMANTIC_READBACK",
}

NOTEBOOK07_SEMANTIC_READBACK_SHA256: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_SEMANTIC_READBACK_PAYLOAD
)


# ------------------------------------------------------------
# Persist and verify readback artifacts
# ------------------------------------------------------------

readback_json_bytes = atomic_write_json(
    NOTEBOOK07_READBACK_AUDIT_JSON_PATH,
    NOTEBOOK07_SEMANTIC_READBACK_PAYLOAD,
)

readback_csv_bytes = atomic_write_dataframe_csv(
    NOTEBOOK07_READBACK_AUDIT_CSV_PATH,
    NOTEBOOK07_SEMANTIC_READBACK_AUDIT,
)

NOTEBOOK07_READBACK_AUDIT_JSON_FILE_SHA256: Final[
    str
] = sha256_bytes(
    readback_json_bytes
)

NOTEBOOK07_READBACK_AUDIT_CSV_FILE_SHA256: Final[
    str
] = sha256_bytes(
    readback_csv_bytes
)

require(
    sha256_file(
        NOTEBOOK07_READBACK_AUDIT_JSON_PATH
    )
    == NOTEBOOK07_READBACK_AUDIT_JSON_FILE_SHA256,
    "The persisted readback JSON failed hash verification.",
)

require(
    sha256_file(
        NOTEBOOK07_READBACK_AUDIT_CSV_PATH
    )
    == NOTEBOOK07_READBACK_AUDIT_CSV_FILE_SHA256,
    "The persisted readback CSV failed hash verification.",
)

readback_json_loaded = readback_json_object(
    NOTEBOOK07_READBACK_AUDIT_JSON_PATH
)

require(
    canonical_json_sha256(
        readback_json_loaded
    )
    == NOTEBOOK07_SEMANTIC_READBACK_SHA256,
    "The persisted readback payload failed semantic verification.",
)


# ------------------------------------------------------------
# Notebook 07-to-Notebook 08 diagnostics handoff
# ------------------------------------------------------------

NOTEBOOK08_REQUIRED_DIAGNOSTICS: Final[
    tuple[str, ...]
] = (
    "RECONCILE_D0_AND_EWMA_BASELINE_SCORES",
    "RUN_PAIRED_DEVELOPMENT_AND_LOCKED_CALIBRATION_COMPARISONS",
    "RUN_TIME_REScaling_OR_EQUIVALENT_POINT_PROCESS_RESIDUAL_DIAGNOSTICS",
    "RETAIN_STRICT_PRE_BATCH_MULTIPLICITY_SEMANTICS",
    "AUDIT_BUY_AND_SELL_RESIDUALS_SEPARATELY",
    "AUDIT_CALIBRATION_SCORE_DEGRADATION",
    "RETAIN_LEFT_EDGE_AND_BURN_IN_SENSITIVITY",
    "RETAIN_ONE_MILLISECOND_TIMESTAMP_SENSITIVITY",
    "TEST_PARAMETER_AND_RESIDUAL_STABILITY_WITHOUT_CALIBRATION_REFIT",
    "WITHHOLD_HAWKES_SUPERIORITY_UNTIL_ALL_DIAGNOSTIC_GATES_PASS",
)

NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PAYLOAD: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_TO_NOTEBOOK_08_HAWKES_HANDOFF"
    ),
    "schema_version": (
        "NOTEBOOK_07_TO_NOTEBOOK_08_HAWKES_HANDOFF_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "consumer": (
        "08_HAWKES_DIAGNOSTICS.ipynb"
    ),
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
    "notebook07_terminal_status": (
        NOTEBOOK07_TERMINAL_STATUS
    ),
    "terminal_decision_sha256": (
        NOTEBOOK07_TERMINAL_DECISION_SHA256
    ),
    "semantic_readback_sha256": (
        NOTEBOOK07_SEMANTIC_READBACK_SHA256
    ),
    "selected_model": {
        "model_id": SELECTED_HAWKES_MODEL_ID,
        "model_family": AUTHORIZED_FIRST_MODEL,
        "kernel_family": AUTHORIZED_KERNEL_FAMILY,
        "estimator_label": (
            PRIMARY_ESTIMATOR_LABEL
        ),
        "primary_event_representation": (
            PRIMARY_EVENT_REPRESENTATION
        ),
        "primary_timestamp_column": (
            PRIMARY_EVENT_TIME_COLUMN
        ),
        "shared_decay": True,
        "cross_excitation_fixed_zero": True,
        "selected_model_package_sha256": (
            SELECTED_HAWKES_MODEL_PACKAGE_SHA256
        ),
    },
    "frozen_parameters": {
        "mu_buy_per_second": (
            SELECTED_HAWKES_PARAMETERS.mu_buy
        ),
        "mu_sell_per_second": (
            SELECTED_HAWKES_PARAMETERS.mu_sell
        ),
        "kappa_buy_to_buy": (
            SELECTED_HAWKES_PARAMETERS.kappa_buy
        ),
        "kappa_sell_to_sell": (
            SELECTED_HAWKES_PARAMETERS.kappa_sell
        ),
        "kappa_buy_to_sell": 0.0,
        "kappa_sell_to_buy": 0.0,
        "beta_buy_per_second": (
            SELECTED_HAWKES_PARAMETERS.beta_buy
        ),
        "beta_sell_per_second": (
            SELECTED_HAWKES_PARAMETERS.beta_sell
        ),
        "buy_half_life_seconds": (
            selected_half_life_buy_seconds
        ),
        "sell_half_life_seconds": (
            selected_half_life_sell_seconds
        ),
        "spectral_radius": float(
            selected_final_validation[
                "spectral_radius"
            ]
        ),
    },
    "primary_scores": {
        "development_log_score_per_event": (
            DEVELOPMENT_PRIMARY_LOG_SCORE_PER_EVENT
        ),
        "chronological_validation_weighted_log_score_per_event": (
            SELECTED_HAWKES_WEIGHTED_VALIDATION_LOG_SCORE_PER_EVENT
        ),
        "locked_calibration_log_score_per_event": (
            CALIBRATION_PRIMARY_LOG_SCORE_PER_EVENT
        ),
        "calibration_minus_development_log_score_per_event": (
            CALIBRATION_PRIMARY_LOG_SCORE_PER_EVENT
            - DEVELOPMENT_PRIMARY_LOG_SCORE_PER_EVENT
        ),
        "calibration_parameter_updates": 0,
    },
    "history_contract": {
        "development_left_censored": True,
        "fabricated_prehistory": False,
        "calibration_initial_state_source": (
            "TERMINAL_DEVELOPMENT_STATE"
        ),
        "calibration_reset_authorized": False,
        "strict_pre_batch_scoring": True,
        "within_batch_zero_lag_excitation": False,
    },
    "baseline_candidates": list(
        AUTHORIZED_BASELINES
        if "AUTHORIZED_BASELINES" in globals()
        else BASELINE_MODEL_IDS
        if "BASELINE_MODEL_IDS" in globals()
        else (
            "D0",
            "EWMA",
        )
    ),
    "required_diagnostics": list(
        NOTEBOOK08_REQUIRED_DIAGNOSTICS
    ),
    "required_artifact_paths": {
        role: str(
            path.relative_to(
                V01_PERSISTENCE_ROOT
            )
        )
        for role, path in (
            NOTEBOOK07_PERSISTED_ROLE_PACKAGE_PATHS.items()
        )
    },
    "readback_audit_path": str(
        NOTEBOOK07_READBACK_AUDIT_JSON_PATH.relative_to(
            V01_PERSISTENCE_ROOT
        )
    ),
    "output_manifest_path": str(
        NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH.relative_to(
            V01_PERSISTENCE_ROOT
        )
    ),
    "claim_limits": {
        "hawkes_superiority_authorized": False,
        "baseline_superiority_authorized": False,
        "unrestricted_cross_excitation_authorized": False,
        "state_dependent_hawkes_authorized": False,
        "directional_price_signal_authorized": False,
        "strategy_or_quoting_authorized": False,
        "fill_simulation_authorized": False,
        "pnl_claim_authorized": False,
        "live_trading_authorized": False,
    },
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "authorization_scope": (
        "NOTEBOOK_08_DIAGNOSTICS_ONLY"
    ),
    "notebook08_authorized_after_manifest_verification": True,
    "status": (
        "PASS_HANDOFF_FOR_HAWKES_DIAGNOSTICS"
    ),
}

NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_SHA256: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PAYLOAD
)

handoff_bytes = atomic_write_json(
    NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PATH,
    NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PAYLOAD,
)

NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_FILE_SHA256: Final[
    str
] = sha256_bytes(
    handoff_bytes
)

require(
    sha256_file(
        NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PATH
    )
    == NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_FILE_SHA256,
    "The Notebook 08 handoff failed file-hash verification.",
)

handoff_loaded = readback_json_object(
    NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PATH
)

require(
    canonical_json_sha256(
        handoff_loaded
    )
    == NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_SHA256,
    "The Notebook 08 handoff failed semantic verification.",
)


# ------------------------------------------------------------
# Final Notebook 07 acceptance
# ------------------------------------------------------------

NOTEBOOK07_FINAL_ACCEPTANCE: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_HAWKES_ESTIMATION_FINAL_ACCEPTANCE"
    ),
    "schema_version": (
        "NOTEBOOK_07_HAWKES_ESTIMATION_FINAL_ACCEPTANCE_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
    "terminal_status": (
        NOTEBOOK07_TERMINAL_STATUS
    ),
    "terminal_decision_sha256": (
        NOTEBOOK07_TERMINAL_DECISION_SHA256
    ),
    "selected_model_id": (
        SELECTED_HAWKES_MODEL_ID
    ),
    "selected_model_package_sha256": (
        SELECTED_HAWKES_MODEL_PACKAGE_SHA256
    ),
    "parameter_and_evaluation_package_sha256": (
        NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256
    ),
    "persistence_index_file_sha256": (
        NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256
    ),
    "semantic_readback_sha256": (
        NOTEBOOK07_SEMANTIC_READBACK_SHA256
    ),
    "notebook07_to_notebook08_handoff_sha256": (
        NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_SHA256
    ),
    "blocking_failure_count": 0,
    "warning_count": (
        NOTEBOOK07_TERMINAL_WARNING_COUNT
    ),
    "core_persistence_completed": True,
    "core_persisted_role_count": len(
        NOTEBOOK07_CORE_PERSISTENCE_ROLES
    ),
    "core_persisted_file_count": len(
        NOTEBOOK07_PERSISTED_FILE_REGISTRY
    ),
    "semantic_readback_completed": True,
    "semantic_readback_passed": True,
    "v0_0_files_unchanged": bool(
        NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT[
            "unchanged"
        ].all()
    ),
    "v0_1_selected_model_unchanged": True,
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "output_manifest_path": str(
        NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH.relative_to(
            V01_PERSISTENCE_ROOT
        )
    ),
    "output_manifest_completion_condition": (
        "THIS_ACCEPTANCE_BECOMES_EFFECTIVE_AFTER_OUTPUT_"
        "MANIFEST_FILE_HASH_VERIFICATION"
    ),
    "notebook08_handoff_completed": True,
    "notebook08_authorized_scope": (
        "HAWKES_DIAGNOSTICS_ONLY"
    ),
    "claim_limits": {
        "hawkes_superiority_authorized": False,
        "baseline_superiority_authorized": False,
        "cross_excitation_authorized": False,
        "state_dependent_hawkes_authorized": False,
        "strategy_or_execution_authorized": False,
        "pnl_claim_authorized": False,
    },
    "next_operation": (
        "BEGIN_NOTEBOOK08_HAWKES_DIAGNOSTICS"
    ),
    "status": (
        "PASS_NOTEBOOK07_COMPLETE_PENDING_MANIFEST_"
        "FILE_VERIFICATION"
    ),
}

NOTEBOOK07_FINAL_ACCEPTANCE_SHA256: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_FINAL_ACCEPTANCE
)

final_acceptance_bytes = atomic_write_json(
    NOTEBOOK07_FINAL_ACCEPTANCE_PATH,
    NOTEBOOK07_FINAL_ACCEPTANCE,
)

NOTEBOOK07_FINAL_ACCEPTANCE_FILE_SHA256: Final[
    str
] = sha256_bytes(
    final_acceptance_bytes
)

require(
    sha256_file(
        NOTEBOOK07_FINAL_ACCEPTANCE_PATH
    )
    == NOTEBOOK07_FINAL_ACCEPTANCE_FILE_SHA256,
    "The final acceptance file failed hash verification.",
)

final_acceptance_loaded = readback_json_object(
    NOTEBOOK07_FINAL_ACCEPTANCE_PATH
)

require(
    canonical_json_sha256(
        final_acceptance_loaded
    )
    == NOTEBOOK07_FINAL_ACCEPTANCE_SHA256,
    "The final acceptance failed semantic verification.",
)


# ------------------------------------------------------------
# Build the pre-manifest file registry
# ------------------------------------------------------------

final_manifest_records: list[
    dict[str, Any]
] = persistence_json_safe(
    NOTEBOOK07_PERSISTED_FILE_REGISTRY
)["records"]


def append_final_manifest_record(
    *,
    artifact_role: str,
    component_name: str,
    artifact_format: str,
    path: Path,
    file_sha256: str,
    semantic_sha256: str | None = None,
    row_count: int | None = None,
    column_count: int | None = None,
) -> None:
    """Append one non-manifest artifact to the final manifest."""
    final_manifest_records.append(
        {
            "artifact_role": artifact_role,
            "component_name": component_name,
            "artifact_format": artifact_format,
            "relative_path": str(
                path.relative_to(
                    V01_PERSISTENCE_ROOT
                )
            ),
            "absolute_path": str(path),
            "byte_count": int(
                path.stat().st_size
            ),
            "sha256": file_sha256,
            "semantic_sha256": (
                semantic_sha256
            ),
            "row_count": (
                row_count
                if row_count is not None
                else None
            ),
            "column_count": (
                column_count
                if column_count is not None
                else None
            ),
            "producer": NOTEBOOK_NAME,
            "source_run_prefix": SOURCE_RUN_PREFIX,
            "v0_1_run_id": V01_RUN_ID,
            "status": "PERSISTED_AND_VERIFIED",
        }
    )


append_final_manifest_record(
    artifact_role="persistence_index",
    component_name="persistence_index",
    artifact_format="JSON",
    path=NOTEBOOK07_PERSISTENCE_INDEX_PATH,
    file_sha256=(
        NOTEBOOK07_PERSISTENCE_INDEX_FILE_SHA256
    ),
    semantic_sha256=(
        NOTEBOOK07_PERSISTENCE_INDEX_SEMANTIC_SHA256
    ),
)

for (
    staging_role,
    staging_component,
    staging_path,
    staging_frame,
) in staging_csv_expectations:
    append_final_manifest_record(
        artifact_role=staging_role,
        component_name=staging_component,
        artifact_format="CSV",
        path=staging_path,
        file_sha256=sha256_file(
            staging_path
        ),
        row_count=len(
            staging_frame
        ),
        column_count=len(
            staging_frame.columns
        ),
    )

append_final_manifest_record(
    artifact_role="readback_audit",
    component_name="semantic_readback_package",
    artifact_format="JSON",
    path=NOTEBOOK07_READBACK_AUDIT_JSON_PATH,
    file_sha256=(
        NOTEBOOK07_READBACK_AUDIT_JSON_FILE_SHA256
    ),
    semantic_sha256=(
        NOTEBOOK07_SEMANTIC_READBACK_SHA256
    ),
)

append_final_manifest_record(
    artifact_role="readback_audit",
    component_name="semantic_readback_table",
    artifact_format="CSV",
    path=NOTEBOOK07_READBACK_AUDIT_CSV_PATH,
    file_sha256=(
        NOTEBOOK07_READBACK_AUDIT_CSV_FILE_SHA256
    ),
    row_count=len(
        NOTEBOOK07_SEMANTIC_READBACK_AUDIT
    ),
    column_count=len(
        NOTEBOOK07_SEMANTIC_READBACK_AUDIT.columns
    ),
)

append_final_manifest_record(
    artifact_role=(
        "notebook07_to_notebook08_handoff"
    ),
    component_name=(
        "notebook07_to_notebook08_handoff"
    ),
    artifact_format="JSON",
    path=(
        NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PATH
    ),
    file_sha256=(
        NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_FILE_SHA256
    ),
    semantic_sha256=(
        NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_SHA256
    ),
)

append_final_manifest_record(
    artifact_role="final_acceptance",
    component_name="final_acceptance",
    artifact_format="JSON",
    path=NOTEBOOK07_FINAL_ACCEPTANCE_PATH,
    file_sha256=(
        NOTEBOOK07_FINAL_ACCEPTANCE_FILE_SHA256
    ),
    semantic_sha256=(
        NOTEBOOK07_FINAL_ACCEPTANCE_SHA256
    ),
)

NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY = pd.DataFrame(
    final_manifest_records
)

require(
    NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY[
        "relative_path"
    ].is_unique,
    "The pre-manifest file registry contains duplicate paths.",
)

require(
    NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY[
        "absolute_path"
    ].is_unique,
    "The pre-manifest file registry contains duplicate absolute paths.",
)

pre_manifest_required_role_coverage = set(
    NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY[
        "artifact_role"
    ]
)

expected_pre_manifest_roles = (
    set(
        NOTEBOOK07_REQUIRED_OUTPUT_ROLES
    )
    - {
        "output_manifest",
    }
)

require(
    expected_pre_manifest_roles.issubset(
        pre_manifest_required_role_coverage
    ),
    (
        "The pre-manifest registry does not cover all required "
        "non-manifest output roles."
    ),
)


# ------------------------------------------------------------
# Final output manifest
#
# The manifest lists every artifact except its own JSON and CSV
# files. Its own hashes are verified externally immediately after
# persistence and included in the in-memory final file registry.
# ------------------------------------------------------------

NOTEBOOK07_FINAL_OUTPUT_MANIFEST_PAYLOAD: Final[
    dict[str, Any]
] = {
    "artifact_type": (
        "NOTEBOOK_07_HAWKES_ESTIMATION_OUTPUT_MANIFEST"
    ),
    "schema_version": (
        "NOTEBOOK_07_HAWKES_ESTIMATION_OUTPUT_MANIFEST_V1"
    ),
    "producer": NOTEBOOK_NAME,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
    "terminal_status": (
        NOTEBOOK07_TERMINAL_STATUS
    ),
    "terminal_decision_sha256": (
        NOTEBOOK07_TERMINAL_DECISION_SHA256
    ),
    "selected_model_id": (
        SELECTED_HAWKES_MODEL_ID
    ),
    "selected_model_package_sha256": (
        SELECTED_HAWKES_MODEL_PACKAGE_SHA256
    ),
    "parameter_and_evaluation_package_sha256": (
        NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256
    ),
    "semantic_readback_sha256": (
        NOTEBOOK07_SEMANTIC_READBACK_SHA256
    ),
    "final_acceptance_sha256": (
        NOTEBOOK07_FINAL_ACCEPTANCE_SHA256
    ),
    "notebook07_to_notebook08_handoff_sha256": (
        NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_SHA256
    ),
    "required_output_roles": list(
        NOTEBOOK07_REQUIRED_OUTPUT_ROLES
    ),
    "required_role_count": len(
        NOTEBOOK07_REQUIRED_OUTPUT_ROLES
    ),
    "non_manifest_file_count": len(
        NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY
    ),
    "non_manifest_total_byte_count": int(
        NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY[
            "byte_count"
        ].sum()
    ),
    "files": persistence_json_safe(
        NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY
    )["records"],
    "output_manifest_self_reference_policy": {
        "manifest_json_listed_inside_itself": False,
        "manifest_csv_listed_inside_itself": False,
        "reason": (
            "AVOID_RECURSIVE_SELF_HASH_DEPENDENCY"
        ),
        "self_files_verified_after_write": True,
    },
    "calibration_parameter_updates": 0,
    "protected_partition_content_loaded": {
        partition: False
        for partition in PROTECTED_PARTITIONS
    },
    "notebook08_authorization_scope": (
        "HAWKES_DIAGNOSTICS_ONLY"
    ),
    "claim_limits": {
        "hawkes_superiority_authorized": False,
        "baseline_superiority_authorized": False,
        "cross_excitation_authorized": False,
        "state_dependent_hawkes_authorized": False,
        "strategy_or_execution_authorized": False,
        "pnl_claim_authorized": False,
    },
    "next_operation": (
        "BEGIN_NOTEBOOK08_HAWKES_DIAGNOSTICS"
    ),
    "status": (
        "PASS_COMPLETE_OUTPUT_MANIFEST"
    ),
}

NOTEBOOK07_FINAL_OUTPUT_MANIFEST_SEMANTIC_SHA256: Final[
    str
] = canonical_json_sha256(
    NOTEBOOK07_FINAL_OUTPUT_MANIFEST_PAYLOAD
)

final_manifest_json_bytes = atomic_write_json(
    NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH,
    NOTEBOOK07_FINAL_OUTPUT_MANIFEST_PAYLOAD,
)

final_manifest_csv_bytes = atomic_write_dataframe_csv(
    NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH,
    NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY,
)

NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_FILE_SHA256: Final[
    str
] = sha256_bytes(
    final_manifest_json_bytes
)

NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_FILE_SHA256: Final[
    str
] = sha256_bytes(
    final_manifest_csv_bytes
)

require(
    sha256_file(
        NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH
    )
    == NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_FILE_SHA256,
    "The output-manifest JSON failed file-hash verification.",
)

require(
    sha256_file(
        NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH
    )
    == NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_FILE_SHA256,
    "The output-manifest CSV failed file-hash verification.",
)

final_manifest_loaded = readback_json_object(
    NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH
)

require(
    canonical_json_sha256(
        final_manifest_loaded
    )
    == NOTEBOOK07_FINAL_OUTPUT_MANIFEST_SEMANTIC_SHA256,
    "The output manifest failed semantic verification.",
)

require(
    final_manifest_loaded.get(
        "status"
    )
    == "PASS_COMPLETE_OUTPUT_MANIFEST",
    "The persisted output manifest has the wrong status.",
)


# ------------------------------------------------------------
# Complete final file registry, including manifest self-files
# ------------------------------------------------------------

NOTEBOOK07_FINAL_FILE_REGISTRY = (
    NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY.copy()
)

manifest_self_records = pd.DataFrame(
    [
        {
            "artifact_role": "output_manifest",
            "component_name": "output_manifest_package",
            "artifact_format": "JSON",
            "relative_path": str(
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH.relative_to(
                    V01_PERSISTENCE_ROOT
                )
            ),
            "absolute_path": str(
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH
            ),
            "byte_count": int(
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH.stat().st_size
            ),
            "sha256": (
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_FILE_SHA256
            ),
            "semantic_sha256": (
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_SEMANTIC_SHA256
            ),
            "row_count": pd.NA,
            "column_count": pd.NA,
            "producer": NOTEBOOK_NAME,
            "source_run_prefix": SOURCE_RUN_PREFIX,
            "v0_1_run_id": V01_RUN_ID,
            "status": "PERSISTED_AND_VERIFIED",
        },
        {
            "artifact_role": "output_manifest",
            "component_name": "output_manifest_table",
            "artifact_format": "CSV",
            "relative_path": str(
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH.relative_to(
                    V01_PERSISTENCE_ROOT
                )
            ),
            "absolute_path": str(
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH
            ),
            "byte_count": int(
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH.stat().st_size
            ),
            "sha256": (
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_FILE_SHA256
            ),
            "semantic_sha256": pd.NA,
            "row_count": len(
                NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY
            ),
            "column_count": len(
                NOTEBOOK07_PRE_MANIFEST_FILE_REGISTRY.columns
            ),
            "producer": NOTEBOOK_NAME,
            "source_run_prefix": SOURCE_RUN_PREFIX,
            "v0_1_run_id": V01_RUN_ID,
            "status": "PERSISTED_AND_VERIFIED",
        },
    ]
)

NOTEBOOK07_FINAL_FILE_REGISTRY = pd.concat(
    [
        NOTEBOOK07_FINAL_FILE_REGISTRY,
        manifest_self_records,
    ],
    axis=0,
    ignore_index=True,
)

require(
    NOTEBOOK07_FINAL_FILE_REGISTRY[
        "relative_path"
    ].is_unique,
    "The final file registry contains duplicate paths.",
)

require(
    set(
        NOTEBOOK07_REQUIRED_OUTPUT_ROLES
    ).issubset(
        set(
            NOTEBOOK07_FINAL_FILE_REGISTRY[
                "artifact_role"
            ]
        )
    ),
    "The final file registry does not cover all required roles.",
)


# ------------------------------------------------------------
# Final physical verification of every registered file
# ------------------------------------------------------------

final_verification_rows: list[
    dict[str, Any]
] = []

for final_row in (
    NOTEBOOK07_FINAL_FILE_REGISTRY
    .itertuples(index=False)
):
    final_path = Path(
        final_row.absolute_path
    )

    final_exists = final_path.is_file()

    final_observed_sha256 = (
        sha256_file(
            final_path
        )
        if final_exists
        else None
    )

    final_expected_sha256 = str(
        final_row.sha256
    )

    final_byte_count = (
        final_path.stat().st_size
        if final_exists
        else -1
    )

    expected_byte_count = int(
        final_row.byte_count
    )

    final_passed = bool(
        final_exists
        and final_observed_sha256
        == final_expected_sha256
        and final_byte_count
        == expected_byte_count
    )

    final_verification_rows.append(
        {
            "artifact_role": (
                final_row.artifact_role
            ),
            "component_name": (
                final_row.component_name
            ),
            "artifact_format": (
                final_row.artifact_format
            ),
            "relative_path": (
                final_row.relative_path
            ),
            "file_exists": final_exists,
            "expected_byte_count": (
                expected_byte_count
            ),
            "observed_byte_count": (
                final_byte_count
            ),
            "byte_count_matches": (
                final_byte_count
                == expected_byte_count
            ),
            "expected_sha256": (
                final_expected_sha256
            ),
            "observed_sha256": (
                final_observed_sha256
            ),
            "sha256_matches": (
                final_observed_sha256
                == final_expected_sha256
            ),
            "passed": final_passed,
            "status": (
                "PASS"
                if final_passed
                else "FAIL"
            ),
        }
    )

NOTEBOOK07_FINAL_FILE_VERIFICATION = pd.DataFrame(
    final_verification_rows
)

require(
    NOTEBOOK07_FINAL_FILE_VERIFICATION[
        "passed"
    ].all(),
    "At least one final Notebook 07 file failed verification.",
)


# ------------------------------------------------------------
# Verify V0.0 and in-memory V0.1 authority one final time
# ------------------------------------------------------------

require(
    NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT[
        "unchanged"
    ].all(),
    "An immutable V0.0 file changed.",
)

require(
    canonical_json_sha256(
        SELECTED_HAWKES_MODEL_PACKAGE
    )
    == SELECTED_HAWKES_MODEL_PACKAGE_SHA256,
    "The selected-model package changed.",
)

require(
    canonical_json_sha256(
        NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE
    )
    == NOTEBOOK07_PARAMETER_AND_EVALUATION_PACKAGE_SHA256,
    "The parameter-and-evaluation package changed.",
)

require(
    canonical_json_sha256(
        NOTEBOOK07_TERMINAL_DECISION_PAYLOAD
    )
    == NOTEBOOK07_TERMINAL_DECISION_SHA256,
    "The terminal decision changed.",
)

require(
    CALIBRATION_PARAMETER_UPDATES_PERFORMED == 0,
    "CALIBRATION parameters changed during finalization.",
)

require(
    not any(
        PROTECTED_PARTITION_CONTENT_LOADED[
            partition
        ]
        for partition in PROTECTED_PARTITIONS
    ),
    "A protected partition was opened during finalization.",
)


# ------------------------------------------------------------
# Final completion gates
# ------------------------------------------------------------

NOTEBOOK07_FINAL_COMPLETION_GATE_LEDGER = pd.DataFrame(
    [
        {
            "gate_id": "TERMINAL_DECISION_PASS",
            "passed": (
                NOTEBOOK07_TERMINAL_DECISION_PASSED
            ),
        },
        {
            "gate_id": "CORE_PERSISTENCE_COMPLETE",
            "passed": (
                NOTEBOOK07_PERSISTENCE_COMPLETED
            ),
        },
        {
            "gate_id": "CORE_FILE_COUNT_RECONCILED",
            "passed": (
                len(
                    NOTEBOOK07_PERSISTED_FILE_REGISTRY
                )
                == 61
            ),
        },
        {
            "gate_id": "SEMANTIC_READBACK_COMPLETE",
            "passed": True,
        },
        {
            "gate_id": "SEMANTIC_READBACK_PASS",
            "passed": (
                NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
                    "passed"
                ].all()
            ),
        },
        {
            "gate_id": "FINAL_ACCEPTANCE_PERSISTED",
            "passed": (
                NOTEBOOK07_FINAL_ACCEPTANCE_PATH
                .is_file()
            ),
        },
        {
            "gate_id": "OUTPUT_MANIFEST_PERSISTED",
            "passed": (
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH
                .is_file()
                and NOTEBOOK07_FINAL_OUTPUT_MANIFEST_CSV_PATH
                .is_file()
            ),
        },
        {
            "gate_id": "HANDOFF_PERSISTED",
            "passed": (
                NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PATH
                .is_file()
            ),
        },
        {
            "gate_id": "ALL_REQUIRED_ROLES_PRESENT",
            "passed": (
                set(
                    NOTEBOOK07_REQUIRED_OUTPUT_ROLES
                ).issubset(
                    set(
                        NOTEBOOK07_FINAL_FILE_REGISTRY[
                            "artifact_role"
                        ]
                    )
                )
            ),
        },
        {
            "gate_id": "ALL_FINAL_FILES_HASH_VERIFIED",
            "passed": (
                NOTEBOOK07_FINAL_FILE_VERIFICATION[
                    "passed"
                ].all()
            ),
        },
        {
            "gate_id": "SELECTED_MODEL_UNCHANGED",
            "passed": (
                canonical_json_sha256(
                    SELECTED_HAWKES_MODEL_PACKAGE
                )
                == SELECTED_HAWKES_MODEL_PACKAGE_SHA256
            ),
        },
        {
            "gate_id": "V0_0_FILES_UNCHANGED",
            "passed": (
                NOTEBOOK07_V00_POST_PERSISTENCE_IMMUTABILITY_AUDIT[
                    "unchanged"
                ].all()
            ),
        },
        {
            "gate_id": "CALIBRATION_ZERO_UPDATES",
            "passed": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
                == 0
            ),
        },
        {
            "gate_id": "PROTECTED_PARTITIONS_UNOPENED",
            "passed": (
                not any(
                    PROTECTED_PARTITION_CONTENT_LOADED[
                        partition
                    ]
                    for partition in PROTECTED_PARTITIONS
                )
            ),
        },
        {
            "gate_id": "HAWKES_SUPERIORITY_STILL_UNAUTHORIZED",
            "passed": (
                not HAWKES_SUPERIORITY_CLAIM_AUTHORIZED
            ),
        },
        {
            "gate_id": "NOTEBOOK08_HANDOFF_SCOPE_DIAGNOSTICS_ONLY",
            "passed": (
                NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_PAYLOAD[
                    "authorization_scope"
                ]
                == "NOTEBOOK_08_DIAGNOSTICS_ONLY"
            ),
        },
    ]
)

NOTEBOOK07_FINAL_COMPLETION_GATE_LEDGER[
    "status"
] = np.where(
    NOTEBOOK07_FINAL_COMPLETION_GATE_LEDGER[
        "passed"
    ],
    "PASS",
    "FAIL",
)

require(
    NOTEBOOK07_FINAL_COMPLETION_GATE_LEDGER[
        "passed"
    ].all(),
    "Notebook 07 final completion failed at least one gate.",
)


# ------------------------------------------------------------
# Final authoritative state
# ------------------------------------------------------------

NOTEBOOK07_READBACK_COMPLETED = True
NOTEBOOK07_READBACK_PASSED: bool = True

NOTEBOOK07_FINAL_ACCEPTANCE_COMPLETED = True
NOTEBOOK07_FINAL_OUTPUT_MANIFEST_COMPLETED = True

NOTEBOOK08_HANDOFF_COMPLETED = True
NOTEBOOK08_AUTHORIZED = True
NOTEBOOK08_AUTHORIZATION_SCOPE: Final[str] = (
    "HAWKES_DIAGNOSTICS_ONLY"
)

NOTEBOOK07_COMPLETED: bool = True
NOTEBOOK07_FINAL_STATUS: Final[str] = (
    "PASS_HAWKES_ESTIMATION_RESTRICTED_FIRST"
)

NOTEBOOK07_NEXT_OPERATION = (
    "BEGIN_NOTEBOOK08_HAWKES_DIAGNOSTICS"
)

HAWKES_SUPERIORITY_CLAIM_AUTHORIZED = False
BASELINE_SUPERIORITY_CLAIM_AUTHORIZED = False

FILESYSTEM_WRITES_PERFORMED = True


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

notebook07_final_summary = pd.DataFrame(
    [
        {
            "field": "notebook07_completed",
            "value": NOTEBOOK07_COMPLETED,
        },
        {
            "field": "final_status",
            "value": NOTEBOOK07_FINAL_STATUS,
        },
        {
            "field": "selected_model_id",
            "value": SELECTED_HAWKES_MODEL_ID,
        },
        {
            "field": "core_persisted_file_count",
            "value": len(
                NOTEBOOK07_PERSISTED_FILE_REGISTRY
            ),
        },
        {
            "field": "semantic_readback_audit_count",
            "value": len(
                NOTEBOOK07_SEMANTIC_READBACK_AUDIT
            ),
        },
        {
            "field": "final_registered_file_count",
            "value": len(
                NOTEBOOK07_FINAL_FILE_REGISTRY
            ),
        },
        {
            "field": "required_output_role_count",
            "value": len(
                NOTEBOOK07_REQUIRED_OUTPUT_ROLES
            ),
        },
        {
            "field": "all_required_roles_present",
            "value": set(
                NOTEBOOK07_REQUIRED_OUTPUT_ROLES
            ).issubset(
                set(
                    NOTEBOOK07_FINAL_FILE_REGISTRY[
                        "artifact_role"
                    ]
                )
            ),
        },
        {
            "field": "all_final_hash_checks_passed",
            "value": (
                NOTEBOOK07_FINAL_FILE_VERIFICATION[
                    "passed"
                ].all()
            ),
        },
        {
            "field": "semantic_readback_completed",
            "value": (
                NOTEBOOK07_READBACK_COMPLETED
            ),
        },
        {
            "field": "final_acceptance_completed",
            "value": (
                NOTEBOOK07_FINAL_ACCEPTANCE_COMPLETED
            ),
        },
        {
            "field": "output_manifest_completed",
            "value": (
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_COMPLETED
            ),
        },
        {
            "field": "notebook08_handoff_completed",
            "value": (
                NOTEBOOK08_HANDOFF_COMPLETED
            ),
        },
        {
            "field": "notebook08_authorized",
            "value": NOTEBOOK08_AUTHORIZED,
        },
        {
            "field": "notebook08_authorization_scope",
            "value": (
                NOTEBOOK08_AUTHORIZATION_SCOPE
            ),
        },
        {
            "field": "output_manifest_path",
            "value": str(
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_PATH
            ),
        },
        {
            "field": "output_manifest_file_sha256",
            "value": (
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_JSON_FILE_SHA256
            ),
        },
        {
            "field": "output_manifest_semantic_sha256",
            "value": (
                NOTEBOOK07_FINAL_OUTPUT_MANIFEST_SEMANTIC_SHA256
            ),
        },
        {
            "field": "final_acceptance_sha256",
            "value": (
                NOTEBOOK07_FINAL_ACCEPTANCE_SHA256
            ),
        },
        {
            "field": "handoff_sha256",
            "value": (
                NOTEBOOK07_TO_NOTEBOOK08_HANDOFF_SHA256
            ),
        },
        {
            "field": "calibration_parameter_updates",
            "value": (
                CALIBRATION_PARAMETER_UPDATES_PERFORMED
            ),
        },
        {
            "field": "hawkes_superiority_claim_authorized",
            "value": (
                HAWKES_SUPERIORITY_CLAIM_AUTHORIZED
            ),
        },
        {
            "field": "validation_content_loaded",
            "value": False,
        },
        {
            "field": (
                "engineering_holdout_content_loaded"
            ),
            "value": False,
        },
        {
            "field": "next_operation",
            "value": NOTEBOOK07_NEXT_OPERATION,
        },
        {
            "field": "filesystem_writes_performed",
            "value": FILESYSTEM_WRITES_PERFORMED,
        },
    ]
)

display(
    notebook07_final_summary
)

display(
    NOTEBOOK07_SEMANTIC_READBACK_SUMMARY
)

display(
    NOTEBOOK07_SEMANTIC_READBACK_AUDIT[
        [
            "artifact_role",
            "component_name",
            "artifact_format",
            "file_exists",
            "byte_hash_matches",
            "parse_passed",
            "semantic_passed",
            "passed",
            "status",
        ]
    ]
)

display(
    NOTEBOOK07_FINAL_FILE_REGISTRY[
        [
            "artifact_role",
            "component_name",
            "artifact_format",
            "relative_path",
            "byte_count",
            "sha256",
            "status",
        ]
    ]
    .sort_values(
        [
            "artifact_role",
            "artifact_format",
            "component_name",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

display(
    NOTEBOOK07_FINAL_COMPLETION_GATE_LEDGER
)

print(
    "Notebook 07 is complete. All persisted Hawkes estimation "
    "artifacts passed physical hash verification and semantic "
    "readback. The final acceptance, complete output manifest, and "
    "Notebook 07-to-Notebook 08 handoff were persisted and verified. "
    "The authoritative result remains the H1 diagonal shared-decay "
    "model fitted on exact Notebook 04 event_time_ns timestamps, "
    "with cross-excitation fixed to zero and locked CALIBRATION "
    "receiving zero parameter updates. V0.0 remains immutable "
    "provenance. Notebook 08 is now authorized for Hawkes diagnostics "
    "only. Hawkes-superiority, strategy, execution, P&L, and live "
    "deployment claims remain unauthorized."
)

,field,value
0,notebook07_completed,True
1,final_status,PASS_HAWKES_ESTIMATION_RESTRICTED_FIRST
2,selected_model_id,H1_DIAGONAL_SHARED_DECAY
3,core_persisted_file_count,61
4,semantic_readback_audit_count,65
5,final_registered_file_count,71
6,required_output_role_count,22
7,all_required_roles_present,True
8,all_final_hash_checks_passed,True
9,semantic_readback_completed,True


,field,value
0,core_file_count,61
1,staging_file_count,4
2,semantic_readback_row_count,65
3,all_files_exist,True
4,all_byte_hashes_match,True
5,all_artifacts_parse,True
6,all_semantic_checks_pass,True
7,calibration_parameter_updates,0
8,validation_content_loaded,False
9,engineering_holdout_content_loaded,False


,artifact_role,component_name,artifact_format,file_exists,byte_hash_matches,parse_passed,semantic_passed,passed,status
0,candidate_fit_summary,HAWKES_MODEL_FIT_SUMMARY,CSV,True,True,True,True,True,PASS
1,candidate_fit_summary,role_package,JSON,True,True,True,True,True,PASS
2,candidate_registry,candidate_registry,CSV,True,True,True,True,True,PASS
3,candidate_registry,role_package,JSON,True,True,True,True,True,PASS
4,candidate_selection,candidate_selection_ledger,CSV,True,True,True,True,True,PASS
...,...,...,...,...,...,...,...,...,...
60,v0_0_hawkes_reconciliation,role_package,JSON,True,True,True,True,True,PASS
61,persistence_index,persistence_index,JSON,True,True,True,True,True,PASS
62,persistence_registry,persisted_file_registry,CSV,True,True,True,True,True,PASS
63,immediate_verification,immediate_persistence_verification,CSV,True,True,True,True,True,PASS


,artifact_role,component_name,artifact_format,relative_path,byte_count,sha256,status
0,candidate_fit_summary,HAWKES_MODEL_FIT_SUMMARY,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2318,692c59030e57acf7e4fb690b16e8fa635151fb56317568...,PERSISTED
1,candidate_fit_summary,role_package,JSON,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,7431,7810ed7e1e15fb35ffb3cc366c504982936056ede18c35...,PERSISTED
2,candidate_registry,candidate_registry,CSV,artifacts\models\BTCUSDT_spot_20260710T063746Z...,1139,4476b92426db646daf391ecab713f3d47ff7607e69b756...,PERSISTED
3,candidate_registry,role_package,JSON,artifacts\models\BTCUSDT_spot_20260710T063746Z...,2887,a2c45f85ac54438f8df1eb79032e1230dab81dc368db59...,PERSISTED
4,candidate_selection,candidate_selection_ledger,CSV,artifacts\models\BTCUSDT_spot_20260710T063746Z...,828,63d89ec4d75d041fbb38e5cec629411c914f1be7416dbe...,PERSISTED
...,...,...,...,...,...,...,...
66,v0_0_hawkes_reconciliation,gate_ledger,CSV,artifacts\reconciliation\BTCUSDT_spot_20260710...,1146,6bb7b6417eee8136bb1b5a9b01b375d64118358474e2d4...,PERSISTED
67,v0_0_hawkes_reconciliation,immutability_audit,CSV,artifacts\reconciliation\BTCUSDT_spot_20260710...,6112,0b4042eb6de8bb98bfe429088a22ed49e3e168fe346469...,PERSISTED
68,v0_0_hawkes_reconciliation,manifest_hash_audit,CSV,artifacts\reconciliation\BTCUSDT_spot_20260710...,3622,71869d65968284537846f3ec22576a4fe52d5d9e84c31e...,PERSISTED
69,v0_0_hawkes_reconciliation,role_package,JSON,artifacts\reconciliation\BTCUSDT_spot_20260710...,21674,70437a9c9037671e71136b956edce1eea80ca7c1810fb5...,PERSISTED


,gate_id,passed,status
0,TERMINAL_DECISION_PASS,True,PASS
1,CORE_PERSISTENCE_COMPLETE,True,PASS
2,CORE_FILE_COUNT_RECONCILED,True,PASS
3,SEMANTIC_READBACK_COMPLETE,True,PASS
4,SEMANTIC_READBACK_PASS,True,PASS
5,FINAL_ACCEPTANCE_PERSISTED,True,PASS
6,OUTPUT_MANIFEST_PERSISTED,True,PASS
7,HANDOFF_PERSISTED,True,PASS
8,ALL_REQUIRED_ROLES_PRESENT,True,PASS
9,ALL_FINAL_FILES_HASH_VERIFIED,True,PASS


Notebook 07 is complete. All persisted Hawkes estimation artifacts passed physical hash verification and semantic readback. The final acceptance, complete output manifest, and Notebook 07-to-Notebook 08 handoff were persisted and verified. The authoritative result remains the H1 diagonal shared-decay model fitted on exact Notebook 04 event_time_ns timestamps, with cross-excitation fixed to zero and locked CALIBRATION receiving zero parameter updates. V0.0 remains immutable provenance. Notebook 08 is now authorized for Hawkes diagnostics only. Hawkes-superiority, strategy, execution, P&L, and live deployment claims remain unauthorized.
